# Diarios de avances de exploración
¿Cuánto aporta cada algoritmo y en qué momento conviene compartir cálculos? Cada sección ejecuta una variante en el mismo puzzle pequeño y después compara las mediciones reales con el paso anterior. Menos instrucciones es mejor.

Las comparaciones usan únicamente IDs resueltos por ambos métodos. La tabla final incluye cobertura para distinguir resultados parciales de entregas completas. Los 1.000 puntuables se miden localmente.

## Datos y funciones de lectura
Se toma la corrida completa más reciente de cada conjunto. Para regenerar los reportes desde la raíz, ejecutar `python3 -m scripts.compare_dag` (práctica) o `python3 -m scripts.scored_dag` (puntuables). Ejecutar el diario solo lee esos reportes y resuelve el ejemplo.

In [1]:
import csv
from pathlib import Path
from scripts.compare_dag import (Puzzle, validate, incremental, rings, bisection, multirings,
    bisection_reuse, multirings_reuse, postprocess, optimize_a4, nested_windows, optimize_dag)

def leer(ruta):
    with ruta.open(newline='') as archivo:
        return list(csv.DictReader(archivo))

def reporte(prefijo, marca):
    archivos = list(Path('reports').glob(f'{prefijo}_*/{marca}'))
    if not archivos:
        raise FileNotFoundError(f'Falta una corrida completa de {prefijo}. Ver comandos arriba.')
    return max(archivos, key=lambda p: p.stat().st_mtime_ns).parent

carpetas = {'Práctica': reporte('dag_practice', 'summary.csv'),
            'Puntuables': reporte('dag_scored', 'submission.csv')}
datos = {k: leer(v / 'comparison.csv') for k, v in carpetas.items()}
ejemplo = Puzzle('ejemplo', [1, 3, 10], [27, 97, 99, 107], 200)
historial = []

def medir(filas, columna, algoritmo=None, etapa=None):
    valores = {}
    for fila in filas:
        if fila['error'] or (algoritmo and fila['algorithm'] != algoritmo):
            continue
        if etapa and fila['insertion'] != etapa:
            continue
        valor = int(fila[columna])
        k = fila['id']
        valores[k] = min(valores.get(k, valor), valor)
    return valores

def comparar(nombre, columna, algoritmo=None, etapa=None):
    actual = {k: medir(filas, columna, algoritmo, etapa) for k, filas in datos.items()}
    for k, valores in actual.items():
        if historial:
            prev = historial[-1][1][k]
            comunes = prev.keys() & valores.keys()
            ahorro = sum(prev[i] - valores[i] for i in comunes)
            print(f'{k}: {len(valores)} resueltos | {len(comunes)} comunes | ahorro frente a {historial[-1][0]}: {ahorro:+,}')
        else:
            print(f'{k}: {len(valores)} resueltos | {sum(valores.values()):,} instrucciones')
    historial.append((nombre, actual))

for k, ruta in carpetas.items():
    print(k, ruta)

Práctica reports/dag_practice_gpvvc01y
Puntuables reports/dag_scored_ojhhopuu


## A1 · Incremental
Conserva un uno y suma hasta alcanzar cada objetivo en orden. Sirve como primera referencia y puede agotar el límite de instrucciones.

In [2]:
programa = incremental(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 107, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Establece la referencia inicial.

In [3]:
comparar('A1 · Incremental', 'before', 'incremental', 'constructor')

Práctica: 16 resueltos | 144 instrucciones
Puntuables: 16 resueltos | 139 instrucciones


## A2 · Anillos
Avanza con múltiplos de una base y completa el resto para alcanzar el objetivo.

In [4]:
programa = rings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [5]:
comparar('A2 · Anillos', 'before', 'rings', 'constructor')

Práctica: 159 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +8
Puntuables: 778 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +4


## A3 · Bisección
Parte del intervalo entre múltiplos y construye valores intermedios para acercarse al objetivo.

In [6]:
programa = bisection(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [7]:
comparar('A3 · Bisección', 'before', 'bisection', 'constructor')

Práctica: 142 resueltos | 133 comunes | ahorro frente a A2 · Anillos: -126,549
Puntuables: 775 resueltos | 684 comunes | ahorro frente a A2 · Anillos: -683,353


## Múltiples anillos
Explora distintas bases iniciales para acortar la construcción.

In [8]:
programa = multirings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 33, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [9]:
comparar('Múltiples anillos', 'before', 'multirings', 'constructor')

Práctica: 163 resueltos | 142 comunes | ahorro frente a A3 · Bisección: +67,763
Puntuables: 853 resueltos | 775 comunes | ahorro frente a A3 · Bisección: +369,669


## Bisección con reutilización
Aprovecha extremos y valores ya presentes en registros durante la bisección.

In [10]:
programa = bisection_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 20, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [11]:
comparar('Bisección con reutilización', 'before', 'bisection_reuse', 'constructor')

Práctica: 200 resueltos | 163 comunes | ahorro frente a Múltiples anillos: +153,313
Puntuables: 1000 resueltos | 853 comunes | ahorro frente a Múltiples anillos: +829,456


## Múltiples anillos con reutilización
Combina la elección de base con el aprovechamiento de valores residentes. Este programa alimenta las tres pasadas siguientes.

In [12]:
programa = multirings_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 14, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [13]:
comparar('Múltiples anillos con reutilización', 'before', 'multirings_reuse', 'constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Bisección con reutilización: +14,062
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Bisección con reutilización: +66,480


## Posprocesamiento
Comprime operaciones y elimina cálculos sin uso.

In [14]:
programa = postprocess(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [15]:
comparar('Posprocesamiento', 'before', 'multirings_reuse', 'postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Múltiples anillos con reutilización: +114,905
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Múltiples anillos con reutilización: +576,348


## A4
Busca atajos que permiten pasarse de un objetivo y corregir por resta o división exacta.

In [16]:
programa = optimize_a4(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [17]:
comparar('A4', 'before', 'multirings_reuse', 'a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Posprocesamiento: +550
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Posprocesamiento: +2,579


## Ventanas de ventanas
Une bloques vecinos para aprovechar valores que cruzan sus límites.

In [18]:
programa = nested_windows(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [19]:
comparar('Ventanas de ventanas', 'before', 'multirings_reuse', 'nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a A4: +437
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a A4: +2,239


## Mejor control por puzzle
Cada constructor válido recibe las tres pasadas. Se elige el programa final más corto por ID para medir el efecto de poder cambiar de constructor.

In [20]:
constructores = (incremental, rings, bisection, multirings, bisection_reuse, multirings_reuse)
pasadas = (postprocess, optimize_a4, nested_windows)

def ruta(constructor, insercion=None):
    programa = constructor(ejemplo)
    for posicion in range(4):
        if posicion == insercion:
            programa = optimize_dag(ejemplo, programa)
        if posicion < len(pasadas):
            programa = pasadas[posicion](ejemplo, programa)
    return validate(ejemplo, programa)['steps']

print(min(ruta(fn) for fn in constructores))

7


### Comparación con ventanas
Cuantifica el cambio de usar el mejor constructor por ID.

In [21]:
comparar('Mejor control sin DAG', 'control_final')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Ventanas de ventanas: +9,800
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Ventanas de ventanas: +44,318


## DAG después de constructor
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [22]:
print(min(ruta(fn, 0) for fn in constructores))

10


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [23]:
comparar('DAG después de constructor', 'dag_final', etapa='constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Mejor control sin DAG: +41,649
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Mejor control sin DAG: +194,289


## DAG después de posprocesamiento
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [24]:
print(min(ruta(fn, 1) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [25]:
comparar('DAG después de posprocesamiento', 'dag_final', etapa='postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de constructor: -26,811
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de constructor: -122,126


## DAG después de A4
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [26]:
print(min(ruta(fn, 2) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [27]:
comparar('DAG después de A4', 'dag_final', etapa='a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de posprocesamiento: -77
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de posprocesamiento: -499


## DAG después de ventanas
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [28]:
print(min(ruta(fn, 3) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [29]:
comparar('DAG después de ventanas', 'dag_final', etapa='nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de A4: +0
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de A4: +0


## Selección final por puzzle
Se conserva el mejor de los controles y las cuatro posiciones del DAG. Para el entregable también compite el programa del CSV previo en cada ID.

In [30]:
seleccion = {}
for k, filas in datos.items():
    control = medir(filas, 'control_final')
    dag = medir(filas, 'dag_final')
    seleccion[k] = {i: min(valor, dag[i]) for i, valor in control.items()}
ganadores = leer(carpetas['Puntuables'] / 'winners.csv')
entrega = {r['id']: int(r['steps']) for r in ganadores}
anterior = {r['id']: int(r['previous']) for r in ganadores}
assert len(entrega) == 1000 and len(seleccion['Práctica']) == 200
assert entrega == {i: min(anterior[i], seleccion['Puntuables'][i]) for i in entrega}
print('Entregable validado:', carpetas['Puntuables'] / 'submission.csv')

Entregable validado: reports/dag_scored_ojhhopuu/submission.csv


### Comparación con la entrega anterior
Cuenta cuántos puzzles mejoraron y cuánto bajó la suma de instrucciones.

In [31]:
print('Puzzles mejorados:', sum(entrega[i] < anterior[i] for i in entrega))
print('Instrucciones ahorradas:', sum(anterior.values()) - sum(entrega.values()))

Puzzles mejorados: 978
Instrucciones ahorradas: 250357


## Tabla de avances
Totales de instrucciones calculados desde los reportes seleccionados. Cada fila indica cuántos puzzles se resolvieron sobre 200 o 1.000; una suma parcial solo describe esos IDs. El signo — indica que falta una medición para ese conjunto.

In [32]:
def celda(valores, cantidad):
    if not valores:
        return '—'
    total = f'{sum(valores.values()):,}'.replace(',', '.')
    return f'{total} ({len(valores)}/{cantidad})'

filas = historial + [
    ('Mejor control o DAG por ID', seleccion),
    ('Entregable anterior', {'Puntuables': anterior}),
    ('Entregable validado', {'Puntuables': entrega})]
from pathlib import Path
import csv

def cargar_pasos(ruta):
    with Path(ruta).open() as archivo:
        return {r["id"]: int(r["final"]) for r in csv.DictReader(archivo)}

rutas_practica = cargar_pasos("reports/partial_practice_i0l79tbp/winners.csv")
rutas_puntuables = cargar_pasos("reports/partial_scored_2jzgqmyl/winners.csv")
con_constructor = cargar_pasos("reports/backward_practice_wyyedfh3/comparison.csv")
assert rutas_practica.keys() == con_constructor.keys()
assert all(con_constructor[k] <= pasos for k, pasos in rutas_practica.items())
constructor_puntuables = cargar_pasos("reports/backward_scored_LECwzt0R/comparison.csv")
assert constructor_puntuables.keys() == rutas_puntuables.keys()
assert all(constructor_puntuables[k] <= pasos for k, pasos in rutas_puntuables.items())
nuevas_filas = [
    ("Cartera de cuatro rutas e histórico", {"Práctica": rutas_practica, "Puntuables": rutas_puntuables}),
    ("Cartera con constructor hacia atrás", {"Práctica": con_constructor, "Puntuables": constructor_puntuables}),
]
filas.extend(nuevas_filas)
from html import escape
from IPython.display import HTML, display

encabezados = ('Algoritmo o entrega', 'Práctica · pasos (cobertura)', 'Puntuables · pasos (cobertura)')
tabla = '<table><thead><tr>' + ''.join(f'<th>{escape(t)}</th>' for t in encabezados) + '</tr></thead><tbody>'
for nombre, valores in filas:
    columnas = (nombre, celda(valores.get('Práctica'), 200), celda(valores.get('Puntuables'), 1000))
    tabla += '<tr>' + ''.join(f'<td>{escape(v)}</td>' for v in columnas) + '</tr>'
display(HTML(tabla + '</tbody></table>'))

Algoritmo o entrega,Práctica · pasos (cobertura),Puntuables · pasos (cobertura)
A1 · Incremental,144 (16/200),139 (16/1000)
A2 · Anillos,234.714 (159/200),1.166.626 (778/1000)
A3 · Bisección,332.967 (142/200),1.929.943 (775/1000)
Múltiples anillos,340.301 (163/200),1.834.813 (853/1000)
Bisección con reutilización,281.552 (200/200),1.372.573 (1000/1000)
Múltiples anillos con reutilización,267.490 (200/200),1.306.093 (1000/1000)
Posprocesamiento,152.585 (200/200),729.745 (1000/1000)
A4,152.035 (200/200),727.166 (1000/1000)
Ventanas de ventanas,151.598 (200/200),724.927 (1000/1000)
Mejor control sin DAG,141.798 (200/200),680.609 (1000/1000)


## Cuatro rutas de búsqueda parcial en práctica
A conserva el original. B elige por longitud inicial y C posprocesa las mismas opciones antes de elegir. D busca sobre los originales ya posprocesados. Cada selección desempata en favor del original. La columna de pasos toma el mínimo por ID entre los seis constructores. Los tiempos suman búsqueda y optimización por constructor, atribuyendo a cada ruta el trabajo reutilizado. Excluyen construcción, indexación de donantes y reconstrucción histórica.
La [tabla por ID](reports/partial_practice_i0l79tbp/winners.csv) registra el ganador frente al histórico. El [detalle por constructor y ruta](reports/partial_practice_i0l79tbp/comparison.csv) incluye fallos de cobertura, índice del candidato (0 para el original) y longitudes por etapa. En D la entrada corresponde al candidato formado después del posprocesamiento.

In [33]:
from pathlib import Path
import csv
from scripts.validate_submission import verify

carpeta = Path("reports/partial_practice_i0l79tbp")
def leer(nombre):
    with (carpeta / nombre).open() as archivo:
        return list(csv.DictReader(archivo))

detalle = [r for r in leer("comparison.csv") if not r["error"]]
ganadores = leer("winners.csv")
validos, total = verify(Path("practice_puzzles.csv"), carpeta / "programs.csv")
por_ruta = {}
print("Ruta   IDs   Candidatos   Pasos   Segundos acumulados")
for ruta in "ABCD":
    filas = [r for r in detalle if r["route"] == ruta]
    mejores = {}
    for r in filas:
        k, pasos = r["id"], int(r["final"])
        mejores[k] = min(mejores.get(k, pasos), pasos)
    por_ruta[ruta] = mejores
    print(ruta, len(mejores), sum(int(r["candidates"]) for r in filas),
          sum(mejores.values()), round(sum(float(r["seconds"]) for r in filas), 1))

for nueva, control in [("C", "B"), ("D", "A")]:
    delta = [por_ruta[control][k] - v for k, v in por_ruta[nueva].items()]
    print(f"{nueva} frente a {control}: {sum(d > 0 for d in delta)} mejoras,"
          f" {delta.count(0)} empates, {sum(d < 0 for d in delta)} retrocesos,"
          f" {sum(delta)} pasos ahorrados.")
    pares = {(r["id"], r["algorithm"]): int(r["final"]) for r in detalle if r["route"] == control}
    cambios = [pares[r["id"], r["algorithm"]] - int(r["final"]) for r in detalle if r["route"] == nueva]
    print("Por par ID-constructor (mejor / igual / peor):", sum(d > 0 for d in cambios),
          cambios.count(0), sum(d < 0 for d in cambios))

previo = sum(int(r["historical"]) for r in ganadores)
assert validos == len(ganadores) == 200 and previo == 98127
for r in ganadores:
    assert int(r["final"]) == min(int(r["historical"]), *(s[r["id"]] for s in por_ruta.values()))
assert total == sum(int(r["final"]) for r in ganadores)
print(f"Cartera validada: {total} pasos en {validos} puzzles. Ahorro histórico: {previo-total}.")
print("IDs mejorados:", sum(int(r["final"]) < int(r["historical"]) for r in ganadores))


Ruta   IDs   Candidatos   Pasos   Segundos acumulados
A 200 880 141798 2948.0
B 200 6988 103295 2786.1
C 200 6988 100957 22459.0
D 200 6953 128287 2974.4
C frente a B: 147 mejoras, 52 empates, 1 retrocesos, 2338 pasos ahorrados.
Por par ID-constructor (mejor / igual / peor): 602 270 8
D frente a A: 173 mejoras, 26 empates, 1 retrocesos, 13511 pasos ahorrados.
Por par ID-constructor (mejor / igual / peor): 733 146 1
Cartera validada: 92881 pasos en 200 puzzles. Ahorro histórico: 5246.
IDs mejorados: 145


## Búsqueda hacia atrás y recombinación de recetas

En la versión revisada de royalacecat, combinar los mejores programas públicos por ID ya suma 132.623 pasos. La búsqueda seleccionada para `scored_0295` lleva ese puzzle de 119 a 118 y el total a 132.622. Nuestro programa tiene 366 pasos en ese ID. La mayor diferencia ya existe al comenzar esa búsqueda. [Notebook de royalacecat, versión 372165224](https://www.kaggle.com/code/royalacecat/santa-2026-public-frontier-score-132-638?scriptVersionId=372165224)

GoodPJW v42 registra 132.636 pasos para la unión pública y 132.634 al incorporar dos programas propios. Su búsqueda local elimina una rama de dependencias y la reconstruye con valores del resto del grafo. Guarda recetas de programas empatados. CP-SAT combina esas recetas, elige un orden válido y permite que dos objetivos compartan un intermedio. El autor reporta ahorros de 45, 28 y 4 pasos en tres pasadas anteriores. Los puntajes públicos son resultados de sus autores. [Notebook de GoodPJW, v42](https://www.kaggle.com/code/goodpjw2008/santa-2026-lns-cp-sat-recombine-lb-132-634?scriptVersionId=372161473)

### Qué hace el constructor de Tony Li

Para cada objetivo prepara un diccionario de expresiones que cuestan hasta dos operaciones desde valores de partida, constantes, objetivos ya alcanzados e intermedios recientes. Si el objetivo no aparece, busca descomposiciones por cociente y resto. Por ejemplo, con 16 y 1 disponibles, construir 97 se reduce a construir 6 mediante la identidad `97 = 6 × 16 + 1`. El haz conserva varias reducciones prometedoras y repite la búsqueda sobre sus cocientes. Después emite solo las dependencias necesarias, asigna registros y puede reconstruir un valor desalojado. Konbu17 amplía el diccionario y combina varias configuraciones por ID. Su total publicado de 142.940 también incluye otros métodos. [Tony Li](https://www.kaggle.com/code/tonylica/elf-calculator-with-register-reuse?scriptVersionId=370866029) · [Konbu17](https://www.kaggle.com/code/konbu17/santa-2026-wider-dictionary-portfolio-163k?scriptVersionId=371106642)

### Experimento sobre los 200 puzzles de práctica

Se compilaron las tres celdas C++ originales de Tony Li con `g++ -std=c++17 -O3`. Cada corrida recibe los mismos objetivos ordenados y solo cambia el ancho del haz (1, 3 o 6). El valor publicado es 3. Cada programa parte de los valores iniciales del puzzle. La referencia local es la mejor cartera por ID después de las cuatro rutas (92.881 instrucciones). La celda siguiente reproduce la comparación desde los programas guardados. La elección del menor resultado por ID requiere ejecutar todas las configuraciones que se comparan. [Fuente compilada](reports/backward_practice_wyyedfh3/solver.cpp)


In [34]:
import csv
import json
from pathlib import Path
from scripts.validate_submission import verify

datos = Path("practice_puzzles.csv")
carpeta = Path("reports/backward_practice_wyyedfh3")
archivos = {
    "Cartera anterior": Path("reports/partial_practice_i0l79tbp/programs.csv"),
    "Ancho 1": carpeta / "programs_w1.csv",
    "Ancho 3": carpeta / "programs.csv",
    "Ancho 6": carpeta / "programs_w6.csv",
}
programas = {}
for nombre, ruta in archivos.items():
    cantidad, total = verify(datos, ruta)
    assert cantidad == 200
    with ruta.open() as fp:
        programas[nombre] = {r["id"]: r["program"] for r in csv.DictReader(fp)}
    print(f"{nombre:16} {total:>7} pasos  ({cantidad}/200)")

with datos.open() as fp:
    puzzles_practica = list(csv.DictReader(fp))
familias = {}
mejores_tres = 0
ganancias = empates = 0
with (carpeta / "comparison.csv").open("w", newline="") as tabla, (carpeta / "portfolio.csv").open("w", newline="") as salida:
    detalle, entrega = csv.writer(tabla), csv.writer(salida)
    detalle.writerow(["id", "familia", *archivos, "ganador", "final"])
    entrega.writerow(["id", "program"])
    for puzzle in puzzles_practica:
        k, familia = puzzle["id"], puzzle["family"]
        pasos = {n: len(json.loads(banco[k])) for n, banco in programas.items()}
        previo = pasos["Cartera anterior"]
        nuevo = min(pasos[n] for n in ("Ancho 1", "Ancho 3", "Ancho 6"))
        mejores_tres += nuevo
        ganancias += nuevo < previo
        empates += nuevo == previo
        ganador = min(pasos, key=pasos.get)
        detalle.writerow([k, familia, *pasos.values(), ganador, pasos[ganador]])
        entrega.writerow([k, programas[ganador][k]])
        suma = familias.setdefault(familia, [0, 0, 0])
        suma[0] += 1
        suma[1] += previo
        suma[2] += nuevo

print(f"Mejor de los tres anchos por ID: {mejores_tres} pasos.")
print(f"Frente al anterior: {ganancias} mejoras, {empates} empates y {200-ganancias-empates} retrocesos.")
for familia, (cuantos, anterior, atras) in sorted(familias.items()):
    print(f"{familia:15} {cuantos:3} puzzles   {anterior:6} → {atras:6}")
cantidad, total = verify(datos, carpeta / "portfolio.csv")
print(f"Cartera combinada validada: {cantidad} programas, {total} pasos.")

Cartera anterior   92881 pasos  (200/200)
Ancho 1            40474 pasos  (200/200)
Ancho 3            39135 pasos  (200/200)
Ancho 6            38648 pasos  (200/200)
Mejor de los tres anchos por ID: 38566 pasos.
Frente al anterior: 183 mejoras, 3 empates y 14 retrocesos.
independent      45 puzzles    32715 →  12973
mixed            48 puzzles    27815 →  11471
multiplicative   40 puzzles    10396 →   5244
shared           51 puzzles    21901 →   8801
tiny             16 puzzles       54 →     77
Cartera combinada validada: 200 programas, 38540 pasos.


### Aporte del constructor a la cartera de práctica
Al sumar los anchos 1, 3 y 6 como opciones por ID, la cartera pasa de 92.881 a 38.540 instrucciones (54.341 menos, 58,51 %). Mejoran 183 IDs y 17 mantienen su longitud. En 14 de esos 17 el programa anterior es estrictamente más corto; en los otros 3 empata. La mayor reducción absoluta aparece en la familia independiente (19.742 instrucciones). En los 16 puzzles diminutos, la selección conserva el total previo de 54. La medición del nuevo constructor cubre los 200 puzzles de práctica.

### Constructor hacia atrás sobre los 1.000 puntuables

| Configuración | Instrucciones |
|---|---:|
| Cartera anterior | 448.063 |
| Ancho 1 | 197.890 |
| Ancho 3 | 191.867 |
| Ancho 6 | 189.141 |
| Cartera combinada | 188.917 |

El mejor de los tres anchos mejora 965 IDs, empata en 11 y queda por detrás en 24. La selección conserva la cartera anterior en los empates. El verificador reprodujo los 1.000 programas de cada configuración y los del CSV final. [Comparación por ID](reports/backward_scored_LECwzt0R/comparison.csv) · [Entrega validada](artifacts/submission_backward_188917.csv)

## Cuatro experimentos sobre el constructor hacia atrás

Cierre validado: 36049 pasos, 2491 ahorrados. [Programas](reports/four_backward_practice/portfolio.csv) · [Ganadores](reports/four_backward_practice/comparison.csv)

Piloto completado: 110 configuraciones en 10 IDs, dos por familia. Suite del entorno: 42 pruebas aprobadas.

| Diccionario y ancho | Control original | Ampliado | Diferencia a favor del ampliado |
|---|---:|---:|---:|
|52-1|40474|37554|2920|
|52-3|39135|36657|2478|
|52-6|38648|36281|2367|
|64-1|40474|37861|2613|
|64-3|39135|37089|2046|
|64-6|38648|36422|2226|

E1 elegir antes: 38436 · E1 elegir después: 38430 · E3 individual: 38491 · E3 con CP-SAT: 38349

| Método | IDs elegidos | Aporte exclusivo |
|---|---:|---:|
|E1|10|11|
|E2|164|2118|
|E3|0|0|
|E4|12|23|

| Familia | E1 | E2 | E3 | E4 | Cartera |
|---|---:|---:|---:|---:|---:|
|independent|12955|12092|12896|12968|12086|
|mixed|11446|10701|11393|11459|10692|
|multiplicative|5213|5051|5238|5149|5030|
|shared|8759|8201|8768|8730|8189|
|tiny|57|77|54|53|52|

Programas reproducidos por método: [E1](reports/four_backward_practice/E1.csv) · [E2](reports/four_backward_practice/E2.csv) · [E3](reports/four_backward_practice/E3.csv) · [E4](reports/four_backward_practice/E4.csv)

Estados de CP-SAT: FEASIBLE 28, OPTIMAL 170, rechazado por registros 2

Referencia validada: 38.540 pasos en 200 IDs. Haz 1/3/6. E2: 52/72 y 64/72. CP-SAT 9.15.6755, semilla 42, un hilo y 30 s. Reparación local: 30 s. Grupo compartido: 14 procesos.

| Método | IDs completos | Histórico | Método | Ahorro al combinar | CPU s | Tiempo acumulado s |
|---|---:|---:|---:|---:|---:|---:|
|E1|200/200|38540|38430|114|94.9|97.8|
|E2|200/200|38540|36122|2447|3200.5|3270.9|
|E3|200/200|38540|38349|191|1102.5|1104.8|
|E4|200/200|38540|38359|181|654.8|660.7|

Comparación directa sobre 200 IDs comunes. E1: 38430 · E2: 36122 · E3: 38349 · E4: 38359

Detalle de resultados recibidos. Las configuraciones restantes están pendientes.

| ID | Método | Configuración | Entrada | Etapas (pasos) | Final validado | Estado |
|---|---|---|---:|---|---:|---|
|practice_0001|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|practice_0001|E1|3|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|practice_0001|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|practice_0001|E2|52-1|158||155|válido|
|practice_0001|E2|52-3|155||154|válido|
|practice_0001|E2|52-6|155||154|válido|
|practice_0001|E2|64-1|158||155|válido|
|practice_0001|E2|64-3|155||154|válido|
|practice_0001|E2|64-6|155||154|válido|
|practice_0001|E3|joint|155|individual 155|155|OPTIMAL|
|practice_0001|E4|repair|155||154|completo|
|practice_0002|E1|1|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|practice_0002|E1|3|299|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|practice_0002|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0002|E2|52-1|321||280|válido|
|practice_0002|E2|52-3|299||269|válido|
|practice_0002|E2|52-6|289||276|válido|
|practice_0002|E2|64-1|321||280|válido|
|practice_0002|E2|64-3|299||267|válido|
|practice_0002|E2|64-6|289||275|válido|
|practice_0002|E3|joint|289|individual 289|288|OPTIMAL|
|practice_0002|E4|repair|289||289|completo|
|practice_0003|E1|1|328|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|practice_0003|E1|3|314|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|practice_0003|E1|6|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|practice_0003|E2|52-1|328||303|válido|
|practice_0003|E2|52-3|314||288|válido|
|practice_0003|E2|52-6|308||285|válido|
|practice_0003|E2|64-1|328||304|válido|
|practice_0003|E2|64-3|314||291|válido|
|practice_0003|E2|64-6|308||289|válido|
|practice_0003|E3|joint|308|individual 307|303|FEASIBLE|
|practice_0003|E4|repair|308||308|completo|
|practice_0004|E1|1|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0004|E1|3|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0004|E1|6|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|practice_0004|E2|52-1|135||133|válido|
|practice_0004|E2|52-3|135||132|válido|
|practice_0004|E2|52-6|134||130|válido|
|practice_0004|E2|64-1|135||133|válido|
|practice_0004|E2|64-3|135||132|válido|
|practice_0004|E2|64-6|134||130|válido|
|practice_0004|E3|joint|134|individual 134|134|OPTIMAL|
|practice_0004|E4|repair|134||134|completo|
|practice_0005|E1|1|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|practice_0005|E1|3|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|practice_0005|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|practice_0005|E2|52-1|252||236|válido|
|practice_0005|E2|52-3|247||232|válido|
|practice_0005|E2|52-6|245||234|válido|
|practice_0005|E2|64-1|252||236|válido|
|practice_0005|E2|64-3|247||236|válido|
|practice_0005|E2|64-6|245||234|válido|
|practice_0005|E3|joint|245|individual 245|244|OPTIMAL|
|practice_0005|E4|repair|245||245|completo|
|practice_0006|E1|1|40|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0006|E1|3|40|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0006|E1|6|40|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0006|E2|52-1|40||40|válido|
|practice_0006|E2|52-3|40||40|válido|
|practice_0006|E2|52-6|40||40|válido|
|practice_0006|E2|64-1|40||40|válido|
|practice_0006|E2|64-3|40||40|válido|
|practice_0006|E2|64-6|40||40|válido|
|practice_0006|E3|joint|40|individual 40|40|OPTIMAL|
|practice_0006|E4|repair|40||38|completo|
|practice_0007|E1|1|107|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|practice_0007|E1|3|107|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|practice_0007|E1|6|107|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|practice_0007|E2|52-1|107||106|válido|
|practice_0007|E2|52-3|107||106|válido|
|practice_0007|E2|52-6|107||106|válido|
|practice_0007|E2|64-1|107||106|válido|
|practice_0007|E2|64-3|107||106|válido|
|practice_0007|E2|64-6|107||106|válido|
|practice_0007|E3|joint|107|individual 107|107|OPTIMAL|
|practice_0007|E4|repair|107||105|completo|
|practice_0008|E1|1|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|practice_0008|E1|3|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0008|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|practice_0008|E2|52-1|287||272|válido|
|practice_0008|E2|52-3|282||258|válido|
|practice_0008|E2|52-6|278||258|válido|
|practice_0008|E2|64-1|287||272|válido|
|practice_0008|E2|64-3|282||260|válido|
|practice_0008|E2|64-6|278||261|válido|
|practice_0008|E3|joint|278|individual 278|278|OPTIMAL|
|practice_0008|E4|repair|278||278|completo|
|practice_0009|E1|1|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|practice_0009|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|practice_0009|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0009|E2|52-1|148||142|válido|
|practice_0009|E2|52-3|147||139|válido|
|practice_0009|E2|52-6|146||139|válido|
|practice_0009|E2|64-1|148||141|válido|
|practice_0009|E2|64-3|147||139|válido|
|practice_0009|E2|64-6|146||139|válido|
|practice_0009|E3|joint|146|individual 146|145|OPTIMAL|
|practice_0009|E4|repair|146||144|completo|
|practice_0010|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0010|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0010|E1|6|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0010|E2|52-1|151||142|válido|
|practice_0010|E2|52-3|151||141|válido|
|practice_0010|E2|52-6|150||141|válido|
|practice_0010|E2|64-1|151||142|válido|
|practice_0010|E2|64-3|151||141|válido|
|practice_0010|E2|64-6|150||141|válido|
|practice_0010|E3|joint|150|individual 150|150|OPTIMAL|
|practice_0010|E4|repair|150||148|completo|
|practice_0011|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0011|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0011|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|practice_0011|E2|52-1|137||128|válido|
|practice_0011|E2|52-3|137||128|válido|
|practice_0011|E2|52-6|136||128|válido|
|practice_0011|E2|64-1|137||128|válido|
|practice_0011|E2|64-3|137||128|válido|
|practice_0011|E2|64-6|136||128|válido|
|practice_0011|E3|joint|136|individual 136|136|OPTIMAL|
|practice_0011|E4|repair|136||133|completo|
|practice_0012|E1|1|295|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0012|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|practice_0012|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|practice_0012|E2|52-1|295||275|válido|
|practice_0012|E2|52-3|286||261|válido|
|practice_0012|E2|52-6|278||259|válido|
|practice_0012|E2|64-1|295||275|válido|
|practice_0012|E2|64-3|286||264|válido|
|practice_0012|E2|64-6|278||259|válido|
|practice_0012|E3|joint|278|individual 277|275|OPTIMAL|
|practice_0012|E4|repair|278||278|completo|
|practice_0013|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|practice_0013|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|practice_0013|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|practice_0013|E2|52-1|243||226|válido|
|practice_0013|E2|52-3|235||223|válido|
|practice_0013|E2|52-6|238||217|válido|
|practice_0013|E2|64-1|243||231|válido|
|practice_0013|E2|64-3|235||223|válido|
|practice_0013|E2|64-6|238||215|válido|
|practice_0013|E3|joint|235|individual 235|234|OPTIMAL|
|practice_0013|E4|repair|235||235|completo|
|practice_0014|E1|1|148|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0014|E1|3|148|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0014|E1|6|148|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0014|E2|52-1|148||138|válido|
|practice_0014|E2|52-3|148||138|válido|
|practice_0014|E2|52-6|148||138|válido|
|practice_0014|E2|64-1|148||139|válido|
|practice_0014|E2|64-3|148||139|válido|
|practice_0014|E2|64-6|148||139|válido|
|practice_0014|E3|joint|148|individual 148|148|OPTIMAL|
|practice_0014|E4|repair|148||147|completo|
|practice_0015|E1|1|242|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|practice_0015|E1|3|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|practice_0015|E1|6|233|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|practice_0015|E2|52-1|242||233|válido|
|practice_0015|E2|52-3|234||222|válido|
|practice_0015|E2|52-6|233||205|válido|
|practice_0015|E2|64-1|242||227|válido|
|practice_0015|E2|64-3|234||221|válido|
|practice_0015|E2|64-6|233||201|válido|
|practice_0015|E3|joint|233|individual 233|232|OPTIMAL|
|practice_0015|E4|repair|233||233|completo|
|practice_0016|E1|1|268|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|practice_0016|E1|3|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|practice_0016|E1|6|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|practice_0016|E2|52-1|268||244|válido|
|practice_0016|E2|52-3|255||233|válido|
|practice_0016|E2|52-6|247||235|válido|
|practice_0016|E2|64-1|268||244|válido|
|practice_0016|E2|64-3|255||231|válido|
|practice_0016|E2|64-6|247||229|válido|
|practice_0016|E3|joint|247|individual 247|246|OPTIMAL|
|practice_0016|E4|repair|247||246|completo|
|practice_0017|E1|1|198|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0017|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0017|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|practice_0017|E2|52-1|198||176|válido|
|practice_0017|E2|52-3|192||176|válido|
|practice_0017|E2|52-6|188||176|válido|
|practice_0017|E2|64-1|198||175|válido|
|practice_0017|E2|64-3|192||175|válido|
|practice_0017|E2|64-6|188||175|válido|
|practice_0017|E3|joint|188|individual 188|187|OPTIMAL|
|practice_0017|E4|repair|188||187|completo|
|practice_0018|E1|1|397|postprocess 397, optimize_a4 397, nested_windows 397|397|válido|
|practice_0018|E1|3|381|postprocess 381, optimize_a4 381, nested_windows 381|381|válido|
|practice_0018|E1|6|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0018|E2|52-1|397||373|válido|
|practice_0018|E2|52-3|381||350|válido|
|practice_0018|E2|52-6|377||342|válido|
|practice_0018|E2|64-1|397||370|válido|
|practice_0018|E2|64-3|381||353|válido|
|practice_0018|E2|64-6|377||350|válido|
|practice_0018|E3|joint|377|individual 377|375|FEASIBLE|
|practice_0018|E4|repair|377||377|completo|
|practice_0019|E1|1|229|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|practice_0019|E1|3|223|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|practice_0019|E1|6|218|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|practice_0019|E2|52-1|229||213|válido|
|practice_0019|E2|52-3|223||209|válido|
|practice_0019|E2|52-6|218||211|válido|
|practice_0019|E2|64-1|229||214|válido|
|practice_0019|E2|64-3|223||212|válido|
|practice_0019|E2|64-6|218||213|válido|
|practice_0019|E3|joint|218|individual 218|217|OPTIMAL|
|practice_0019|E4|repair|218||218|completo|
|practice_0020|E1|1|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|practice_0020|E1|3|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|practice_0020|E1|6|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|practice_0020|E2|52-1|118||115|válido|
|practice_0020|E2|52-3|118||114|válido|
|practice_0020|E2|52-6|118||114|válido|
|practice_0020|E2|64-1|118||113|válido|
|practice_0020|E2|64-3|118||112|válido|
|practice_0020|E2|64-6|118||112|válido|
|practice_0020|E3|joint|118|individual 118|118|OPTIMAL|
|practice_0020|E4|repair|118||113|completo|
|practice_0021|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0021|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|practice_0021|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|practice_0021|E2|52-1|141||136|válido|
|practice_0021|E2|52-3|140||136|válido|
|practice_0021|E2|52-6|139||136|válido|
|practice_0021|E2|64-1|141||136|válido|
|practice_0021|E2|64-3|140||136|válido|
|practice_0021|E2|64-6|139||136|válido|
|practice_0021|E3|joint|139|individual 139|139|OPTIMAL|
|practice_0021|E4|repair|139||139|completo|
|practice_0022|E1|1|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0022|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0022|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0022|E2|52-1|152||143|válido|
|practice_0022|E2|52-3|152||143|válido|
|practice_0022|E2|52-6|152||143|válido|
|practice_0022|E2|64-1|152||147|válido|
|practice_0022|E2|64-3|152||147|válido|
|practice_0022|E2|64-6|152||147|válido|
|practice_0022|E3|joint|152|individual 152|152|OPTIMAL|
|practice_0022|E4|repair|152||152|completo|
|practice_0023|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|practice_0023|E1|3|205|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|practice_0023|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0023|E2|52-1|218||195|válido|
|practice_0023|E2|52-3|205||194|válido|
|practice_0023|E2|52-6|204||193|válido|
|practice_0023|E2|64-1|218||195|válido|
|practice_0023|E2|64-3|205||195|válido|
|practice_0023|E2|64-6|204||193|válido|
|practice_0023|E3|joint|204|individual 204|202|OPTIMAL|
|practice_0023|E4|repair|204||204|completo|
|practice_0024|E1|1|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|practice_0024|E1|3|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|practice_0024|E1|6|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|practice_0024|E2|52-1|43||43|válido|
|practice_0024|E2|52-3|43||43|válido|
|practice_0024|E2|52-6|43||43|válido|
|practice_0024|E2|64-1|43||43|válido|
|practice_0024|E2|64-3|43||43|válido|
|practice_0024|E2|64-6|43||43|válido|
|practice_0024|E3|joint|43|individual 43|43|OPTIMAL|
|practice_0024|E4|repair|43||43|completo|
|practice_0025|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0025|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0025|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0025|E2|52-1|204||195|válido|
|practice_0025|E2|52-3|199||190|válido|
|practice_0025|E2|52-6|199||182|válido|
|practice_0025|E2|64-1|204||196|válido|
|practice_0025|E2|64-3|199||190|válido|
|practice_0025|E2|64-6|199||182|válido|
|practice_0025|E3|joint|199|individual 199|198|OPTIMAL|
|practice_0025|E4|repair|199||199|completo|
|practice_0026|E1|1|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|practice_0026|E1|3|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|practice_0026|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|practice_0026|E2|52-1|164||159|válido|
|practice_0026|E2|52-3|161||153|válido|
|practice_0026|E2|52-6|160||153|válido|
|practice_0026|E2|64-1|164||158|válido|
|practice_0026|E2|64-3|161||154|válido|
|practice_0026|E2|64-6|160||154|válido|
|practice_0026|E3|joint|160|individual 159|158|OPTIMAL|
|practice_0026|E4|repair|160||159|completo|
|practice_0027|E1|1|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0027|E1|3|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|practice_0027|E1|6|221|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|practice_0027|E2|52-1|228||210|válido|
|practice_0027|E2|52-3|223||206|válido|
|practice_0027|E2|52-6|221||205|válido|
|practice_0027|E2|64-1|228||214|válido|
|practice_0027|E2|64-3|223||208|válido|
|practice_0027|E2|64-6|221||206|válido|
|practice_0027|E3|joint|221|individual 220|220|OPTIMAL|
|practice_0027|E4|repair|221||221|completo|
|practice_0028|E1|1|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0028|E1|3|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|practice_0028|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|practice_0028|E2|52-1|283||257|válido|
|practice_0028|E2|52-3|262||244|válido|
|practice_0028|E2|52-6|254||240|válido|
|practice_0028|E2|64-1|283||258|válido|
|practice_0028|E2|64-3|262||246|válido|
|practice_0028|E2|64-6|254||242|válido|
|practice_0028|E3|joint|254|individual 254|253|OPTIMAL|
|practice_0028|E4|repair|254||254|completo|
|practice_0029|E1|1|244|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|practice_0029|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|practice_0029|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0029|E2|52-1|244||226|válido|
|practice_0029|E2|52-3|233||212|válido|
|practice_0029|E2|52-6|227||214|válido|
|practice_0029|E2|64-1|244||224|válido|
|practice_0029|E2|64-3|233||212|válido|
|practice_0029|E2|64-6|227||213|válido|
|practice_0029|E3|joint|227|individual 227|226|OPTIMAL|
|practice_0029|E4|repair|227||227|completo|
|practice_0030|E1|1|144|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0030|E1|3|144|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0030|E1|6|144|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0030|E2|52-1|144||146|válido|
|practice_0030|E2|52-3|144||142|válido|
|practice_0030|E2|52-6|144||142|válido|
|practice_0030|E2|64-1|144||143|válido|
|practice_0030|E2|64-3|144||143|válido|
|practice_0030|E2|64-6|144||143|válido|
|practice_0030|E3|joint|144|individual 144|144|OPTIMAL|
|practice_0030|E4|repair|144||140|completo|
|practice_0031|E1|1|264|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|practice_0031|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|practice_0031|E1|6|245|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|practice_0031|E2|52-1|264||238|válido|
|practice_0031|E2|52-3|250||229|válido|
|practice_0031|E2|52-6|245||226|válido|
|practice_0031|E2|64-1|264||238|válido|
|practice_0031|E2|64-3|250||229|válido|
|practice_0031|E2|64-6|245||226|válido|
|practice_0031|E3|joint|245|individual 245|244|OPTIMAL|
|practice_0031|E4|repair|245||245|completo|
|practice_0032|E1|1|147|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|practice_0032|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|practice_0032|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|practice_0032|E2|52-1|147||140|válido|
|practice_0032|E2|52-3|147||140|válido|
|practice_0032|E2|52-6|147||140|válido|
|practice_0032|E2|64-1|147||140|válido|
|practice_0032|E2|64-3|147||140|válido|
|practice_0032|E2|64-6|147||140|válido|
|practice_0032|E3|joint|147|individual 147|147|OPTIMAL|
|practice_0032|E4|repair|147||146|completo|
|practice_0033|E1|1|42|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|practice_0033|E1|3|42|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|practice_0033|E1|6|42|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|practice_0033|E2|52-1|42||42|válido|
|practice_0033|E2|52-3|42||42|válido|
|practice_0033|E2|52-6|42||42|válido|
|practice_0033|E2|64-1|42||42|válido|
|practice_0033|E2|64-3|42||42|válido|
|practice_0033|E2|64-6|42||42|válido|
|practice_0033|E3|joint|42|individual 42|42|OPTIMAL|
|practice_0033|E4|repair|42||42|completo|
|practice_0034|E1|1|372|postprocess 371, optimize_a4 371, nested_windows 371|371|válido|
|practice_0034|E1|3|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|practice_0034|E1|6|341|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|practice_0034|E2|52-1|372||338|válido|
|practice_0034|E2|52-3|355||322|válido|
|practice_0034|E2|52-6|341||319|válido|
|practice_0034|E2|64-1|372||337|válido|
|practice_0034|E2|64-3|355||325|válido|
|practice_0034|E2|64-6|341||318|válido|
|practice_0034|E3|joint|341|individual 341|341|FEASIBLE|
|practice_0034|E4|repair|341||341|completo|
|practice_0035|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0035|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|practice_0035|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0035|E2|52-1|301||273|válido|
|practice_0035|E2|52-3|286||265|válido|
|practice_0035|E2|52-6|282||265|válido|
|practice_0035|E2|64-1|301||275|válido|
|practice_0035|E2|64-3|286||265|válido|
|practice_0035|E2|64-6|282||264|válido|
|practice_0035|E3|joint|282|individual 282|282|OPTIMAL|
|practice_0035|E4|repair|282||282|completo|
|practice_0036|E1|1|139|postprocess 139, optimize_a4 138, nested_windows 138|138|válido|
|practice_0036|E1|3|137|postprocess 137, optimize_a4 136, nested_windows 136|136|válido|
|practice_0036|E1|6|137|postprocess 137, optimize_a4 136, nested_windows 136|136|válido|
|practice_0036|E2|52-1|139||133|válido|
|practice_0036|E2|52-3|137||133|válido|
|practice_0036|E2|52-6|137||133|válido|
|practice_0036|E2|64-1|139||134|válido|
|practice_0036|E2|64-3|137||133|válido|
|practice_0036|E2|64-6|137||133|válido|
|practice_0036|E3|joint|137|individual 137|137|OPTIMAL|
|practice_0036|E4|repair|137||136|completo|
|practice_0037|E1|1|338|postprocess 335, optimize_a4 335, nested_windows 335|335|válido|
|practice_0037|E1|3|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|practice_0037|E1|6|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|practice_0037|E2|52-1|338||313|válido|
|practice_0037|E2|52-3|322||301|válido|
|practice_0037|E2|52-6|313||293|válido|
|practice_0037|E2|64-1|338||315|válido|
|practice_0037|E2|64-3|322||301|válido|
|practice_0037|E2|64-6|313||294|válido|
|practice_0037|E3|joint|313|individual 313|312|FEASIBLE|
|practice_0037|E4|repair|313||313|completo|
|practice_0038|E1|1|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|practice_0038|E1|3|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|practice_0038|E1|6|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|practice_0038|E2|52-1|45||45|válido|
|practice_0038|E2|52-3|45||45|válido|
|practice_0038|E2|52-6|45||45|válido|
|practice_0038|E2|64-1|45||45|válido|
|practice_0038|E2|64-3|45||45|válido|
|practice_0038|E2|64-6|45||45|válido|
|practice_0038|E3|joint|45|individual 45|45|OPTIMAL|
|practice_0038|E4|repair|45||44|completo|
|practice_0039|E1|1|9|postprocess 8, optimize_a4 8, nested_windows 8|8|válido|
|practice_0039|E1|3|9|postprocess 8, optimize_a4 8, nested_windows 8|8|válido|
|practice_0039|E1|6|9|postprocess 8, optimize_a4 8, nested_windows 8|8|válido|
|practice_0039|E2|52-1|9||9|válido|
|practice_0039|E2|52-3|9||9|válido|
|practice_0039|E2|52-6|9||9|válido|
|practice_0039|E2|64-1|9||9|válido|
|practice_0039|E2|64-3|9||9|válido|
|practice_0039|E2|64-6|9||9|válido|
|practice_0039|E3|joint|6|individual 6|6|OPTIMAL|
|practice_0039|E4|repair|6||6|completo|
|practice_0040|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|practice_0040|E1|3|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0040|E1|6|210|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|practice_0040|E2|52-1|223||206|válido|
|practice_0040|E2|52-3|213||199|válido|
|practice_0040|E2|52-6|210||203|válido|
|practice_0040|E2|64-1|223||208|válido|
|practice_0040|E2|64-3|213||198|válido|
|practice_0040|E2|64-6|210||200|válido|
|practice_0040|E3|joint|210|individual 210|209|OPTIMAL|
|practice_0040|E4|repair|210||210|completo|
|practice_0041|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0041|E1|3|197|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0041|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0041|E2|52-1|206||189|válido|
|practice_0041|E2|52-3|197||192|válido|
|practice_0041|E2|52-6|197||188|válido|
|practice_0041|E2|64-1|206||190|válido|
|practice_0041|E2|64-3|197||191|válido|
|practice_0041|E2|64-6|197||196|válido|
|practice_0041|E3|joint|197|individual 196|196|OPTIMAL|
|practice_0041|E4|repair|197||197|completo|
|practice_0042|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0042|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0042|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0042|E2|52-1|204||190|válido|
|practice_0042|E2|52-3|199||189|válido|
|practice_0042|E2|52-6|197||185|válido|
|practice_0042|E2|64-1|204||189|válido|
|practice_0042|E2|64-3|199||189|válido|
|practice_0042|E2|64-6|197||186|válido|
|practice_0042|E3|joint|197|individual 197|197|OPTIMAL|
|practice_0042|E4|repair|197||197|completo|
|practice_0043|E1|1|134|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|practice_0043|E1|3|134|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|practice_0043|E1|6|133|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0043|E2|52-1|134||127|válido|
|practice_0043|E2|52-3|134||127|válido|
|practice_0043|E2|52-6|133||127|válido|
|practice_0043|E2|64-1|134||127|válido|
|practice_0043|E2|64-3|134||127|válido|
|practice_0043|E2|64-6|133||127|válido|
|practice_0043|E3|joint|133|individual 133|133|OPTIMAL|
|practice_0043|E4|repair|133||130|completo|
|practice_0044|E1|1|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|practice_0044|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0044|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0044|E2|52-1|259||241|válido|
|practice_0044|E2|52-3|246||230|válido|
|practice_0044|E2|52-6|246||233|válido|
|practice_0044|E2|64-1|259||241|válido|
|practice_0044|E2|64-3|246||232|válido|
|practice_0044|E2|64-6|246||232|válido|
|practice_0044|E3|joint|246|individual 245|243|OPTIMAL|
|practice_0044|E4|repair|246||245|completo|
|practice_0045|E1|1|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|practice_0045|E1|3|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|practice_0045|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0045|E2|52-1|162||151|válido|
|practice_0045|E2|52-3|158||150|válido|
|practice_0045|E2|52-6|156||150|válido|
|practice_0045|E2|64-1|162||152|válido|
|practice_0045|E2|64-3|158||150|válido|
|practice_0045|E2|64-6|156||150|válido|
|practice_0045|E3|joint|156|individual 155|155|OPTIMAL|
|practice_0045|E4|repair|156||155|completo|
|practice_0046|E1|1|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0046|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|practice_0046|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0046|E2|52-1|312||280|válido|
|practice_0046|E2|52-3|303||276|válido|
|practice_0046|E2|52-6|289||271|válido|
|practice_0046|E2|64-1|312||283|válido|
|practice_0046|E2|64-3|303||276|válido|
|practice_0046|E2|64-6|289||271|válido|
|practice_0046|E3|joint|289|individual 289|289|FEASIBLE|
|practice_0046|E4|repair|289||289|completo|
|practice_0047|E1|1|5|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0047|E1|3|5|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0047|E1|6|5|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0047|E2|52-1|5||5|válido|
|practice_0047|E2|52-3|5||5|válido|
|practice_0047|E2|52-6|5||5|válido|
|practice_0047|E2|64-1|5||5|válido|
|practice_0047|E2|64-3|5||5|válido|
|practice_0047|E2|64-6|5||5|válido|
|practice_0047|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0047|E4|repair|2||2|completo|
|practice_0048|E1|1|131|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0048|E1|3|131|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0048|E1|6|131|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0048|E2|52-1|131||125|válido|
|practice_0048|E2|52-3|131||125|válido|
|practice_0048|E2|52-6|131||124|válido|
|practice_0048|E2|64-1|131||127|válido|
|practice_0048|E2|64-3|131||127|válido|
|practice_0048|E2|64-6|131||127|válido|
|practice_0048|E3|joint|131|individual 131|131|OPTIMAL|
|practice_0048|E4|repair|131||126|completo|
|practice_0049|E1|1|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|practice_0049|E1|3|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|practice_0049|E1|6|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0049|E2|52-1|137||131|válido|
|practice_0049|E2|52-3|137||131|válido|
|practice_0049|E2|52-6|136||131|válido|
|practice_0049|E2|64-1|137||131|válido|
|practice_0049|E2|64-3|137||131|válido|
|practice_0049|E2|64-6|136||131|válido|
|practice_0049|E3|joint|136|individual 136|136|OPTIMAL|
|practice_0049|E4|repair|136||135|completo|
|practice_0050|E1|1|5|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|practice_0050|E1|3|5|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|practice_0050|E1|6|5|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|practice_0050|E2|52-1|5||5|válido|
|practice_0050|E2|52-3|5||5|válido|
|practice_0050|E2|52-6|5||5|válido|
|practice_0050|E2|64-1|5||5|válido|
|practice_0050|E2|64-3|5||5|válido|
|practice_0050|E2|64-6|5||5|válido|
|practice_0050|E3|joint|3|individual 3|3|OPTIMAL|
|practice_0050|E4|repair|3||3|completo|
|practice_0051|E1|1|7|postprocess 7, optimize_a4 5, nested_windows 5|5|válido|
|practice_0051|E1|3|7|postprocess 7, optimize_a4 5, nested_windows 5|5|válido|
|practice_0051|E1|6|7|postprocess 7, optimize_a4 5, nested_windows 5|5|válido|
|practice_0051|E2|52-1|7||7|válido|
|practice_0051|E2|52-3|7||7|válido|
|practice_0051|E2|52-6|7||7|válido|
|practice_0051|E2|64-1|7||7|válido|
|practice_0051|E2|64-3|7||7|válido|
|practice_0051|E2|64-6|7||7|válido|
|practice_0051|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0051|E4|repair|5||5|completo|
|practice_0052|E1|1|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0052|E1|3|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0052|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|practice_0052|E2|52-1|230||216|válido|
|practice_0052|E2|52-3|228||211|válido|
|practice_0052|E2|52-6|226||214|válido|
|practice_0052|E2|64-1|230||215|válido|
|practice_0052|E2|64-3|228||209|válido|
|practice_0052|E2|64-6|226||213|válido|
|practice_0052|E3|joint|226|individual 226|223|OPTIMAL|
|practice_0052|E4|repair|226||226|completo|
|practice_0053|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|practice_0053|E1|3|294|postprocess 293, optimize_a4 293, nested_windows 293|293|válido|
|practice_0053|E1|6|286|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|practice_0053|E2|52-1|306||275|válido|
|practice_0053|E2|52-3|294||267|válido|
|practice_0053|E2|52-6|286||267|válido|
|practice_0053|E2|64-1|306||275|válido|
|practice_0053|E2|64-3|294||271|válido|
|practice_0053|E2|64-6|286||268|válido|
|practice_0053|E3|joint|286|individual 285|281|OPTIMAL|
|practice_0053|E4|repair|286||286|completo|
|practice_0054|E1|1|413|postprocess 411, optimize_a4 411, nested_windows 411|411|válido|
|practice_0054|E1|3|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|practice_0054|E1|6|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0054|E2|52-1|413||375|válido|
|practice_0054|E2|52-3|385||348|válido|
|practice_0054|E2|52-6|377||346|válido|
|practice_0054|E2|64-1|413||382|válido|
|practice_0054|E2|64-3|385||352|válido|
|practice_0054|E2|64-6|377||346|válido|
|practice_0054|E3|joint|377|individual 376|375|FEASIBLE|
|practice_0054|E4|repair|377||377|completo|
|practice_0055|E1|1|407|postprocess 404, optimize_a4 404, nested_windows 404|404|válido|
|practice_0055|E1|3|387|postprocess 387, optimize_a4 387, nested_windows 387|387|válido|
|practice_0055|E1|6|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0055|E2|52-1|407||366|válido|
|practice_0055|E2|52-3|387||356|válido|
|practice_0055|E2|52-6|377||349|válido|
|practice_0055|E2|64-1|407||369|válido|
|practice_0055|E2|64-3|387||358|válido|
|practice_0055|E2|64-6|377||349|válido|
|practice_0055|E3|joint|377|individual 376|371|FEASIBLE|
|practice_0055|E4|repair|377||377|completo|
|practice_0056|E1|1|7|postprocess 7, optimize_a4 6, nested_windows 6|6|válido|
|practice_0056|E1|3|7|postprocess 7, optimize_a4 6, nested_windows 6|6|válido|
|practice_0056|E1|6|7|postprocess 7, optimize_a4 6, nested_windows 6|6|válido|
|practice_0056|E2|52-1|7||7|válido|
|practice_0056|E2|52-3|7||7|válido|
|practice_0056|E2|52-6|7||7|válido|
|practice_0056|E2|64-1|7||7|válido|
|practice_0056|E2|64-3|7||7|válido|
|practice_0056|E2|64-6|7||7|válido|
|practice_0056|E3|joint|4|individual 4|4|OPTIMAL|
|practice_0056|E4|repair|4||4|completo|
|practice_0057|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|practice_0057|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|practice_0057|E1|6|285|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|practice_0057|E2|52-1|295||269|válido|
|practice_0057|E2|52-3|286||261|válido|
|practice_0057|E2|52-6|285||256|válido|
|practice_0057|E2|64-1|295||270|válido|
|practice_0057|E2|64-3|286||262|válido|
|practice_0057|E2|64-6|285||256|válido|
|practice_0057|E3|joint|285|individual 279|277|FEASIBLE|
|practice_0057|E4|repair|285||285|completo|
|practice_0058|E1|1|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|practice_0058|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0058|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|practice_0058|E2|52-1|310||291|válido|
|practice_0058|E2|52-3|294||276|válido|
|practice_0058|E2|52-6|292||267|válido|
|practice_0058|E2|64-1|310||292|válido|
|practice_0058|E2|64-3|294||276|válido|
|practice_0058|E2|64-6|292||273|válido|
|practice_0058|E3|joint|292|individual 292|291|OPTIMAL|
|practice_0058|E4|repair|292||292|completo|
|practice_0059|E1|1|157|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|practice_0059|E1|3|156|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|practice_0059|E1|6|156|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|practice_0059|E2|52-1|157||153|válido|
|practice_0059|E2|52-3|156||151|válido|
|practice_0059|E2|52-6|156||151|válido|
|practice_0059|E2|64-1|157||153|válido|
|practice_0059|E2|64-3|156||150|válido|
|practice_0059|E2|64-6|156||150|válido|
|practice_0059|E3|joint|156|individual 156|156|OPTIMAL|
|practice_0059|E4|repair|156||154|completo|
|practice_0060|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0060|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|practice_0060|E1|6|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|practice_0060|E2|52-1|301||277|válido|
|practice_0060|E2|52-3|290||268|válido|
|practice_0060|E2|52-6|290||265|válido|
|practice_0060|E2|64-1|301||281|válido|
|practice_0060|E2|64-3|290||268|válido|
|practice_0060|E2|64-6|290||268|válido|
|practice_0060|E3|joint|290|individual 289|286|FEASIBLE|
|practice_0060|E4|repair|290||290|completo|
|practice_0061|E1|1|328|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|practice_0061|E1|3|316|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|practice_0061|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|practice_0061|E2|52-1|328||298|válido|
|practice_0061|E2|52-3|316||291|válido|
|practice_0061|E2|52-6|297||282|válido|
|practice_0061|E2|64-1|328||300|válido|
|practice_0061|E2|64-3|316||287|válido|
|practice_0061|E2|64-6|297||282|válido|
|practice_0061|E3|joint|297|individual 297|297|OPTIMAL|
|practice_0061|E4|repair|297||297|completo|
|practice_0062|E1|1|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|practice_0062|E1|3|349|postprocess 349, optimize_a4 349, nested_windows 349|349|válido|
|practice_0062|E1|6|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|practice_0062|E2|52-1|358||342|válido|
|practice_0062|E2|52-3|349||324|válido|
|practice_0062|E2|52-6|336||313|válido|
|practice_0062|E2|64-1|358||339|válido|
|practice_0062|E2|64-3|349||325|válido|
|practice_0062|E2|64-6|336||314|válido|
|practice_0062|E3|joint|336|individual 336|334|OPTIMAL|
|practice_0062|E4|repair|336||336|completo|
|practice_0063|E1|1|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0063|E1|3|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0063|E1|6|135|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|practice_0063|E2|52-1|135||131|válido|
|practice_0063|E2|52-3|135||131|válido|
|practice_0063|E2|52-6|135||131|válido|
|practice_0063|E2|64-1|135||131|válido|
|practice_0063|E2|64-3|135||131|válido|
|practice_0063|E2|64-6|135||131|válido|
|practice_0063|E3|joint|135|individual 135|135|OPTIMAL|
|practice_0063|E4|repair|135||132|completo|
|practice_0064|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0064|E1|3|286|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|practice_0064|E1|6|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0064|E2|52-1|301||274|válido|
|practice_0064|E2|52-3|286||264|válido|
|practice_0064|E2|52-6|283||255|válido|
|practice_0064|E2|64-1|301||275|válido|
|practice_0064|E2|64-3|286||263|válido|
|practice_0064|E2|64-6|283||259|válido|
|practice_0064|E3|joint|283|individual 283|280|OPTIMAL|
|practice_0064|E4|repair|283||282|completo|
|practice_0065|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|practice_0065|E1|3|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|practice_0065|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|practice_0065|E2|52-1|273||249|válido|
|practice_0065|E2|52-3|262||241|válido|
|practice_0065|E2|52-6|257||236|válido|
|practice_0065|E2|64-1|273||246|válido|
|practice_0065|E2|64-3|262||244|válido|
|practice_0065|E2|64-6|257||242|válido|
|practice_0065|E3|joint|257|individual 256|256|OPTIMAL|
|practice_0065|E4|repair|257||257|completo|
|practice_0066|E1|1|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|practice_0066|E1|3|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|practice_0066|E1|6|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|practice_0066|E2|52-1|112||112|válido|
|practice_0066|E2|52-3|112||112|válido|
|practice_0066|E2|52-6|112||112|válido|
|practice_0066|E2|64-1|112||111|válido|
|practice_0066|E2|64-3|112||111|válido|
|practice_0066|E2|64-6|112||111|válido|
|practice_0066|E3|joint|112|individual 112|112|OPTIMAL|
|practice_0066|E4|repair|112||109|completo|
|practice_0067|E1|1|191|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|practice_0067|E1|3|189|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|practice_0067|E1|6|189|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|practice_0067|E2|52-1|191||173|válido|
|practice_0067|E2|52-3|189||173|válido|
|practice_0067|E2|52-6|189||173|válido|
|practice_0067|E2|64-1|191||174|válido|
|practice_0067|E2|64-3|189||175|válido|
|practice_0067|E2|64-6|189||175|válido|
|practice_0067|E3|joint|189|individual 189|184|OPTIMAL|
|practice_0067|E4|repair|189||186|completo|
|practice_0068|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|practice_0068|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|practice_0068|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0068|E2|52-1|215||206|válido|
|practice_0068|E2|52-3|214||197|válido|
|practice_0068|E2|52-6|212||198|válido|
|practice_0068|E2|64-1|215||207|válido|
|practice_0068|E2|64-3|214||198|válido|
|practice_0068|E2|64-6|212||196|válido|
|practice_0068|E3|joint|212|individual 212|212|OPTIMAL|
|practice_0068|E4|repair|212||212|completo|
|practice_0069|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|practice_0069|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|practice_0069|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|practice_0069|E2|52-1|240||229|válido|
|practice_0069|E2|52-3|232||220|válido|
|practice_0069|E2|52-6|233||220|válido|
|practice_0069|E2|64-1|240||229|válido|
|practice_0069|E2|64-3|232||222|válido|
|practice_0069|E2|64-6|233||221|válido|
|practice_0069|E3|joint|232|individual 232|231|OPTIMAL|
|practice_0069|E4|repair|232||232|completo|
|practice_0070|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|practice_0070|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|practice_0070|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0070|E2|52-1|310||287|válido|
|practice_0070|E2|52-3|288||275|válido|
|practice_0070|E2|52-6|289||271|válido|
|practice_0070|E2|64-1|310||288|válido|
|practice_0070|E2|64-3|288||276|válido|
|practice_0070|E2|64-6|289||270|válido|
|practice_0070|E3|joint|288|individual 288|287|FEASIBLE|
|practice_0070|E4|repair|288||288|completo|
|practice_0071|E1|1|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|practice_0071|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0071|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0071|E2|52-1|188||173|válido|
|practice_0071|E2|52-3|185||172|válido|
|practice_0071|E2|52-6|185||169|válido|
|practice_0071|E2|64-1|188||176|válido|
|practice_0071|E2|64-3|185||173|válido|
|practice_0071|E2|64-6|185||170|válido|
|practice_0071|E3|joint|185|individual 184|184|OPTIMAL|
|practice_0071|E4|repair|185||183|completo|
|practice_0072|E1|1|8|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|practice_0072|E1|3|8|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|practice_0072|E1|6|8|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|practice_0072|E2|52-1|8||8|válido|
|practice_0072|E2|52-3|8||8|válido|
|practice_0072|E2|52-6|8||8|válido|
|practice_0072|E2|64-1|8||8|válido|
|practice_0072|E2|64-3|8||8|válido|
|practice_0072|E2|64-6|8||8|válido|
|practice_0072|E3|joint|6|individual 6|6|OPTIMAL|
|practice_0072|E4|repair|6||5|completo|
|practice_0073|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|practice_0073|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0073|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|practice_0073|E2|52-1|169||160|válido|
|practice_0073|E2|52-3|166||159|válido|
|practice_0073|E2|52-6|166||159|válido|
|practice_0073|E2|64-1|169||160|válido|
|practice_0073|E2|64-3|166||159|válido|
|practice_0073|E2|64-6|166||159|válido|
|practice_0073|E3|joint|166|individual 165|165|OPTIMAL|
|practice_0073|E4|repair|166||165|completo|
|practice_0074|E1|1|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|practice_0074|E1|3|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|practice_0074|E1|6|238|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|practice_0074|E2|52-1|260||237|válido|
|practice_0074|E2|52-3|243||225|válido|
|practice_0074|E2|52-6|238||226|válido|
|practice_0074|E2|64-1|260||240|válido|
|practice_0074|E2|64-3|243||224|válido|
|practice_0074|E2|64-6|238||220|válido|
|practice_0074|E3|joint|238|individual 238|236|OPTIMAL|
|practice_0074|E4|repair|238||238|completo|
|practice_0075|E1|1|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|practice_0075|E1|3|212|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|practice_0075|E1|6|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|practice_0075|E2|52-1|223||207|válido|
|practice_0075|E2|52-3|212||204|válido|
|practice_0075|E2|52-6|214||199|válido|
|practice_0075|E2|64-1|223||210|válido|
|practice_0075|E2|64-3|212||205|válido|
|practice_0075|E2|64-6|214||198|válido|
|practice_0075|E3|joint|212|individual 212|210|OPTIMAL|
|practice_0075|E4|repair|212||211|completo|
|practice_0076|E1|1|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0076|E1|3|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|practice_0076|E1|6|361|postprocess 361, optimize_a4 361, nested_windows 361|361|válido|
|practice_0076|E2|52-1|377||349|válido|
|practice_0076|E2|52-3|364||332|válido|
|practice_0076|E2|52-6|361||337|válido|
|practice_0076|E2|64-1|377||352|válido|
|practice_0076|E2|64-3|364||363|válido|
|practice_0076|E2|64-6|361||338|válido|
|practice_0076|E3|joint|361|individual 360|355|FEASIBLE|
|practice_0076|E4|repair|361||361|completo|
|practice_0077|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0077|E1|3|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0077|E1|6|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0077|E2|52-1|119||115|válido|
|practice_0077|E2|52-3|119||115|válido|
|practice_0077|E2|52-6|119||115|válido|
|practice_0077|E2|64-1|119||114|válido|
|practice_0077|E2|64-3|119||114|válido|
|practice_0077|E2|64-6|119||114|válido|
|practice_0077|E3|joint|119|individual 119|119|OPTIMAL|
|practice_0077|E4|repair|119||116|completo|
|practice_0078|E1|1|331|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|practice_0078|E1|3|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|practice_0078|E1|6|369|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|practice_0078|E2|52-1|331||300|válido|
|practice_0078|E2|52-3|317||291|válido|
|practice_0078|E2|52-6|369||284|válido|
|practice_0078|E2|64-1|331||305|válido|
|practice_0078|E2|64-3|317||293|válido|
|practice_0078|E2|64-6|369||286|válido|
|practice_0078|E3|joint|317|individual 311|311|rechazado por registros|
|practice_0078|E4|repair|317||317|completo|
|practice_0079|E1|1|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|practice_0079|E1|3|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0079|E1|6|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0079|E2|52-1|142||138|válido|
|practice_0079|E2|52-3|141||138|válido|
|practice_0079|E2|52-6|141||138|válido|
|practice_0079|E2|64-1|142||138|válido|
|practice_0079|E2|64-3|141||137|válido|
|practice_0079|E2|64-6|141||137|válido|
|practice_0079|E3|joint|141|individual 141|141|OPTIMAL|
|practice_0079|E4|repair|141||139|completo|
|practice_0080|E1|1|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0080|E1|3|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0080|E1|6|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0080|E2|52-1|125||124|válido|
|practice_0080|E2|52-3|125||124|válido|
|practice_0080|E2|52-6|125||124|válido|
|practice_0080|E2|64-1|125||123|válido|
|practice_0080|E2|64-3|125||123|válido|
|practice_0080|E2|64-6|125||123|válido|
|practice_0080|E3|joint|125|individual 125|125|OPTIMAL|
|practice_0080|E4|repair|125||120|completo|
|practice_0081|E1|1|218|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|practice_0081|E1|3|221|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|practice_0081|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|practice_0081|E2|52-1|218||203|válido|
|practice_0081|E2|52-3|221||197|válido|
|practice_0081|E2|52-6|214||203|válido|
|practice_0081|E2|64-1|218||201|válido|
|practice_0081|E2|64-3|221||196|válido|
|practice_0081|E2|64-6|214||197|válido|
|practice_0081|E3|joint|214|individual 214|213|OPTIMAL|
|practice_0081|E4|repair|214||210|completo|
|practice_0082|E1|1|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|practice_0082|E1|3|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|practice_0082|E1|6|253|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|practice_0082|E2|52-1|268||240|válido|
|practice_0082|E2|52-3|256||243|válido|
|practice_0082|E2|52-6|253||243|válido|
|practice_0082|E2|64-1|268||244|válido|
|practice_0082|E2|64-3|256||243|válido|
|practice_0082|E2|64-6|253||239|válido|
|practice_0082|E3|joint|253|individual 253|250|OPTIMAL|
|practice_0082|E4|repair|253||253|completo|
|practice_0083|E1|1|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0083|E1|3|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|practice_0083|E1|6|158|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0083|E2|52-1|157||154|válido|
|practice_0083|E2|52-3|165||152|válido|
|practice_0083|E2|52-6|158||155|válido|
|practice_0083|E2|64-1|157||154|válido|
|practice_0083|E2|64-3|165||152|válido|
|practice_0083|E2|64-6|158||154|válido|
|practice_0083|E3|joint|157|individual 157|156|OPTIMAL|
|practice_0083|E4|repair|157||155|completo|
|practice_0084|E1|1|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|practice_0084|E1|3|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|practice_0084|E1|6|170|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|practice_0084|E2|52-1|180||163|válido|
|practice_0084|E2|52-3|173||163|válido|
|practice_0084|E2|52-6|170||168|válido|
|practice_0084|E2|64-1|180||164|válido|
|practice_0084|E2|64-3|173||162|válido|
|practice_0084|E2|64-6|170||169|válido|
|practice_0084|E3|joint|170|individual 170|170|OPTIMAL|
|practice_0084|E4|repair|170||170|completo|
|practice_0085|E1|1|343|postprocess 342, optimize_a4 342, nested_windows 342|342|válido|
|practice_0085|E1|3|331|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|practice_0085|E1|6|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|practice_0085|E2|52-1|343||309|válido|
|practice_0085|E2|52-3|331||313|válido|
|practice_0085|E2|52-6|318||304|válido|
|practice_0085|E2|64-1|343||354|válido|
|practice_0085|E2|64-3|331||316|válido|
|practice_0085|E2|64-6|318||304|válido|
|practice_0085|E3|joint|318|individual 317|314|FEASIBLE|
|practice_0085|E4|repair|318||318|completo|
|practice_0086|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|practice_0086|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0086|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|practice_0086|E2|52-1|168||160|válido|
|practice_0086|E2|52-3|166||157|válido|
|practice_0086|E2|52-6|163||153|válido|
|practice_0086|E2|64-1|168||158|válido|
|practice_0086|E2|64-3|166||154|válido|
|practice_0086|E2|64-6|163||151|válido|
|practice_0086|E3|joint|163|individual 163|163|OPTIMAL|
|practice_0086|E4|repair|163||162|completo|
|practice_0087|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|practice_0087|E1|3|280|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|practice_0087|E1|6|280|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|practice_0087|E2|52-1|303||264|válido|
|practice_0087|E2|52-3|280||260|válido|
|practice_0087|E2|52-6|280||255|válido|
|practice_0087|E2|64-1|303||267|válido|
|practice_0087|E2|64-3|280||262|válido|
|practice_0087|E2|64-6|280||256|válido|
|practice_0087|E3|joint|280|individual 278|276|OPTIMAL|
|practice_0087|E4|repair|280||280|completo|
|practice_0088|E1|1|276|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|practice_0088|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|practice_0088|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|practice_0088|E2|52-1|276||248|válido|
|practice_0088|E2|52-3|260||234|válido|
|practice_0088|E2|52-6|257||233|válido|
|practice_0088|E2|64-1|276||250|válido|
|practice_0088|E2|64-3|260||240|válido|
|practice_0088|E2|64-6|257||238|válido|
|practice_0088|E3|joint|257|individual 257|257|OPTIMAL|
|practice_0088|E4|repair|257||257|completo|
|practice_0089|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0089|E1|3|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|practice_0089|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|practice_0089|E2|52-1|204||180|válido|
|practice_0089|E2|52-3|198||186|válido|
|practice_0089|E2|52-6|191||165|válido|
|practice_0089|E2|64-1|204||183|válido|
|practice_0089|E2|64-3|198||187|válido|
|practice_0089|E2|64-6|191||165|válido|
|practice_0089|E3|joint|191|individual 191|191|OPTIMAL|
|practice_0089|E4|repair|191||191|completo|
|practice_0090|E1|1|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|practice_0090|E1|3|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|practice_0090|E1|6|167|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0090|E2|52-1|169||159|válido|
|practice_0090|E2|52-3|168||159|válido|
|practice_0090|E2|52-6|167||155|válido|
|practice_0090|E2|64-1|169||159|válido|
|practice_0090|E2|64-3|168||160|válido|
|practice_0090|E2|64-6|167||156|válido|
|practice_0090|E3|joint|167|individual 166|166|OPTIMAL|
|practice_0090|E4|repair|167||166|completo|
|practice_0091|E1|1|259|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|practice_0091|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0091|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|practice_0091|E2|52-1|259||228|válido|
|practice_0091|E2|52-3|246||224|válido|
|practice_0091|E2|52-6|245||230|válido|
|practice_0091|E2|64-1|259||235|válido|
|practice_0091|E2|64-3|246||237|válido|
|practice_0091|E2|64-6|245||226|válido|
|practice_0091|E3|joint|245|individual 244|244|OPTIMAL|
|practice_0091|E4|repair|245||244|completo|
|practice_0092|E1|1|204|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|practice_0092|E1|3|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|practice_0092|E1|6|201|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0092|E2|52-1|204||189|válido|
|practice_0092|E2|52-3|199||183|válido|
|practice_0092|E2|52-6|201||183|válido|
|practice_0092|E2|64-1|204||190|válido|
|practice_0092|E2|64-3|199||183|válido|
|practice_0092|E2|64-6|201||181|válido|
|practice_0092|E3|joint|199|individual 199|199|OPTIMAL|
|practice_0092|E4|repair|199||199|completo|
|practice_0093|E1|1|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|practice_0093|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|practice_0093|E1|6|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|practice_0093|E2|52-1|266||252|válido|
|practice_0093|E2|52-3|260||239|válido|
|practice_0093|E2|52-6|251||238|válido|
|practice_0093|E2|64-1|266||252|válido|
|practice_0093|E2|64-3|260||239|válido|
|practice_0093|E2|64-6|251||240|válido|
|practice_0093|E3|joint|251|individual 251|251|OPTIMAL|
|practice_0093|E4|repair|251||251|completo|
|practice_0094|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0094|E1|3|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0094|E1|6|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0094|E2|52-1|150||142|válido|
|practice_0094|E2|52-3|149||141|válido|
|practice_0094|E2|52-6|149||141|válido|
|practice_0094|E2|64-1|150||141|válido|
|practice_0094|E2|64-3|149||141|válido|
|practice_0094|E2|64-6|149||141|válido|
|practice_0094|E3|joint|149|individual 149|149|OPTIMAL|
|practice_0094|E4|repair|149||149|completo|
|practice_0095|E1|1|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0095|E1|3|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0095|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0095|E2|52-1|294||266|válido|
|practice_0095|E2|52-3|283||262|válido|
|practice_0095|E2|52-6|282||261|válido|
|practice_0095|E2|64-1|294||267|válido|
|practice_0095|E2|64-3|283||263|válido|
|practice_0095|E2|64-6|282||259|válido|
|practice_0095|E3|joint|282|individual 282|275|OPTIMAL|
|practice_0095|E4|repair|282||282|completo|
|practice_0096|E1|1|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|practice_0096|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|practice_0096|E1|6|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0096|E2|52-1|239||225|válido|
|practice_0096|E2|52-3|235||220|válido|
|practice_0096|E2|52-6|230||214|válido|
|practice_0096|E2|64-1|239||226|válido|
|practice_0096|E2|64-3|235||220|válido|
|practice_0096|E2|64-6|230||216|válido|
|practice_0096|E3|joint|230|individual 230|229|FEASIBLE|
|practice_0096|E4|repair|230||230|completo|
|practice_0097|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0097|E1|3|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0097|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|practice_0097|E2|52-1|141||133|válido|
|practice_0097|E2|52-3|141||133|válido|
|practice_0097|E2|52-6|139||133|válido|
|practice_0097|E2|64-1|141||136|válido|
|practice_0097|E2|64-3|141||136|válido|
|practice_0097|E2|64-6|139||136|válido|
|practice_0097|E3|joint|139|individual 139|139|OPTIMAL|
|practice_0097|E4|repair|139||138|completo|
|practice_0098|E1|1|2|postprocess 2, optimize_a4 1, nested_windows 1|1|válido|
|practice_0098|E1|3|2|postprocess 2, optimize_a4 1, nested_windows 1|1|válido|
|practice_0098|E1|6|2|postprocess 2, optimize_a4 1, nested_windows 1|1|válido|
|practice_0098|E2|52-1|2||2|válido|
|practice_0098|E2|52-3|2||2|válido|
|practice_0098|E2|52-6|2||2|válido|
|practice_0098|E2|64-1|2||2|válido|
|practice_0098|E2|64-3|2||2|válido|
|practice_0098|E2|64-6|2||2|válido|
|practice_0098|E3|joint|1|individual 1|1|OPTIMAL|
|practice_0098|E4|repair|1||1|completo|
|practice_0099|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0099|E1|3|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0099|E1|6|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0099|E2|52-1|137||132|válido|
|practice_0099|E2|52-3|135||132|válido|
|practice_0099|E2|52-6|135||132|válido|
|practice_0099|E2|64-1|137||132|válido|
|practice_0099|E2|64-3|135||132|válido|
|practice_0099|E2|64-6|135||132|válido|
|practice_0099|E3|joint|135|individual 135|135|OPTIMAL|
|practice_0099|E4|repair|135||135|completo|
|practice_0100|E1|1|385|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|practice_0100|E1|3|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|practice_0100|E1|6|366|postprocess 365, optimize_a4 365, nested_windows 365|365|válido|
|practice_0100|E2|52-1|385||363|válido|
|practice_0100|E2|52-3|369||341|válido|
|practice_0100|E2|52-6|366||342|válido|
|practice_0100|E2|64-1|385||361|válido|
|practice_0100|E2|64-3|369||345|válido|
|practice_0100|E2|64-6|366||344|válido|
|practice_0100|E3|joint|366|individual 365|362|FEASIBLE|
|practice_0100|E4|repair|366||366|completo|
|practice_0101|E1|1|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|practice_0101|E1|3|176|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|practice_0101|E1|6|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|practice_0101|E2|52-1|180||168|válido|
|practice_0101|E2|52-3|176||168|válido|
|practice_0101|E2|52-6|173||168|válido|
|practice_0101|E2|64-1|180||168|válido|
|practice_0101|E2|64-3|176||167|válido|
|practice_0101|E2|64-6|173||166|válido|
|practice_0101|E3|joint|173|individual 173|173|OPTIMAL|
|practice_0101|E4|repair|173||173|completo|
|practice_0102|E1|1|348|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|practice_0102|E1|3|338|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|practice_0102|E1|6|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|practice_0102|E2|52-1|348||324|válido|
|practice_0102|E2|52-3|338||316|válido|
|practice_0102|E2|52-6|327||307|válido|
|practice_0102|E2|64-1|348||323|válido|
|practice_0102|E2|64-3|338||315|válido|
|practice_0102|E2|64-6|327||306|válido|
|practice_0102|E3|joint|327|individual 326|326|OPTIMAL|
|practice_0102|E4|repair|327||327|completo|
|practice_0103|E1|1|6|postprocess 6, optimize_a4 5, nested_windows 5|5|válido|
|practice_0103|E1|3|6|postprocess 6, optimize_a4 5, nested_windows 5|5|válido|
|practice_0103|E1|6|6|postprocess 6, optimize_a4 5, nested_windows 5|5|válido|
|practice_0103|E2|52-1|6||6|válido|
|practice_0103|E2|52-3|6||6|válido|
|practice_0103|E2|52-6|6||6|válido|
|practice_0103|E2|64-1|6||6|válido|
|practice_0103|E2|64-3|6||6|válido|
|practice_0103|E2|64-6|6||6|válido|
|practice_0103|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0103|E4|repair|5||5|completo|
|practice_0104|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|practice_0104|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|practice_0104|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|practice_0104|E2|52-1|251||236|válido|
|practice_0104|E2|52-3|243||238|válido|
|practice_0104|E2|52-6|244||230|válido|
|practice_0104|E2|64-1|251||240|válido|
|practice_0104|E2|64-3|243||239|válido|
|practice_0104|E2|64-6|244||230|válido|
|practice_0104|E3|joint|243|individual 243|241|OPTIMAL|
|practice_0104|E4|repair|243||242|completo|
|practice_0105|E1|1|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|practice_0105|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0105|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|practice_0105|E2|52-1|214||196|válido|
|practice_0105|E2|52-3|206||197|válido|
|practice_0105|E2|52-6|202||195|válido|
|practice_0105|E2|64-1|214||196|válido|
|practice_0105|E2|64-3|206||196|válido|
|practice_0105|E2|64-6|202||195|válido|
|practice_0105|E3|joint|202|individual 202|202|OPTIMAL|
|practice_0105|E4|repair|202||200|completo|
|practice_0106|E1|1|200|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0106|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0106|E1|6|191|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|practice_0106|E2|52-1|200||187|válido|
|practice_0106|E2|52-3|197||189|válido|
|practice_0106|E2|52-6|191||180|válido|
|practice_0106|E2|64-1|200||190|válido|
|practice_0106|E2|64-3|197||189|válido|
|practice_0106|E2|64-6|191||180|válido|
|practice_0106|E3|joint|191|individual 191|191|OPTIMAL|
|practice_0106|E4|repair|191||190|completo|
|practice_0107|E1|1|132|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0107|E1|3|132|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0107|E1|6|132|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0107|E2|52-1|132||125|válido|
|practice_0107|E2|52-3|132||125|válido|
|practice_0107|E2|52-6|132||125|válido|
|practice_0107|E2|64-1|132||125|válido|
|practice_0107|E2|64-3|132||125|válido|
|practice_0107|E2|64-6|132||125|válido|
|practice_0107|E3|joint|132|individual 132|132|OPTIMAL|
|practice_0107|E4|repair|132||130|completo|
|practice_0108|E1|1|121|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0108|E1|3|121|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0108|E1|6|121|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0108|E2|52-1|121||119|válido|
|practice_0108|E2|52-3|121||119|válido|
|practice_0108|E2|52-6|121||119|válido|
|practice_0108|E2|64-1|121||119|válido|
|practice_0108|E2|64-3|121||119|válido|
|practice_0108|E2|64-6|121||119|válido|
|practice_0108|E3|joint|121|individual 121|121|OPTIMAL|
|practice_0108|E4|repair|121||116|completo|
|practice_0109|E1|1|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0109|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0109|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0109|E2|52-1|153||151|válido|
|practice_0109|E2|52-3|152||150|válido|
|practice_0109|E2|52-6|152||150|válido|
|practice_0109|E2|64-1|153||152|válido|
|practice_0109|E2|64-3|152||151|válido|
|practice_0109|E2|64-6|152||150|válido|
|practice_0109|E3|joint|152|individual 152|152|OPTIMAL|
|practice_0109|E4|repair|152||150|completo|
|practice_0110|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|practice_0110|E1|3|283|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0110|E1|6|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|practice_0110|E2|52-1|296||281|válido|
|practice_0110|E2|52-3|283||268|válido|
|practice_0110|E2|52-6|279||268|válido|
|practice_0110|E2|64-1|296||281|válido|
|practice_0110|E2|64-3|283||390|válido|
|practice_0110|E2|64-6|279||270|válido|
|practice_0110|E3|joint|279|individual 279|276|FEASIBLE|
|practice_0110|E4|repair|279||279|completo|
|practice_0111|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0111|E1|3|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0111|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|practice_0111|E2|52-1|197||185|válido|
|practice_0111|E2|52-3|193||181|válido|
|practice_0111|E2|52-6|193||183|válido|
|practice_0111|E2|64-1|197||187|válido|
|practice_0111|E2|64-3|193||180|válido|
|practice_0111|E2|64-6|193||180|válido|
|practice_0111|E3|joint|193|individual 193|192|OPTIMAL|
|practice_0111|E4|repair|193||193|completo|
|practice_0112|E1|1|132|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0112|E1|3|131|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0112|E1|6|131|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0112|E2|52-1|132||129|válido|
|practice_0112|E2|52-3|131||129|válido|
|practice_0112|E2|52-6|131||129|válido|
|practice_0112|E2|64-1|132||129|válido|
|practice_0112|E2|64-3|131||129|válido|
|practice_0112|E2|64-6|131||129|válido|
|practice_0112|E3|joint|131|individual 131|131|OPTIMAL|
|practice_0112|E4|repair|131||128|completo|
|practice_0113|E1|1|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|practice_0113|E1|3|331|postprocess 331, optimize_a4 331, nested_windows 331|331|válido|
|practice_0113|E1|6|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|practice_0113|E2|52-1|345||324|válido|
|practice_0113|E2|52-3|331||310|válido|
|practice_0113|E2|52-6|330||306|válido|
|practice_0113|E2|64-1|345||329|válido|
|practice_0113|E2|64-3|331||308|válido|
|practice_0113|E2|64-6|330||308|válido|
|practice_0113|E3|joint|330|individual 330|329|FEASIBLE|
|practice_0113|E4|repair|330||330|completo|
|practice_0114|E1|1|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|practice_0114|E1|3|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|practice_0114|E1|6|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|practice_0114|E2|52-1|56||57|válido|
|practice_0114|E2|52-3|56||57|válido|
|practice_0114|E2|52-6|56||57|válido|
|practice_0114|E2|64-1|56||57|válido|
|practice_0114|E2|64-3|56||57|válido|
|practice_0114|E2|64-6|56||57|válido|
|practice_0114|E3|joint|56|individual 56|56|OPTIMAL|
|practice_0114|E4|repair|56||56|completo|
|practice_0115|E1|1|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|practice_0115|E1|3|201|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|practice_0115|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|practice_0115|E2|52-1|214||201|válido|
|practice_0115|E2|52-3|201||192|válido|
|practice_0115|E2|52-6|203||189|válido|
|practice_0115|E2|64-1|214||201|válido|
|practice_0115|E2|64-3|201||190|válido|
|practice_0115|E2|64-6|203||190|válido|
|practice_0115|E3|joint|201|individual 201|201|OPTIMAL|
|practice_0115|E4|repair|201||201|completo|
|practice_0116|E1|1|74|postprocess 73, optimize_a4 73, nested_windows 73|73|válido|
|practice_0116|E1|3|74|postprocess 73, optimize_a4 73, nested_windows 73|73|válido|
|practice_0116|E1|6|69|postprocess 67, optimize_a4 67, nested_windows 67|67|válido|
|practice_0116|E2|52-1|74||72|válido|
|practice_0116|E2|52-3|74||72|válido|
|practice_0116|E2|52-6|69||71|válido|
|practice_0116|E2|64-1|74||72|válido|
|practice_0116|E2|64-3|74||72|válido|
|practice_0116|E2|64-6|69||71|válido|
|practice_0116|E3|joint|69|individual 69|69|OPTIMAL|
|practice_0116|E4|repair|69||69|completo|
|practice_0117|E1|1|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0117|E1|3|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0117|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0117|E2|52-1|156||144|válido|
|practice_0117|E2|52-3|156||143|válido|
|practice_0117|E2|52-6|156||143|válido|
|practice_0117|E2|64-1|156||143|válido|
|practice_0117|E2|64-3|156||143|válido|
|practice_0117|E2|64-6|156||143|válido|
|practice_0117|E3|joint|156|individual 156|156|OPTIMAL|
|practice_0117|E4|repair|156||155|completo|
|practice_0118|E1|1|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0118|E1|3|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0118|E1|6|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0118|E2|52-1|43||43|válido|
|practice_0118|E2|52-3|43||43|válido|
|practice_0118|E2|52-6|43||43|válido|
|practice_0118|E2|64-1|43||43|válido|
|practice_0118|E2|64-3|43||43|válido|
|practice_0118|E2|64-6|43||43|válido|
|practice_0118|E3|joint|43|individual 43|43|OPTIMAL|
|practice_0118|E4|repair|43||43|completo|
|practice_0119|E1|1|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0119|E1|3|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0119|E1|6|289|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|practice_0119|E2|52-1|311||281|válido|
|practice_0119|E2|52-3|289||274|válido|
|practice_0119|E2|52-6|289||267|válido|
|practice_0119|E2|64-1|311||282|válido|
|practice_0119|E2|64-3|289||273|válido|
|practice_0119|E2|64-6|289||266|válido|
|practice_0119|E3|joint|289|individual 289|285|FEASIBLE|
|practice_0119|E4|repair|289||289|completo|
|practice_0120|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|practice_0120|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|practice_0120|E1|6|170|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|practice_0120|E2|52-1|176||169|válido|
|practice_0120|E2|52-3|173||167|válido|
|practice_0120|E2|52-6|170||167|válido|
|practice_0120|E2|64-1|176||170|válido|
|practice_0120|E2|64-3|173||166|válido|
|practice_0120|E2|64-6|170||166|válido|
|practice_0120|E3|joint|170|individual 170|170|OPTIMAL|
|practice_0120|E4|repair|170||166|completo|
|practice_0121|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|practice_0121|E1|3|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|practice_0121|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|practice_0121|E2|52-1|319||296|válido|
|practice_0121|E2|52-3|305||289|válido|
|practice_0121|E2|52-6|295||280|válido|
|practice_0121|E2|64-1|319||294|válido|
|practice_0121|E2|64-3|305||290|válido|
|practice_0121|E2|64-6|295||283|válido|
|practice_0121|E3|joint|295|individual 295|295|OPTIMAL|
|practice_0121|E4|repair|295||295|completo|
|practice_0122|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|practice_0122|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|practice_0122|E1|6|346|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|practice_0122|E2|52-1|370||345|válido|
|practice_0122|E2|52-3|350||322|válido|
|practice_0122|E2|52-6|346||319|válido|
|practice_0122|E2|64-1|370||460|válido|
|practice_0122|E2|64-3|350||324|válido|
|practice_0122|E2|64-6|346||319|válido|
|practice_0122|E3|joint|346|individual 346|345|FEASIBLE|
|practice_0122|E4|repair|346||346|completo|
|practice_0123|E1|1|3|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0123|E1|3|3|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0123|E1|6|3|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0123|E2|52-1|3||3|válido|
|practice_0123|E2|52-3|3||3|válido|
|practice_0123|E2|52-6|3||3|válido|
|practice_0123|E2|64-1|3||3|válido|
|practice_0123|E2|64-3|3||3|válido|
|practice_0123|E2|64-6|3||3|válido|
|practice_0123|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0123|E4|repair|2||2|completo|
|practice_0124|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|practice_0124|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|practice_0124|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|practice_0124|E2|52-1|253||234|válido|
|practice_0124|E2|52-3|239||229|válido|
|practice_0124|E2|52-6|242||226|válido|
|practice_0124|E2|64-1|253||236|válido|
|practice_0124|E2|64-3|239||230|válido|
|practice_0124|E2|64-6|242||226|válido|
|practice_0124|E3|joint|239|individual 238|237|OPTIMAL|
|practice_0124|E4|repair|239||239|completo|
|practice_0125|E1|1|4|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|practice_0125|E1|3|4|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|practice_0125|E1|6|4|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|practice_0125|E2|52-1|4||4|válido|
|practice_0125|E2|52-3|4||4|válido|
|practice_0125|E2|52-6|4||4|válido|
|practice_0125|E2|64-1|4||4|válido|
|practice_0125|E2|64-3|4||4|válido|
|practice_0125|E2|64-6|4||4|válido|
|practice_0125|E3|joint|3|individual 3|3|OPTIMAL|
|practice_0125|E4|repair|3||3|completo|
|practice_0126|E1|1|181|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|practice_0126|E1|3|176|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|practice_0126|E1|6|173|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|practice_0126|E2|52-1|181||165|válido|
|practice_0126|E2|52-3|176||163|válido|
|practice_0126|E2|52-6|173||156|válido|
|practice_0126|E2|64-1|181||171|válido|
|practice_0126|E2|64-3|176||166|válido|
|practice_0126|E2|64-6|173||158|válido|
|practice_0126|E3|joint|173|individual 173|173|OPTIMAL|
|practice_0126|E4|repair|173||170|completo|
|practice_0127|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|practice_0127|E1|3|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|practice_0127|E1|6|207|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0127|E2|52-1|216||201|válido|
|practice_0127|E2|52-3|210||199|válido|
|practice_0127|E2|52-6|207||196|válido|
|practice_0127|E2|64-1|216||200|válido|
|practice_0127|E2|64-3|210||198|válido|
|practice_0127|E2|64-6|207||197|válido|
|practice_0127|E3|joint|207|individual 207|207|OPTIMAL|
|practice_0127|E4|repair|207||206|completo|
|practice_0128|E1|1|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0128|E1|3|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0128|E1|6|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0128|E2|52-1|4||4|válido|
|practice_0128|E2|52-3|4||4|válido|
|practice_0128|E2|52-6|4||4|válido|
|practice_0128|E2|64-1|4||4|válido|
|practice_0128|E2|64-3|4||4|válido|
|practice_0128|E2|64-6|4||4|válido|
|practice_0128|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0128|E4|repair|2||2|completo|
|practice_0129|E1|1|133|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0129|E1|3|133|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0129|E1|6|131|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0129|E2|52-1|133||125|válido|
|practice_0129|E2|52-3|133||125|válido|
|practice_0129|E2|52-6|131||125|válido|
|practice_0129|E2|64-1|133||127|válido|
|practice_0129|E2|64-3|133||127|válido|
|practice_0129|E2|64-6|131||127|válido|
|practice_0129|E3|joint|131|individual 131|131|OPTIMAL|
|practice_0129|E4|repair|131||127|completo|
|practice_0130|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0130|E1|3|120|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|practice_0130|E1|6|120|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|practice_0130|E2|52-1|119||111|válido|
|practice_0130|E2|52-3|120||111|válido|
|practice_0130|E2|52-6|120||111|válido|
|practice_0130|E2|64-1|119||114|válido|
|practice_0130|E2|64-3|120||114|válido|
|practice_0130|E2|64-6|120||113|válido|
|practice_0130|E3|joint|119|individual 119|119|OPTIMAL|
|practice_0130|E4|repair|119||116|completo|
|practice_0131|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|practice_0131|E1|3|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|practice_0131|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|practice_0131|E2|52-1|254||234|válido|
|practice_0131|E2|52-3|244||229|válido|
|practice_0131|E2|52-6|238||228|válido|
|practice_0131|E2|64-1|254||234|válido|
|practice_0131|E2|64-3|244||229|válido|
|practice_0131|E2|64-6|238||226|válido|
|practice_0131|E3|joint|238|individual 238|238|OPTIMAL|
|practice_0131|E4|repair|238||238|completo|
|practice_0132|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0132|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0132|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|practice_0132|E2|52-1|199||183|válido|
|practice_0132|E2|52-3|196||181|válido|
|practice_0132|E2|52-6|193||179|válido|
|practice_0132|E2|64-1|199||183|válido|
|practice_0132|E2|64-3|196||180|válido|
|practice_0132|E2|64-6|193||180|válido|
|practice_0132|E3|joint|193|individual 192|192|OPTIMAL|
|practice_0132|E4|repair|193||193|completo|
|practice_0133|E1|1|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|practice_0133|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0133|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|practice_0133|E2|52-1|208||190|válido|
|practice_0133|E2|52-3|197||188|válido|
|practice_0133|E2|52-6|198||189|válido|
|practice_0133|E2|64-1|208||190|válido|
|practice_0133|E2|64-3|197||188|válido|
|practice_0133|E2|64-6|198||189|válido|
|practice_0133|E3|joint|197|individual 197|197|OPTIMAL|
|practice_0133|E4|repair|197||195|completo|
|practice_0134|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|practice_0134|E1|3|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0134|E1|6|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0134|E2|52-1|237||225|válido|
|practice_0134|E2|52-3|230||218|válido|
|practice_0134|E2|52-6|228||215|válido|
|practice_0134|E2|64-1|237||225|válido|
|practice_0134|E2|64-3|230||221|válido|
|practice_0134|E2|64-6|228||215|válido|
|practice_0134|E3|joint|228|individual 228|225|OPTIMAL|
|practice_0134|E4|repair|228||226|completo|
|practice_0135|E1|1|401|postprocess 401, optimize_a4 401, nested_windows 401|401|válido|
|practice_0135|E1|3|386|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|practice_0135|E1|6|373|postprocess 373, optimize_a4 373, nested_windows 373|373|válido|
|practice_0135|E2|52-1|401||366|válido|
|practice_0135|E2|52-3|386||354|válido|
|practice_0135|E2|52-6|373||344|válido|
|practice_0135|E2|64-1|401||371|válido|
|practice_0135|E2|64-3|386||359|válido|
|practice_0135|E2|64-6|373||346|válido|
|practice_0135|E3|joint|373|individual 373|373|FEASIBLE|
|practice_0135|E4|repair|373||373|completo|
|practice_0136|E1|1|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|practice_0136|E1|3|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|practice_0136|E1|6|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|practice_0136|E2|52-1|56||56|válido|
|practice_0136|E2|52-3|56||56|válido|
|practice_0136|E2|52-6|56||56|válido|
|practice_0136|E2|64-1|56||56|válido|
|practice_0136|E2|64-3|56||56|válido|
|practice_0136|E2|64-6|56||56|válido|
|practice_0136|E3|joint|56|individual 56|56|OPTIMAL|
|practice_0136|E4|repair|56||54|completo|
|practice_0137|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0137|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0137|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0137|E2|52-1|113||106|válido|
|practice_0137|E2|52-3|113||106|válido|
|practice_0137|E2|52-6|113||106|válido|
|practice_0137|E2|64-1|113||106|válido|
|practice_0137|E2|64-3|113||106|válido|
|practice_0137|E2|64-6|113||106|válido|
|practice_0137|E3|joint|113|individual 113|113|OPTIMAL|
|practice_0137|E4|repair|113||110|completo|
|practice_0138|E1|1|402|postprocess 402, optimize_a4 402, nested_windows 402|402|válido|
|practice_0138|E1|3|378|postprocess 378, optimize_a4 378, nested_windows 378|378|válido|
|practice_0138|E1|6|367|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|practice_0138|E2|52-1|402||365|válido|
|practice_0138|E2|52-3|378||356|válido|
|practice_0138|E2|52-6|367||346|válido|
|practice_0138|E2|64-1|402||369|válido|
|practice_0138|E2|64-3|378||359|válido|
|practice_0138|E2|64-6|367||345|válido|
|practice_0138|E3|joint|367|individual 365|365|FEASIBLE|
|practice_0138|E4|repair|367||367|completo|
|practice_0139|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|practice_0139|E1|3|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|practice_0139|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|practice_0139|E2|52-1|247||224|válido|
|practice_0139|E2|52-3|236||221|válido|
|practice_0139|E2|52-6|237||218|válido|
|practice_0139|E2|64-1|247||223|válido|
|practice_0139|E2|64-3|236||222|válido|
|practice_0139|E2|64-6|237||218|válido|
|practice_0139|E3|joint|236|individual 236|236|OPTIMAL|
|practice_0139|E4|repair|236||236|completo|
|practice_0140|E1|1|1|postprocess 1, optimize_a4 1, nested_windows 1|1|válido|
|practice_0140|E1|3|1|postprocess 1, optimize_a4 1, nested_windows 1|1|válido|
|practice_0140|E1|6|1|postprocess 1, optimize_a4 1, nested_windows 1|1|válido|
|practice_0140|E2|52-1|1||1|válido|
|practice_0140|E2|52-3|1||1|válido|
|practice_0140|E2|52-6|1||1|válido|
|practice_0140|E2|64-1|1||1|válido|
|practice_0140|E2|64-3|1||1|válido|
|practice_0140|E2|64-6|1||1|válido|
|practice_0140|E3|joint|1|individual 1|1|OPTIMAL|
|practice_0140|E4|repair|1||1|completo|
|practice_0141|E1|1|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|practice_0141|E1|3|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|practice_0141|E1|6|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|practice_0141|E2|52-1|170||160|válido|
|practice_0141|E2|52-3|168||159|válido|
|practice_0141|E2|52-6|167||157|válido|
|practice_0141|E2|64-1|170||159|válido|
|practice_0141|E2|64-3|168||159|válido|
|practice_0141|E2|64-6|167||156|válido|
|practice_0141|E3|joint|167|individual 167|167|OPTIMAL|
|practice_0141|E4|repair|167||167|completo|
|practice_0142|E1|1|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|practice_0142|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|practice_0142|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|practice_0142|E2|52-1|302||281|válido|
|practice_0142|E2|52-3|277||276|válido|
|practice_0142|E2|52-6|281||264|válido|
|practice_0142|E2|64-1|302||283|válido|
|practice_0142|E2|64-3|277||275|válido|
|practice_0142|E2|64-6|281||268|válido|
|practice_0142|E3|joint|277|individual 277|276|OPTIMAL|
|practice_0142|E4|repair|277||277|completo|
|practice_0143|E1|1|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|practice_0143|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|practice_0143|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|practice_0143|E2|52-1|178||161|válido|
|practice_0143|E2|52-3|174||159|válido|
|practice_0143|E2|52-6|172||156|válido|
|practice_0143|E2|64-1|178||163|válido|
|practice_0143|E2|64-3|174||161|válido|
|practice_0143|E2|64-6|172||156|válido|
|practice_0143|E3|joint|172|individual 172|172|OPTIMAL|
|practice_0143|E4|repair|172||172|completo|
|practice_0144|E1|1|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0144|E1|3|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|practice_0144|E1|6|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|practice_0144|E2|52-1|125||116|válido|
|practice_0144|E2|52-3|124||116|válido|
|practice_0144|E2|52-6|124||116|válido|
|practice_0144|E2|64-1|125||116|válido|
|practice_0144|E2|64-3|124||115|válido|
|practice_0144|E2|64-6|124||115|válido|
|practice_0144|E3|joint|124|individual 124|124|OPTIMAL|
|practice_0144|E4|repair|124||117|completo|
|practice_0145|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|practice_0145|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|practice_0145|E1|6|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|practice_0145|E2|52-1|229||213|válido|
|practice_0145|E2|52-3|225||211|válido|
|practice_0145|E2|52-6|219||207|válido|
|practice_0145|E2|64-1|229||208|válido|
|practice_0145|E2|64-3|225||201|válido|
|practice_0145|E2|64-6|219||202|válido|
|practice_0145|E3|joint|219|individual 218|218|OPTIMAL|
|practice_0145|E4|repair|219||212|completo|
|practice_0146|E1|1|47|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|practice_0146|E1|3|47|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|practice_0146|E1|6|47|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|practice_0146|E2|52-1|47||47|válido|
|practice_0146|E2|52-3|47||47|válido|
|practice_0146|E2|52-6|47||47|válido|
|practice_0146|E2|64-1|47||47|válido|
|practice_0146|E2|64-3|47||47|válido|
|practice_0146|E2|64-6|47||47|válido|
|practice_0146|E3|joint|47|individual 47|47|OPTIMAL|
|practice_0146|E4|repair|47||46|completo|
|practice_0147|E1|1|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|practice_0147|E1|3|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0147|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0147|E2|52-1|205||192|válido|
|practice_0147|E2|52-3|205||188|válido|
|practice_0147|E2|52-6|197||188|válido|
|practice_0147|E2|64-1|205||193|válido|
|practice_0147|E2|64-3|205||188|válido|
|practice_0147|E2|64-6|197||189|válido|
|practice_0147|E3|joint|197|individual 197|197|OPTIMAL|
|practice_0147|E4|repair|197||194|completo|
|practice_0148|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0148|E1|3|186|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0148|E1|6|185|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|practice_0148|E2|52-1|192||181|válido|
|practice_0148|E2|52-3|186||177|válido|
|practice_0148|E2|52-6|185||176|válido|
|practice_0148|E2|64-1|192||180|válido|
|practice_0148|E2|64-3|186||177|válido|
|practice_0148|E2|64-6|185||177|válido|
|practice_0148|E3|joint|185|individual 185|185|OPTIMAL|
|practice_0148|E4|repair|185||182|completo|
|practice_0149|E1|1|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0149|E1|3|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0149|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0149|E2|52-1|230||211|válido|
|practice_0149|E2|52-3|212||205|válido|
|practice_0149|E2|52-6|212||203|válido|
|practice_0149|E2|64-1|230||214|válido|
|practice_0149|E2|64-3|212||209|válido|
|practice_0149|E2|64-6|212||201|válido|
|practice_0149|E3|joint|212|individual 212|209|OPTIMAL|
|practice_0149|E4|repair|212||212|completo|
|practice_0150|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|practice_0150|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|practice_0150|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|practice_0150|E2|52-1|228||209|válido|
|practice_0150|E2|52-3|219||208|válido|
|practice_0150|E2|52-6|217||203|válido|
|practice_0150|E2|64-1|228||209|válido|
|practice_0150|E2|64-3|219||208|válido|
|practice_0150|E2|64-6|217||204|válido|
|practice_0150|E3|joint|217|individual 216|216|OPTIMAL|
|practice_0150|E4|repair|217||217|completo|
|practice_0151|E1|1|379|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|practice_0151|E1|3|365|postprocess 365, optimize_a4 365, nested_windows 365|365|válido|
|practice_0151|E1|6|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|practice_0151|E2|52-1|379||344|válido|
|practice_0151|E2|52-3|365||337|válido|
|practice_0151|E2|52-6|355||333|válido|
|practice_0151|E2|64-1|379||349|válido|
|practice_0151|E2|64-3|365||343|válido|
|practice_0151|E2|64-6|355||336|válido|
|practice_0151|E3|joint|355|individual 355|355|FEASIBLE|
|practice_0151|E4|repair|355||355|completo|
|practice_0152|E1|1|187|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0152|E1|3|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|practice_0152|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|practice_0152|E2|52-1|187||167|válido|
|practice_0152|E2|52-3|175||164|válido|
|practice_0152|E2|52-6|172||162|válido|
|practice_0152|E2|64-1|187||168|válido|
|practice_0152|E2|64-3|175||162|válido|
|practice_0152|E2|64-6|172||161|válido|
|practice_0152|E3|joint|172|individual 171|171|OPTIMAL|
|practice_0152|E4|repair|172||172|completo|
|practice_0153|E1|1|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|practice_0153|E1|3|163|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|practice_0153|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|practice_0153|E2|52-1|171||163|válido|
|practice_0153|E2|52-3|163||153|válido|
|practice_0153|E2|52-6|162||149|válido|
|practice_0153|E2|64-1|171||165|válido|
|practice_0153|E2|64-3|163||154|válido|
|practice_0153|E2|64-6|162||153|válido|
|practice_0153|E3|joint|162|individual 162|162|OPTIMAL|
|practice_0153|E4|repair|162||161|completo|
|practice_0154|E1|1|383|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|practice_0154|E1|3|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|practice_0154|E1|6|360|postprocess 360, optimize_a4 360, nested_windows 360|360|válido|
|practice_0154|E2|52-1|383||351|válido|
|practice_0154|E2|52-3|369||339|válido|
|practice_0154|E2|52-6|360||336|válido|
|practice_0154|E2|64-1|383||354|válido|
|practice_0154|E2|64-3|369||342|válido|
|practice_0154|E2|64-6|360||340|válido|
|practice_0154|E3|joint|360|individual 360|359|OPTIMAL|
|practice_0154|E4|repair|360||360|completo|
|practice_0155|E1|1|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|practice_0155|E1|3|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|practice_0155|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0155|E2|52-1|174||163|válido|
|practice_0155|E2|52-3|170||165|válido|
|practice_0155|E2|52-6|166||160|válido|
|practice_0155|E2|64-1|174||165|válido|
|practice_0155|E2|64-3|170||163|válido|
|practice_0155|E2|64-6|166||160|válido|
|practice_0155|E3|joint|166|individual 166|166|OPTIMAL|
|practice_0155|E4|repair|166||164|completo|
|practice_0156|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0156|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0156|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0156|E2|52-1|113||111|válido|
|practice_0156|E2|52-3|113||111|válido|
|practice_0156|E2|52-6|113||111|válido|
|practice_0156|E2|64-1|113||111|válido|
|practice_0156|E2|64-3|113||111|válido|
|practice_0156|E2|64-6|113||111|válido|
|practice_0156|E3|joint|113|individual 113|113|OPTIMAL|
|practice_0156|E4|repair|113||112|completo|
|practice_0157|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|practice_0157|E1|3|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0157|E1|6|152|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|practice_0157|E2|52-1|158||145|válido|
|practice_0157|E2|52-3|152||143|válido|
|practice_0157|E2|52-6|152||146|válido|
|practice_0157|E2|64-1|158||147|válido|
|practice_0157|E2|64-3|152||143|válido|
|practice_0157|E2|64-6|152||147|válido|
|practice_0157|E3|joint|152|individual 152|151|OPTIMAL|
|practice_0157|E4|repair|152||150|completo|
|practice_0158|E1|1|276|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|practice_0158|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|practice_0158|E1|6|260|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|practice_0158|E2|52-1|276||251|válido|
|practice_0158|E2|52-3|270||244|válido|
|practice_0158|E2|52-6|260||239|válido|
|practice_0158|E2|64-1|276||252|válido|
|practice_0158|E2|64-3|270||249|válido|
|practice_0158|E2|64-6|260||236|válido|
|practice_0158|E3|joint|260|individual 259|257|OPTIMAL|
|practice_0158|E4|repair|260||260|completo|
|practice_0159|E1|1|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|practice_0159|E1|3|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|practice_0159|E1|6|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|practice_0159|E2|52-1|39||39|válido|
|practice_0159|E2|52-3|39||39|válido|
|practice_0159|E2|52-6|39||39|válido|
|practice_0159|E2|64-1|39||39|válido|
|practice_0159|E2|64-3|39||39|válido|
|practice_0159|E2|64-6|39||39|válido|
|practice_0159|E3|joint|39|individual 39|39|OPTIMAL|
|practice_0159|E4|repair|39||34|completo|
|practice_0160|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|practice_0160|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0160|E1|6|197|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0160|E2|52-1|202||189|válido|
|practice_0160|E2|52-3|196||187|válido|
|practice_0160|E2|52-6|197||187|válido|
|practice_0160|E2|64-1|202||190|válido|
|practice_0160|E2|64-3|196||186|válido|
|practice_0160|E2|64-6|197||187|válido|
|practice_0160|E3|joint|196|individual 195|194|OPTIMAL|
|practice_0160|E4|repair|196||196|completo|
|practice_0161|E1|1|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0161|E1|3|296|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0161|E1|6|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|practice_0161|E2|52-1|312||284|válido|
|practice_0161|E2|52-3|296||274|válido|
|practice_0161|E2|52-6|300||274|válido|
|practice_0161|E2|64-1|312||285|válido|
|practice_0161|E2|64-3|296||277|válido|
|practice_0161|E2|64-6|300||274|válido|
|practice_0161|E3|joint|296|individual 296|295|FEASIBLE|
|practice_0161|E4|repair|296||296|completo|
|practice_0162|E1|1|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|practice_0162|E1|3|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|practice_0162|E1|6|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0162|E2|52-1|315||287|válido|
|practice_0162|E2|52-3|305||287|válido|
|practice_0162|E2|52-6|302||285|válido|
|practice_0162|E2|64-1|315||295|válido|
|practice_0162|E2|64-3|305||291|válido|
|practice_0162|E2|64-6|302||288|válido|
|practice_0162|E3|joint|302|individual 302|299|OPTIMAL|
|practice_0162|E4|repair|302||302|completo|
|practice_0163|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0163|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|practice_0163|E1|6|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|practice_0163|E2|52-1|301||277|válido|
|practice_0163|E2|52-3|281||267|válido|
|practice_0163|E2|52-6|276||263|válido|
|practice_0163|E2|64-1|301||276|válido|
|practice_0163|E2|64-3|281||264|válido|
|practice_0163|E2|64-6|276||263|válido|
|practice_0163|E3|joint|276|individual 276|276|FEASIBLE|
|practice_0163|E4|repair|276||276|completo|
|practice_0164|E1|1|305|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|practice_0164|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|practice_0164|E1|6|276|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|practice_0164|E2|52-1|305||280|válido|
|practice_0164|E2|52-3|288||263|válido|
|practice_0164|E2|52-6|276||256|válido|
|practice_0164|E2|64-1|305||277|válido|
|practice_0164|E2|64-3|288||270|válido|
|practice_0164|E2|64-6|276||255|válido|
|practice_0164|E3|joint|276|individual 276|276|OPTIMAL|
|practice_0164|E4|repair|276||275|completo|
|practice_0165|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|practice_0165|E1|3|260|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|practice_0165|E1|6|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|practice_0165|E2|52-1|273||248|válido|
|practice_0165|E2|52-3|260||239|válido|
|practice_0165|E2|52-6|256||244|válido|
|practice_0165|E2|64-1|273||251|válido|
|practice_0165|E2|64-3|260||243|válido|
|practice_0165|E2|64-6|256||241|válido|
|practice_0165|E3|joint|256|individual 256|256|OPTIMAL|
|practice_0165|E4|repair|256||256|completo|
|practice_0166|E1|1|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|practice_0166|E1|3|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|practice_0166|E1|6|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|practice_0166|E2|52-1|54||54|válido|
|practice_0166|E2|52-3|54||54|válido|
|practice_0166|E2|52-6|54||54|válido|
|practice_0166|E2|64-1|54||54|válido|
|practice_0166|E2|64-3|54||54|válido|
|practice_0166|E2|64-6|54||54|válido|
|practice_0166|E3|joint|54|individual 54|54|OPTIMAL|
|practice_0166|E4|repair|54||51|completo|
|practice_0167|E1|1|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|practice_0167|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|practice_0167|E1|6|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|practice_0167|E2|52-1|339||310|válido|
|practice_0167|E2|52-3|329||305|válido|
|practice_0167|E2|52-6|326||300|válido|
|practice_0167|E2|64-1|339||312|válido|
|practice_0167|E2|64-3|329||306|válido|
|practice_0167|E2|64-6|326||303|válido|
|practice_0167|E3|joint|326|individual 324|320|FEASIBLE|
|practice_0167|E4|repair|326||326|completo|
|practice_0168|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|practice_0168|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|practice_0168|E1|6|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|practice_0168|E2|52-1|133||131|válido|
|practice_0168|E2|52-3|133||131|válido|
|practice_0168|E2|52-6|132||130|válido|
|practice_0168|E2|64-1|133||132|válido|
|practice_0168|E2|64-3|133||132|válido|
|practice_0168|E2|64-6|132||130|válido|
|practice_0168|E3|joint|132|individual 132|132|OPTIMAL|
|practice_0168|E4|repair|132||131|completo|
|practice_0169|E1|1|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|practice_0169|E1|3|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|practice_0169|E1|6|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|practice_0169|E2|52-1|6||6|válido|
|practice_0169|E2|52-3|6||6|válido|
|practice_0169|E2|52-6|6||6|válido|
|practice_0169|E2|64-1|6||6|válido|
|practice_0169|E2|64-3|6||6|válido|
|practice_0169|E2|64-6|6||6|válido|
|practice_0169|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0169|E4|repair|5||5|completo|
|practice_0170|E1|1|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|practice_0170|E1|3|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|practice_0170|E1|6|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|practice_0170|E2|52-1|41||41|válido|
|practice_0170|E2|52-3|41||41|válido|
|practice_0170|E2|52-6|41||41|válido|
|practice_0170|E2|64-1|41||41|válido|
|practice_0170|E2|64-3|41||41|válido|
|practice_0170|E2|64-6|41||41|válido|
|practice_0170|E3|joint|38|individual 38|38|OPTIMAL|
|practice_0170|E4|repair|38||36|completo|
|practice_0171|E1|1|342|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|practice_0171|E1|3|327|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|practice_0171|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0171|E2|52-1|342||302|válido|
|practice_0171|E2|52-3|327||295|válido|
|practice_0171|E2|52-6|311||292|válido|
|practice_0171|E2|64-1|342||305|válido|
|practice_0171|E2|64-3|327||297|válido|
|practice_0171|E2|64-6|311||292|válido|
|practice_0171|E3|joint|311|individual 310|309|OPTIMAL|
|practice_0171|E4|repair|311||310|completo|
|practice_0172|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0172|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0172|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0172|E2|52-1|2||2|válido|
|practice_0172|E2|52-3|2||2|válido|
|practice_0172|E2|52-6|2||2|válido|
|practice_0172|E2|64-1|2||2|válido|
|practice_0172|E2|64-3|2||2|válido|
|practice_0172|E2|64-6|2||2|válido|
|practice_0172|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0172|E4|repair|2||2|completo|
|practice_0173|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|practice_0173|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|practice_0173|E1|6|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|practice_0173|E2|52-1|165||153|válido|
|practice_0173|E2|52-3|159||147|válido|
|practice_0173|E2|52-6|159||147|válido|
|practice_0173|E2|64-1|165||157|válido|
|practice_0173|E2|64-3|159||154|válido|
|practice_0173|E2|64-6|159||152|válido|
|practice_0173|E3|joint|159|individual 159|159|OPTIMAL|
|practice_0173|E4|repair|159||157|completo|
|practice_0174|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0174|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0174|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0174|E2|52-1|2||2|válido|
|practice_0174|E2|52-3|2||2|válido|
|practice_0174|E2|52-6|2||2|válido|
|practice_0174|E2|64-1|2||2|válido|
|practice_0174|E2|64-3|2||2|válido|
|practice_0174|E2|64-6|2||2|válido|
|practice_0174|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0174|E4|repair|2||2|completo|
|practice_0175|E1|1|6|postprocess 5, optimize_a4 5, nested_windows 5|5|válido|
|practice_0175|E1|3|6|postprocess 5, optimize_a4 5, nested_windows 5|5|válido|
|practice_0175|E1|6|6|postprocess 5, optimize_a4 5, nested_windows 5|5|válido|
|practice_0175|E2|52-1|6||6|válido|
|practice_0175|E2|52-3|6||6|válido|
|practice_0175|E2|52-6|6||6|válido|
|practice_0175|E2|64-1|6||6|válido|
|practice_0175|E2|64-3|6||6|válido|
|practice_0175|E2|64-6|6||6|válido|
|practice_0175|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0175|E4|repair|5||5|completo|
|practice_0176|E1|1|362|postprocess 362, optimize_a4 362, nested_windows 362|362|válido|
|practice_0176|E1|3|340|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|practice_0176|E1|6|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|practice_0176|E2|52-1|362||334|válido|
|practice_0176|E2|52-3|340||325|válido|
|practice_0176|E2|52-6|337||318|válido|
|practice_0176|E2|64-1|362||340|válido|
|practice_0176|E2|64-3|340||326|válido|
|practice_0176|E2|64-6|337||322|válido|
|practice_0176|E3|joint|337|individual 336|335|FEASIBLE|
|practice_0176|E4|repair|337||337|completo|
|practice_0177|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|practice_0177|E1|3|197|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0177|E1|6|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0177|E2|52-1|200||189|válido|
|practice_0177|E2|52-3|197||179|válido|
|practice_0177|E2|52-6|193||179|válido|
|practice_0177|E2|64-1|200||188|válido|
|practice_0177|E2|64-3|197||179|válido|
|practice_0177|E2|64-6|193||179|válido|
|practice_0177|E3|joint|193|individual 193|193|OPTIMAL|
|practice_0177|E4|repair|193||192|completo|
|practice_0178|E1|1|97|postprocess 95, optimize_a4 95, nested_windows 95|95|válido|
|practice_0178|E1|3|97|postprocess 95, optimize_a4 95, nested_windows 95|95|válido|
|practice_0178|E1|6|97|postprocess 95, optimize_a4 95, nested_windows 95|95|válido|
|practice_0178|E2|52-1|97||93|válido|
|practice_0178|E2|52-3|97||93|válido|
|practice_0178|E2|52-6|97||93|válido|
|practice_0178|E2|64-1|97||93|válido|
|practice_0178|E2|64-3|97||93|válido|
|practice_0178|E2|64-6|97||93|válido|
|practice_0178|E3|joint|97|individual 97|97|OPTIMAL|
|practice_0178|E4|repair|97||95|completo|
|practice_0179|E1|1|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|practice_0179|E1|3|378|postprocess 378, optimize_a4 378, nested_windows 378|378|válido|
|practice_0179|E1|6|371|postprocess 371, optimize_a4 371, nested_windows 371|371|válido|
|practice_0179|E2|52-1|385||363|válido|
|practice_0179|E2|52-3|378||350|válido|
|practice_0179|E2|52-6|371||342|válido|
|practice_0179|E2|64-1|385||367|válido|
|practice_0179|E2|64-3|378||514|válido|
|practice_0179|E2|64-6|371||430|válido|
|practice_0179|E3|joint|371|individual 370|368|OPTIMAL|
|practice_0179|E4|repair|371||371|completo|
|practice_0180|E1|1|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0180|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|practice_0180|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|practice_0180|E2|52-1|166||156|válido|
|practice_0180|E2|52-3|160||155|válido|
|practice_0180|E2|52-6|160||155|válido|
|practice_0180|E2|64-1|166||157|válido|
|practice_0180|E2|64-3|160||157|válido|
|practice_0180|E2|64-6|160||157|válido|
|practice_0180|E3|joint|160|individual 160|160|OPTIMAL|
|practice_0180|E4|repair|160||159|completo|
|practice_0181|E1|1|329|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|practice_0181|E1|3|311|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|practice_0181|E1|6|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|practice_0181|E2|52-1|329||297|válido|
|practice_0181|E2|52-3|311||286|válido|
|practice_0181|E2|52-6|300||281|válido|
|practice_0181|E2|64-1|329||297|válido|
|practice_0181|E2|64-3|311||285|válido|
|practice_0181|E2|64-6|300||285|válido|
|practice_0181|E3|joint|300|individual 300|300|FEASIBLE|
|practice_0181|E4|repair|300||300|completo|
|practice_0182|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|practice_0182|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|practice_0182|E1|6|230|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|practice_0182|E2|52-1|253||226|válido|
|practice_0182|E2|52-3|242||221|válido|
|practice_0182|E2|52-6|230||220|válido|
|practice_0182|E2|64-1|253||225|válido|
|practice_0182|E2|64-3|242||221|válido|
|practice_0182|E2|64-6|230||220|válido|
|practice_0182|E3|joint|230|individual 230|230|OPTIMAL|
|practice_0182|E4|repair|230||230|completo|
|practice_0183|E1|1|332|postprocess 332, optimize_a4 332, nested_windows 332|332|válido|
|practice_0183|E1|3|321|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|practice_0183|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0183|E2|52-1|332||287|válido|
|practice_0183|E2|52-3|321||286|válido|
|practice_0183|E2|52-6|311||287|válido|
|practice_0183|E2|64-1|332||294|válido|
|practice_0183|E2|64-3|321||290|válido|
|practice_0183|E2|64-6|311||292|válido|
|practice_0183|E3|joint|311|individual 311|311|rechazado por registros|
|practice_0183|E4|repair|311||311|completo|
|practice_0184|E1|1|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0184|E1|3|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|practice_0184|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|practice_0184|E2|52-1|283||260|válido|
|practice_0184|E2|52-3|275||255|válido|
|practice_0184|E2|52-6|269||256|válido|
|practice_0184|E2|64-1|283||261|válido|
|practice_0184|E2|64-3|275||255|válido|
|practice_0184|E2|64-6|269||254|válido|
|practice_0184|E3|joint|269|individual 269|267|OPTIMAL|
|practice_0184|E4|repair|269||269|completo|
|practice_0185|E1|1|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|practice_0185|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|practice_0185|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|practice_0185|E2|52-1|191||181|válido|
|practice_0185|E2|52-3|191||180|válido|
|practice_0185|E2|52-6|189||177|válido|
|practice_0185|E2|64-1|191||183|válido|
|practice_0185|E2|64-3|191||181|válido|
|practice_0185|E2|64-6|189||172|válido|
|practice_0185|E3|joint|189|individual 189|188|OPTIMAL|
|practice_0185|E4|repair|189||186|completo|
|practice_0186|E1|1|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|practice_0186|E1|3|128|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|practice_0186|E1|6|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|practice_0186|E2|52-1|126||121|válido|
|practice_0186|E2|52-3|128||120|válido|
|practice_0186|E2|52-6|125||118|válido|
|practice_0186|E2|64-1|126||121|válido|
|practice_0186|E2|64-3|128||120|válido|
|practice_0186|E2|64-6|125||118|válido|
|practice_0186|E3|joint|125|individual 125|125|OPTIMAL|
|practice_0186|E4|repair|125||125|completo|
|practice_0187|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|practice_0187|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|practice_0187|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|practice_0187|E2|52-1|240||225|válido|
|practice_0187|E2|52-3|232||218|válido|
|practice_0187|E2|52-6|232||218|válido|
|practice_0187|E2|64-1|240||223|válido|
|practice_0187|E2|64-3|232||219|válido|
|practice_0187|E2|64-6|232||218|válido|
|practice_0187|E3|joint|232|individual 231|230|OPTIMAL|
|practice_0187|E4|repair|232||232|completo|
|practice_0188|E1|1|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0188|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|practice_0188|E1|6|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|practice_0188|E2|52-1|246||235|válido|
|practice_0188|E2|52-3|241||226|válido|
|practice_0188|E2|52-6|240||224|válido|
|practice_0188|E2|64-1|246||231|válido|
|practice_0188|E2|64-3|241||226|válido|
|practice_0188|E2|64-6|240||227|válido|
|practice_0188|E3|joint|240|individual 240|239|OPTIMAL|
|practice_0188|E4|repair|240||240|completo|
|practice_0189|E1|1|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0189|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0189|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|practice_0189|E2|52-1|212||189|válido|
|practice_0189|E2|52-3|206||188|válido|
|practice_0189|E2|52-6|203||187|válido|
|practice_0189|E2|64-1|212||187|válido|
|practice_0189|E2|64-3|206||184|válido|
|practice_0189|E2|64-6|203||188|válido|
|practice_0189|E3|joint|203|individual 203|202|OPTIMAL|
|practice_0189|E4|repair|203||203|completo|
|practice_0190|E1|1|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|practice_0190|E1|3|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|practice_0190|E1|6|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|practice_0190|E2|52-1|343||312|válido|
|practice_0190|E2|52-3|326||297|válido|
|practice_0190|E2|52-6|316||295|válido|
|practice_0190|E2|64-1|343||312|válido|
|practice_0190|E2|64-3|326||300|válido|
|practice_0190|E2|64-6|316||298|válido|
|practice_0190|E3|joint|316|individual 316|316|OPTIMAL|
|practice_0190|E4|repair|316||316|completo|
|practice_0191|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|practice_0191|E1|3|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|practice_0191|E1|6|337|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|practice_0191|E2|52-1|370||332|válido|
|practice_0191|E2|52-3|348||315|válido|
|practice_0191|E2|52-6|337||313|válido|
|practice_0191|E2|64-1|370||332|válido|
|practice_0191|E2|64-3|348||318|válido|
|practice_0191|E2|64-6|337||313|válido|
|practice_0191|E3|joint|337|individual 337|337|OPTIMAL|
|practice_0191|E4|repair|337||337|completo|
|practice_0192|E1|1|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0192|E1|3|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|practice_0192|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|practice_0192|E2|52-1|129||126|válido|
|practice_0192|E2|52-3|127||125|válido|
|practice_0192|E2|52-6|127||125|válido|
|practice_0192|E2|64-1|129||126|válido|
|practice_0192|E2|64-3|127||126|válido|
|practice_0192|E2|64-6|127||125|válido|
|practice_0192|E3|joint|127|individual 127|127|OPTIMAL|
|practice_0192|E4|repair|127||124|completo|
|practice_0193|E1|1|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0193|E1|3|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|practice_0193|E1|6|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0193|E2|52-1|138||136|válido|
|practice_0193|E2|52-3|139||141|válido|
|practice_0193|E2|52-6|138||141|válido|
|practice_0193|E2|64-1|138||135|válido|
|practice_0193|E2|64-3|139||139|válido|
|practice_0193|E2|64-6|138||138|válido|
|practice_0193|E3|joint|138|individual 138|137|OPTIMAL|
|practice_0193|E4|repair|138||136|completo|
|practice_0194|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0194|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|practice_0194|E1|6|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|practice_0194|E2|52-1|289||259|válido|
|practice_0194|E2|52-3|278||255|válido|
|practice_0194|E2|52-6|274||249|válido|
|practice_0194|E2|64-1|289||263|válido|
|practice_0194|E2|64-3|278||256|válido|
|practice_0194|E2|64-6|274||251|válido|
|practice_0194|E3|joint|274|individual 274|271|FEASIBLE|
|practice_0194|E4|repair|274||274|completo|
|practice_0195|E1|1|296|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|practice_0195|E1|3|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|practice_0195|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0195|E2|52-1|296||280|válido|
|practice_0195|E2|52-3|291||269|válido|
|practice_0195|E2|52-6|289||262|válido|
|practice_0195|E2|64-1|296||284|válido|
|practice_0195|E2|64-3|291||270|válido|
|practice_0195|E2|64-6|289||263|válido|
|practice_0195|E3|joint|289|individual 289|286|FEASIBLE|
|practice_0195|E4|repair|289||289|completo|
|practice_0196|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|practice_0196|E1|3|145|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0196|E1|6|143|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0196|E2|52-1|144||140|válido|
|practice_0196|E2|52-3|145||140|válido|
|practice_0196|E2|52-6|143||137|válido|
|practice_0196|E2|64-1|144||140|válido|
|practice_0196|E2|64-3|145||140|válido|
|practice_0196|E2|64-6|143||137|válido|
|practice_0196|E3|joint|143|individual 143|142|OPTIMAL|
|practice_0196|E4|repair|143||137|completo|
|practice_0197|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|practice_0197|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|practice_0197|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|practice_0197|E2|52-1|235||218|válido|
|practice_0197|E2|52-3|223||213|válido|
|practice_0197|E2|52-6|224||209|válido|
|practice_0197|E2|64-1|235||218|válido|
|practice_0197|E2|64-3|223||211|válido|
|practice_0197|E2|64-6|224||207|válido|
|practice_0197|E3|joint|223|individual 223|223|OPTIMAL|
|practice_0197|E4|repair|223||223|completo|
|practice_0198|E1|1|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|practice_0198|E1|3|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|practice_0198|E1|6|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|practice_0198|E2|52-1|47||47|válido|
|practice_0198|E2|52-3|47||47|válido|
|practice_0198|E2|52-6|47||47|válido|
|practice_0198|E2|64-1|47||47|válido|
|practice_0198|E2|64-3|47||47|válido|
|practice_0198|E2|64-6|47||47|válido|
|practice_0198|E3|joint|47|individual 47|47|OPTIMAL|
|practice_0198|E4|repair|47||44|completo|
|practice_0199|E1|1|338|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|practice_0199|E1|3|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|practice_0199|E1|6|317|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|practice_0199|E2|52-1|338||310|válido|
|practice_0199|E2|52-3|316||298|válido|
|practice_0199|E2|52-6|317||291|válido|
|practice_0199|E2|64-1|338||314|válido|
|practice_0199|E2|64-3|316||297|válido|
|practice_0199|E2|64-6|317||292|válido|
|practice_0199|E3|joint|316|individual 316|313|OPTIMAL|
|practice_0199|E4|repair|316||315|completo|
|practice_0200|E1|1|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|practice_0200|E1|3|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0200|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0200|E2|52-1|155||144|válido|
|practice_0200|E2|52-3|153||145|válido|
|practice_0200|E2|52-6|152||145|válido|
|practice_0200|E2|64-1|155||141|válido|
|practice_0200|E2|64-3|153||140|válido|
|practice_0200|E2|64-6|152||139|válido|
|practice_0200|E3|joint|152|individual 152|152|OPTIMAL|
|practice_0200|E4|repair|152||152|completo|

## Cuatro experimentos sobre los 1.000 puntuables

Cierre validado: 177352 pasos, 11565 ahorrados. [Programas](reports/four_backward_scored/portfolio.csv) · [Ganadores](reports/four_backward_scored/comparison.csv)

Piloto completado: 22 configuraciones en 2 IDs (puntuables sin etiqueta de familia). Suite del entorno: 42 pruebas aprobadas.

| Diccionario y ancho | Control original | Ampliado | Diferencia a favor del ampliado |
|---|---:|---:|---:|
|52-1|197890|184113|13777|
|52-3|191867|180225|11642|
|52-6|189141|178565|10576|
|64-1|197890|184973|12917|
|64-3|191867|181202|10665|
|64-6|189141|179345|9796|

E1 elegir antes: 188505 · E1 elegir después: 188484 · E3 individual: 188677 · E3 con CP-SAT: 188118

| Método | IDs elegidos | Aporte exclusivo |
|---|---:|---:|
|E1|80|125|
|E2|828|9782|
|E3|3|4|
|E4|62|117|

| Familia | E1 | E2 | E3 | E4 | Cartera |
|---|---:|---:|---:|---:|---:|
|Sin etiqueta|188484|177730|188118|187980|177352|

Programas reproducidos por método: [E1](reports/four_backward_scored/E1.csv) · [E2](reports/four_backward_scored/E2.csv) · [E3](reports/four_backward_scored/E3.csv) · [E4](reports/four_backward_scored/E4.csv)

Estados de CP-SAT: FEASIBLE 82, OPTIMAL 910, rechazado por registros 8

| Método | IDs que mejoran el histórico | Pasos ahorrados en esos IDs |
|---|---:|---:|
|E1|304|450|
|E2|861|11256|
|E3|414|799|
|E4|402|937|

Referencia validada: 188.917 pasos en 1000 IDs. Haz 1/3/6. E2: 52/72 y 64/72. CP-SAT 9.15.6755, semilla 42, un hilo y 30 s. Reparación local: 30 s. Grupo compartido: 14 procesos.

| Método | IDs completos | Histórico | Método | Ahorro al combinar | CPU s | Tiempo acumulado s |
|---|---:|---:|---:|---:|---:|---:|
|E1|1000/1000|188917|188484|450|458.2|467.4|
|E2|1000/1000|188917|177730|11256|15886.2|16127.5|
|E3|1000/1000|188917|188118|799|3918.3|3924.4|
|E4|1000/1000|188917|187980|937|3016.4|3028.5|

Comparación directa sobre 1000 IDs comunes. E1: 188484 · E2: 177730 · E3: 188118 · E4: 187980

Detalle de resultados recibidos. Las configuraciones restantes están pendientes.

| ID | Método | Configuración | Entrada | Etapas (pasos) | Final validado | Estado |
|---|---|---|---:|---|---:|---|
|scored_0001|E1|1|318|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0001|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0001|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0001|E2|52-1|318||298|válido|
|scored_0001|E2|52-3|310||297|válido|
|scored_0001|E2|52-6|304||289|válido|
|scored_0001|E2|64-1|318||301|válido|
|scored_0001|E2|64-3|310||297|válido|
|scored_0001|E2|64-6|304||291|válido|
|scored_0001|E3|joint|304|individual 304|302|FEASIBLE|
|scored_0001|E4|repair|304||304|completo|
|scored_0002|E1|1|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0002|E1|3|385|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0002|E1|6|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0002|E2|52-1|348||316|válido|
|scored_0002|E2|52-3|385||311|válido|
|scored_0002|E2|52-6|322||304|válido|
|scored_0002|E2|64-1|348||321|válido|
|scored_0002|E2|64-3|385||313|válido|
|scored_0002|E2|64-6|322||305|válido|
|scored_0002|E3|joint|322|individual 322|321|FEASIBLE|
|scored_0002|E4|repair|322||322|completo|
|scored_0003|E1|1|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0003|E1|3|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0003|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0003|E2|52-1|310||281|válido|
|scored_0003|E2|52-3|296||271|válido|
|scored_0003|E2|52-6|284||266|válido|
|scored_0003|E2|64-1|310||279|válido|
|scored_0003|E2|64-3|296||270|válido|
|scored_0003|E2|64-6|284||265|válido|
|scored_0003|E3|joint|284|individual 284|284|OPTIMAL|
|scored_0003|E4|repair|284||284|completo|
|scored_0004|E1|1|300|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0004|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0004|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0004|E2|52-1|300||276|válido|
|scored_0004|E2|52-3|281||275|válido|
|scored_0004|E2|52-6|282||269|válido|
|scored_0004|E2|64-1|300||276|válido|
|scored_0004|E2|64-3|281||276|válido|
|scored_0004|E2|64-6|282||268|válido|
|scored_0004|E3|joint|281|individual 280|280|OPTIMAL|
|scored_0004|E4|repair|281||281|completo|
|scored_0005|E1|1|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0005|E1|3|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0005|E1|6|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0005|E2|52-1|71||70|válido|
|scored_0005|E2|52-3|71||70|válido|
|scored_0005|E2|52-6|71||70|válido|
|scored_0005|E2|64-1|71||70|válido|
|scored_0005|E2|64-3|71||70|válido|
|scored_0005|E2|64-6|71||70|válido|
|scored_0005|E3|joint|71|individual 71|71|OPTIMAL|
|scored_0005|E4|repair|71||69|completo|
|scored_0006|E1|1|189|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0006|E1|3|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0006|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0006|E2|52-1|189||176|válido|
|scored_0006|E2|52-3|184||174|válido|
|scored_0006|E2|52-6|183||175|válido|
|scored_0006|E2|64-1|189||177|válido|
|scored_0006|E2|64-3|184||178|válido|
|scored_0006|E2|64-6|183||174|válido|
|scored_0006|E3|joint|183|individual 183|182|OPTIMAL|
|scored_0006|E4|repair|183||179|completo|
|scored_0007|E1|1|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0007|E1|3|244|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0007|E1|6|241|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0007|E2|52-1|263||247|válido|
|scored_0007|E2|52-3|244||225|válido|
|scored_0007|E2|52-6|241||222|válido|
|scored_0007|E2|64-1|263||252|válido|
|scored_0007|E2|64-3|244||225|válido|
|scored_0007|E2|64-6|241||224|válido|
|scored_0007|E3|joint|241|individual 241|238|OPTIMAL|
|scored_0007|E4|repair|241||241|completo|
|scored_0008|E1|1|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0008|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0008|E1|6|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0008|E2|52-1|173||163|válido|
|scored_0008|E2|52-3|171||163|válido|
|scored_0008|E2|52-6|171||163|válido|
|scored_0008|E2|64-1|173||163|válido|
|scored_0008|E2|64-3|171||163|válido|
|scored_0008|E2|64-6|171||163|válido|
|scored_0008|E3|joint|171|individual 171|170|OPTIMAL|
|scored_0008|E4|repair|171||171|completo|
|scored_0009|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0009|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0009|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0009|E2|52-1|50||50|válido|
|scored_0009|E2|52-3|50||50|válido|
|scored_0009|E2|52-6|50||50|válido|
|scored_0009|E2|64-1|50||50|válido|
|scored_0009|E2|64-3|50||50|válido|
|scored_0009|E2|64-6|50||50|válido|
|scored_0009|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0009|E4|repair|50||50|completo|
|scored_0010|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0010|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0010|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0010|E2|52-1|250||234|válido|
|scored_0010|E2|52-3|241||225|válido|
|scored_0010|E2|52-6|241||224|válido|
|scored_0010|E2|64-1|250||236|válido|
|scored_0010|E2|64-3|241||226|válido|
|scored_0010|E2|64-6|241||224|válido|
|scored_0010|E3|joint|241|individual 241|240|OPTIMAL|
|scored_0010|E4|repair|241||241|completo|
|scored_0011|E1|1|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0011|E1|3|200|postprocess 197, optimize_a4 196, nested_windows 196|196|válido|
|scored_0011|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0011|E2|52-1|211||194|válido|
|scored_0011|E2|52-3|200||189|válido|
|scored_0011|E2|52-6|190||181|válido|
|scored_0011|E2|64-1|211||189|válido|
|scored_0011|E2|64-3|200||184|válido|
|scored_0011|E2|64-6|190||175|válido|
|scored_0011|E3|joint|190|individual 189|189|OPTIMAL|
|scored_0011|E4|repair|190||189|completo|
|scored_0012|E1|1|254|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0012|E1|3|246|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0012|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0012|E2|52-1|254||232|válido|
|scored_0012|E2|52-3|246||230|válido|
|scored_0012|E2|52-6|245||226|válido|
|scored_0012|E2|64-1|254||231|válido|
|scored_0012|E2|64-3|246||232|válido|
|scored_0012|E2|64-6|245||226|válido|
|scored_0012|E3|joint|245|individual 245|244|OPTIMAL|
|scored_0012|E4|repair|245||245|completo|
|scored_0013|E1|1|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0013|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0013|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0013|E2|52-1|239||217|válido|
|scored_0013|E2|52-3|239||219|válido|
|scored_0013|E2|52-6|237||222|válido|
|scored_0013|E2|64-1|239||220|válido|
|scored_0013|E2|64-3|239||219|válido|
|scored_0013|E2|64-6|237||221|válido|
|scored_0013|E3|joint|237|individual 236|235|OPTIMAL|
|scored_0013|E4|repair|237||237|completo|
|scored_0014|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0014|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0014|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0014|E2|52-1|41||41|válido|
|scored_0014|E2|52-3|41||41|válido|
|scored_0014|E2|52-6|41||41|válido|
|scored_0014|E2|64-1|41||41|válido|
|scored_0014|E2|64-3|41||41|válido|
|scored_0014|E2|64-6|41||41|válido|
|scored_0014|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0014|E4|repair|41||41|completo|
|scored_0015|E1|1|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0015|E1|3|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0015|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0015|E2|52-1|317||291|válido|
|scored_0015|E2|52-3|295||276|válido|
|scored_0015|E2|52-6|298||276|válido|
|scored_0015|E2|64-1|317||293|válido|
|scored_0015|E2|64-3|295||277|válido|
|scored_0015|E2|64-6|298||276|válido|
|scored_0015|E3|joint|295|individual 294|291|FEASIBLE|
|scored_0015|E4|repair|295||295|completo|
|scored_0016|E1|1|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0016|E1|3|231|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0016|E1|6|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0016|E2|52-1|242||222|válido|
|scored_0016|E2|52-3|231||207|válido|
|scored_0016|E2|52-6|225||203|válido|
|scored_0016|E2|64-1|242||222|válido|
|scored_0016|E2|64-3|231||210|válido|
|scored_0016|E2|64-6|225||205|válido|
|scored_0016|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0016|E4|repair|225||225|completo|
|scored_0017|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0017|E1|3|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0017|E1|6|172|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0017|E2|52-1|176||168|válido|
|scored_0017|E2|52-3|176||165|válido|
|scored_0017|E2|52-6|172||161|válido|
|scored_0017|E2|64-1|176||166|válido|
|scored_0017|E2|64-3|176||165|válido|
|scored_0017|E2|64-6|172||161|válido|
|scored_0017|E3|joint|172|individual 171|171|OPTIMAL|
|scored_0017|E4|repair|172||172|completo|
|scored_0018|E1|1|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0018|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0018|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0018|E2|52-1|175||170|válido|
|scored_0018|E2|52-3|173||165|válido|
|scored_0018|E2|52-6|172||169|válido|
|scored_0018|E2|64-1|175||170|válido|
|scored_0018|E2|64-3|173||167|válido|
|scored_0018|E2|64-6|172||169|válido|
|scored_0018|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0018|E4|repair|172||172|completo|
|scored_0019|E1|1|36|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0019|E1|3|36|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0019|E1|6|36|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0019|E2|52-1|36||36|válido|
|scored_0019|E2|52-3|36||36|válido|
|scored_0019|E2|52-6|36||36|válido|
|scored_0019|E2|64-1|36||36|válido|
|scored_0019|E2|64-3|36||36|válido|
|scored_0019|E2|64-6|36||36|válido|
|scored_0019|E3|joint|36|individual 36|36|OPTIMAL|
|scored_0019|E4|repair|36||36|completo|
|scored_0020|E1|1|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0020|E1|3|330|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0020|E1|6|324|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0020|E2|52-1|358||315|válido|
|scored_0020|E2|52-3|330||311|válido|
|scored_0020|E2|52-6|324||303|válido|
|scored_0020|E2|64-1|358||319|válido|
|scored_0020|E2|64-3|330||309|válido|
|scored_0020|E2|64-6|324||301|válido|
|scored_0020|E3|joint|324|individual 324|324|OPTIMAL|
|scored_0020|E4|repair|324||324|completo|
|scored_0021|E1|1|55|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0021|E1|3|55|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0021|E1|6|55|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0021|E2|52-1|55||55|válido|
|scored_0021|E2|52-3|55||55|válido|
|scored_0021|E2|52-6|55||55|válido|
|scored_0021|E2|64-1|55||55|válido|
|scored_0021|E2|64-3|55||55|válido|
|scored_0021|E2|64-6|55||55|válido|
|scored_0021|E3|joint|55|individual 55|55|OPTIMAL|
|scored_0021|E4|repair|55||54|completo|
|scored_0022|E1|1|170|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0022|E1|3|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0022|E1|6|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0022|E2|52-1|170||156|válido|
|scored_0022|E2|52-3|169||156|válido|
|scored_0022|E2|52-6|168||156|válido|
|scored_0022|E2|64-1|170||155|válido|
|scored_0022|E2|64-3|169||155|válido|
|scored_0022|E2|64-6|168||155|válido|
|scored_0022|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0022|E4|repair|168||166|completo|
|scored_0023|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0023|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0023|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0023|E2|52-1|220||207|válido|
|scored_0023|E2|52-3|215||204|válido|
|scored_0023|E2|52-6|214||201|válido|
|scored_0023|E2|64-1|220||209|válido|
|scored_0023|E2|64-3|215||205|válido|
|scored_0023|E2|64-6|214||201|válido|
|scored_0023|E3|joint|214|individual 214|212|OPTIMAL|
|scored_0023|E4|repair|214||214|completo|
|scored_0024|E1|1|43|postprocess 43, optimize_a4 42, nested_windows 42|42|válido|
|scored_0024|E1|3|43|postprocess 43, optimize_a4 42, nested_windows 42|42|válido|
|scored_0024|E1|6|43|postprocess 43, optimize_a4 42, nested_windows 42|42|válido|
|scored_0024|E2|52-1|43||43|válido|
|scored_0024|E2|52-3|43||43|válido|
|scored_0024|E2|52-6|43||43|válido|
|scored_0024|E2|64-1|43||43|válido|
|scored_0024|E2|64-3|43||43|válido|
|scored_0024|E2|64-6|43||43|válido|
|scored_0024|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0024|E4|repair|43||43|completo|
|scored_0025|E1|1|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0025|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0025|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0025|E2|52-1|194||181|válido|
|scored_0025|E2|52-3|191||177|válido|
|scored_0025|E2|52-6|187||177|válido|
|scored_0025|E2|64-1|194||181|válido|
|scored_0025|E2|64-3|191||177|válido|
|scored_0025|E2|64-6|187||177|válido|
|scored_0025|E3|joint|187|individual 187|186|OPTIMAL|
|scored_0025|E4|repair|187||185|completo|
|scored_0026|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0026|E1|3|242|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0026|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0026|E2|52-1|250||234|válido|
|scored_0026|E2|52-3|242||232|válido|
|scored_0026|E2|52-6|242||227|válido|
|scored_0026|E2|64-1|250||237|válido|
|scored_0026|E2|64-3|242||236|válido|
|scored_0026|E2|64-6|242||231|válido|
|scored_0026|E3|joint|242|individual 241|239|OPTIMAL|
|scored_0026|E4|repair|242||242|completo|
|scored_0027|E1|1|149|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0027|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0027|E1|6|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0027|E2|52-1|149||145|válido|
|scored_0027|E2|52-3|148||140|válido|
|scored_0027|E2|52-6|146||139|válido|
|scored_0027|E2|64-1|149||143|válido|
|scored_0027|E2|64-3|148||140|válido|
|scored_0027|E2|64-6|146||138|válido|
|scored_0027|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0027|E4|repair|146||140|completo|
|scored_0028|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0028|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0028|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0028|E2|52-1|211||190|válido|
|scored_0028|E2|52-3|202||189|válido|
|scored_0028|E2|52-6|202||188|válido|
|scored_0028|E2|64-1|211||195|válido|
|scored_0028|E2|64-3|202||190|válido|
|scored_0028|E2|64-6|202||188|válido|
|scored_0028|E3|joint|202|individual 202|201|OPTIMAL|
|scored_0028|E4|repair|202||200|completo|
|scored_0029|E1|1|54|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0029|E1|3|54|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0029|E1|6|54|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0029|E2|52-1|54||54|válido|
|scored_0029|E2|52-3|54||54|válido|
|scored_0029|E2|52-6|54||54|válido|
|scored_0029|E2|64-1|54||54|válido|
|scored_0029|E2|64-3|54||54|válido|
|scored_0029|E2|64-6|54||54|válido|
|scored_0029|E3|joint|54|individual 54|54|OPTIMAL|
|scored_0029|E4|repair|54||53|completo|
|scored_0030|E1|1|279|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0030|E1|3|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0030|E1|6|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0030|E2|52-1|279||264|válido|
|scored_0030|E2|52-3|274||257|válido|
|scored_0030|E2|52-6|265||250|válido|
|scored_0030|E2|64-1|279||265|válido|
|scored_0030|E2|64-3|274||259|válido|
|scored_0030|E2|64-6|265||253|válido|
|scored_0030|E3|joint|265|individual 265|265|OPTIMAL|
|scored_0030|E4|repair|265||265|completo|
|scored_0031|E1|1|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0031|E1|3|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0031|E1|6|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0031|E2|52-1|347||318|válido|
|scored_0031|E2|52-3|327||308|válido|
|scored_0031|E2|52-6|321||301|válido|
|scored_0031|E2|64-1|347||323|válido|
|scored_0031|E2|64-3|327||310|válido|
|scored_0031|E2|64-6|321||302|válido|
|scored_0031|E3|joint|321|individual 320|320|FEASIBLE|
|scored_0031|E4|repair|321||321|completo|
|scored_0032|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0032|E1|3|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0032|E1|6|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0032|E2|52-1|145||138|válido|
|scored_0032|E2|52-3|142||138|válido|
|scored_0032|E2|52-6|142||136|válido|
|scored_0032|E2|64-1|145||138|válido|
|scored_0032|E2|64-3|142||138|válido|
|scored_0032|E2|64-6|142||136|válido|
|scored_0032|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0032|E4|repair|142||142|completo|
|scored_0033|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0033|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0033|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0033|E2|52-1|306||277|válido|
|scored_0033|E2|52-3|288||268|válido|
|scored_0033|E2|52-6|284||263|válido|
|scored_0033|E2|64-1|306||275|válido|
|scored_0033|E2|64-3|288||269|válido|
|scored_0033|E2|64-6|284||260|válido|
|scored_0033|E3|joint|284|individual 284|283|FEASIBLE|
|scored_0033|E4|repair|284||284|completo|
|scored_0034|E1|1|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0034|E1|3|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0034|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0034|E2|52-1|231||219|válido|
|scored_0034|E2|52-3|221||217|válido|
|scored_0034|E2|52-6|220||214|válido|
|scored_0034|E2|64-1|231||221|válido|
|scored_0034|E2|64-3|221||218|válido|
|scored_0034|E2|64-6|220||216|válido|
|scored_0034|E3|joint|220|individual 220|220|OPTIMAL|
|scored_0034|E4|repair|220||220|completo|
|scored_0035|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0035|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0035|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0035|E2|52-1|150||139|válido|
|scored_0035|E2|52-3|148||142|válido|
|scored_0035|E2|52-6|148||142|válido|
|scored_0035|E2|64-1|150||138|válido|
|scored_0035|E2|64-3|148||140|válido|
|scored_0035|E2|64-6|148||140|válido|
|scored_0035|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0035|E4|repair|148||148|completo|
|scored_0036|E1|1|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|scored_0036|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0036|E1|6|313|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0036|E2|52-1|343||310|válido|
|scored_0036|E2|52-3|329||298|válido|
|scored_0036|E2|52-6|313||295|válido|
|scored_0036|E2|64-1|343||308|válido|
|scored_0036|E2|64-3|329||300|válido|
|scored_0036|E2|64-6|313||295|válido|
|scored_0036|E3|joint|313|individual 313|313|FEASIBLE|
|scored_0036|E4|repair|313||313|completo|
|scored_0037|E1|1|278|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0037|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0037|E1|6|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0037|E2|52-1|278||259|válido|
|scored_0037|E2|52-3|265||253|válido|
|scored_0037|E2|52-6|262||251|válido|
|scored_0037|E2|64-1|278||260|válido|
|scored_0037|E2|64-3|265||255|válido|
|scored_0037|E2|64-6|262||252|válido|
|scored_0037|E3|joint|262|individual 262|260|OPTIMAL|
|scored_0037|E4|repair|262||262|completo|
|scored_0038|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0038|E1|3|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0038|E1|6|193|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0038|E2|52-1|199||187|válido|
|scored_0038|E2|52-3|193||186|válido|
|scored_0038|E2|52-6|193||179|válido|
|scored_0038|E2|64-1|199||187|válido|
|scored_0038|E2|64-3|193||186|válido|
|scored_0038|E2|64-6|193||179|válido|
|scored_0038|E3|joint|193|individual 192|192|OPTIMAL|
|scored_0038|E4|repair|193||192|completo|
|scored_0039|E1|1|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0039|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0039|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0039|E2|52-1|143||140|válido|
|scored_0039|E2|52-3|143||140|válido|
|scored_0039|E2|52-6|143||140|válido|
|scored_0039|E2|64-1|143||138|válido|
|scored_0039|E2|64-3|143||138|válido|
|scored_0039|E2|64-6|143||138|válido|
|scored_0039|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0039|E4|repair|143||142|completo|
|scored_0040|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0040|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0040|E1|6|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0040|E2|52-1|240||223|válido|
|scored_0040|E2|52-3|239||227|válido|
|scored_0040|E2|52-6|235||217|válido|
|scored_0040|E2|64-1|240||232|válido|
|scored_0040|E2|64-3|239||226|válido|
|scored_0040|E2|64-6|235||222|válido|
|scored_0040|E3|joint|235|individual 234|234|OPTIMAL|
|scored_0040|E4|repair|235||235|completo|
|scored_0041|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0041|E1|3|197|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0041|E1|6|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0041|E2|52-1|200||188|válido|
|scored_0041|E2|52-3|197||182|válido|
|scored_0041|E2|52-6|194||183|válido|
|scored_0041|E2|64-1|200||187|válido|
|scored_0041|E2|64-3|197||182|válido|
|scored_0041|E2|64-6|194||183|válido|
|scored_0041|E3|joint|194|individual 194|191|OPTIMAL|
|scored_0041|E4|repair|194||194|completo|
|scored_0042|E1|1|229|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0042|E1|3|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0042|E1|6|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0042|E2|52-1|229||205|válido|
|scored_0042|E2|52-3|217||203|válido|
|scored_0042|E2|52-6|213||202|válido|
|scored_0042|E2|64-1|229||205|válido|
|scored_0042|E2|64-3|217||203|válido|
|scored_0042|E2|64-6|213||204|válido|
|scored_0042|E3|joint|213|individual 213|213|OPTIMAL|
|scored_0042|E4|repair|213||213|completo|
|scored_0043|E1|1|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0043|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0043|E1|6|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0043|E2|52-1|297||280|válido|
|scored_0043|E2|52-3|288||270|válido|
|scored_0043|E2|52-6|285||268|válido|
|scored_0043|E2|64-1|297||279|válido|
|scored_0043|E2|64-3|288||270|válido|
|scored_0043|E2|64-6|285||267|válido|
|scored_0043|E3|joint|285|individual 285|284|OPTIMAL|
|scored_0043|E4|repair|285||285|completo|
|scored_0044|E1|1|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0044|E1|3|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0044|E1|6|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0044|E2|52-1|108||105|válido|
|scored_0044|E2|52-3|108||105|válido|
|scored_0044|E2|52-6|108||105|válido|
|scored_0044|E2|64-1|108||107|válido|
|scored_0044|E2|64-3|108||107|válido|
|scored_0044|E2|64-6|108||107|válido|
|scored_0044|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0044|E4|repair|108||102|completo|
|scored_0045|E1|1|222|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0045|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0045|E1|6|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0045|E2|52-1|222||206|válido|
|scored_0045|E2|52-3|215||203|válido|
|scored_0045|E2|52-6|214||201|válido|
|scored_0045|E2|64-1|222||207|válido|
|scored_0045|E2|64-3|215||204|válido|
|scored_0045|E2|64-6|214||205|válido|
|scored_0045|E3|joint|214|individual 213|212|OPTIMAL|
|scored_0045|E4|repair|214||214|completo|
|scored_0046|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0046|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0046|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0046|E2|52-1|41||41|válido|
|scored_0046|E2|52-3|41||41|válido|
|scored_0046|E2|52-6|41||41|válido|
|scored_0046|E2|64-1|41||41|válido|
|scored_0046|E2|64-3|41||41|válido|
|scored_0046|E2|64-6|41||41|válido|
|scored_0046|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0046|E4|repair|41||41|completo|
|scored_0047|E1|1|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0047|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0047|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0047|E2|52-1|242||219|válido|
|scored_0047|E2|52-3|232||215|válido|
|scored_0047|E2|52-6|226||209|válido|
|scored_0047|E2|64-1|242||221|válido|
|scored_0047|E2|64-3|232||217|válido|
|scored_0047|E2|64-6|226||208|válido|
|scored_0047|E3|joint|226|individual 226|225|OPTIMAL|
|scored_0047|E4|repair|226||226|completo|
|scored_0048|E1|1|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0048|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0048|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0048|E2|52-1|202||186|válido|
|scored_0048|E2|52-3|200||188|válido|
|scored_0048|E2|52-6|193||183|válido|
|scored_0048|E2|64-1|202||189|válido|
|scored_0048|E2|64-3|200||187|válido|
|scored_0048|E2|64-6|193||183|válido|
|scored_0048|E3|joint|193|individual 192|192|OPTIMAL|
|scored_0048|E4|repair|193||193|completo|
|scored_0049|E1|1|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0049|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0049|E1|6|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0049|E2|52-1|212||198|válido|
|scored_0049|E2|52-3|213||195|válido|
|scored_0049|E2|52-6|205||194|válido|
|scored_0049|E2|64-1|212||199|válido|
|scored_0049|E2|64-3|213||196|válido|
|scored_0049|E2|64-6|205||193|válido|
|scored_0049|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0049|E4|repair|205||205|completo|
|scored_0050|E1|1|44|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0050|E1|3|44|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0050|E1|6|44|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0050|E2|52-1|44||44|válido|
|scored_0050|E2|52-3|44||44|válido|
|scored_0050|E2|52-6|44||44|válido|
|scored_0050|E2|64-1|44||44|válido|
|scored_0050|E2|64-3|44||44|válido|
|scored_0050|E2|64-6|44||44|válido|
|scored_0050|E3|joint|43|individual 43|41|OPTIMAL|
|scored_0050|E4|repair|43||36|completo|
|scored_0051|E1|1|414|postprocess 413, optimize_a4 413, nested_windows 413|413|válido|
|scored_0051|E1|3|386|postprocess 386, optimize_a4 386, nested_windows 386|386|válido|
|scored_0051|E1|6|379|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|scored_0051|E2|52-1|414||372|válido|
|scored_0051|E2|52-3|386||358|válido|
|scored_0051|E2|52-6|379||352|válido|
|scored_0051|E2|64-1|414||421|válido|
|scored_0051|E2|64-3|386||363|válido|
|scored_0051|E2|64-6|379||357|válido|
|scored_0051|E3|joint|379|individual 378|378|FEASIBLE|
|scored_0051|E4|repair|379||379|completo|
|scored_0052|E1|1|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0052|E1|3|144|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0052|E1|6|144|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0052|E2|52-1|143||141|válido|
|scored_0052|E2|52-3|144||139|válido|
|scored_0052|E2|52-6|144||139|válido|
|scored_0052|E2|64-1|143||139|válido|
|scored_0052|E2|64-3|144||139|válido|
|scored_0052|E2|64-6|144||139|válido|
|scored_0052|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0052|E4|repair|143||142|completo|
|scored_0053|E1|1|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0053|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0053|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0053|E2|52-1|225||199|válido|
|scored_0053|E2|52-3|220||197|válido|
|scored_0053|E2|52-6|215||197|válido|
|scored_0053|E2|64-1|225||199|válido|
|scored_0053|E2|64-3|220||195|válido|
|scored_0053|E2|64-6|215||193|válido|
|scored_0053|E3|joint|215|individual 213|211|OPTIMAL|
|scored_0053|E4|repair|215||213|completo|
|scored_0054|E1|1|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0054|E1|3|265|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0054|E1|6|256|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0054|E2|52-1|269||253|válido|
|scored_0054|E2|52-3|265||244|válido|
|scored_0054|E2|52-6|256||242|válido|
|scored_0054|E2|64-1|269||253|válido|
|scored_0054|E2|64-3|265||245|válido|
|scored_0054|E2|64-6|256||242|válido|
|scored_0054|E3|joint|256|individual 255|255|FEASIBLE|
|scored_0054|E4|repair|256||256|completo|
|scored_0055|E1|1|104|postprocess 104, optimize_a4 104, nested_windows 104|104|válido|
|scored_0055|E1|3|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0055|E1|6|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0055|E2|52-1|104||100|válido|
|scored_0055|E2|52-3|103||100|válido|
|scored_0055|E2|52-6|103||100|válido|
|scored_0055|E2|64-1|104||100|válido|
|scored_0055|E2|64-3|103||100|válido|
|scored_0055|E2|64-6|103||100|válido|
|scored_0055|E3|joint|103|individual 103|103|OPTIMAL|
|scored_0055|E4|repair|103||103|completo|
|scored_0056|E1|1|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0056|E1|3|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0056|E1|6|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0056|E2|52-1|37||37|válido|
|scored_0056|E2|52-3|37||37|válido|
|scored_0056|E2|52-6|37||37|válido|
|scored_0056|E2|64-1|37||37|válido|
|scored_0056|E2|64-3|37||37|válido|
|scored_0056|E2|64-6|37||37|válido|
|scored_0056|E3|joint|37|individual 37|37|OPTIMAL|
|scored_0056|E4|repair|37||37|completo|
|scored_0057|E1|1|305|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0057|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0057|E1|6|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0057|E2|52-1|305||281|válido|
|scored_0057|E2|52-3|288||276|válido|
|scored_0057|E2|52-6|288||270|válido|
|scored_0057|E2|64-1|305||288|válido|
|scored_0057|E2|64-3|288||279|válido|
|scored_0057|E2|64-6|288||270|válido|
|scored_0057|E3|joint|288|individual 287|285|OPTIMAL|
|scored_0057|E4|repair|288||288|completo|
|scored_0058|E1|1|375|postprocess 375, optimize_a4 375, nested_windows 375|375|válido|
|scored_0058|E1|3|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|scored_0058|E1|6|349|postprocess 349, optimize_a4 349, nested_windows 349|349|válido|
|scored_0058|E2|52-1|375||351|válido|
|scored_0058|E2|52-3|355||339|válido|
|scored_0058|E2|52-6|349||333|válido|
|scored_0058|E2|64-1|375||349|válido|
|scored_0058|E2|64-3|355||344|válido|
|scored_0058|E2|64-6|349||336|válido|
|scored_0058|E3|joint|349|individual 349|349|FEASIBLE|
|scored_0058|E4|repair|349||349|completo|
|scored_0059|E1|1|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0059|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0059|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0059|E2|52-1|198||185|válido|
|scored_0059|E2|52-3|195||187|válido|
|scored_0059|E2|52-6|192||186|válido|
|scored_0059|E2|64-1|198||183|válido|
|scored_0059|E2|64-3|195||189|válido|
|scored_0059|E2|64-6|192||187|válido|
|scored_0059|E3|joint|192|individual 191|191|OPTIMAL|
|scored_0059|E4|repair|192||192|completo|
|scored_0060|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0060|E1|3|355|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0060|E1|6|342|postprocess 342, optimize_a4 342, nested_windows 342|342|válido|
|scored_0060|E2|52-1|370||330|válido|
|scored_0060|E2|52-3|355||326|válido|
|scored_0060|E2|52-6|342||317|válido|
|scored_0060|E2|64-1|370||330|válido|
|scored_0060|E2|64-3|355||326|válido|
|scored_0060|E2|64-6|342||320|válido|
|scored_0060|E3|joint|342|individual 340|339|OPTIMAL|
|scored_0060|E4|repair|342||342|completo|
|scored_0061|E1|1|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0061|E1|3|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0061|E1|6|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0061|E2|52-1|4||4|válido|
|scored_0061|E2|52-3|4||4|válido|
|scored_0061|E2|52-6|4||4|válido|
|scored_0061|E2|64-1|4||4|válido|
|scored_0061|E2|64-3|4||4|válido|
|scored_0061|E2|64-6|4||4|válido|
|scored_0061|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0061|E4|repair|4||4|completo|
|scored_0062|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0062|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0062|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0062|E2|52-1|192||185|válido|
|scored_0062|E2|52-3|188||184|válido|
|scored_0062|E2|52-6|190||182|válido|
|scored_0062|E2|64-1|192||185|válido|
|scored_0062|E2|64-3|188||184|válido|
|scored_0062|E2|64-6|190||182|válido|
|scored_0062|E3|joint|188|individual 188|188|OPTIMAL|
|scored_0062|E4|repair|188||188|completo|
|scored_0063|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0063|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0063|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0063|E2|52-1|167||160|válido|
|scored_0063|E2|52-3|166||160|válido|
|scored_0063|E2|52-6|165||154|válido|
|scored_0063|E2|64-1|167||160|válido|
|scored_0063|E2|64-3|166||160|válido|
|scored_0063|E2|64-6|165||157|válido|
|scored_0063|E3|joint|165|individual 164|164|OPTIMAL|
|scored_0063|E4|repair|165||165|completo|
|scored_0064|E1|1|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0064|E1|3|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0064|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0064|E2|52-1|185||170|válido|
|scored_0064|E2|52-3|182||170|válido|
|scored_0064|E2|52-6|183||168|válido|
|scored_0064|E2|64-1|185||172|válido|
|scored_0064|E2|64-3|182||172|válido|
|scored_0064|E2|64-6|183||172|válido|
|scored_0064|E3|joint|182|individual 182|182|OPTIMAL|
|scored_0064|E4|repair|182||179|completo|
|scored_0065|E1|1|50|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0065|E1|3|50|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0065|E1|6|50|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0065|E2|52-1|50||50|válido|
|scored_0065|E2|52-3|50||50|válido|
|scored_0065|E2|52-6|50||50|válido|
|scored_0065|E2|64-1|50||50|válido|
|scored_0065|E2|64-3|50||50|válido|
|scored_0065|E2|64-6|50||50|válido|
|scored_0065|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0065|E4|repair|50||48|completo|
|scored_0066|E1|1|55|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0066|E1|3|55|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0066|E1|6|55|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0066|E2|52-1|55||55|válido|
|scored_0066|E2|52-3|55||55|válido|
|scored_0066|E2|52-6|55||55|válido|
|scored_0066|E2|64-1|55||55|válido|
|scored_0066|E2|64-3|55||55|válido|
|scored_0066|E2|64-6|55||55|válido|
|scored_0066|E3|joint|55|individual 55|55|OPTIMAL|
|scored_0066|E4|repair|55||55|completo|
|scored_0067|E1|1|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0067|E1|3|346|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0067|E1|6|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|scored_0067|E2|52-1|377||346|válido|
|scored_0067|E2|52-3|346||324|válido|
|scored_0067|E2|52-6|343||321|válido|
|scored_0067|E2|64-1|377||344|válido|
|scored_0067|E2|64-3|346||326|válido|
|scored_0067|E2|64-6|343||322|válido|
|scored_0067|E3|joint|343|individual 343|341|OPTIMAL|
|scored_0067|E4|repair|343||343|completo|
|scored_0068|E1|1|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0068|E1|3|208|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0068|E1|6|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0068|E2|52-1|225||204|válido|
|scored_0068|E2|52-3|208||201|válido|
|scored_0068|E2|52-6|203||195|válido|
|scored_0068|E2|64-1|225||208|válido|
|scored_0068|E2|64-3|208||200|válido|
|scored_0068|E2|64-6|203||194|válido|
|scored_0068|E3|joint|203|individual 203|203|OPTIMAL|
|scored_0068|E4|repair|203||203|completo|
|scored_0069|E1|1|405|postprocess 405, optimize_a4 405, nested_windows 405|405|válido|
|scored_0069|E1|3|375|postprocess 375, optimize_a4 375, nested_windows 375|375|válido|
|scored_0069|E1|6|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0069|E2|52-1|405||366|válido|
|scored_0069|E2|52-3|375||355|válido|
|scored_0069|E2|52-6|370||353|válido|
|scored_0069|E2|64-1|405||374|válido|
|scored_0069|E2|64-3|375||361|válido|
|scored_0069|E2|64-6|370||356|válido|
|scored_0069|E3|joint|370|individual 370|369|OPTIMAL|
|scored_0069|E4|repair|370||370|completo|
|scored_0070|E1|1|212|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0070|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0070|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0070|E2|52-1|212||191|válido|
|scored_0070|E2|52-3|207||186|válido|
|scored_0070|E2|52-6|201||184|válido|
|scored_0070|E2|64-1|212||194|válido|
|scored_0070|E2|64-3|207||185|válido|
|scored_0070|E2|64-6|201||185|válido|
|scored_0070|E3|joint|201|individual 201|201|OPTIMAL|
|scored_0070|E4|repair|201||201|completo|
|scored_0071|E1|1|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0071|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0071|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0071|E2|52-1|148||146|válido|
|scored_0071|E2|52-3|148||146|válido|
|scored_0071|E2|52-6|148||146|válido|
|scored_0071|E2|64-1|148||147|válido|
|scored_0071|E2|64-3|148||147|válido|
|scored_0071|E2|64-6|148||147|válido|
|scored_0071|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0071|E4|repair|148||146|completo|
|scored_0072|E1|1|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0072|E1|3|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0072|E1|6|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0072|E2|52-1|188||173|válido|
|scored_0072|E2|52-3|181||162|válido|
|scored_0072|E2|52-6|180||161|válido|
|scored_0072|E2|64-1|188||174|válido|
|scored_0072|E2|64-3|181||162|válido|
|scored_0072|E2|64-6|180||163|válido|
|scored_0072|E3|joint|180|individual 180|180|OPTIMAL|
|scored_0072|E4|repair|180||179|completo|
|scored_0073|E1|1|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0073|E1|3|163|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0073|E1|6|163|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0073|E2|52-1|168||158|válido|
|scored_0073|E2|52-3|163||157|válido|
|scored_0073|E2|52-6|163||160|válido|
|scored_0073|E2|64-1|168||159|válido|
|scored_0073|E2|64-3|163||158|válido|
|scored_0073|E2|64-6|163||156|válido|
|scored_0073|E3|joint|163|individual 162|162|OPTIMAL|
|scored_0073|E4|repair|163||161|completo|
|scored_0074|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0074|E1|3|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0074|E1|6|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0074|E2|52-1|296||270|válido|
|scored_0074|E2|52-3|284||266|válido|
|scored_0074|E2|52-6|277||260|válido|
|scored_0074|E2|64-1|296||273|válido|
|scored_0074|E2|64-3|284||268|válido|
|scored_0074|E2|64-6|277||259|válido|
|scored_0074|E3|joint|277|individual 277|276|OPTIMAL|
|scored_0074|E4|repair|277||277|completo|
|scored_0075|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0075|E1|3|359|postprocess 359, optimize_a4 359, nested_windows 359|359|válido|
|scored_0075|E1|6|341|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0075|E2|52-1|370||337|válido|
|scored_0075|E2|52-3|359||323|válido|
|scored_0075|E2|52-6|341||318|válido|
|scored_0075|E2|64-1|370||336|válido|
|scored_0075|E2|64-3|359||325|válido|
|scored_0075|E2|64-6|341||318|válido|
|scored_0075|E3|joint|341|individual 341|338|FEASIBLE|
|scored_0075|E4|repair|341||341|completo|
|scored_0076|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0076|E1|3|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0076|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0076|E2|52-1|190||182|válido|
|scored_0076|E2|52-3|187||180|válido|
|scored_0076|E2|52-6|185||178|válido|
|scored_0076|E2|64-1|190||181|válido|
|scored_0076|E2|64-3|187||176|válido|
|scored_0076|E2|64-6|185||176|válido|
|scored_0076|E3|joint|185|individual 185|185|OPTIMAL|
|scored_0076|E4|repair|185||185|completo|
|scored_0077|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0077|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0077|E1|6|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0077|E2|52-1|144||135|válido|
|scored_0077|E2|52-3|143||135|válido|
|scored_0077|E2|52-6|142||135|válido|
|scored_0077|E2|64-1|144||137|válido|
|scored_0077|E2|64-3|143||137|válido|
|scored_0077|E2|64-6|142||136|válido|
|scored_0077|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0077|E4|repair|142||139|completo|
|scored_0078|E1|1|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0078|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0078|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0078|E2|52-1|139||130|válido|
|scored_0078|E2|52-3|137||130|válido|
|scored_0078|E2|52-6|136||130|válido|
|scored_0078|E2|64-1|139||130|válido|
|scored_0078|E2|64-3|137||130|válido|
|scored_0078|E2|64-6|136||130|válido|
|scored_0078|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0078|E4|repair|136||133|completo|
|scored_0079|E1|1|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0079|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0079|E1|6|207|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0079|E2|52-1|209||201|válido|
|scored_0079|E2|52-3|209||199|válido|
|scored_0079|E2|52-6|207||191|válido|
|scored_0079|E2|64-1|209||202|válido|
|scored_0079|E2|64-3|209||197|válido|
|scored_0079|E2|64-6|207||193|válido|
|scored_0079|E3|joint|207|individual 207|205|OPTIMAL|
|scored_0079|E4|repair|207||204|completo|
|scored_0080|E1|1|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0080|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0080|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0080|E2|52-1|203||190|válido|
|scored_0080|E2|52-3|196||186|válido|
|scored_0080|E2|52-6|193||186|válido|
|scored_0080|E2|64-1|203||190|válido|
|scored_0080|E2|64-3|196||186|válido|
|scored_0080|E2|64-6|193||184|válido|
|scored_0080|E3|joint|193|individual 193|193|OPTIMAL|
|scored_0080|E4|repair|193||193|completo|
|scored_0081|E1|1|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0081|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0081|E1|6|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0081|E2|52-1|161||154|válido|
|scored_0081|E2|52-3|160||154|válido|
|scored_0081|E2|52-6|157||149|válido|
|scored_0081|E2|64-1|161||154|válido|
|scored_0081|E2|64-3|160||154|válido|
|scored_0081|E2|64-6|157||148|válido|
|scored_0081|E3|joint|157|individual 157|157|OPTIMAL|
|scored_0081|E4|repair|157||156|completo|
|scored_0082|E1|1|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0082|E1|3|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0082|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0082|E2|52-1|168||164|válido|
|scored_0082|E2|52-3|164||156|válido|
|scored_0082|E2|52-6|166||158|válido|
|scored_0082|E2|64-1|168||164|válido|
|scored_0082|E2|64-3|164||156|válido|
|scored_0082|E2|64-6|166||158|válido|
|scored_0082|E3|joint|164|individual 164|156|OPTIMAL|
|scored_0082|E4|repair|164||161|completo|
|scored_0083|E1|1|242|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0083|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0083|E1|6|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0083|E2|52-1|242||228|válido|
|scored_0083|E2|52-3|235||225|válido|
|scored_0083|E2|52-6|236||219|válido|
|scored_0083|E2|64-1|242||227|válido|
|scored_0083|E2|64-3|235||225|válido|
|scored_0083|E2|64-6|236||221|válido|
|scored_0083|E3|joint|235|individual 235|234|OPTIMAL|
|scored_0083|E4|repair|235||235|completo|
|scored_0084|E1|1|52|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0084|E1|3|52|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0084|E1|6|52|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0084|E2|52-1|52||52|válido|
|scored_0084|E2|52-3|52||52|válido|
|scored_0084|E2|52-6|52||52|válido|
|scored_0084|E2|64-1|52||52|válido|
|scored_0084|E2|64-3|52||52|válido|
|scored_0084|E2|64-6|52||52|válido|
|scored_0084|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0084|E4|repair|52||52|completo|
|scored_0085|E1|1|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0085|E1|3|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0085|E1|6|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0085|E2|52-1|122||116|válido|
|scored_0085|E2|52-3|122||116|válido|
|scored_0085|E2|52-6|122||116|válido|
|scored_0085|E2|64-1|122||116|válido|
|scored_0085|E2|64-3|122||116|válido|
|scored_0085|E2|64-6|122||116|válido|
|scored_0085|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0085|E4|repair|122||119|completo|
|scored_0086|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0086|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0086|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0086|E2|52-1|137||132|válido|
|scored_0086|E2|52-3|137||132|válido|
|scored_0086|E2|52-6|136||132|válido|
|scored_0086|E2|64-1|137||132|válido|
|scored_0086|E2|64-3|137||132|válido|
|scored_0086|E2|64-6|136||132|válido|
|scored_0086|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0086|E4|repair|136||134|completo|
|scored_0087|E1|1|356|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|scored_0087|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0087|E1|6|325|postprocess 325, optimize_a4 325, nested_windows 325|325|válido|
|scored_0087|E2|52-1|356||319|válido|
|scored_0087|E2|52-3|329||311|válido|
|scored_0087|E2|52-6|325||306|válido|
|scored_0087|E2|64-1|356||319|válido|
|scored_0087|E2|64-3|329||317|válido|
|scored_0087|E2|64-6|325||304|válido|
|scored_0087|E3|joint|325|individual 323|323|FEASIBLE|
|scored_0087|E4|repair|325||325|completo|
|scored_0088|E1|1|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0088|E1|3|290|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0088|E1|6|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0088|E2|52-1|302||271|válido|
|scored_0088|E2|52-3|290||265|válido|
|scored_0088|E2|52-6|285||264|válido|
|scored_0088|E2|64-1|302||274|válido|
|scored_0088|E2|64-3|290||264|válido|
|scored_0088|E2|64-6|285||266|válido|
|scored_0088|E3|joint|285|individual 285|285|OPTIMAL|
|scored_0088|E4|repair|285||285|completo|
|scored_0089|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0089|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0089|E1|6|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0089|E2|52-1|163||156|válido|
|scored_0089|E2|52-3|159||155|válido|
|scored_0089|E2|52-6|159||154|válido|
|scored_0089|E2|64-1|163||154|válido|
|scored_0089|E2|64-3|159||154|válido|
|scored_0089|E2|64-6|159||153|válido|
|scored_0089|E3|joint|159|individual 159|159|OPTIMAL|
|scored_0089|E4|repair|159||159|completo|
|scored_0090|E1|1|271|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0090|E1|3|259|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0090|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0090|E2|52-1|271||248|válido|
|scored_0090|E2|52-3|259||238|válido|
|scored_0090|E2|52-6|254||238|válido|
|scored_0090|E2|64-1|271||247|válido|
|scored_0090|E2|64-3|259||242|válido|
|scored_0090|E2|64-6|254||239|válido|
|scored_0090|E3|joint|254|individual 253|253|OPTIMAL|
|scored_0090|E4|repair|254||254|completo|
|scored_0091|E1|1|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0091|E1|3|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0091|E1|6|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0091|E2|52-1|42||42|válido|
|scored_0091|E2|52-3|42||42|válido|
|scored_0091|E2|52-6|42||42|válido|
|scored_0091|E2|64-1|42||42|válido|
|scored_0091|E2|64-3|42||42|válido|
|scored_0091|E2|64-6|42||42|válido|
|scored_0091|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0091|E4|repair|42||42|completo|
|scored_0092|E1|1|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|scored_0092|E1|3|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|scored_0092|E1|6|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|scored_0092|E2|52-1|56||56|válido|
|scored_0092|E2|52-3|56||56|válido|
|scored_0092|E2|52-6|56||56|válido|
|scored_0092|E2|64-1|56||56|válido|
|scored_0092|E2|64-3|56||56|válido|
|scored_0092|E2|64-6|56||56|válido|
|scored_0092|E3|joint|56|individual 56|56|OPTIMAL|
|scored_0092|E4|repair|56||54|completo|
|scored_0093|E1|1|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0093|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0093|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0093|E2|52-1|147||138|válido|
|scored_0093|E2|52-3|146||138|válido|
|scored_0093|E2|52-6|145||138|válido|
|scored_0093|E2|64-1|147||139|válido|
|scored_0093|E2|64-3|146||139|válido|
|scored_0093|E2|64-6|145||139|válido|
|scored_0093|E3|joint|145|individual 145|145|OPTIMAL|
|scored_0093|E4|repair|145||142|completo|
|scored_0094|E1|1|356|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0094|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0094|E1|6|331|postprocess 331, optimize_a4 331, nested_windows 331|331|válido|
|scored_0094|E2|52-1|356||327|válido|
|scored_0094|E2|52-3|337||315|válido|
|scored_0094|E2|52-6|331||313|válido|
|scored_0094|E2|64-1|356||331|válido|
|scored_0094|E2|64-3|337||316|válido|
|scored_0094|E2|64-6|331||316|válido|
|scored_0094|E3|joint|331|individual 331|331|OPTIMAL|
|scored_0094|E4|repair|331||331|completo|
|scored_0095|E1|1|47|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0095|E1|3|47|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0095|E1|6|47|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0095|E2|52-1|47||47|válido|
|scored_0095|E2|52-3|47||47|válido|
|scored_0095|E2|52-6|47||47|válido|
|scored_0095|E2|64-1|47||47|válido|
|scored_0095|E2|64-3|47||47|válido|
|scored_0095|E2|64-6|47||47|válido|
|scored_0095|E3|joint|47|individual 47|47|OPTIMAL|
|scored_0095|E4|repair|47||47|completo|
|scored_0096|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0096|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0096|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0096|E2|52-1|240||223|válido|
|scored_0096|E2|52-3|235||222|válido|
|scored_0096|E2|52-6|233||221|válido|
|scored_0096|E2|64-1|240||223|válido|
|scored_0096|E2|64-3|235||223|válido|
|scored_0096|E2|64-6|233||223|válido|
|scored_0096|E3|joint|233|individual 233|232|OPTIMAL|
|scored_0096|E4|repair|233||231|completo|
|scored_0097|E1|1|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0097|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0097|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0097|E2|52-1|131||127|válido|
|scored_0097|E2|52-3|131||127|válido|
|scored_0097|E2|52-6|131||127|válido|
|scored_0097|E2|64-1|131||125|válido|
|scored_0097|E2|64-3|131||125|válido|
|scored_0097|E2|64-6|131||125|válido|
|scored_0097|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0097|E4|repair|131||129|completo|
|scored_0098|E1|1|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0098|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0098|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0098|E2|52-1|258||235|válido|
|scored_0098|E2|52-3|249||235|válido|
|scored_0098|E2|52-6|246||235|válido|
|scored_0098|E2|64-1|258||237|válido|
|scored_0098|E2|64-3|249||233|válido|
|scored_0098|E2|64-6|246||236|válido|
|scored_0098|E3|joint|246|individual 246|244|OPTIMAL|
|scored_0098|E4|repair|246||246|completo|
|scored_0099|E1|1|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0099|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0099|E1|6|217|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0099|E2|52-1|224||193|válido|
|scored_0099|E2|52-3|220||192|válido|
|scored_0099|E2|52-6|217||197|válido|
|scored_0099|E2|64-1|224||197|válido|
|scored_0099|E2|64-3|220||191|válido|
|scored_0099|E2|64-6|217||199|válido|
|scored_0099|E3|joint|217|individual 217|217|OPTIMAL|
|scored_0099|E4|repair|217||214|completo|
|scored_0100|E1|1|204|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0100|E1|3|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0100|E1|6|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0100|E2|52-1|204||187|válido|
|scored_0100|E2|52-3|199||189|válido|
|scored_0100|E2|52-6|196||188|válido|
|scored_0100|E2|64-1|204||186|válido|
|scored_0100|E2|64-3|199||186|válido|
|scored_0100|E2|64-6|196||185|válido|
|scored_0100|E3|joint|196|individual 194|193|OPTIMAL|
|scored_0100|E4|repair|196||196|completo|
|scored_0101|E1|1|164|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0101|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0101|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0101|E2|52-1|164||156|válido|
|scored_0101|E2|52-3|162||154|válido|
|scored_0101|E2|52-6|162||154|válido|
|scored_0101|E2|64-1|164||153|válido|
|scored_0101|E2|64-3|162||153|válido|
|scored_0101|E2|64-6|162||153|válido|
|scored_0101|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0101|E4|repair|162||160|completo|
|scored_0102|E1|1|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0102|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0102|E1|6|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0102|E2|52-1|213||200|válido|
|scored_0102|E2|52-3|208||193|válido|
|scored_0102|E2|52-6|203||190|válido|
|scored_0102|E2|64-1|213||200|válido|
|scored_0102|E2|64-3|208||194|válido|
|scored_0102|E2|64-6|203||192|válido|
|scored_0102|E3|joint|203|individual 203|202|OPTIMAL|
|scored_0102|E4|repair|203||203|completo|
|scored_0103|E1|1|61|postprocess 59, optimize_a4 59, nested_windows 59|59|válido|
|scored_0103|E1|3|61|postprocess 59, optimize_a4 59, nested_windows 59|59|válido|
|scored_0103|E1|6|61|postprocess 59, optimize_a4 59, nested_windows 59|59|válido|
|scored_0103|E2|52-1|61||61|válido|
|scored_0103|E2|52-3|61||61|válido|
|scored_0103|E2|52-6|61||61|válido|
|scored_0103|E2|64-1|61||61|válido|
|scored_0103|E2|64-3|61||61|válido|
|scored_0103|E2|64-6|61||61|válido|
|scored_0103|E3|joint|61|individual 61|61|OPTIMAL|
|scored_0103|E4|repair|61||61|completo|
|scored_0104|E1|1|329|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0104|E1|3|312|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0104|E1|6|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0104|E2|52-1|329||301|válido|
|scored_0104|E2|52-3|312||289|válido|
|scored_0104|E2|52-6|306||286|válido|
|scored_0104|E2|64-1|329||373|válido|
|scored_0104|E2|64-3|312||295|válido|
|scored_0104|E2|64-6|306||286|válido|
|scored_0104|E3|joint|306|individual 306|306|FEASIBLE|
|scored_0104|E4|repair|306||306|completo|
|scored_0105|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0105|E1|3|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0105|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0105|E2|52-1|207||188|válido|
|scored_0105|E2|52-3|201||190|válido|
|scored_0105|E2|52-6|201||189|válido|
|scored_0105|E2|64-1|207||190|válido|
|scored_0105|E2|64-3|201||192|válido|
|scored_0105|E2|64-6|201||189|válido|
|scored_0105|E3|joint|201|individual 200|200|OPTIMAL|
|scored_0105|E4|repair|201||201|completo|
|scored_0106|E1|1|341|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0106|E1|3|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0106|E1|6|309|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0106|E2|52-1|341||302|válido|
|scored_0106|E2|52-3|323||301|válido|
|scored_0106|E2|52-6|309||289|válido|
|scored_0106|E2|64-1|341||327|válido|
|scored_0106|E2|64-3|323||303|válido|
|scored_0106|E2|64-6|309||291|válido|
|scored_0106|E3|joint|309|individual 309|309|OPTIMAL|
|scored_0106|E4|repair|309||308|completo|
|scored_0107|E1|1|351|postprocess 351, optimize_a4 351, nested_windows 351|351|válido|
|scored_0107|E1|3|328|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0107|E1|6|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0107|E2|52-1|351||327|válido|
|scored_0107|E2|52-3|328||311|válido|
|scored_0107|E2|52-6|319||300|válido|
|scored_0107|E2|64-1|351||327|válido|
|scored_0107|E2|64-3|328||310|válido|
|scored_0107|E2|64-6|319||302|válido|
|scored_0107|E3|joint|319|individual 319|318|FEASIBLE|
|scored_0107|E4|repair|319||319|completo|
|scored_0108|E1|1|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0108|E1|3|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0108|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0108|E2|52-1|249||232|válido|
|scored_0108|E2|52-3|248||233|válido|
|scored_0108|E2|52-6|245||228|válido|
|scored_0108|E2|64-1|249||233|válido|
|scored_0108|E2|64-3|248||236|válido|
|scored_0108|E2|64-6|245||229|válido|
|scored_0108|E3|joint|245|individual 245|245|OPTIMAL|
|scored_0108|E4|repair|245||245|completo|
|scored_0109|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0109|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0109|E1|6|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0109|E2|52-1|199||186|válido|
|scored_0109|E2|52-3|197||183|válido|
|scored_0109|E2|52-6|194||177|válido|
|scored_0109|E2|64-1|199||188|válido|
|scored_0109|E2|64-3|197||181|válido|
|scored_0109|E2|64-6|194||178|válido|
|scored_0109|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0109|E4|repair|194||194|completo|
|scored_0110|E1|1|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0110|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0110|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0110|E2|52-1|242||226|válido|
|scored_0110|E2|52-3|238||215|válido|
|scored_0110|E2|52-6|232||213|válido|
|scored_0110|E2|64-1|242||226|válido|
|scored_0110|E2|64-3|238||215|válido|
|scored_0110|E2|64-6|232||213|válido|
|scored_0110|E3|joint|232|individual 231|231|OPTIMAL|
|scored_0110|E4|repair|232||232|completo|
|scored_0111|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0111|E1|3|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0111|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0111|E2|52-1|310||279|válido|
|scored_0111|E2|52-3|298||268|válido|
|scored_0111|E2|52-6|287||266|válido|
|scored_0111|E2|64-1|310||281|válido|
|scored_0111|E2|64-3|298||269|válido|
|scored_0111|E2|64-6|287||266|válido|
|scored_0111|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0111|E4|repair|287||287|completo|
|scored_0112|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0112|E1|3|275|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0112|E1|6|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0112|E2|52-1|298||280|válido|
|scored_0112|E2|52-3|275||262|válido|
|scored_0112|E2|52-6|272||264|válido|
|scored_0112|E2|64-1|298||280|válido|
|scored_0112|E2|64-3|275||265|válido|
|scored_0112|E2|64-6|272||265|válido|
|scored_0112|E3|joint|272|individual 272|272|OPTIMAL|
|scored_0112|E4|repair|272||272|completo|
|scored_0113|E1|1|40|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0113|E1|3|40|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0113|E1|6|40|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0113|E2|52-1|40||40|válido|
|scored_0113|E2|52-3|40||40|válido|
|scored_0113|E2|52-6|40||40|válido|
|scored_0113|E2|64-1|40||40|válido|
|scored_0113|E2|64-3|40||40|válido|
|scored_0113|E2|64-6|40||40|válido|
|scored_0113|E3|joint|40|individual 40|40|OPTIMAL|
|scored_0113|E4|repair|40||39|completo|
|scored_0114|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0114|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0114|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0114|E2|52-1|226||215|válido|
|scored_0114|E2|52-3|223||211|válido|
|scored_0114|E2|52-6|222||211|válido|
|scored_0114|E2|64-1|226||216|válido|
|scored_0114|E2|64-3|223||211|válido|
|scored_0114|E2|64-6|222||212|válido|
|scored_0114|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0114|E4|repair|222||222|completo|
|scored_0115|E1|1|282|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0115|E1|3|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0115|E1|6|271|postprocess 271, optimize_a4 271, nested_windows 271|271|válido|
|scored_0115|E2|52-1|282||266|válido|
|scored_0115|E2|52-3|279||254|válido|
|scored_0115|E2|52-6|271||259|válido|
|scored_0115|E2|64-1|282||268|válido|
|scored_0115|E2|64-3|279||254|válido|
|scored_0115|E2|64-6|271||258|válido|
|scored_0115|E3|joint|271|individual 271|271|OPTIMAL|
|scored_0115|E4|repair|271||271|completo|
|scored_0116|E1|1|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0116|E1|3|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0116|E1|6|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0116|E2|52-1|117||111|válido|
|scored_0116|E2|52-3|117||111|válido|
|scored_0116|E2|52-6|117||111|válido|
|scored_0116|E2|64-1|117||112|válido|
|scored_0116|E2|64-3|117||112|válido|
|scored_0116|E2|64-6|117||110|válido|
|scored_0116|E3|joint|117|individual 117|117|OPTIMAL|
|scored_0116|E4|repair|117||114|completo|
|scored_0117|E1|1|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0117|E1|3|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0117|E1|6|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0117|E2|52-1|272||254|válido|
|scored_0117|E2|52-3|259||249|válido|
|scored_0117|E2|52-6|261||242|válido|
|scored_0117|E2|64-1|272||254|válido|
|scored_0117|E2|64-3|259||249|válido|
|scored_0117|E2|64-6|261||246|válido|
|scored_0117|E3|joint|259|individual 258|258|rechazado por registros|
|scored_0117|E4|repair|259||259|completo|
|scored_0118|E1|1|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0118|E1|3|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0118|E1|6|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0118|E2|52-1|137||132|válido|
|scored_0118|E2|52-3|136||132|válido|
|scored_0118|E2|52-6|136||132|válido|
|scored_0118|E2|64-1|137||132|válido|
|scored_0118|E2|64-3|136||131|válido|
|scored_0118|E2|64-6|136||131|válido|
|scored_0118|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0118|E4|repair|136||134|completo|
|scored_0119|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0119|E1|3|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0119|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0119|E2|52-1|330||302|válido|
|scored_0119|E2|52-3|311||296|válido|
|scored_0119|E2|52-6|303||287|válido|
|scored_0119|E2|64-1|330||303|válido|
|scored_0119|E2|64-3|311||294|válido|
|scored_0119|E2|64-6|303||288|válido|
|scored_0119|E3|joint|303|individual 303|301|OPTIMAL|
|scored_0119|E4|repair|303||303|completo|
|scored_0120|E1|1|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0120|E1|3|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0120|E1|6|271|postprocess 271, optimize_a4 271, nested_windows 271|271|válido|
|scored_0120|E2|52-1|282||262|válido|
|scored_0120|E2|52-3|268||251|válido|
|scored_0120|E2|52-6|271||247|válido|
|scored_0120|E2|64-1|282||263|válido|
|scored_0120|E2|64-3|268||256|válido|
|scored_0120|E2|64-6|271||250|válido|
|scored_0120|E3|joint|268|individual 268|267|OPTIMAL|
|scored_0120|E4|repair|268||268|completo|
|scored_0121|E1|1|66|postprocess 66, optimize_a4 66, nested_windows 66|66|válido|
|scored_0121|E1|3|66|postprocess 66, optimize_a4 66, nested_windows 66|66|válido|
|scored_0121|E1|6|66|postprocess 66, optimize_a4 66, nested_windows 66|66|válido|
|scored_0121|E2|52-1|66||67|válido|
|scored_0121|E2|52-3|66||67|válido|
|scored_0121|E2|52-6|66||67|válido|
|scored_0121|E2|64-1|66||67|válido|
|scored_0121|E2|64-3|66||67|válido|
|scored_0121|E2|64-6|66||67|válido|
|scored_0121|E3|joint|66|individual 66|66|OPTIMAL|
|scored_0121|E4|repair|66||64|completo|
|scored_0122|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0122|E1|3|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0122|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0122|E2|52-1|219||205|válido|
|scored_0122|E2|52-3|210||200|válido|
|scored_0122|E2|52-6|211||199|válido|
|scored_0122|E2|64-1|219||208|válido|
|scored_0122|E2|64-3|210||199|válido|
|scored_0122|E2|64-6|211||201|válido|
|scored_0122|E3|joint|210|individual 210|209|OPTIMAL|
|scored_0122|E4|repair|210||210|completo|
|scored_0123|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0123|E1|3|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0123|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0123|E2|52-1|303||289|válido|
|scored_0123|E2|52-3|291||273|válido|
|scored_0123|E2|52-6|292||271|válido|
|scored_0123|E2|64-1|303||290|válido|
|scored_0123|E2|64-3|291||274|válido|
|scored_0123|E2|64-6|292||269|válido|
|scored_0123|E3|joint|291|individual 289|284|FEASIBLE|
|scored_0123|E4|repair|291||291|completo|
|scored_0124|E1|1|37|postprocess 35, optimize_a4 35, nested_windows 35|35|válido|
|scored_0124|E1|3|37|postprocess 35, optimize_a4 35, nested_windows 35|35|válido|
|scored_0124|E1|6|37|postprocess 35, optimize_a4 35, nested_windows 35|35|válido|
|scored_0124|E2|52-1|37||37|válido|
|scored_0124|E2|52-3|37||37|válido|
|scored_0124|E2|52-6|37||37|válido|
|scored_0124|E2|64-1|37||37|válido|
|scored_0124|E2|64-3|37||37|válido|
|scored_0124|E2|64-6|37||37|válido|
|scored_0124|E3|joint|37|individual 37|37|OPTIMAL|
|scored_0124|E4|repair|37||37|completo|
|scored_0125|E1|1|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0125|E1|3|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0125|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0125|E2|52-1|157||149|válido|
|scored_0125|E2|52-3|153||148|válido|
|scored_0125|E2|52-6|152||148|válido|
|scored_0125|E2|64-1|157||150|válido|
|scored_0125|E2|64-3|153||148|válido|
|scored_0125|E2|64-6|152||149|válido|
|scored_0125|E3|joint|152|individual 151|151|OPTIMAL|
|scored_0125|E4|repair|152||150|completo|
|scored_0126|E1|1|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0126|E1|3|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0126|E1|6|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0126|E2|52-1|5||5|válido|
|scored_0126|E2|52-3|5||5|válido|
|scored_0126|E2|52-6|5||5|válido|
|scored_0126|E2|64-1|5||5|válido|
|scored_0126|E2|64-3|5||5|válido|
|scored_0126|E2|64-6|5||5|válido|
|scored_0126|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0126|E4|repair|4||4|completo|
|scored_0127|E1|1|172|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0127|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0127|E1|6|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0127|E2|52-1|172||149|válido|
|scored_0127|E2|52-3|169||148|válido|
|scored_0127|E2|52-6|164||146|válido|
|scored_0127|E2|64-1|172||155|válido|
|scored_0127|E2|64-3|169||155|válido|
|scored_0127|E2|64-6|164||151|válido|
|scored_0127|E3|joint|164|individual 164|164|OPTIMAL|
|scored_0127|E4|repair|164||156|completo|
|scored_0128|E1|1|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0128|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0128|E1|6|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0128|E2|52-1|172||161|válido|
|scored_0128|E2|52-3|169||161|válido|
|scored_0128|E2|52-6|168||163|válido|
|scored_0128|E2|64-1|172||161|válido|
|scored_0128|E2|64-3|169||161|válido|
|scored_0128|E2|64-6|168||160|válido|
|scored_0128|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0128|E4|repair|168||167|completo|
|scored_0129|E1|1|382|postprocess 382, optimize_a4 382, nested_windows 382|382|válido|
|scored_0129|E1|3|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|scored_0129|E1|6|366|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0129|E2|52-1|382||356|válido|
|scored_0129|E2|52-3|369||346|válido|
|scored_0129|E2|52-6|366||340|válido|
|scored_0129|E2|64-1|382||358|válido|
|scored_0129|E2|64-3|369||349|válido|
|scored_0129|E2|64-6|366||455|válido|
|scored_0129|E3|joint|366|individual 366|365|FEASIBLE|
|scored_0129|E4|repair|366||366|completo|
|scored_0130|E1|1|39|postprocess 36, optimize_a4 35, nested_windows 35|35|válido|
|scored_0130|E1|3|39|postprocess 36, optimize_a4 35, nested_windows 35|35|válido|
|scored_0130|E1|6|39|postprocess 36, optimize_a4 35, nested_windows 35|35|válido|
|scored_0130|E2|52-1|39||39|válido|
|scored_0130|E2|52-3|39||39|válido|
|scored_0130|E2|52-6|39||39|válido|
|scored_0130|E2|64-1|39||39|válido|
|scored_0130|E2|64-3|39||39|válido|
|scored_0130|E2|64-6|39||39|válido|
|scored_0130|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0130|E4|repair|39||39|completo|
|scored_0131|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0131|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0131|E1|6|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0131|E2|52-1|125||122|válido|
|scored_0131|E2|52-3|125||122|válido|
|scored_0131|E2|52-6|125||122|válido|
|scored_0131|E2|64-1|125||121|válido|
|scored_0131|E2|64-3|125||121|válido|
|scored_0131|E2|64-6|125||121|válido|
|scored_0131|E3|joint|125|individual 125|125|OPTIMAL|
|scored_0131|E4|repair|125||121|completo|
|scored_0132|E1|1|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0132|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0132|E1|6|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0132|E2|52-1|138||133|válido|
|scored_0132|E2|52-3|138||133|válido|
|scored_0132|E2|52-6|138||133|válido|
|scored_0132|E2|64-1|138||133|válido|
|scored_0132|E2|64-3|138||133|válido|
|scored_0132|E2|64-6|138||133|válido|
|scored_0132|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0132|E4|repair|138||138|completo|
|scored_0133|E1|1|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0133|E1|3|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0133|E1|6|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0133|E2|52-1|348||305|válido|
|scored_0133|E2|52-3|315||293|válido|
|scored_0133|E2|52-6|308||289|válido|
|scored_0133|E2|64-1|348||308|válido|
|scored_0133|E2|64-3|315||295|válido|
|scored_0133|E2|64-6|308||292|válido|
|scored_0133|E3|joint|308|individual 307|305|FEASIBLE|
|scored_0133|E4|repair|308||308|completo|
|scored_0134|E1|1|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0134|E1|3|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0134|E1|6|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0134|E2|52-1|44||44|válido|
|scored_0134|E2|52-3|44||44|válido|
|scored_0134|E2|52-6|44||44|válido|
|scored_0134|E2|64-1|44||44|válido|
|scored_0134|E2|64-3|44||44|válido|
|scored_0134|E2|64-6|44||44|válido|
|scored_0134|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0134|E4|repair|44||44|completo|
|scored_0135|E1|1|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0135|E1|3|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0135|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0135|E2|52-1|123||120|válido|
|scored_0135|E2|52-3|123||120|válido|
|scored_0135|E2|52-6|123||120|válido|
|scored_0135|E2|64-1|123||119|válido|
|scored_0135|E2|64-3|123||119|válido|
|scored_0135|E2|64-6|123||119|válido|
|scored_0135|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0135|E4|repair|123||122|completo|
|scored_0136|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0136|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0136|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0136|E2|52-1|192||163|válido|
|scored_0136|E2|52-3|188||167|válido|
|scored_0136|E2|52-6|188||165|válido|
|scored_0136|E2|64-1|192||160|válido|
|scored_0136|E2|64-3|188||160|válido|
|scored_0136|E2|64-6|188||157|válido|
|scored_0136|E3|joint|188|individual 187|187|OPTIMAL|
|scored_0136|E4|repair|188||187|completo|
|scored_0137|E1|1|76|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0137|E1|3|76|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0137|E1|6|76|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0137|E2|52-1|76||75|válido|
|scored_0137|E2|52-3|76||75|válido|
|scored_0137|E2|52-6|76||75|válido|
|scored_0137|E2|64-1|76||75|válido|
|scored_0137|E2|64-3|76||75|válido|
|scored_0137|E2|64-6|76||75|válido|
|scored_0137|E3|joint|76|individual 76|76|OPTIMAL|
|scored_0137|E4|repair|76||72|completo|
|scored_0138|E1|1|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0138|E1|3|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0138|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0138|E2|52-1|124||119|válido|
|scored_0138|E2|52-3|124||119|válido|
|scored_0138|E2|52-6|122||119|válido|
|scored_0138|E2|64-1|124||119|válido|
|scored_0138|E2|64-3|124||119|válido|
|scored_0138|E2|64-6|122||119|válido|
|scored_0138|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0138|E4|repair|122||120|completo|
|scored_0139|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0139|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0139|E1|6|246|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0139|E2|52-1|251||238|válido|
|scored_0139|E2|52-3|249||233|válido|
|scored_0139|E2|52-6|246||228|válido|
|scored_0139|E2|64-1|251||238|válido|
|scored_0139|E2|64-3|249||231|válido|
|scored_0139|E2|64-6|246||228|válido|
|scored_0139|E3|joint|246|individual 246|243|OPTIMAL|
|scored_0139|E4|repair|246||246|completo|
|scored_0140|E1|1|226|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0140|E1|3|215|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0140|E1|6|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0140|E2|52-1|226||209|válido|
|scored_0140|E2|52-3|215||200|válido|
|scored_0140|E2|52-6|209||199|válido|
|scored_0140|E2|64-1|226||208|válido|
|scored_0140|E2|64-3|215||201|válido|
|scored_0140|E2|64-6|209||204|válido|
|scored_0140|E3|joint|209|individual 209|209|OPTIMAL|
|scored_0140|E4|repair|209||209|completo|
|scored_0141|E1|1|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0141|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0141|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0141|E2|52-1|174||168|válido|
|scored_0141|E2|52-3|173||168|válido|
|scored_0141|E2|52-6|172||168|válido|
|scored_0141|E2|64-1|174||170|válido|
|scored_0141|E2|64-3|173||167|válido|
|scored_0141|E2|64-6|172||168|válido|
|scored_0141|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0141|E4|repair|172||170|completo|
|scored_0142|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0142|E1|3|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0142|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0142|E2|52-1|158||148|válido|
|scored_0142|E2|52-3|156||148|válido|
|scored_0142|E2|52-6|156||148|válido|
|scored_0142|E2|64-1|158||149|válido|
|scored_0142|E2|64-3|156||149|válido|
|scored_0142|E2|64-6|156||149|válido|
|scored_0142|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0142|E4|repair|156||156|completo|
|scored_0143|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0143|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0143|E1|6|221|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0143|E2|52-1|236||202|válido|
|scored_0143|E2|52-3|233||207|válido|
|scored_0143|E2|52-6|221||190|válido|
|scored_0143|E2|64-1|236||204|válido|
|scored_0143|E2|64-3|233||203|válido|
|scored_0143|E2|64-6|221||188|válido|
|scored_0143|E3|joint|221|individual 221|220|OPTIMAL|
|scored_0143|E4|repair|221||220|completo|
|scored_0144|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0144|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0144|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0144|E2|52-1|151||147|válido|
|scored_0144|E2|52-3|151||145|válido|
|scored_0144|E2|52-6|148||144|válido|
|scored_0144|E2|64-1|151||142|válido|
|scored_0144|E2|64-3|151||142|válido|
|scored_0144|E2|64-6|148||142|válido|
|scored_0144|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0144|E4|repair|148||141|completo|
|scored_0145|E1|1|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0145|E1|3|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0145|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0145|E2|52-1|308||280|válido|
|scored_0145|E2|52-3|297||276|válido|
|scored_0145|E2|52-6|287||269|válido|
|scored_0145|E2|64-1|308||282|válido|
|scored_0145|E2|64-3|297||276|válido|
|scored_0145|E2|64-6|287||273|válido|
|scored_0145|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0145|E4|repair|287||287|completo|
|scored_0146|E1|1|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0146|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0146|E1|6|344|postprocess 344, optimize_a4 344, nested_windows 344|344|válido|
|scored_0146|E2|52-1|364||338|válido|
|scored_0146|E2|52-3|350||330|válido|
|scored_0146|E2|52-6|344||329|válido|
|scored_0146|E2|64-1|364||335|válido|
|scored_0146|E2|64-3|350||328|válido|
|scored_0146|E2|64-6|344||327|válido|
|scored_0146|E3|joint|344|individual 344|343|OPTIMAL|
|scored_0146|E4|repair|344||344|completo|
|scored_0147|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0147|E1|3|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0147|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0147|E2|52-1|123||123|válido|
|scored_0147|E2|52-3|122||123|válido|
|scored_0147|E2|52-6|122||123|válido|
|scored_0147|E2|64-1|123||122|válido|
|scored_0147|E2|64-3|122||122|válido|
|scored_0147|E2|64-6|122||122|válido|
|scored_0147|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0147|E4|repair|122||119|completo|
|scored_0148|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0148|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0148|E1|6|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0148|E2|52-1|248||228|válido|
|scored_0148|E2|52-3|239||222|válido|
|scored_0148|E2|52-6|234||217|válido|
|scored_0148|E2|64-1|248||230|válido|
|scored_0148|E2|64-3|239||221|válido|
|scored_0148|E2|64-6|234||221|válido|
|scored_0148|E3|joint|234|individual 234|233|OPTIMAL|
|scored_0148|E4|repair|234||234|completo|
|scored_0149|E1|1|30|postprocess 28, optimize_a4 27, nested_windows 26|26|válido|
|scored_0149|E1|3|30|postprocess 28, optimize_a4 27, nested_windows 26|26|válido|
|scored_0149|E1|6|30|postprocess 28, optimize_a4 27, nested_windows 26|26|válido|
|scored_0149|E2|52-1|30||30|válido|
|scored_0149|E2|52-3|30||30|válido|
|scored_0149|E2|52-6|30||30|válido|
|scored_0149|E2|64-1|30||30|válido|
|scored_0149|E2|64-3|30||30|válido|
|scored_0149|E2|64-6|30||30|válido|
|scored_0149|E3|joint|28|individual 27|26|OPTIMAL|
|scored_0149|E4|repair|28||28|completo|
|scored_0150|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0150|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0150|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0150|E2|52-1|228||216|válido|
|scored_0150|E2|52-3|223||208|válido|
|scored_0150|E2|52-6|217||208|válido|
|scored_0150|E2|64-1|228||217|válido|
|scored_0150|E2|64-3|223||211|válido|
|scored_0150|E2|64-6|217||205|válido|
|scored_0150|E3|joint|217|individual 217|217|OPTIMAL|
|scored_0150|E4|repair|217||217|completo|
|scored_0151|E1|1|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0151|E1|3|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0151|E1|6|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0151|E2|52-1|44||44|válido|
|scored_0151|E2|52-3|44||44|válido|
|scored_0151|E2|52-6|44||44|válido|
|scored_0151|E2|64-1|44||44|válido|
|scored_0151|E2|64-3|44||44|válido|
|scored_0151|E2|64-6|44||44|válido|
|scored_0151|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0151|E4|repair|44||44|completo|
|scored_0152|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0152|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0152|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0152|E2|52-1|159||159|válido|
|scored_0152|E2|52-3|157||158|válido|
|scored_0152|E2|52-6|156||158|válido|
|scored_0152|E2|64-1|159||158|válido|
|scored_0152|E2|64-3|157||156|válido|
|scored_0152|E2|64-6|156||156|válido|
|scored_0152|E3|joint|156|individual 155|155|OPTIMAL|
|scored_0152|E4|repair|156||154|completo|
|scored_0153|E1|1|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0153|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0153|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0153|E2|52-1|142||138|válido|
|scored_0153|E2|52-3|145||138|válido|
|scored_0153|E2|52-6|145||138|válido|
|scored_0153|E2|64-1|142||138|válido|
|scored_0153|E2|64-3|145||138|válido|
|scored_0153|E2|64-6|145||138|válido|
|scored_0153|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0153|E4|repair|142||141|completo|
|scored_0154|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0154|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0154|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0154|E2|52-1|233||213|válido|
|scored_0154|E2|52-3|225||215|válido|
|scored_0154|E2|52-6|222||206|válido|
|scored_0154|E2|64-1|233||214|válido|
|scored_0154|E2|64-3|225||215|válido|
|scored_0154|E2|64-6|222||206|válido|
|scored_0154|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0154|E4|repair|222||222|completo|
|scored_0155|E1|1|321|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0155|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0155|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0155|E2|52-1|321||289|válido|
|scored_0155|E2|52-3|308||286|válido|
|scored_0155|E2|52-6|298||285|válido|
|scored_0155|E2|64-1|321||291|válido|
|scored_0155|E2|64-3|308||289|válido|
|scored_0155|E2|64-6|298||282|válido|
|scored_0155|E3|joint|298|individual 298|298|FEASIBLE|
|scored_0155|E4|repair|298||298|completo|
|scored_0156|E1|1|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0156|E1|3|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0156|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0156|E2|52-1|187||175|válido|
|scored_0156|E2|52-3|182||177|válido|
|scored_0156|E2|52-6|182||173|válido|
|scored_0156|E2|64-1|187||176|válido|
|scored_0156|E2|64-3|182||177|válido|
|scored_0156|E2|64-6|182||171|válido|
|scored_0156|E3|joint|182|individual 182|181|OPTIMAL|
|scored_0156|E4|repair|182||182|completo|
|scored_0157|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0157|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0157|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0157|E2|52-1|216||205|válido|
|scored_0157|E2|52-3|206||200|válido|
|scored_0157|E2|52-6|208||198|válido|
|scored_0157|E2|64-1|216||205|válido|
|scored_0157|E2|64-3|206||203|válido|
|scored_0157|E2|64-6|208||200|válido|
|scored_0157|E3|joint|206|individual 205|205|OPTIMAL|
|scored_0157|E4|repair|206||206|completo|
|scored_0158|E1|1|204|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0158|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0158|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0158|E2|52-1|204||197|válido|
|scored_0158|E2|52-3|200||194|válido|
|scored_0158|E2|52-6|199||192|válido|
|scored_0158|E2|64-1|204||197|válido|
|scored_0158|E2|64-3|200||193|válido|
|scored_0158|E2|64-6|199||192|válido|
|scored_0158|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0158|E4|repair|199||199|completo|
|scored_0159|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0159|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0159|E1|6|307|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0159|E2|52-1|330||300|válido|
|scored_0159|E2|52-3|304||288|válido|
|scored_0159|E2|52-6|307||286|válido|
|scored_0159|E2|64-1|330||301|válido|
|scored_0159|E2|64-3|304||312|válido|
|scored_0159|E2|64-6|307||285|válido|
|scored_0159|E3|joint|304|individual 304|301|OPTIMAL|
|scored_0159|E4|repair|304||304|completo|
|scored_0160|E1|1|256|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0160|E1|3|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0160|E1|6|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0160|E2|52-1|256||238|válido|
|scored_0160|E2|52-3|253||231|válido|
|scored_0160|E2|52-6|248||229|válido|
|scored_0160|E2|64-1|256||237|válido|
|scored_0160|E2|64-3|253||236|válido|
|scored_0160|E2|64-6|248||229|válido|
|scored_0160|E3|joint|248|individual 248|248|OPTIMAL|
|scored_0160|E4|repair|248||248|completo|
|scored_0161|E1|1|213|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0161|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0161|E1|6|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0161|E2|52-1|213||201|válido|
|scored_0161|E2|52-3|213||201|válido|
|scored_0161|E2|52-6|209||197|válido|
|scored_0161|E2|64-1|213||201|válido|
|scored_0161|E2|64-3|213||200|válido|
|scored_0161|E2|64-6|209||196|válido|
|scored_0161|E3|joint|209|individual 209|209|OPTIMAL|
|scored_0161|E4|repair|209||208|completo|
|scored_0162|E1|1|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0162|E1|3|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0162|E1|6|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0162|E2|52-1|264||250|válido|
|scored_0162|E2|52-3|254||239|válido|
|scored_0162|E2|52-6|250||229|válido|
|scored_0162|E2|64-1|264||242|válido|
|scored_0162|E2|64-3|254||231|válido|
|scored_0162|E2|64-6|250||231|válido|
|scored_0162|E3|joint|250|individual 250|249|OPTIMAL|
|scored_0162|E4|repair|250||250|completo|
|scored_0163|E1|1|352|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0163|E1|3|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0163|E1|6|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0163|E2|52-1|352||328|válido|
|scored_0163|E2|52-3|333||312|válido|
|scored_0163|E2|52-6|333||314|válido|
|scored_0163|E2|64-1|352||329|válido|
|scored_0163|E2|64-3|333||313|válido|
|scored_0163|E2|64-6|333||315|válido|
|scored_0163|E3|joint|333|individual 331|327|FEASIBLE|
|scored_0163|E4|repair|333||333|completo|
|scored_0164|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0164|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0164|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0164|E2|52-1|169||161|válido|
|scored_0164|E2|52-3|165||161|válido|
|scored_0164|E2|52-6|165||161|válido|
|scored_0164|E2|64-1|169||161|válido|
|scored_0164|E2|64-3|165||161|válido|
|scored_0164|E2|64-6|165||161|válido|
|scored_0164|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0164|E4|repair|165||164|completo|
|scored_0165|E1|1|346|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|scored_0165|E1|3|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0165|E1|6|324|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0165|E2|52-1|346||311|válido|
|scored_0165|E2|52-3|330||304|válido|
|scored_0165|E2|52-6|324||302|válido|
|scored_0165|E2|64-1|346||314|válido|
|scored_0165|E2|64-3|330||304|válido|
|scored_0165|E2|64-6|324||303|válido|
|scored_0165|E3|joint|324|individual 323|319|OPTIMAL|
|scored_0165|E4|repair|324||324|completo|
|scored_0166|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0166|E1|3|288|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0166|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0166|E2|52-1|298||274|válido|
|scored_0166|E2|52-3|288||262|válido|
|scored_0166|E2|52-6|282||257|válido|
|scored_0166|E2|64-1|298||274|válido|
|scored_0166|E2|64-3|288||264|válido|
|scored_0166|E2|64-6|282||257|válido|
|scored_0166|E3|joint|282|individual 282|282|OPTIMAL|
|scored_0166|E4|repair|282||282|completo|
|scored_0167|E1|1|356|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0167|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0167|E1|6|333|postprocess 332, optimize_a4 332, nested_windows 332|332|válido|
|scored_0167|E2|52-1|356||327|válido|
|scored_0167|E2|52-3|337||317|válido|
|scored_0167|E2|52-6|333||312|válido|
|scored_0167|E2|64-1|356||328|válido|
|scored_0167|E2|64-3|337||317|válido|
|scored_0167|E2|64-6|333||310|válido|
|scored_0167|E3|joint|333|individual 333|333|FEASIBLE|
|scored_0167|E4|repair|333||333|completo|
|scored_0168|E1|1|389|postprocess 388, optimize_a4 388, nested_windows 388|388|válido|
|scored_0168|E1|3|364|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0168|E1|6|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0168|E2|52-1|389||361|válido|
|scored_0168|E2|52-3|364||343|válido|
|scored_0168|E2|52-6|364||337|válido|
|scored_0168|E2|64-1|389||363|válido|
|scored_0168|E2|64-3|364||343|válido|
|scored_0168|E2|64-6|364||341|válido|
|scored_0168|E3|joint|364|individual 363|357|FEASIBLE|
|scored_0168|E4|repair|364||364|completo|
|scored_0169|E1|1|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0169|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0169|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0169|E2|52-1|140||131|válido|
|scored_0169|E2|52-3|140||131|válido|
|scored_0169|E2|52-6|139||132|válido|
|scored_0169|E2|64-1|140||133|válido|
|scored_0169|E2|64-3|140||133|válido|
|scored_0169|E2|64-6|139||133|válido|
|scored_0169|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0169|E4|repair|139||136|completo|
|scored_0170|E1|1|320|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0170|E1|3|309|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0170|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0170|E2|52-1|320||301|válido|
|scored_0170|E2|52-3|309||284|válido|
|scored_0170|E2|52-6|298||285|válido|
|scored_0170|E2|64-1|320||298|válido|
|scored_0170|E2|64-3|309||283|válido|
|scored_0170|E2|64-6|298||283|válido|
|scored_0170|E3|joint|298|individual 298|296|OPTIMAL|
|scored_0170|E4|repair|298||297|completo|
|scored_0171|E1|1|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0171|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0171|E1|6|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0171|E2|52-1|175||164|válido|
|scored_0171|E2|52-3|173||159|válido|
|scored_0171|E2|52-6|170||164|válido|
|scored_0171|E2|64-1|175||164|válido|
|scored_0171|E2|64-3|173||161|válido|
|scored_0171|E2|64-6|170||159|válido|
|scored_0171|E3|joint|170|individual 170|170|OPTIMAL|
|scored_0171|E4|repair|170||170|completo|
|scored_0172|E1|1|237|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0172|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0172|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0172|E2|52-1|237||214|válido|
|scored_0172|E2|52-3|223||206|válido|
|scored_0172|E2|52-6|219||204|válido|
|scored_0172|E2|64-1|237||214|válido|
|scored_0172|E2|64-3|223||206|válido|
|scored_0172|E2|64-6|219||206|válido|
|scored_0172|E3|joint|219|individual 219|219|OPTIMAL|
|scored_0172|E4|repair|219||219|completo|
|scored_0173|E1|1|267|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0173|E1|3|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0173|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0173|E2|52-1|267||241|válido|
|scored_0173|E2|52-3|262||235|válido|
|scored_0173|E2|52-6|254||236|válido|
|scored_0173|E2|64-1|267||242|válido|
|scored_0173|E2|64-3|262||234|válido|
|scored_0173|E2|64-6|254||236|válido|
|scored_0173|E3|joint|254|individual 254|254|OPTIMAL|
|scored_0173|E4|repair|254||254|completo|
|scored_0174|E1|1|190|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0174|E1|3|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0174|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0174|E2|52-1|190||180|válido|
|scored_0174|E2|52-3|187||178|válido|
|scored_0174|E2|52-6|183||175|válido|
|scored_0174|E2|64-1|190||180|válido|
|scored_0174|E2|64-3|187||178|válido|
|scored_0174|E2|64-6|183||177|válido|
|scored_0174|E3|joint|183|individual 183|183|OPTIMAL|
|scored_0174|E4|repair|183||181|completo|
|scored_0175|E1|1|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0175|E1|3|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0175|E1|6|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0175|E2|52-1|136||133|válido|
|scored_0175|E2|52-3|136||133|válido|
|scored_0175|E2|52-6|136||133|válido|
|scored_0175|E2|64-1|136||133|válido|
|scored_0175|E2|64-3|136||133|válido|
|scored_0175|E2|64-6|136||133|válido|
|scored_0175|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0175|E4|repair|136||133|completo|
|scored_0176|E1|1|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0176|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0176|E1|6|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0176|E2|52-1|152||151|válido|
|scored_0176|E2|52-3|153||144|válido|
|scored_0176|E2|52-6|155||144|válido|
|scored_0176|E2|64-1|152||151|válido|
|scored_0176|E2|64-3|153||144|válido|
|scored_0176|E2|64-6|155||144|válido|
|scored_0176|E3|joint|152|individual 151|150|OPTIMAL|
|scored_0176|E4|repair|152||152|completo|
|scored_0177|E1|1|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0177|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0177|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0177|E2|52-1|189||181|válido|
|scored_0177|E2|52-3|188||175|válido|
|scored_0177|E2|52-6|187||171|válido|
|scored_0177|E2|64-1|189||181|válido|
|scored_0177|E2|64-3|188||176|válido|
|scored_0177|E2|64-6|187||172|válido|
|scored_0177|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0177|E4|repair|187||186|completo|
|scored_0178|E1|1|145|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0178|E1|3|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0178|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0178|E2|52-1|145||138|válido|
|scored_0178|E2|52-3|146||137|válido|
|scored_0178|E2|52-6|143||136|válido|
|scored_0178|E2|64-1|145||136|válido|
|scored_0178|E2|64-3|146||136|válido|
|scored_0178|E2|64-6|143||136|válido|
|scored_0178|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0178|E4|repair|143||143|completo|
|scored_0179|E1|1|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0179|E1|3|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0179|E1|6|280|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0179|E2|52-1|294||270|válido|
|scored_0179|E2|52-3|292||270|válido|
|scored_0179|E2|52-6|280||263|válido|
|scored_0179|E2|64-1|294||276|válido|
|scored_0179|E2|64-3|292||270|válido|
|scored_0179|E2|64-6|280||262|válido|
|scored_0179|E3|joint|280|individual 280|278|OPTIMAL|
|scored_0179|E4|repair|280||280|completo|
|scored_0180|E1|1|114|postprocess 114, optimize_a4 113, nested_windows 113|113|válido|
|scored_0180|E1|3|114|postprocess 114, optimize_a4 113, nested_windows 113|113|válido|
|scored_0180|E1|6|114|postprocess 114, optimize_a4 113, nested_windows 113|113|válido|
|scored_0180|E2|52-1|114||113|válido|
|scored_0180|E2|52-3|114||113|válido|
|scored_0180|E2|52-6|114||113|válido|
|scored_0180|E2|64-1|114||114|válido|
|scored_0180|E2|64-3|114||114|válido|
|scored_0180|E2|64-6|114||114|válido|
|scored_0180|E3|joint|114|individual 114|114|OPTIMAL|
|scored_0180|E4|repair|114||112|completo|
|scored_0181|E1|1|40|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0181|E1|3|40|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0181|E1|6|40|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0181|E2|52-1|40||40|válido|
|scored_0181|E2|52-3|40||40|válido|
|scored_0181|E2|52-6|40||40|válido|
|scored_0181|E2|64-1|40||40|válido|
|scored_0181|E2|64-3|40||40|válido|
|scored_0181|E2|64-6|40||40|válido|
|scored_0181|E3|joint|40|individual 40|40|OPTIMAL|
|scored_0181|E4|repair|40||40|completo|
|scored_0182|E1|1|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0182|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0182|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0182|E2|52-1|225||208|válido|
|scored_0182|E2|52-3|218||207|válido|
|scored_0182|E2|52-6|217||204|válido|
|scored_0182|E2|64-1|225||208|válido|
|scored_0182|E2|64-3|218||208|válido|
|scored_0182|E2|64-6|217||204|válido|
|scored_0182|E3|joint|217|individual 216|216|OPTIMAL|
|scored_0182|E4|repair|217||217|completo|
|scored_0183|E1|1|409|postprocess 409, optimize_a4 409, nested_windows 409|409|válido|
|scored_0183|E1|3|393|postprocess 393, optimize_a4 393, nested_windows 393|393|válido|
|scored_0183|E1|6|379|postprocess 378, optimize_a4 378, nested_windows 378|378|válido|
|scored_0183|E2|52-1|409||373|válido|
|scored_0183|E2|52-3|393||358|válido|
|scored_0183|E2|52-6|379||352|válido|
|scored_0183|E2|64-1|409||375|válido|
|scored_0183|E2|64-3|393||359|válido|
|scored_0183|E2|64-6|379||354|válido|
|scored_0183|E3|joint|379|individual 379|377|FEASIBLE|
|scored_0183|E4|repair|379||379|completo|
|scored_0184|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0184|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0184|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0184|E2|52-1|190||181|válido|
|scored_0184|E2|52-3|185||173|válido|
|scored_0184|E2|52-6|182||175|válido|
|scored_0184|E2|64-1|190||182|válido|
|scored_0184|E2|64-3|185||175|válido|
|scored_0184|E2|64-6|182||178|válido|
|scored_0184|E3|joint|182|individual 182|180|OPTIMAL|
|scored_0184|E4|repair|182||182|completo|
|scored_0185|E1|1|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0185|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0185|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0185|E2|52-1|246||224|válido|
|scored_0185|E2|52-3|238||233|válido|
|scored_0185|E2|52-6|233||226|válido|
|scored_0185|E2|64-1|246||229|válido|
|scored_0185|E2|64-3|238||233|válido|
|scored_0185|E2|64-6|233||226|válido|
|scored_0185|E3|joint|233|individual 231|231|OPTIMAL|
|scored_0185|E4|repair|233||232|completo|
|scored_0186|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0186|E1|3|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0186|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0186|E2|52-1|190||179|válido|
|scored_0186|E2|52-3|187||174|válido|
|scored_0186|E2|52-6|181||168|válido|
|scored_0186|E2|64-1|190||182|válido|
|scored_0186|E2|64-3|187||175|válido|
|scored_0186|E2|64-6|181||170|válido|
|scored_0186|E3|joint|181|individual 181|181|OPTIMAL|
|scored_0186|E4|repair|181||179|completo|
|scored_0187|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0187|E1|3|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0187|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0187|E2|52-1|123||122|válido|
|scored_0187|E2|52-3|123||122|válido|
|scored_0187|E2|52-6|124||122|válido|
|scored_0187|E2|64-1|123||121|válido|
|scored_0187|E2|64-3|123||121|válido|
|scored_0187|E2|64-6|124||121|válido|
|scored_0187|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0187|E4|repair|123||121|completo|
|scored_0188|E1|1|127|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0188|E1|3|127|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0188|E1|6|126|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0188|E2|52-1|127||124|válido|
|scored_0188|E2|52-3|127||124|válido|
|scored_0188|E2|52-6|126||124|válido|
|scored_0188|E2|64-1|127||125|válido|
|scored_0188|E2|64-3|127||125|válido|
|scored_0188|E2|64-6|126||125|válido|
|scored_0188|E3|joint|126|individual 126|126|OPTIMAL|
|scored_0188|E4|repair|126||124|completo|
|scored_0189|E1|1|225|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0189|E1|3|205|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0189|E1|6|205|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0189|E2|52-1|225||205|válido|
|scored_0189|E2|52-3|205||201|válido|
|scored_0189|E2|52-6|205||197|válido|
|scored_0189|E2|64-1|225||205|válido|
|scored_0189|E2|64-3|205||204|válido|
|scored_0189|E2|64-6|205||197|válido|
|scored_0189|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0189|E4|repair|205||205|completo|
|scored_0190|E1|1|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0190|E1|3|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0190|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0190|E2|52-1|171||164|válido|
|scored_0190|E2|52-3|170||164|válido|
|scored_0190|E2|52-6|166||156|válido|
|scored_0190|E2|64-1|171||162|válido|
|scored_0190|E2|64-3|170||162|válido|
|scored_0190|E2|64-6|166||154|válido|
|scored_0190|E3|joint|166|individual 166|165|OPTIMAL|
|scored_0190|E4|repair|166||166|completo|
|scored_0191|E1|1|181|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0191|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0191|E1|6|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0191|E2|52-1|181||176|válido|
|scored_0191|E2|52-3|171||165|válido|
|scored_0191|E2|52-6|169||161|válido|
|scored_0191|E2|64-1|181||178|válido|
|scored_0191|E2|64-3|171||165|válido|
|scored_0191|E2|64-6|169||160|válido|
|scored_0191|E3|joint|169|individual 169|167|OPTIMAL|
|scored_0191|E4|repair|169||168|completo|
|scored_0192|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0192|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0192|E1|6|200|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0192|E2|52-1|205||193|válido|
|scored_0192|E2|52-3|202||188|válido|
|scored_0192|E2|52-6|200||182|válido|
|scored_0192|E2|64-1|205||191|válido|
|scored_0192|E2|64-3|202||188|válido|
|scored_0192|E2|64-6|200||189|válido|
|scored_0192|E3|joint|200|individual 200|198|OPTIMAL|
|scored_0192|E4|repair|200||198|completo|
|scored_0193|E1|1|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0193|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0193|E1|6|158|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0193|E2|52-1|162||155|válido|
|scored_0193|E2|52-3|159||155|válido|
|scored_0193|E2|52-6|158||153|válido|
|scored_0193|E2|64-1|162||154|válido|
|scored_0193|E2|64-3|159||154|válido|
|scored_0193|E2|64-6|158||154|válido|
|scored_0193|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0193|E4|repair|158||157|completo|
|scored_0194|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0194|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0194|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0194|E2|52-1|210||196|válido|
|scored_0194|E2|52-3|206||189|válido|
|scored_0194|E2|52-6|204||190|válido|
|scored_0194|E2|64-1|210||194|válido|
|scored_0194|E2|64-3|206||188|válido|
|scored_0194|E2|64-6|204||190|válido|
|scored_0194|E3|joint|204|individual 203|202|OPTIMAL|
|scored_0194|E4|repair|204||204|completo|
|scored_0195|E1|1|172|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0195|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0195|E1|6|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0195|E2|52-1|172||156|válido|
|scored_0195|E2|52-3|169||156|válido|
|scored_0195|E2|52-6|167||156|válido|
|scored_0195|E2|64-1|172||157|válido|
|scored_0195|E2|64-3|169||155|válido|
|scored_0195|E2|64-6|167||155|válido|
|scored_0195|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0195|E4|repair|167||166|completo|
|scored_0196|E1|1|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0196|E1|3|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0196|E1|6|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0196|E2|52-1|41||41|válido|
|scored_0196|E2|52-3|41||41|válido|
|scored_0196|E2|52-6|41||41|válido|
|scored_0196|E2|64-1|41||41|válido|
|scored_0196|E2|64-3|41||41|válido|
|scored_0196|E2|64-6|41||41|válido|
|scored_0196|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0196|E4|repair|41||40|completo|
|scored_0197|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0197|E1|3|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0197|E1|6|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0197|E2|52-1|165||158|válido|
|scored_0197|E2|52-3|162||158|válido|
|scored_0197|E2|52-6|160||155|válido|
|scored_0197|E2|64-1|165||157|válido|
|scored_0197|E2|64-3|162||157|válido|
|scored_0197|E2|64-6|160||155|válido|
|scored_0197|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0197|E4|repair|160||159|completo|
|scored_0198|E1|1|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0198|E1|3|292|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0198|E1|6|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0198|E2|52-1|305||281|válido|
|scored_0198|E2|52-3|292||277|válido|
|scored_0198|E2|52-6|290||271|válido|
|scored_0198|E2|64-1|305||283|válido|
|scored_0198|E2|64-3|292||276|válido|
|scored_0198|E2|64-6|290||269|válido|
|scored_0198|E3|joint|290|individual 288|287|OPTIMAL|
|scored_0198|E4|repair|290||290|completo|
|scored_0199|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0199|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0199|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0199|E2|52-1|125||117|válido|
|scored_0199|E2|52-3|125||117|válido|
|scored_0199|E2|52-6|124||117|válido|
|scored_0199|E2|64-1|125||117|válido|
|scored_0199|E2|64-3|125||117|válido|
|scored_0199|E2|64-6|124||117|válido|
|scored_0199|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0199|E4|repair|124||124|completo|
|scored_0200|E1|1|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0200|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0200|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0200|E2|52-1|201||179|válido|
|scored_0200|E2|52-3|194||178|válido|
|scored_0200|E2|52-6|185||173|válido|
|scored_0200|E2|64-1|201||177|válido|
|scored_0200|E2|64-3|194||175|válido|
|scored_0200|E2|64-6|185||173|válido|
|scored_0200|E3|joint|185|individual 185|185|OPTIMAL|
|scored_0200|E4|repair|185||185|completo|
|scored_0201|E1|1|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0201|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0201|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0201|E2|52-1|208||198|válido|
|scored_0201|E2|52-3|207||197|válido|
|scored_0201|E2|52-6|203||195|válido|
|scored_0201|E2|64-1|208||200|válido|
|scored_0201|E2|64-3|207||195|válido|
|scored_0201|E2|64-6|203||194|válido|
|scored_0201|E3|joint|203|individual 203|200|OPTIMAL|
|scored_0201|E4|repair|203||203|completo|
|scored_0202|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0202|E1|3|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0202|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0202|E2|52-1|165||152|válido|
|scored_0202|E2|52-3|164||149|válido|
|scored_0202|E2|52-6|161||149|válido|
|scored_0202|E2|64-1|165||151|válido|
|scored_0202|E2|64-3|164||149|válido|
|scored_0202|E2|64-6|161||148|válido|
|scored_0202|E3|joint|161|individual 161|161|OPTIMAL|
|scored_0202|E4|repair|161||159|completo|
|scored_0203|E1|1|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0203|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0203|E1|6|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0203|E2|52-1|315||290|válido|
|scored_0203|E2|52-3|304||277|válido|
|scored_0203|E2|52-6|305||275|válido|
|scored_0203|E2|64-1|315||293|válido|
|scored_0203|E2|64-3|304||275|válido|
|scored_0203|E2|64-6|305||277|válido|
|scored_0203|E3|joint|304|individual 303|302|OPTIMAL|
|scored_0203|E4|repair|304||303|completo|
|scored_0204|E1|1|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0204|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0204|E1|6|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0204|E2|52-1|308||283|válido|
|scored_0204|E2|52-3|304||276|válido|
|scored_0204|E2|52-6|290||273|válido|
|scored_0204|E2|64-1|308||289|válido|
|scored_0204|E2|64-3|304||276|válido|
|scored_0204|E2|64-6|290||271|válido|
|scored_0204|E3|joint|290|individual 289|288|OPTIMAL|
|scored_0204|E4|repair|290||290|completo|
|scored_0205|E1|1|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0205|E1|3|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0205|E1|6|178|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0205|E2|52-1|181||175|válido|
|scored_0205|E2|52-3|179||174|válido|
|scored_0205|E2|52-6|178||174|válido|
|scored_0205|E2|64-1|181||172|válido|
|scored_0205|E2|64-3|179||166|válido|
|scored_0205|E2|64-6|178||168|válido|
|scored_0205|E3|joint|178|individual 177|176|OPTIMAL|
|scored_0205|E4|repair|178||178|completo|
|scored_0206|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0206|E1|3|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0206|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0206|E2|52-1|216||205|válido|
|scored_0206|E2|52-3|211||200|válido|
|scored_0206|E2|52-6|207||199|válido|
|scored_0206|E2|64-1|216||205|válido|
|scored_0206|E2|64-3|211||198|válido|
|scored_0206|E2|64-6|207||198|válido|
|scored_0206|E3|joint|207|individual 207|206|OPTIMAL|
|scored_0206|E4|repair|207||207|completo|
|scored_0207|E1|1|320|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0207|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0207|E1|6|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0207|E2|52-1|320||290|válido|
|scored_0207|E2|52-3|303||280|válido|
|scored_0207|E2|52-6|301||276|válido|
|scored_0207|E2|64-1|320||293|válido|
|scored_0207|E2|64-3|303||280|válido|
|scored_0207|E2|64-6|301||278|válido|
|scored_0207|E3|joint|301|individual 300|299|OPTIMAL|
|scored_0207|E4|repair|301||301|completo|
|scored_0208|E1|1|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0208|E1|3|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0208|E1|6|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0208|E2|52-1|326||295|válido|
|scored_0208|E2|52-3|312||288|válido|
|scored_0208|E2|52-6|299||285|válido|
|scored_0208|E2|64-1|326||300|válido|
|scored_0208|E2|64-3|312||289|válido|
|scored_0208|E2|64-6|299||286|válido|
|scored_0208|E3|joint|299|individual 297|297|OPTIMAL|
|scored_0208|E4|repair|299||299|completo|
|scored_0209|E1|1|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0209|E1|3|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0209|E1|6|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0209|E2|52-1|312||288|válido|
|scored_0209|E2|52-3|300||281|válido|
|scored_0209|E2|52-6|297||281|válido|
|scored_0209|E2|64-1|312||290|válido|
|scored_0209|E2|64-3|300||332|válido|
|scored_0209|E2|64-6|297||285|válido|
|scored_0209|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0209|E4|repair|297||297|completo|
|scored_0210|E1|1|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0210|E1|3|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0210|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0210|E2|52-1|123||123|válido|
|scored_0210|E2|52-3|123||123|válido|
|scored_0210|E2|52-6|123||123|válido|
|scored_0210|E2|64-1|123||122|válido|
|scored_0210|E2|64-3|123||122|válido|
|scored_0210|E2|64-6|123||122|válido|
|scored_0210|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0210|E4|repair|123||121|completo|
|scored_0211|E1|1|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0211|E1|3|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0211|E1|6|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0211|E2|52-1|54||54|válido|
|scored_0211|E2|52-3|54||54|válido|
|scored_0211|E2|52-6|54||54|válido|
|scored_0211|E2|64-1|54||54|válido|
|scored_0211|E2|64-3|54||54|válido|
|scored_0211|E2|64-6|54||54|válido|
|scored_0211|E3|joint|54|individual 54|54|OPTIMAL|
|scored_0211|E4|repair|54||53|completo|
|scored_0212|E1|1|30|postprocess 28, optimize_a4 28, nested_windows 28|28|válido|
|scored_0212|E1|3|30|postprocess 28, optimize_a4 28, nested_windows 28|28|válido|
|scored_0212|E1|6|30|postprocess 28, optimize_a4 28, nested_windows 28|28|válido|
|scored_0212|E2|52-1|30||30|válido|
|scored_0212|E2|52-3|30||30|válido|
|scored_0212|E2|52-6|30||30|válido|
|scored_0212|E2|64-1|30||30|válido|
|scored_0212|E2|64-3|30||30|válido|
|scored_0212|E2|64-6|30||30|válido|
|scored_0212|E3|joint|26|individual 26|26|OPTIMAL|
|scored_0212|E4|repair|26||25|completo|
|scored_0213|E1|1|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0213|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0213|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0213|E2|52-1|152||149|válido|
|scored_0213|E2|52-3|151||148|válido|
|scored_0213|E2|52-6|151||148|válido|
|scored_0213|E2|64-1|152||150|válido|
|scored_0213|E2|64-3|151||149|válido|
|scored_0213|E2|64-6|151||149|válido|
|scored_0213|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0213|E4|repair|151||146|completo|
|scored_0214|E1|1|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0214|E1|3|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0214|E1|6|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0214|E2|52-1|243||227|válido|
|scored_0214|E2|52-3|237||213|válido|
|scored_0214|E2|52-6|236||215|válido|
|scored_0214|E2|64-1|243||228|válido|
|scored_0214|E2|64-3|237||216|válido|
|scored_0214|E2|64-6|236||215|válido|
|scored_0214|E3|joint|236|individual 236|233|OPTIMAL|
|scored_0214|E4|repair|236||236|completo|
|scored_0215|E1|1|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0215|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0215|E1|6|215|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0215|E2|52-1|225||207|válido|
|scored_0215|E2|52-3|215||208|válido|
|scored_0215|E2|52-6|215||203|válido|
|scored_0215|E2|64-1|225||207|válido|
|scored_0215|E2|64-3|215||209|válido|
|scored_0215|E2|64-6|215||203|válido|
|scored_0215|E3|joint|215|individual 215|214|OPTIMAL|
|scored_0215|E4|repair|215||215|completo|
|scored_0216|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0216|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0216|E1|6|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0216|E2|52-1|211||202|válido|
|scored_0216|E2|52-3|205||191|válido|
|scored_0216|E2|52-6|202||190|válido|
|scored_0216|E2|64-1|211||204|válido|
|scored_0216|E2|64-3|205||192|válido|
|scored_0216|E2|64-6|202||189|válido|
|scored_0216|E3|joint|202|individual 202|201|OPTIMAL|
|scored_0216|E4|repair|202||202|completo|
|scored_0217|E1|1|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0217|E1|3|324|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0217|E1|6|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0217|E2|52-1|364||318|válido|
|scored_0217|E2|52-3|324||303|válido|
|scored_0217|E2|52-6|315||301|válido|
|scored_0217|E2|64-1|364||320|válido|
|scored_0217|E2|64-3|324||303|válido|
|scored_0217|E2|64-6|315||306|válido|
|scored_0217|E3|joint|315|individual 315|314|OPTIMAL|
|scored_0217|E4|repair|315||315|completo|
|scored_0218|E1|1|412|postprocess 411, optimize_a4 411, nested_windows 411|411|válido|
|scored_0218|E1|3|392|postprocess 392, optimize_a4 392, nested_windows 392|392|válido|
|scored_0218|E1|6|380|postprocess 380, optimize_a4 380, nested_windows 380|380|válido|
|scored_0218|E2|52-1|412||371|válido|
|scored_0218|E2|52-3|392||353|válido|
|scored_0218|E2|52-6|380||348|válido|
|scored_0218|E2|64-1|412||374|válido|
|scored_0218|E2|64-3|392||361|válido|
|scored_0218|E2|64-6|380||352|válido|
|scored_0218|E3|joint|380|individual 380|379|FEASIBLE|
|scored_0218|E4|repair|380||380|completo|
|scored_0219|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0219|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0219|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0219|E2|52-1|167||156|válido|
|scored_0219|E2|52-3|165||154|válido|
|scored_0219|E2|52-6|162||154|válido|
|scored_0219|E2|64-1|167||157|válido|
|scored_0219|E2|64-3|165||151|válido|
|scored_0219|E2|64-6|162||151|válido|
|scored_0219|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0219|E4|repair|162||158|completo|
|scored_0220|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0220|E1|3|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0220|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0220|E2|52-1|145||141|válido|
|scored_0220|E2|52-3|143||141|válido|
|scored_0220|E2|52-6|143||141|válido|
|scored_0220|E2|64-1|145||141|válido|
|scored_0220|E2|64-3|143||141|válido|
|scored_0220|E2|64-6|143||140|válido|
|scored_0220|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0220|E4|repair|143||142|completo|
|scored_0221|E1|1|51|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0221|E1|3|51|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0221|E1|6|51|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0221|E2|52-1|51||51|válido|
|scored_0221|E2|52-3|51||51|válido|
|scored_0221|E2|52-6|51||51|válido|
|scored_0221|E2|64-1|51||51|válido|
|scored_0221|E2|64-3|51||51|válido|
|scored_0221|E2|64-6|51||51|válido|
|scored_0221|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0221|E4|repair|51||51|completo|
|scored_0222|E1|1|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0222|E1|3|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0222|E1|6|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0222|E2|52-1|51||51|válido|
|scored_0222|E2|52-3|51||51|válido|
|scored_0222|E2|52-6|51||51|válido|
|scored_0222|E2|64-1|51||51|válido|
|scored_0222|E2|64-3|51||51|válido|
|scored_0222|E2|64-6|51||51|válido|
|scored_0222|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0222|E4|repair|51||49|completo|
|scored_0223|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0223|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0223|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0223|E2|52-1|303||272|válido|
|scored_0223|E2|52-3|290||276|válido|
|scored_0223|E2|52-6|289||263|válido|
|scored_0223|E2|64-1|303||279|válido|
|scored_0223|E2|64-3|290||277|válido|
|scored_0223|E2|64-6|289||269|válido|
|scored_0223|E3|joint|289|individual 289|284|OPTIMAL|
|scored_0223|E4|repair|289||289|completo|
|scored_0224|E1|1|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0224|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0224|E1|6|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0224|E2|52-1|252||232|válido|
|scored_0224|E2|52-3|242||226|válido|
|scored_0224|E2|52-6|239||225|válido|
|scored_0224|E2|64-1|252||232|válido|
|scored_0224|E2|64-3|242||227|válido|
|scored_0224|E2|64-6|239||226|válido|
|scored_0224|E3|joint|239|individual 239|239|OPTIMAL|
|scored_0224|E4|repair|239||239|completo|
|scored_0225|E1|1|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0225|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0225|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0225|E2|52-1|143||132|válido|
|scored_0225|E2|52-3|138||132|válido|
|scored_0225|E2|52-6|137||132|válido|
|scored_0225|E2|64-1|143||132|válido|
|scored_0225|E2|64-3|138||132|válido|
|scored_0225|E2|64-6|137||132|válido|
|scored_0225|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0225|E4|repair|137||133|completo|
|scored_0226|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0226|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0226|E1|6|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0226|E2|52-1|167||159|válido|
|scored_0226|E2|52-3|166||154|válido|
|scored_0226|E2|52-6|164||154|válido|
|scored_0226|E2|64-1|167||161|válido|
|scored_0226|E2|64-3|166||154|válido|
|scored_0226|E2|64-6|164||154|válido|
|scored_0226|E3|joint|164|individual 162|161|OPTIMAL|
|scored_0226|E4|repair|164||163|completo|
|scored_0227|E1|1|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0227|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0227|E1|6|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0227|E2|52-1|212||196|válido|
|scored_0227|E2|52-3|206||196|válido|
|scored_0227|E2|52-6|205||189|válido|
|scored_0227|E2|64-1|212||203|válido|
|scored_0227|E2|64-3|206||196|válido|
|scored_0227|E2|64-6|205||196|válido|
|scored_0227|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0227|E4|repair|205||203|completo|
|scored_0228|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0228|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0228|E1|6|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0228|E2|52-1|167||161|válido|
|scored_0228|E2|52-3|165||150|válido|
|scored_0228|E2|52-6|164||146|válido|
|scored_0228|E2|64-1|167||161|válido|
|scored_0228|E2|64-3|165||152|válido|
|scored_0228|E2|64-6|164||151|válido|
|scored_0228|E3|joint|164|individual 164|164|OPTIMAL|
|scored_0228|E4|repair|164||161|completo|
|scored_0229|E1|1|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0229|E1|3|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0229|E1|6|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0229|E2|52-1|193||182|válido|
|scored_0229|E2|52-3|189||182|válido|
|scored_0229|E2|52-6|184||182|válido|
|scored_0229|E2|64-1|193||183|válido|
|scored_0229|E2|64-3|189||180|válido|
|scored_0229|E2|64-6|184||180|válido|
|scored_0229|E3|joint|184|individual 184|184|OPTIMAL|
|scored_0229|E4|repair|184||184|completo|
|scored_0230|E1|1|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0230|E1|3|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0230|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0230|E2|52-1|299||277|válido|
|scored_0230|E2|52-3|292||275|válido|
|scored_0230|E2|52-6|282||266|válido|
|scored_0230|E2|64-1|299||276|válido|
|scored_0230|E2|64-3|292||272|válido|
|scored_0230|E2|64-6|282||269|válido|
|scored_0230|E3|joint|282|individual 282|281|FEASIBLE|
|scored_0230|E4|repair|282||282|completo|
|scored_0231|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0231|E1|3|236|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0231|E1|6|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0231|E2|52-1|248||234|válido|
|scored_0231|E2|52-3|236||226|válido|
|scored_0231|E2|52-6|231||221|válido|
|scored_0231|E2|64-1|248||234|válido|
|scored_0231|E2|64-3|236||228|válido|
|scored_0231|E2|64-6|231||226|válido|
|scored_0231|E3|joint|231|individual 231|229|OPTIMAL|
|scored_0231|E4|repair|231||231|completo|
|scored_0232|E1|1|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0232|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0232|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0232|E2|52-1|224||212|válido|
|scored_0232|E2|52-3|218||207|válido|
|scored_0232|E2|52-6|219||203|válido|
|scored_0232|E2|64-1|224||210|válido|
|scored_0232|E2|64-3|218||208|válido|
|scored_0232|E2|64-6|219||203|válido|
|scored_0232|E3|joint|218|individual 218|216|OPTIMAL|
|scored_0232|E4|repair|218||218|completo|
|scored_0233|E1|1|52|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0233|E1|3|52|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0233|E1|6|52|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0233|E2|52-1|52||52|válido|
|scored_0233|E2|52-3|52||52|válido|
|scored_0233|E2|52-6|52||52|válido|
|scored_0233|E2|64-1|52||52|válido|
|scored_0233|E2|64-3|52||52|válido|
|scored_0233|E2|64-6|52||52|válido|
|scored_0233|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0233|E4|repair|52||52|completo|
|scored_0234|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0234|E1|3|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0234|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0234|E2|52-1|247||235|válido|
|scored_0234|E2|52-3|245||230|válido|
|scored_0234|E2|52-6|244||227|válido|
|scored_0234|E2|64-1|247||236|válido|
|scored_0234|E2|64-3|245||230|válido|
|scored_0234|E2|64-6|244||229|válido|
|scored_0234|E3|joint|244|individual 244|244|OPTIMAL|
|scored_0234|E4|repair|244||244|completo|
|scored_0235|E1|1|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0235|E1|3|287|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0235|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0235|E2|52-1|297||273|válido|
|scored_0235|E2|52-3|287||266|válido|
|scored_0235|E2|52-6|282||264|válido|
|scored_0235|E2|64-1|297||272|válido|
|scored_0235|E2|64-3|287||267|válido|
|scored_0235|E2|64-6|282||264|válido|
|scored_0235|E3|joint|282|individual 281|280|OPTIMAL|
|scored_0235|E4|repair|282||282|completo|
|scored_0236|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0236|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0236|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0236|E2|52-1|145||137|válido|
|scored_0236|E2|52-3|144||137|válido|
|scored_0236|E2|52-6|144||137|válido|
|scored_0236|E2|64-1|145||136|válido|
|scored_0236|E2|64-3|144||136|válido|
|scored_0236|E2|64-6|144||136|válido|
|scored_0236|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0236|E4|repair|144||144|completo|
|scored_0237|E1|1|46|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0237|E1|3|46|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0237|E1|6|46|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0237|E2|52-1|46||46|válido|
|scored_0237|E2|52-3|46||46|válido|
|scored_0237|E2|52-6|46||46|válido|
|scored_0237|E2|64-1|46||46|válido|
|scored_0237|E2|64-3|46||46|válido|
|scored_0237|E2|64-6|46||46|válido|
|scored_0237|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0237|E4|repair|46||45|completo|
|scored_0238|E1|1|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0238|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0238|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0238|E2|52-1|195||184|válido|
|scored_0238|E2|52-3|195||183|válido|
|scored_0238|E2|52-6|192||183|válido|
|scored_0238|E2|64-1|195||184|válido|
|scored_0238|E2|64-3|195||183|válido|
|scored_0238|E2|64-6|192||183|válido|
|scored_0238|E3|joint|192|individual 191|191|OPTIMAL|
|scored_0238|E4|repair|192||192|completo|
|scored_0239|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0239|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0239|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0239|E2|52-1|238||222|válido|
|scored_0239|E2|52-3|235||219|válido|
|scored_0239|E2|52-6|228||212|válido|
|scored_0239|E2|64-1|238||222|válido|
|scored_0239|E2|64-3|235||220|válido|
|scored_0239|E2|64-6|228||212|válido|
|scored_0239|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0239|E4|repair|228||228|completo|
|scored_0240|E1|1|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0240|E1|3|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0240|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0240|E2|52-1|160||156|válido|
|scored_0240|E2|52-3|158||154|válido|
|scored_0240|E2|52-6|155||154|válido|
|scored_0240|E2|64-1|160||153|válido|
|scored_0240|E2|64-3|158||153|válido|
|scored_0240|E2|64-6|155||153|válido|
|scored_0240|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0240|E4|repair|155||155|completo|
|scored_0241|E1|1|43|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0241|E1|3|43|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0241|E1|6|43|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0241|E2|52-1|43||43|válido|
|scored_0241|E2|52-3|43||43|válido|
|scored_0241|E2|52-6|43||43|válido|
|scored_0241|E2|64-1|43||43|válido|
|scored_0241|E2|64-3|43||43|válido|
|scored_0241|E2|64-6|43||43|válido|
|scored_0241|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0241|E4|repair|43||40|completo|
|scored_0242|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0242|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0242|E1|6|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0242|E2|52-1|220||207|válido|
|scored_0242|E2|52-3|213||205|válido|
|scored_0242|E2|52-6|214||201|válido|
|scored_0242|E2|64-1|220||210|válido|
|scored_0242|E2|64-3|213||203|válido|
|scored_0242|E2|64-6|214||202|válido|
|scored_0242|E3|joint|213|individual 213|212|OPTIMAL|
|scored_0242|E4|repair|213||213|completo|
|scored_0243|E1|1|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0243|E1|3|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0243|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0243|E2|52-1|162||155|válido|
|scored_0243|E2|52-3|159||155|válido|
|scored_0243|E2|52-6|161||153|válido|
|scored_0243|E2|64-1|162||152|válido|
|scored_0243|E2|64-3|159||151|válido|
|scored_0243|E2|64-6|161||151|válido|
|scored_0243|E3|joint|159|individual 159|159|OPTIMAL|
|scored_0243|E4|repair|159||158|completo|
|scored_0244|E1|1|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0244|E1|3|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0244|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0244|E2|52-1|276||247|válido|
|scored_0244|E2|52-3|259||237|válido|
|scored_0244|E2|52-6|254||233|válido|
|scored_0244|E2|64-1|276||247|válido|
|scored_0244|E2|64-3|259||237|válido|
|scored_0244|E2|64-6|254||234|válido|
|scored_0244|E3|joint|254|individual 253|253|OPTIMAL|
|scored_0244|E4|repair|254||253|completo|
|scored_0245|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0245|E1|3|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0245|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0245|E2|52-1|153||146|válido|
|scored_0245|E2|52-3|150||145|válido|
|scored_0245|E2|52-6|148||143|válido|
|scored_0245|E2|64-1|153||145|válido|
|scored_0245|E2|64-3|150||143|válido|
|scored_0245|E2|64-6|148||143|válido|
|scored_0245|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0245|E4|repair|148||148|completo|
|scored_0246|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0246|E1|3|177|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0246|E1|6|177|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0246|E2|52-1|177||171|válido|
|scored_0246|E2|52-3|177||167|válido|
|scored_0246|E2|52-6|177||167|válido|
|scored_0246|E2|64-1|177||169|válido|
|scored_0246|E2|64-3|177||169|válido|
|scored_0246|E2|64-6|177||168|válido|
|scored_0246|E3|joint|177|individual 177|177|OPTIMAL|
|scored_0246|E4|repair|177||177|completo|
|scored_0247|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0247|E1|3|195|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0247|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0247|E2|52-1|197||179|válido|
|scored_0247|E2|52-3|195||178|válido|
|scored_0247|E2|52-6|187||177|válido|
|scored_0247|E2|64-1|197||179|válido|
|scored_0247|E2|64-3|195||179|válido|
|scored_0247|E2|64-6|187||176|válido|
|scored_0247|E3|joint|187|individual 186|186|OPTIMAL|
|scored_0247|E4|repair|187||187|completo|
|scored_0248|E1|1|242|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0248|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0248|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0248|E2|52-1|242||226|válido|
|scored_0248|E2|52-3|235||221|válido|
|scored_0248|E2|52-6|233||217|válido|
|scored_0248|E2|64-1|242||227|válido|
|scored_0248|E2|64-3|235||220|válido|
|scored_0248|E2|64-6|233||220|válido|
|scored_0248|E3|joint|233|individual 233|231|OPTIMAL|
|scored_0248|E4|repair|233||233|completo|
|scored_0249|E1|1|373|postprocess 373, optimize_a4 373, nested_windows 373|373|válido|
|scored_0249|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0249|E1|6|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0249|E2|52-1|373||336|válido|
|scored_0249|E2|52-3|350||330|válido|
|scored_0249|E2|52-6|347||323|válido|
|scored_0249|E2|64-1|373||343|válido|
|scored_0249|E2|64-3|350||330|válido|
|scored_0249|E2|64-6|347||325|válido|
|scored_0249|E3|joint|347|individual 347|344|OPTIMAL|
|scored_0249|E4|repair|347||347|completo|
|scored_0250|E1|1|34|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0250|E1|3|34|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0250|E1|6|34|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0250|E2|52-1|34||34|válido|
|scored_0250|E2|52-3|34||34|válido|
|scored_0250|E2|52-6|34||34|válido|
|scored_0250|E2|64-1|34||34|válido|
|scored_0250|E2|64-3|34||34|válido|
|scored_0250|E2|64-6|34||34|válido|
|scored_0250|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0250|E4|repair|34||31|completo|
|scored_0251|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0251|E1|3|301|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0251|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0251|E2|52-1|319||286|válido|
|scored_0251|E2|52-3|301||274|válido|
|scored_0251|E2|52-6|297||272|válido|
|scored_0251|E2|64-1|319||286|válido|
|scored_0251|E2|64-3|301||276|válido|
|scored_0251|E2|64-6|297||267|válido|
|scored_0251|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0251|E4|repair|297||297|completo|
|scored_0252|E1|1|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0252|E1|3|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0252|E1|6|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0252|E2|52-1|90||89|válido|
|scored_0252|E2|52-3|90||89|válido|
|scored_0252|E2|52-6|90||89|válido|
|scored_0252|E2|64-1|90||89|válido|
|scored_0252|E2|64-3|90||89|válido|
|scored_0252|E2|64-6|90||89|válido|
|scored_0252|E3|joint|90|individual 90|90|OPTIMAL|
|scored_0252|E4|repair|90||90|completo|
|scored_0253|E1|1|106|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0253|E1|3|106|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0253|E1|6|106|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0253|E2|52-1|106||102|válido|
|scored_0253|E2|52-3|106||102|válido|
|scored_0253|E2|52-6|106||102|válido|
|scored_0253|E2|64-1|106||102|válido|
|scored_0253|E2|64-3|106||102|válido|
|scored_0253|E2|64-6|106||102|válido|
|scored_0253|E3|joint|106|individual 106|106|OPTIMAL|
|scored_0253|E4|repair|106||106|completo|
|scored_0254|E1|1|340|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0254|E1|3|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0254|E1|6|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0254|E2|52-1|340||300|válido|
|scored_0254|E2|52-3|318||290|válido|
|scored_0254|E2|52-6|317||286|válido|
|scored_0254|E2|64-1|340||306|válido|
|scored_0254|E2|64-3|318||295|válido|
|scored_0254|E2|64-6|317||288|válido|
|scored_0254|E3|joint|317|individual 316|306|OPTIMAL|
|scored_0254|E4|repair|317||317|completo|
|scored_0255|E1|1|387|postprocess 387, optimize_a4 387, nested_windows 387|387|válido|
|scored_0255|E1|3|380|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|scored_0255|E1|6|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0255|E2|52-1|387||368|válido|
|scored_0255|E2|52-3|380||349|válido|
|scored_0255|E2|52-6|363||346|válido|
|scored_0255|E2|64-1|387||367|válido|
|scored_0255|E2|64-3|380||352|válido|
|scored_0255|E2|64-6|363||447|válido|
|scored_0255|E3|joint|363|individual 362|361|FEASIBLE|
|scored_0255|E4|repair|363||363|completo|
|scored_0256|E1|1|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0256|E1|3|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0256|E1|6|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0256|E2|52-1|118||113|válido|
|scored_0256|E2|52-3|118||113|válido|
|scored_0256|E2|52-6|118||113|válido|
|scored_0256|E2|64-1|118||113|válido|
|scored_0256|E2|64-3|118||113|válido|
|scored_0256|E2|64-6|118||113|válido|
|scored_0256|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0256|E4|repair|118||112|completo|
|scored_0257|E1|1|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0257|E1|3|260|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0257|E1|6|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0257|E2|52-1|270||252|válido|
|scored_0257|E2|52-3|260||241|válido|
|scored_0257|E2|52-6|258||242|válido|
|scored_0257|E2|64-1|270||253|válido|
|scored_0257|E2|64-3|260||241|válido|
|scored_0257|E2|64-6|258||243|válido|
|scored_0257|E3|joint|258|individual 258|255|OPTIMAL|
|scored_0257|E4|repair|258||258|completo|
|scored_0258|E1|1|402|postprocess 402, optimize_a4 402, nested_windows 402|402|válido|
|scored_0258|E1|3|371|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|scored_0258|E1|6|366|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0258|E2|52-1|402||350|válido|
|scored_0258|E2|52-3|371||348|válido|
|scored_0258|E2|52-6|366||342|válido|
|scored_0258|E2|64-1|402||356|válido|
|scored_0258|E2|64-3|371||348|válido|
|scored_0258|E2|64-6|366||346|válido|
|scored_0258|E3|joint|366|individual 364|361|OPTIMAL|
|scored_0258|E4|repair|366||366|completo|
|scored_0259|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0259|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0259|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0259|E2|52-1|145||136|válido|
|scored_0259|E2|52-3|146||136|válido|
|scored_0259|E2|52-6|143||136|válido|
|scored_0259|E2|64-1|145||137|válido|
|scored_0259|E2|64-3|146||137|válido|
|scored_0259|E2|64-6|143||137|válido|
|scored_0259|E3|joint|143|individual 141|141|OPTIMAL|
|scored_0259|E4|repair|143||137|completo|
|scored_0260|E1|1|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0260|E1|3|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0260|E1|6|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0260|E2|52-1|122||119|válido|
|scored_0260|E2|52-3|122||119|válido|
|scored_0260|E2|52-6|122||119|válido|
|scored_0260|E2|64-1|122||121|válido|
|scored_0260|E2|64-3|122||121|válido|
|scored_0260|E2|64-6|122||121|válido|
|scored_0260|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0260|E4|repair|122||120|completo|
|scored_0261|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0261|E1|3|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0261|E1|6|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0261|E2|52-1|233||217|válido|
|scored_0261|E2|52-3|228||213|válido|
|scored_0261|E2|52-6|225||214|válido|
|scored_0261|E2|64-1|233||214|válido|
|scored_0261|E2|64-3|228||216|válido|
|scored_0261|E2|64-6|225||215|válido|
|scored_0261|E3|joint|225|individual 225|225|OPTIMAL|
|scored_0261|E4|repair|225||224|completo|
|scored_0262|E1|1|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0262|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0262|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0262|E2|52-1|307||290|válido|
|scored_0262|E2|52-3|303||279|válido|
|scored_0262|E2|52-6|295||277|válido|
|scored_0262|E2|64-1|307||292|válido|
|scored_0262|E2|64-3|303||277|válido|
|scored_0262|E2|64-6|295||279|válido|
|scored_0262|E3|joint|295|individual 295|290|OPTIMAL|
|scored_0262|E4|repair|295||295|completo|
|scored_0263|E1|1|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0263|E1|3|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0263|E1|6|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0263|E2|52-1|336||326|válido|
|scored_0263|E2|52-3|323||303|válido|
|scored_0263|E2|52-6|319||299|válido|
|scored_0263|E2|64-1|336||335|válido|
|scored_0263|E2|64-3|323||305|válido|
|scored_0263|E2|64-6|319||301|válido|
|scored_0263|E3|joint|319|individual 318|318|OPTIMAL|
|scored_0263|E4|repair|319||319|completo|
|scored_0264|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0264|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0264|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0264|E2|52-1|318||283|válido|
|scored_0264|E2|52-3|299||279|válido|
|scored_0264|E2|52-6|295||269|válido|
|scored_0264|E2|64-1|318||291|válido|
|scored_0264|E2|64-3|299||274|válido|
|scored_0264|E2|64-6|295||271|válido|
|scored_0264|E3|joint|295|individual 295|292|OPTIMAL|
|scored_0264|E4|repair|295||295|completo|
|scored_0265|E1|1|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0265|E1|3|276|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0265|E1|6|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0265|E2|52-1|281||254|válido|
|scored_0265|E2|52-3|276||247|válido|
|scored_0265|E2|52-6|263||239|válido|
|scored_0265|E2|64-1|281||256|válido|
|scored_0265|E2|64-3|276||245|válido|
|scored_0265|E2|64-6|263||234|válido|
|scored_0265|E3|joint|263|individual 263|263|OPTIMAL|
|scored_0265|E4|repair|263||263|completo|
|scored_0266|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0266|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0266|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0266|E2|52-1|151||144|válido|
|scored_0266|E2|52-3|144||144|válido|
|scored_0266|E2|52-6|147||143|válido|
|scored_0266|E2|64-1|151||146|válido|
|scored_0266|E2|64-3|144||143|válido|
|scored_0266|E2|64-6|147||143|válido|
|scored_0266|E3|joint|144|individual 143|143|OPTIMAL|
|scored_0266|E4|repair|144||142|completo|
|scored_0267|E1|1|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0267|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0267|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0267|E2|52-1|156||150|válido|
|scored_0267|E2|52-3|151||149|válido|
|scored_0267|E2|52-6|151||149|válido|
|scored_0267|E2|64-1|156||149|válido|
|scored_0267|E2|64-3|151||149|válido|
|scored_0267|E2|64-6|151||149|válido|
|scored_0267|E3|joint|151|individual 150|150|OPTIMAL|
|scored_0267|E4|repair|151||149|completo|
|scored_0268|E1|1|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0268|E1|3|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0268|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0268|E2|52-1|291||266|válido|
|scored_0268|E2|52-3|285||263|válido|
|scored_0268|E2|52-6|281||260|válido|
|scored_0268|E2|64-1|291||268|válido|
|scored_0268|E2|64-3|285||262|válido|
|scored_0268|E2|64-6|281||261|válido|
|scored_0268|E3|joint|281|individual 280|280|OPTIMAL|
|scored_0268|E4|repair|281||281|completo|
|scored_0269|E1|1|408|postprocess 408, optimize_a4 408, nested_windows 408|408|válido|
|scored_0269|E1|3|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0269|E1|6|371|postprocess 371, optimize_a4 371, nested_windows 371|371|válido|
|scored_0269|E2|52-1|408||377|válido|
|scored_0269|E2|52-3|377||351|válido|
|scored_0269|E2|52-6|371||347|válido|
|scored_0269|E2|64-1|408||383|válido|
|scored_0269|E2|64-3|377||352|válido|
|scored_0269|E2|64-6|371||356|válido|
|scored_0269|E3|joint|371|individual 371|371|FEASIBLE|
|scored_0269|E4|repair|371||371|completo|
|scored_0270|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0270|E1|3|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0270|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0270|E2|52-1|169||159|válido|
|scored_0270|E2|52-3|168||158|válido|
|scored_0270|E2|52-6|167||157|válido|
|scored_0270|E2|64-1|169||157|válido|
|scored_0270|E2|64-3|168||157|válido|
|scored_0270|E2|64-6|167||157|válido|
|scored_0270|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0270|E4|repair|167||166|completo|
|scored_0271|E1|1|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0271|E1|3|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0271|E1|6|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0271|E2|52-1|59||59|válido|
|scored_0271|E2|52-3|59||59|válido|
|scored_0271|E2|52-6|59||59|válido|
|scored_0271|E2|64-1|59||59|válido|
|scored_0271|E2|64-3|59||59|válido|
|scored_0271|E2|64-6|59||59|válido|
|scored_0271|E3|joint|59|individual 59|59|OPTIMAL|
|scored_0271|E4|repair|59||57|completo|
|scored_0272|E1|1|34|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0272|E1|3|34|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0272|E1|6|34|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0272|E2|52-1|34||34|válido|
|scored_0272|E2|52-3|34||34|válido|
|scored_0272|E2|52-6|34||34|válido|
|scored_0272|E2|64-1|34||34|válido|
|scored_0272|E2|64-3|34||34|válido|
|scored_0272|E2|64-6|34||34|válido|
|scored_0272|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0272|E4|repair|34||34|completo|
|scored_0273|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0273|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0273|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0273|E2|52-1|254||235|válido|
|scored_0273|E2|52-3|246||231|válido|
|scored_0273|E2|52-6|242||231|válido|
|scored_0273|E2|64-1|254||236|válido|
|scored_0273|E2|64-3|246||234|válido|
|scored_0273|E2|64-6|242||230|válido|
|scored_0273|E3|joint|242|individual 242|240|OPTIMAL|
|scored_0273|E4|repair|242||242|completo|
|scored_0274|E1|1|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0274|E1|3|230|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0274|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0274|E2|52-1|231||219|válido|
|scored_0274|E2|52-3|230||217|válido|
|scored_0274|E2|52-6|228||211|válido|
|scored_0274|E2|64-1|231||220|válido|
|scored_0274|E2|64-3|230||218|válido|
|scored_0274|E2|64-6|228||208|válido|
|scored_0274|E3|joint|228|individual 228|226|OPTIMAL|
|scored_0274|E4|repair|228||228|completo|
|scored_0275|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0275|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0275|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0275|E2|52-1|202||193|válido|
|scored_0275|E2|52-3|196||186|válido|
|scored_0275|E2|52-6|192||184|válido|
|scored_0275|E2|64-1|202||194|válido|
|scored_0275|E2|64-3|196||185|válido|
|scored_0275|E2|64-6|192||184|válido|
|scored_0275|E3|joint|192|individual 192|192|OPTIMAL|
|scored_0275|E4|repair|192||192|completo|
|scored_0276|E1|1|95|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0276|E1|3|95|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0276|E1|6|95|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0276|E2|52-1|95||91|válido|
|scored_0276|E2|52-3|95||91|válido|
|scored_0276|E2|52-6|95||91|válido|
|scored_0276|E2|64-1|95||91|válido|
|scored_0276|E2|64-3|95||91|válido|
|scored_0276|E2|64-6|95||91|válido|
|scored_0276|E3|joint|95|individual 95|95|OPTIMAL|
|scored_0276|E4|repair|95||95|completo|
|scored_0277|E1|1|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0277|E1|3|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0277|E1|6|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0277|E2|52-1|182||175|válido|
|scored_0277|E2|52-3|175||172|válido|
|scored_0277|E2|52-6|175||171|válido|
|scored_0277|E2|64-1|182||177|válido|
|scored_0277|E2|64-3|175||172|válido|
|scored_0277|E2|64-6|175||171|válido|
|scored_0277|E3|joint|175|individual 175|175|OPTIMAL|
|scored_0277|E4|repair|175||175|completo|
|scored_0278|E1|1|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0278|E1|3|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0278|E1|6|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0278|E2|52-1|107||104|válido|
|scored_0278|E2|52-3|107||104|válido|
|scored_0278|E2|52-6|107||104|válido|
|scored_0278|E2|64-1|107||104|válido|
|scored_0278|E2|64-3|107||104|válido|
|scored_0278|E2|64-6|107||104|válido|
|scored_0278|E3|joint|107|individual 107|107|OPTIMAL|
|scored_0278|E4|repair|107||105|completo|
|scored_0279|E1|1|366|postprocess 365, optimize_a4 365, nested_windows 365|365|válido|
|scored_0279|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0279|E1|6|344|postprocess 344, optimize_a4 344, nested_windows 344|344|válido|
|scored_0279|E2|52-1|366||340|válido|
|scored_0279|E2|52-3|350||330|válido|
|scored_0279|E2|52-6|344||320|válido|
|scored_0279|E2|64-1|366||340|válido|
|scored_0279|E2|64-3|350||330|válido|
|scored_0279|E2|64-6|344||321|válido|
|scored_0279|E3|joint|344|individual 344|341|OPTIMAL|
|scored_0279|E4|repair|344||344|completo|
|scored_0280|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0280|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0280|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0280|E2|52-1|211||194|válido|
|scored_0280|E2|52-3|202||186|válido|
|scored_0280|E2|52-6|199||190|válido|
|scored_0280|E2|64-1|211||195|válido|
|scored_0280|E2|64-3|202||186|válido|
|scored_0280|E2|64-6|199||188|válido|
|scored_0280|E3|joint|199|individual 199|197|OPTIMAL|
|scored_0280|E4|repair|199||194|completo|
|scored_0281|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0281|E1|3|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0281|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0281|E2|52-1|251||234|válido|
|scored_0281|E2|52-3|245||232|válido|
|scored_0281|E2|52-6|244||230|válido|
|scored_0281|E2|64-1|251||233|válido|
|scored_0281|E2|64-3|245||232|válido|
|scored_0281|E2|64-6|244||232|válido|
|scored_0281|E3|joint|244|individual 244|239|FEASIBLE|
|scored_0281|E4|repair|244||244|completo|
|scored_0282|E1|1|63|postprocess 62, optimize_a4 62, nested_windows 62|62|válido|
|scored_0282|E1|3|63|postprocess 62, optimize_a4 62, nested_windows 62|62|válido|
|scored_0282|E1|6|63|postprocess 62, optimize_a4 62, nested_windows 62|62|válido|
|scored_0282|E2|52-1|63||62|válido|
|scored_0282|E2|52-3|63||62|válido|
|scored_0282|E2|52-6|63||62|válido|
|scored_0282|E2|64-1|63||62|válido|
|scored_0282|E2|64-3|63||62|válido|
|scored_0282|E2|64-6|63||62|válido|
|scored_0282|E3|joint|63|individual 63|63|OPTIMAL|
|scored_0282|E4|repair|63||60|completo|
|scored_0283|E1|1|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0283|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0283|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0283|E2|52-1|305||279|válido|
|scored_0283|E2|52-3|294||270|válido|
|scored_0283|E2|52-6|282||273|válido|
|scored_0283|E2|64-1|305||279|válido|
|scored_0283|E2|64-3|294||271|válido|
|scored_0283|E2|64-6|282||274|válido|
|scored_0283|E3|joint|282|individual 282|282|OPTIMAL|
|scored_0283|E4|repair|282||282|completo|
|scored_0284|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0284|E1|3|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0284|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0284|E2|52-1|200||188|válido|
|scored_0284|E2|52-3|193||178|válido|
|scored_0284|E2|52-6|189||176|válido|
|scored_0284|E2|64-1|200||188|válido|
|scored_0284|E2|64-3|193||178|válido|
|scored_0284|E2|64-6|189||178|válido|
|scored_0284|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0284|E4|repair|189||189|completo|
|scored_0285|E1|1|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0285|E1|3|280|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0285|E1|6|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0285|E2|52-1|300||267|válido|
|scored_0285|E2|52-3|280||261|válido|
|scored_0285|E2|52-6|275||256|válido|
|scored_0285|E2|64-1|300||272|válido|
|scored_0285|E2|64-3|280||262|válido|
|scored_0285|E2|64-6|275||257|válido|
|scored_0285|E3|joint|275|individual 274|274|FEASIBLE|
|scored_0285|E4|repair|275||275|completo|
|scored_0286|E1|1|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0286|E1|3|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0286|E1|6|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0286|E2|52-1|178||159|válido|
|scored_0286|E2|52-3|177||159|válido|
|scored_0286|E2|52-6|177||162|válido|
|scored_0286|E2|64-1|178||170|válido|
|scored_0286|E2|64-3|177||165|válido|
|scored_0286|E2|64-6|177||165|válido|
|scored_0286|E3|joint|177|individual 177|175|OPTIMAL|
|scored_0286|E4|repair|177||173|completo|
|scored_0287|E1|1|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0287|E1|3|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0287|E1|6|266|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0287|E2|52-1|276||252|válido|
|scored_0287|E2|52-3|275||245|válido|
|scored_0287|E2|52-6|266||244|válido|
|scored_0287|E2|64-1|276||252|válido|
|scored_0287|E2|64-3|275||246|válido|
|scored_0287|E2|64-6|266||243|válido|
|scored_0287|E3|joint|266|individual 265|264|OPTIMAL|
|scored_0287|E4|repair|266||266|completo|
|scored_0288|E1|1|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0288|E1|3|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0288|E1|6|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0288|E2|52-1|185||175|válido|
|scored_0288|E2|52-3|181||169|válido|
|scored_0288|E2|52-6|180||170|válido|
|scored_0288|E2|64-1|185||174|válido|
|scored_0288|E2|64-3|181||168|válido|
|scored_0288|E2|64-6|180||170|válido|
|scored_0288|E3|joint|180|individual 180|180|OPTIMAL|
|scored_0288|E4|repair|180||178|completo|
|scored_0289|E1|1|27|postprocess 25, optimize_a4 25, nested_windows 25|25|válido|
|scored_0289|E1|3|27|postprocess 25, optimize_a4 25, nested_windows 25|25|válido|
|scored_0289|E1|6|27|postprocess 25, optimize_a4 25, nested_windows 25|25|válido|
|scored_0289|E2|52-1|27||27|válido|
|scored_0289|E2|52-3|27||27|válido|
|scored_0289|E2|52-6|27||27|válido|
|scored_0289|E2|64-1|27||27|válido|
|scored_0289|E2|64-3|27||27|válido|
|scored_0289|E2|64-6|27||27|válido|
|scored_0289|E3|joint|26|individual 26|26|OPTIMAL|
|scored_0289|E4|repair|26||24|completo|
|scored_0290|E1|1|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0290|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0290|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0290|E2|52-1|194||188|válido|
|scored_0290|E2|52-3|192||186|válido|
|scored_0290|E2|52-6|189||181|válido|
|scored_0290|E2|64-1|194||188|válido|
|scored_0290|E2|64-3|192||186|válido|
|scored_0290|E2|64-6|189||182|válido|
|scored_0290|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0290|E4|repair|189||189|completo|
|scored_0291|E1|1|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0291|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0291|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0291|E2|52-1|233||210|válido|
|scored_0291|E2|52-3|225||206|válido|
|scored_0291|E2|52-6|221||207|válido|
|scored_0291|E2|64-1|233||211|válido|
|scored_0291|E2|64-3|225||204|válido|
|scored_0291|E2|64-6|221||204|válido|
|scored_0291|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0291|E4|repair|221||221|completo|
|scored_0292|E1|1|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0292|E1|3|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0292|E1|6|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0292|E2|52-1|71||66|válido|
|scored_0292|E2|52-3|71||66|válido|
|scored_0292|E2|52-6|71||66|válido|
|scored_0292|E2|64-1|71||66|válido|
|scored_0292|E2|64-3|71||66|válido|
|scored_0292|E2|64-6|71||66|válido|
|scored_0292|E3|joint|71|individual 71|71|OPTIMAL|
|scored_0292|E4|repair|71||71|completo|
|scored_0293|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0293|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0293|E1|6|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0293|E2|52-1|273||254|válido|
|scored_0293|E2|52-3|270||245|válido|
|scored_0293|E2|52-6|265||245|válido|
|scored_0293|E2|64-1|273||255|válido|
|scored_0293|E2|64-3|270||247|válido|
|scored_0293|E2|64-6|265||248|válido|
|scored_0293|E3|joint|265|individual 265|262|OPTIMAL|
|scored_0293|E4|repair|265||265|completo|
|scored_0294|E1|1|221|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0294|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0294|E1|6|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0294|E2|52-1|221||203|válido|
|scored_0294|E2|52-3|213||196|válido|
|scored_0294|E2|52-6|210||195|válido|
|scored_0294|E2|64-1|221||204|válido|
|scored_0294|E2|64-3|213||195|válido|
|scored_0294|E2|64-6|210||196|válido|
|scored_0294|E3|joint|210|individual 209|209|OPTIMAL|
|scored_0294|E4|repair|210||210|completo|
|scored_0295|E1|1|176|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0295|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0295|E1|6|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0295|E2|52-1|176||168|válido|
|scored_0295|E2|52-3|174||167|válido|
|scored_0295|E2|52-6|173||163|válido|
|scored_0295|E2|64-1|176||168|válido|
|scored_0295|E2|64-3|174||167|válido|
|scored_0295|E2|64-6|173||164|válido|
|scored_0295|E3|joint|173|individual 172|172|OPTIMAL|
|scored_0295|E4|repair|173||170|completo|
|scored_0296|E1|1|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0296|E1|3|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0296|E1|6|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0296|E2|52-1|56||56|válido|
|scored_0296|E2|52-3|56||56|válido|
|scored_0296|E2|52-6|56||56|válido|
|scored_0296|E2|64-1|56||56|válido|
|scored_0296|E2|64-3|56||56|válido|
|scored_0296|E2|64-6|56||56|válido|
|scored_0296|E3|joint|56|individual 56|56|OPTIMAL|
|scored_0296|E4|repair|56||56|completo|
|scored_0297|E1|1|168|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0297|E1|3|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0297|E1|6|166|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0297|E2|52-1|168||158|válido|
|scored_0297|E2|52-3|167||155|válido|
|scored_0297|E2|52-6|166||155|válido|
|scored_0297|E2|64-1|168||158|válido|
|scored_0297|E2|64-3|167||158|válido|
|scored_0297|E2|64-6|166||156|válido|
|scored_0297|E3|joint|166|individual 165|165|OPTIMAL|
|scored_0297|E4|repair|166||166|completo|
|scored_0298|E1|1|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0298|E1|3|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0298|E1|6|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0298|E2|52-1|136||128|válido|
|scored_0298|E2|52-3|134||128|válido|
|scored_0298|E2|52-6|134||128|válido|
|scored_0298|E2|64-1|136||127|válido|
|scored_0298|E2|64-3|134||127|válido|
|scored_0298|E2|64-6|134||127|válido|
|scored_0298|E3|joint|134|individual 134|134|OPTIMAL|
|scored_0298|E4|repair|134||133|completo|
|scored_0299|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0299|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0299|E1|6|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0299|E2|52-1|119||118|válido|
|scored_0299|E2|52-3|120||118|válido|
|scored_0299|E2|52-6|119||117|válido|
|scored_0299|E2|64-1|119||118|válido|
|scored_0299|E2|64-3|120||118|válido|
|scored_0299|E2|64-6|119||117|válido|
|scored_0299|E3|joint|119|individual 119|119|OPTIMAL|
|scored_0299|E4|repair|119||118|completo|
|scored_0300|E1|1|57|postprocess 57, optimize_a4 57, nested_windows 57|57|válido|
|scored_0300|E1|3|57|postprocess 57, optimize_a4 57, nested_windows 57|57|válido|
|scored_0300|E1|6|57|postprocess 57, optimize_a4 57, nested_windows 57|57|válido|
|scored_0300|E2|52-1|57||56|válido|
|scored_0300|E2|52-3|57||56|válido|
|scored_0300|E2|52-6|57||56|válido|
|scored_0300|E2|64-1|57||56|válido|
|scored_0300|E2|64-3|57||56|válido|
|scored_0300|E2|64-6|57||56|válido|
|scored_0300|E3|joint|57|individual 57|57|OPTIMAL|
|scored_0300|E4|repair|57||56|completo|
|scored_0301|E1|1|5|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0301|E1|3|5|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0301|E1|6|5|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0301|E2|52-1|5||5|válido|
|scored_0301|E2|52-3|5||5|válido|
|scored_0301|E2|52-6|5||5|válido|
|scored_0301|E2|64-1|5||5|válido|
|scored_0301|E2|64-3|5||5|válido|
|scored_0301|E2|64-6|5||5|válido|
|scored_0301|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0301|E4|repair|4||4|completo|
|scored_0302|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0302|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0302|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0302|E2|52-1|236||206|válido|
|scored_0302|E2|52-3|226||202|válido|
|scored_0302|E2|52-6|223||207|válido|
|scored_0302|E2|64-1|236||209|válido|
|scored_0302|E2|64-3|226||205|válido|
|scored_0302|E2|64-6|223||200|válido|
|scored_0302|E3|joint|223|individual 222|218|OPTIMAL|
|scored_0302|E4|repair|223||223|completo|
|scored_0303|E1|1|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0303|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0303|E1|6|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0303|E2|52-1|285||266|válido|
|scored_0303|E2|52-3|278||260|válido|
|scored_0303|E2|52-6|265||255|válido|
|scored_0303|E2|64-1|285||268|válido|
|scored_0303|E2|64-3|278||260|válido|
|scored_0303|E2|64-6|265||256|válido|
|scored_0303|E3|joint|265|individual 265|265|OPTIMAL|
|scored_0303|E4|repair|265||265|completo|
|scored_0304|E1|1|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0304|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0304|E1|6|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0304|E2|52-1|219||199|válido|
|scored_0304|E2|52-3|213||198|válido|
|scored_0304|E2|52-6|213||197|válido|
|scored_0304|E2|64-1|219||199|válido|
|scored_0304|E2|64-3|213||198|válido|
|scored_0304|E2|64-6|213||196|válido|
|scored_0304|E3|joint|213|individual 213|212|OPTIMAL|
|scored_0304|E4|repair|213||211|completo|
|scored_0305|E1|1|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0305|E1|3|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0305|E1|6|219|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0305|E2|52-1|228||211|válido|
|scored_0305|E2|52-3|224||209|válido|
|scored_0305|E2|52-6|219||205|válido|
|scored_0305|E2|64-1|228||210|válido|
|scored_0305|E2|64-3|224||208|válido|
|scored_0305|E2|64-6|219||208|válido|
|scored_0305|E3|joint|219|individual 218|217|OPTIMAL|
|scored_0305|E4|repair|219||218|completo|
|scored_0306|E1|1|381|postprocess 380, optimize_a4 380, nested_windows 380|380|válido|
|scored_0306|E1|3|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0306|E1|6|352|postprocess 352, optimize_a4 352, nested_windows 352|352|válido|
|scored_0306|E2|52-1|381||346|válido|
|scored_0306|E2|52-3|353||332|válido|
|scored_0306|E2|52-6|352||328|válido|
|scored_0306|E2|64-1|381||353|válido|
|scored_0306|E2|64-3|353||337|válido|
|scored_0306|E2|64-6|352||331|válido|
|scored_0306|E3|joint|352|individual 351|349|FEASIBLE|
|scored_0306|E4|repair|352||352|completo|
|scored_0307|E1|1|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0307|E1|3|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0307|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0307|E2|52-1|182||163|válido|
|scored_0307|E2|52-3|180||163|válido|
|scored_0307|E2|52-6|174||165|válido|
|scored_0307|E2|64-1|182||163|válido|
|scored_0307|E2|64-3|180||163|válido|
|scored_0307|E2|64-6|174||165|válido|
|scored_0307|E3|joint|174|individual 174|174|OPTIMAL|
|scored_0307|E4|repair|174||171|completo|
|scored_0308|E1|1|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0308|E1|3|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0308|E1|6|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0308|E2|52-1|44||44|válido|
|scored_0308|E2|52-3|44||44|válido|
|scored_0308|E2|52-6|44||44|válido|
|scored_0308|E2|64-1|44||44|válido|
|scored_0308|E2|64-3|44||44|válido|
|scored_0308|E2|64-6|44||44|válido|
|scored_0308|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0308|E4|repair|44||42|completo|
|scored_0309|E1|1|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0309|E1|3|178|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0309|E1|6|177|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0309|E2|52-1|182||164|válido|
|scored_0309|E2|52-3|178||164|válido|
|scored_0309|E2|52-6|177||162|válido|
|scored_0309|E2|64-1|182||167|válido|
|scored_0309|E2|64-3|178||164|válido|
|scored_0309|E2|64-6|177||165|válido|
|scored_0309|E3|joint|177|individual 177|177|OPTIMAL|
|scored_0309|E4|repair|177||173|completo|
|scored_0310|E1|1|159|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0310|E1|3|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0310|E1|6|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0310|E2|52-1|159||149|válido|
|scored_0310|E2|52-3|157||148|válido|
|scored_0310|E2|52-6|157||146|válido|
|scored_0310|E2|64-1|159||148|válido|
|scored_0310|E2|64-3|157||146|válido|
|scored_0310|E2|64-6|157||145|válido|
|scored_0310|E3|joint|157|individual 157|157|OPTIMAL|
|scored_0310|E4|repair|157||157|completo|
|scored_0311|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0311|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0311|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0311|E2|52-1|146||137|válido|
|scored_0311|E2|52-3|145||136|válido|
|scored_0311|E2|52-6|144||136|válido|
|scored_0311|E2|64-1|146||137|válido|
|scored_0311|E2|64-3|145||137|válido|
|scored_0311|E2|64-6|144||137|válido|
|scored_0311|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0311|E4|repair|144||143|completo|
|scored_0312|E1|1|354|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0312|E1|3|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0312|E1|6|330|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0312|E2|52-1|354||330|válido|
|scored_0312|E2|52-3|330||313|válido|
|scored_0312|E2|52-6|330||307|válido|
|scored_0312|E2|64-1|354||329|válido|
|scored_0312|E2|64-3|330||311|válido|
|scored_0312|E2|64-6|330||308|válido|
|scored_0312|E3|joint|330|individual 328|324|FEASIBLE|
|scored_0312|E4|repair|330||330|completo|
|scored_0313|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0313|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0313|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0313|E2|52-1|238||222|válido|
|scored_0313|E2|52-3|229||217|válido|
|scored_0313|E2|52-6|226||212|válido|
|scored_0313|E2|64-1|238||223|válido|
|scored_0313|E2|64-3|229||217|válido|
|scored_0313|E2|64-6|226||212|válido|
|scored_0313|E3|joint|226|individual 226|225|OPTIMAL|
|scored_0313|E4|repair|226||226|completo|
|scored_0314|E1|1|279|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0314|E1|3|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0314|E1|6|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0314|E2|52-1|279||252|válido|
|scored_0314|E2|52-3|264||248|válido|
|scored_0314|E2|52-6|252||243|válido|
|scored_0314|E2|64-1|279||255|válido|
|scored_0314|E2|64-3|264||247|válido|
|scored_0314|E2|64-6|252||243|válido|
|scored_0314|E3|joint|252|individual 252|252|OPTIMAL|
|scored_0314|E4|repair|252||252|completo|
|scored_0315|E1|1|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0315|E1|3|299|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0315|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0315|E2|52-1|316||297|válido|
|scored_0315|E2|52-3|299||285|válido|
|scored_0315|E2|52-6|297||283|válido|
|scored_0315|E2|64-1|316||300|válido|
|scored_0315|E2|64-3|299||288|válido|
|scored_0315|E2|64-6|297||286|válido|
|scored_0315|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0315|E4|repair|297||297|completo|
|scored_0316|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0316|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0316|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0316|E2|52-1|145||141|válido|
|scored_0316|E2|52-3|145||141|válido|
|scored_0316|E2|52-6|144||141|válido|
|scored_0316|E2|64-1|145||142|válido|
|scored_0316|E2|64-3|145||141|válido|
|scored_0316|E2|64-6|144||141|válido|
|scored_0316|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0316|E4|repair|144||144|completo|
|scored_0317|E1|1|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0317|E1|3|237|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0317|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0317|E2|52-1|246||229|válido|
|scored_0317|E2|52-3|237||226|válido|
|scored_0317|E2|52-6|237||224|válido|
|scored_0317|E2|64-1|246||231|válido|
|scored_0317|E2|64-3|237||227|válido|
|scored_0317|E2|64-6|237||226|válido|
|scored_0317|E3|joint|237|individual 237|235|OPTIMAL|
|scored_0317|E4|repair|237||237|completo|
|scored_0318|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0318|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0318|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0318|E2|52-1|243||233|válido|
|scored_0318|E2|52-3|239||228|válido|
|scored_0318|E2|52-6|238||226|válido|
|scored_0318|E2|64-1|243||230|válido|
|scored_0318|E2|64-3|239||228|válido|
|scored_0318|E2|64-6|238||226|válido|
|scored_0318|E3|joint|238|individual 238|238|OPTIMAL|
|scored_0318|E4|repair|238||238|completo|
|scored_0319|E1|1|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0319|E1|3|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0319|E1|6|313|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0319|E2|52-1|333||293|válido|
|scored_0319|E2|52-3|318||290|válido|
|scored_0319|E2|52-6|313||282|válido|
|scored_0319|E2|64-1|333||298|válido|
|scored_0319|E2|64-3|318||290|válido|
|scored_0319|E2|64-6|313||283|válido|
|scored_0319|E3|joint|313|individual 313|309|FEASIBLE|
|scored_0319|E4|repair|313||313|completo|
|scored_0320|E1|1|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0320|E1|3|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0320|E1|6|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0320|E2|52-1|42||42|válido|
|scored_0320|E2|52-3|42||42|válido|
|scored_0320|E2|52-6|42||42|válido|
|scored_0320|E2|64-1|42||42|válido|
|scored_0320|E2|64-3|42||42|válido|
|scored_0320|E2|64-6|42||42|válido|
|scored_0320|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0320|E4|repair|42||41|completo|
|scored_0321|E1|1|400|postprocess 400, optimize_a4 400, nested_windows 400|400|válido|
|scored_0321|E1|3|373|postprocess 373, optimize_a4 373, nested_windows 373|373|válido|
|scored_0321|E1|6|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0321|E2|52-1|400||358|válido|
|scored_0321|E2|52-3|373||347|válido|
|scored_0321|E2|52-6|364||340|válido|
|scored_0321|E2|64-1|400||363|válido|
|scored_0321|E2|64-3|373||350|válido|
|scored_0321|E2|64-6|364||346|válido|
|scored_0321|E3|joint|364|individual 364|360|FEASIBLE|
|scored_0321|E4|repair|364||364|completo|
|scored_0322|E1|1|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0322|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0322|E1|6|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0322|E2|52-1|249||228|válido|
|scored_0322|E2|52-3|241||221|válido|
|scored_0322|E2|52-6|236||222|válido|
|scored_0322|E2|64-1|249||228|válido|
|scored_0322|E2|64-3|241||220|válido|
|scored_0322|E2|64-6|236||222|válido|
|scored_0322|E3|joint|236|individual 236|235|FEASIBLE|
|scored_0322|E4|repair|236||236|completo|
|scored_0323|E1|1|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0323|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0323|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0323|E2|52-1|203||191|válido|
|scored_0323|E2|52-3|199||184|válido|
|scored_0323|E2|52-6|198||184|válido|
|scored_0323|E2|64-1|203||191|válido|
|scored_0323|E2|64-3|199||188|válido|
|scored_0323|E2|64-6|198||188|válido|
|scored_0323|E3|joint|198|individual 197|197|OPTIMAL|
|scored_0323|E4|repair|198||196|completo|
|scored_0324|E1|1|256|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0324|E1|3|245|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0324|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0324|E2|52-1|256||234|válido|
|scored_0324|E2|52-3|245||226|válido|
|scored_0324|E2|52-6|245||224|válido|
|scored_0324|E2|64-1|256||234|válido|
|scored_0324|E2|64-3|245||226|válido|
|scored_0324|E2|64-6|245||223|válido|
|scored_0324|E3|joint|245|individual 245|241|FEASIBLE|
|scored_0324|E4|repair|245||245|completo|
|scored_0325|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0325|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0325|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0325|E2|52-1|168||162|válido|
|scored_0325|E2|52-3|165||161|válido|
|scored_0325|E2|52-6|163||155|válido|
|scored_0325|E2|64-1|168||163|válido|
|scored_0325|E2|64-3|165||160|válido|
|scored_0325|E2|64-6|163||156|válido|
|scored_0325|E3|joint|163|individual 162|161|OPTIMAL|
|scored_0325|E4|repair|163||162|completo|
|scored_0326|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0326|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0326|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0326|E2|52-1|202||186|válido|
|scored_0326|E2|52-3|195||184|válido|
|scored_0326|E2|52-6|193||182|válido|
|scored_0326|E2|64-1|202||187|válido|
|scored_0326|E2|64-3|195||180|válido|
|scored_0326|E2|64-6|193||180|válido|
|scored_0326|E3|joint|193|individual 193|192|OPTIMAL|
|scored_0326|E4|repair|193||192|completo|
|scored_0327|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0327|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0327|E1|6|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0327|E2|52-1|177||170|válido|
|scored_0327|E2|52-3|174||170|válido|
|scored_0327|E2|52-6|176||164|válido|
|scored_0327|E2|64-1|177||174|válido|
|scored_0327|E2|64-3|174||168|válido|
|scored_0327|E2|64-6|176||166|válido|
|scored_0327|E3|joint|174|individual 173|172|OPTIMAL|
|scored_0327|E4|repair|174||174|completo|
|scored_0328|E1|1|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0328|E1|3|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0328|E1|6|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0328|E2|52-1|329||304|válido|
|scored_0328|E2|52-3|322||288|válido|
|scored_0328|E2|52-6|312||286|válido|
|scored_0328|E2|64-1|329||305|válido|
|scored_0328|E2|64-3|322||291|válido|
|scored_0328|E2|64-6|312||288|válido|
|scored_0328|E3|joint|312|individual 312|309|OPTIMAL|
|scored_0328|E4|repair|312||312|completo|
|scored_0329|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0329|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0329|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0329|E2|52-1|169||158|válido|
|scored_0329|E2|52-3|165||158|válido|
|scored_0329|E2|52-6|163||158|válido|
|scored_0329|E2|64-1|169||158|válido|
|scored_0329|E2|64-3|165||156|válido|
|scored_0329|E2|64-6|163||156|válido|
|scored_0329|E3|joint|163|individual 163|163|OPTIMAL|
|scored_0329|E4|repair|163||163|completo|
|scored_0330|E1|1|236|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0330|E1|3|230|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0330|E1|6|224|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0330|E2|52-1|236||197|válido|
|scored_0330|E2|52-3|230||197|válido|
|scored_0330|E2|52-6|224||215|válido|
|scored_0330|E2|64-1|236||199|válido|
|scored_0330|E2|64-3|230||208|válido|
|scored_0330|E2|64-6|224||199|válido|
|scored_0330|E3|joint|224|individual 224|224|OPTIMAL|
|scored_0330|E4|repair|224||219|completo|
|scored_0331|E1|1|357|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0331|E1|3|341|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|scored_0331|E1|6|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0331|E2|52-1|357||323|válido|
|scored_0331|E2|52-3|341||320|válido|
|scored_0331|E2|52-6|333||309|válido|
|scored_0331|E2|64-1|357||326|válido|
|scored_0331|E2|64-3|341||545|válido|
|scored_0331|E2|64-6|333||313|válido|
|scored_0331|E3|joint|333|individual 332|331|OPTIMAL|
|scored_0331|E4|repair|333||333|completo|
|scored_0332|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0332|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0332|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0332|E2|52-1|207||192|válido|
|scored_0332|E2|52-3|203||190|válido|
|scored_0332|E2|52-6|197||181|válido|
|scored_0332|E2|64-1|207||192|válido|
|scored_0332|E2|64-3|203||191|válido|
|scored_0332|E2|64-6|197||184|válido|
|scored_0332|E3|joint|197|individual 197|197|OPTIMAL|
|scored_0332|E4|repair|197||195|completo|
|scored_0333|E1|1|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0333|E1|3|287|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0333|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0333|E2|52-1|304||285|válido|
|scored_0333|E2|52-3|287||271|válido|
|scored_0333|E2|52-6|281||267|válido|
|scored_0333|E2|64-1|304||285|válido|
|scored_0333|E2|64-3|287||272|válido|
|scored_0333|E2|64-6|281||265|válido|
|scored_0333|E3|joint|281|individual 281|280|FEASIBLE|
|scored_0333|E4|repair|281||281|completo|
|scored_0334|E1|1|347|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0334|E1|3|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0334|E1|6|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0334|E2|52-1|347||301|válido|
|scored_0334|E2|52-3|321||295|válido|
|scored_0334|E2|52-6|318||285|válido|
|scored_0334|E2|64-1|347||305|válido|
|scored_0334|E2|64-3|321||295|válido|
|scored_0334|E2|64-6|318||287|válido|
|scored_0334|E3|joint|318|individual 318|316|OPTIMAL|
|scored_0334|E4|repair|318||318|completo|
|scored_0335|E1|1|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0335|E1|3|257|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|scored_0335|E1|6|252|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0335|E2|52-1|270||246|válido|
|scored_0335|E2|52-3|257||237|válido|
|scored_0335|E2|52-6|252||233|válido|
|scored_0335|E2|64-1|270||248|válido|
|scored_0335|E2|64-3|257||232|válido|
|scored_0335|E2|64-6|252||234|válido|
|scored_0335|E3|joint|252|individual 251|249|OPTIMAL|
|scored_0335|E4|repair|252||252|completo|
|scored_0336|E1|1|212|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0336|E1|3|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0336|E1|6|205|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0336|E2|52-1|212||190|válido|
|scored_0336|E2|52-3|209||190|válido|
|scored_0336|E2|52-6|205||193|válido|
|scored_0336|E2|64-1|212||191|válido|
|scored_0336|E2|64-3|209||190|válido|
|scored_0336|E2|64-6|205||193|válido|
|scored_0336|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0336|E4|repair|205||205|completo|
|scored_0337|E1|1|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0337|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0337|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0337|E2|52-1|164||155|válido|
|scored_0337|E2|52-3|162||155|válido|
|scored_0337|E2|52-6|162||154|válido|
|scored_0337|E2|64-1|164||151|válido|
|scored_0337|E2|64-3|162||151|válido|
|scored_0337|E2|64-6|162||150|válido|
|scored_0337|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0337|E4|repair|162||162|completo|
|scored_0338|E1|1|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0338|E1|3|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0338|E1|6|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0338|E2|52-1|245||230|válido|
|scored_0338|E2|52-3|236||222|válido|
|scored_0338|E2|52-6|235||221|válido|
|scored_0338|E2|64-1|245||227|válido|
|scored_0338|E2|64-3|236||223|válido|
|scored_0338|E2|64-6|235||221|válido|
|scored_0338|E3|joint|235|individual 235|234|OPTIMAL|
|scored_0338|E4|repair|235||235|completo|
|scored_0339|E1|1|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0339|E1|3|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0339|E1|6|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0339|E2|52-1|124||117|válido|
|scored_0339|E2|52-3|124||117|válido|
|scored_0339|E2|52-6|124||117|válido|
|scored_0339|E2|64-1|124||113|válido|
|scored_0339|E2|64-3|124||113|válido|
|scored_0339|E2|64-6|124||113|válido|
|scored_0339|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0339|E4|repair|124||115|completo|
|scored_0340|E1|1|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0340|E1|3|154|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0340|E1|6|154|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0340|E2|52-1|160||148|válido|
|scored_0340|E2|52-3|154||148|válido|
|scored_0340|E2|52-6|154||148|válido|
|scored_0340|E2|64-1|160||148|válido|
|scored_0340|E2|64-3|154||146|válido|
|scored_0340|E2|64-6|154||146|válido|
|scored_0340|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0340|E4|repair|154||154|completo|
|scored_0341|E1|1|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0341|E1|3|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0341|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0341|E2|52-1|183||195|válido|
|scored_0341|E2|52-3|179||187|válido|
|scored_0341|E2|52-6|174||194|válido|
|scored_0341|E2|64-1|183||187|válido|
|scored_0341|E2|64-3|179||185|válido|
|scored_0341|E2|64-6|174||185|válido|
|scored_0341|E3|joint|174|individual 174|174|OPTIMAL|
|scored_0341|E4|repair|174||173|completo|
|scored_0342|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0342|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0342|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0342|E2|52-1|310||283|válido|
|scored_0342|E2|52-3|290||270|válido|
|scored_0342|E2|52-6|292||270|válido|
|scored_0342|E2|64-1|310||288|válido|
|scored_0342|E2|64-3|290||271|válido|
|scored_0342|E2|64-6|292||270|válido|
|scored_0342|E3|joint|290|individual 290|288|OPTIMAL|
|scored_0342|E4|repair|290||289|completo|
|scored_0343|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0343|E1|3|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0343|E1|6|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0343|E2|52-1|155||147|válido|
|scored_0343|E2|52-3|153||146|válido|
|scored_0343|E2|52-6|153||145|válido|
|scored_0343|E2|64-1|155||146|válido|
|scored_0343|E2|64-3|153||147|válido|
|scored_0343|E2|64-6|153||146|válido|
|scored_0343|E3|joint|153|individual 152|152|OPTIMAL|
|scored_0343|E4|repair|153||153|completo|
|scored_0344|E1|1|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0344|E1|3|216|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0344|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0344|E2|52-1|223||202|válido|
|scored_0344|E2|52-3|216||206|válido|
|scored_0344|E2|52-6|211||203|válido|
|scored_0344|E2|64-1|223||202|válido|
|scored_0344|E2|64-3|216||206|válido|
|scored_0344|E2|64-6|211||205|válido|
|scored_0344|E3|joint|211|individual 210|210|OPTIMAL|
|scored_0344|E4|repair|211||210|completo|
|scored_0345|E1|1|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0345|E1|3|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0345|E1|6|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0345|E2|52-1|279||255|válido|
|scored_0345|E2|52-3|269||257|válido|
|scored_0345|E2|52-6|267||246|válido|
|scored_0345|E2|64-1|279||257|válido|
|scored_0345|E2|64-3|269||255|válido|
|scored_0345|E2|64-6|267||245|válido|
|scored_0345|E3|joint|267|individual 267|263|OPTIMAL|
|scored_0345|E4|repair|267||267|completo|
|scored_0346|E1|1|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0346|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0346|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0346|E2|52-1|234||221|válido|
|scored_0346|E2|52-3|226||213|válido|
|scored_0346|E2|52-6|227||213|válido|
|scored_0346|E2|64-1|234||222|válido|
|scored_0346|E2|64-3|226||216|válido|
|scored_0346|E2|64-6|227||214|válido|
|scored_0346|E3|joint|226|individual 226|226|OPTIMAL|
|scored_0346|E4|repair|226||225|completo|
|scored_0347|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0347|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0347|E1|6|155|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0347|E2|52-1|159||145|válido|
|scored_0347|E2|52-3|159||145|válido|
|scored_0347|E2|52-6|155||145|válido|
|scored_0347|E2|64-1|159||146|válido|
|scored_0347|E2|64-3|159||141|válido|
|scored_0347|E2|64-6|155||142|válido|
|scored_0347|E3|joint|155|individual 154|154|OPTIMAL|
|scored_0347|E4|repair|155||155|completo|
|scored_0348|E1|1|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0348|E1|3|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0348|E1|6|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0348|E2|52-1|3||3|válido|
|scored_0348|E2|52-3|3||3|válido|
|scored_0348|E2|52-6|3||3|válido|
|scored_0348|E2|64-1|3||3|válido|
|scored_0348|E2|64-3|3||3|válido|
|scored_0348|E2|64-6|3||3|válido|
|scored_0348|E3|joint|3|individual 3|3|OPTIMAL|
|scored_0348|E4|repair|3||3|completo|
|scored_0349|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0349|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0349|E1|6|258|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0349|E2|52-1|270||243|válido|
|scored_0349|E2|52-3|261||237|válido|
|scored_0349|E2|52-6|258||235|válido|
|scored_0349|E2|64-1|270||244|válido|
|scored_0349|E2|64-3|261||238|válido|
|scored_0349|E2|64-6|258||236|válido|
|scored_0349|E3|joint|258|individual 258|257|OPTIMAL|
|scored_0349|E4|repair|258||258|completo|
|scored_0350|E1|1|77|postprocess 76, optimize_a4 76, nested_windows 76|76|válido|
|scored_0350|E1|3|77|postprocess 76, optimize_a4 76, nested_windows 76|76|válido|
|scored_0350|E1|6|77|postprocess 76, optimize_a4 76, nested_windows 76|76|válido|
|scored_0350|E2|52-1|77||74|válido|
|scored_0350|E2|52-3|77||74|válido|
|scored_0350|E2|52-6|77||74|válido|
|scored_0350|E2|64-1|77||74|válido|
|scored_0350|E2|64-3|77||74|válido|
|scored_0350|E2|64-6|77||74|válido|
|scored_0350|E3|joint|77|individual 77|77|OPTIMAL|
|scored_0350|E4|repair|77||77|completo|
|scored_0351|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0351|E1|3|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0351|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0351|E2|52-1|226||207|válido|
|scored_0351|E2|52-3|216||201|válido|
|scored_0351|E2|52-6|214||202|válido|
|scored_0351|E2|64-1|226||204|válido|
|scored_0351|E2|64-3|216||201|válido|
|scored_0351|E2|64-6|214||202|válido|
|scored_0351|E3|joint|214|individual 214|214|OPTIMAL|
|scored_0351|E4|repair|214||210|completo|
|scored_0352|E1|1|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0352|E1|3|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0352|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0352|E2|52-1|327||296|válido|
|scored_0352|E2|52-3|307||288|válido|
|scored_0352|E2|52-6|303||288|válido|
|scored_0352|E2|64-1|327||294|válido|
|scored_0352|E2|64-3|307||290|válido|
|scored_0352|E2|64-6|303||288|válido|
|scored_0352|E3|joint|303|individual 303|299|OPTIMAL|
|scored_0352|E4|repair|303||303|completo|
|scored_0353|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0353|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0353|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0353|E2|52-1|200||183|válido|
|scored_0353|E2|52-3|192||183|válido|
|scored_0353|E2|52-6|192||183|válido|
|scored_0353|E2|64-1|200||183|válido|
|scored_0353|E2|64-3|192||183|válido|
|scored_0353|E2|64-6|192||182|válido|
|scored_0353|E3|joint|192|individual 191|191|OPTIMAL|
|scored_0353|E4|repair|192||192|completo|
|scored_0354|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0354|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0354|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0354|E2|52-1|211||201|válido|
|scored_0354|E2|52-3|202||191|válido|
|scored_0354|E2|52-6|199||189|válido|
|scored_0354|E2|64-1|211||194|válido|
|scored_0354|E2|64-3|202||184|válido|
|scored_0354|E2|64-6|199||182|válido|
|scored_0354|E3|joint|199|individual 198|198|OPTIMAL|
|scored_0354|E4|repair|199||195|completo|
|scored_0355|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0355|E1|3|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0355|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0355|E2|52-1|158||148|válido|
|scored_0355|E2|52-3|156||148|válido|
|scored_0355|E2|52-6|156||148|válido|
|scored_0355|E2|64-1|158||149|válido|
|scored_0355|E2|64-3|156||149|válido|
|scored_0355|E2|64-6|156||149|válido|
|scored_0355|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0355|E4|repair|156||155|completo|
|scored_0356|E1|1|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0356|E1|3|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0356|E1|6|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0356|E2|52-1|108||104|válido|
|scored_0356|E2|52-3|108||104|válido|
|scored_0356|E2|52-6|108||104|válido|
|scored_0356|E2|64-1|108||106|válido|
|scored_0356|E2|64-3|108||106|válido|
|scored_0356|E2|64-6|108||106|válido|
|scored_0356|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0356|E4|repair|108||108|completo|
|scored_0357|E1|1|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0357|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0357|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0357|E2|52-1|321||296|válido|
|scored_0357|E2|52-3|310||290|válido|
|scored_0357|E2|52-6|304||285|válido|
|scored_0357|E2|64-1|321||299|válido|
|scored_0357|E2|64-3|310||293|válido|
|scored_0357|E2|64-6|304||290|válido|
|scored_0357|E3|joint|304|individual 303|302|OPTIMAL|
|scored_0357|E4|repair|304||304|completo|
|scored_0358|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0358|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0358|E1|6|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0358|E2|52-1|210||187|válido|
|scored_0358|E2|52-3|203||182|válido|
|scored_0358|E2|52-6|203||188|válido|
|scored_0358|E2|64-1|210||186|válido|
|scored_0358|E2|64-3|203||182|válido|
|scored_0358|E2|64-6|203||184|válido|
|scored_0358|E3|joint|203|individual 203|201|OPTIMAL|
|scored_0358|E4|repair|203||200|completo|
|scored_0359|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0359|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0359|E1|6|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0359|E2|52-1|151||146|válido|
|scored_0359|E2|52-3|151||146|válido|
|scored_0359|E2|52-6|149||146|válido|
|scored_0359|E2|64-1|151||145|válido|
|scored_0359|E2|64-3|151||145|válido|
|scored_0359|E2|64-6|149||145|válido|
|scored_0359|E3|joint|149|individual 149|149|OPTIMAL|
|scored_0359|E4|repair|149||147|completo|
|scored_0360|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0360|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0360|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0360|E2|52-1|167||162|válido|
|scored_0360|E2|52-3|166||161|válido|
|scored_0360|E2|52-6|166||161|válido|
|scored_0360|E2|64-1|167||161|válido|
|scored_0360|E2|64-3|166||161|válido|
|scored_0360|E2|64-6|166||161|válido|
|scored_0360|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0360|E4|repair|166||163|completo|
|scored_0361|E1|1|244|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0361|E1|3|240|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0361|E1|6|237|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0361|E2|52-1|244||233|válido|
|scored_0361|E2|52-3|240||224|válido|
|scored_0361|E2|52-6|237||226|válido|
|scored_0361|E2|64-1|244||237|válido|
|scored_0361|E2|64-3|240||224|válido|
|scored_0361|E2|64-6|237||228|válido|
|scored_0361|E3|joint|237|individual 237|233|OPTIMAL|
|scored_0361|E4|repair|237||234|completo|
|scored_0362|E1|1|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0362|E1|3|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0362|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0362|E2|52-1|148||147|válido|
|scored_0362|E2|52-3|146||144|válido|
|scored_0362|E2|52-6|143||144|válido|
|scored_0362|E2|64-1|148||147|válido|
|scored_0362|E2|64-3|146||144|válido|
|scored_0362|E2|64-6|143||142|válido|
|scored_0362|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0362|E4|repair|143||140|completo|
|scored_0363|E1|1|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0363|E1|3|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0363|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0363|E2|52-1|222||208|válido|
|scored_0363|E2|52-3|223||207|válido|
|scored_0363|E2|52-6|217||205|válido|
|scored_0363|E2|64-1|222||208|válido|
|scored_0363|E2|64-3|223||205|válido|
|scored_0363|E2|64-6|217||205|válido|
|scored_0363|E3|joint|217|individual 217|215|OPTIMAL|
|scored_0363|E4|repair|217||217|completo|
|scored_0364|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0364|E1|3|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0364|E1|6|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0364|E2|52-1|253||228|válido|
|scored_0364|E2|52-3|240||225|válido|
|scored_0364|E2|52-6|233||217|válido|
|scored_0364|E2|64-1|253||223|válido|
|scored_0364|E2|64-3|240||222|válido|
|scored_0364|E2|64-6|233||219|válido|
|scored_0364|E3|joint|233|individual 232|232|OPTIMAL|
|scored_0364|E4|repair|233||232|completo|
|scored_0365|E1|1|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0365|E1|3|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0365|E1|6|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0365|E2|52-1|5||5|válido|
|scored_0365|E2|52-3|5||5|válido|
|scored_0365|E2|52-6|5||5|válido|
|scored_0365|E2|64-1|5||5|válido|
|scored_0365|E2|64-3|5||5|válido|
|scored_0365|E2|64-6|5||5|válido|
|scored_0365|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0365|E4|repair|4||4|completo|
|scored_0366|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0366|E1|3|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0366|E1|6|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0366|E2|52-1|158||157|válido|
|scored_0366|E2|52-3|154||155|válido|
|scored_0366|E2|52-6|154||151|válido|
|scored_0366|E2|64-1|158||155|válido|
|scored_0366|E2|64-3|154||155|válido|
|scored_0366|E2|64-6|154||154|válido|
|scored_0366|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0366|E4|repair|154||148|completo|
|scored_0367|E1|1|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0367|E1|3|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0367|E1|6|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0367|E2|52-1|158||153|válido|
|scored_0367|E2|52-3|158||153|válido|
|scored_0367|E2|52-6|158||153|válido|
|scored_0367|E2|64-1|158||150|válido|
|scored_0367|E2|64-3|158||151|válido|
|scored_0367|E2|64-6|158||151|válido|
|scored_0367|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0367|E4|repair|158||157|completo|
|scored_0368|E1|1|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0368|E1|3|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0368|E1|6|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0368|E2|52-1|39||39|válido|
|scored_0368|E2|52-3|39||39|válido|
|scored_0368|E2|52-6|39||39|válido|
|scored_0368|E2|64-1|39||39|válido|
|scored_0368|E2|64-3|39||39|válido|
|scored_0368|E2|64-6|39||39|válido|
|scored_0368|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0368|E4|repair|39||39|completo|
|scored_0369|E1|1|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0369|E1|3|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0369|E1|6|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0369|E2|52-1|345||301|válido|
|scored_0369|E2|52-3|316||304|válido|
|scored_0369|E2|52-6|310||298|válido|
|scored_0369|E2|64-1|345||302|válido|
|scored_0369|E2|64-3|316||302|válido|
|scored_0369|E2|64-6|310||294|válido|
|scored_0369|E3|joint|310|individual 310|310|OPTIMAL|
|scored_0369|E4|repair|310||310|completo|
|scored_0370|E1|1|374|postprocess 374, optimize_a4 374, nested_windows 374|374|válido|
|scored_0370|E1|3|357|postprocess 357, optimize_a4 357, nested_windows 357|357|válido|
|scored_0370|E1|6|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|scored_0370|E2|52-1|374||343|válido|
|scored_0370|E2|52-3|357||328|válido|
|scored_0370|E2|52-6|355||327|válido|
|scored_0370|E2|64-1|374||473|válido|
|scored_0370|E2|64-3|357||334|válido|
|scored_0370|E2|64-6|355||334|válido|
|scored_0370|E3|joint|355|individual 355|354|OPTIMAL|
|scored_0370|E4|repair|355||355|completo|
|scored_0371|E1|1|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0371|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0371|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0371|E2|52-1|179||172|válido|
|scored_0371|E2|52-3|173||168|válido|
|scored_0371|E2|52-6|172||161|válido|
|scored_0371|E2|64-1|179||164|válido|
|scored_0371|E2|64-3|173||162|válido|
|scored_0371|E2|64-6|172||160|válido|
|scored_0371|E3|joint|172|individual 172|170|OPTIMAL|
|scored_0371|E4|repair|172||164|completo|
|scored_0372|E1|1|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0372|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0372|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0372|E2|52-1|148||145|válido|
|scored_0372|E2|52-3|148||144|válido|
|scored_0372|E2|52-6|147||144|válido|
|scored_0372|E2|64-1|148||141|válido|
|scored_0372|E2|64-3|148||141|válido|
|scored_0372|E2|64-6|147||141|válido|
|scored_0372|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0372|E4|repair|147||147|completo|
|scored_0373|E1|1|338|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0373|E1|3|300|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0373|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0373|E2|52-1|338||297|válido|
|scored_0373|E2|52-3|300||275|válido|
|scored_0373|E2|52-6|303||281|válido|
|scored_0373|E2|64-1|338||296|válido|
|scored_0373|E2|64-3|300||275|válido|
|scored_0373|E2|64-6|303||281|válido|
|scored_0373|E3|joint|300|individual 300|299|FEASIBLE|
|scored_0373|E4|repair|300||300|completo|
|scored_0374|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0374|E1|3|302|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0374|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0374|E2|52-1|318||281|válido|
|scored_0374|E2|52-3|302||277|válido|
|scored_0374|E2|52-6|295||274|válido|
|scored_0374|E2|64-1|318||324|válido|
|scored_0374|E2|64-3|302||278|válido|
|scored_0374|E2|64-6|295||280|válido|
|scored_0374|E3|joint|295|individual 295|295|OPTIMAL|
|scored_0374|E4|repair|295||295|completo|
|scored_0375|E1|1|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0375|E1|3|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0375|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0375|E2|52-1|308||280|válido|
|scored_0375|E2|52-3|296||269|válido|
|scored_0375|E2|52-6|287||263|válido|
|scored_0375|E2|64-1|308||280|válido|
|scored_0375|E2|64-3|296||273|válido|
|scored_0375|E2|64-6|287||263|válido|
|scored_0375|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0375|E4|repair|287||287|completo|
|scored_0376|E1|1|37|postprocess 33, optimize_a4 33, nested_windows 33|33|válido|
|scored_0376|E1|3|37|postprocess 33, optimize_a4 33, nested_windows 33|33|válido|
|scored_0376|E1|6|37|postprocess 33, optimize_a4 33, nested_windows 33|33|válido|
|scored_0376|E2|52-1|37||37|válido|
|scored_0376|E2|52-3|37||37|válido|
|scored_0376|E2|52-6|37||37|válido|
|scored_0376|E2|64-1|37||37|válido|
|scored_0376|E2|64-3|37||37|válido|
|scored_0376|E2|64-6|37||37|válido|
|scored_0376|E3|joint|37|individual 37|37|OPTIMAL|
|scored_0376|E4|repair|37||37|completo|
|scored_0377|E1|1|111|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0377|E1|3|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0377|E1|6|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0377|E2|52-1|111||107|válido|
|scored_0377|E2|52-3|108||105|válido|
|scored_0377|E2|52-6|108||105|válido|
|scored_0377|E2|64-1|111||108|válido|
|scored_0377|E2|64-3|108||105|válido|
|scored_0377|E2|64-6|108||105|válido|
|scored_0377|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0377|E4|repair|108||106|completo|
|scored_0378|E1|1|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0378|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0378|E1|6|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0378|E2|52-1|112||105|válido|
|scored_0378|E2|52-3|111||108|válido|
|scored_0378|E2|52-6|112||109|válido|
|scored_0378|E2|64-1|112||105|válido|
|scored_0378|E2|64-3|111||108|válido|
|scored_0378|E2|64-6|112||109|válido|
|scored_0378|E3|joint|111|individual 111|110|OPTIMAL|
|scored_0378|E4|repair|111||111|completo|
|scored_0379|E1|1|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0379|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0379|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0379|E2|52-1|154||150|válido|
|scored_0379|E2|52-3|152||149|válido|
|scored_0379|E2|52-6|152||148|válido|
|scored_0379|E2|64-1|154||150|válido|
|scored_0379|E2|64-3|152||149|válido|
|scored_0379|E2|64-6|152||149|válido|
|scored_0379|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0379|E4|repair|152||152|completo|
|scored_0380|E1|1|216|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0380|E1|3|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0380|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0380|E2|52-1|216||202|válido|
|scored_0380|E2|52-3|212||196|válido|
|scored_0380|E2|52-6|211||195|válido|
|scored_0380|E2|64-1|216||206|válido|
|scored_0380|E2|64-3|212||196|válido|
|scored_0380|E2|64-6|211||198|válido|
|scored_0380|E3|joint|211|individual 211|211|OPTIMAL|
|scored_0380|E4|repair|211||209|completo|
|scored_0381|E1|1|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0381|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0381|E1|6|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0381|E2|52-1|284||262|válido|
|scored_0381|E2|52-3|270||256|válido|
|scored_0381|E2|52-6|267||245|válido|
|scored_0381|E2|64-1|284||262|válido|
|scored_0381|E2|64-3|270||257|válido|
|scored_0381|E2|64-6|267||253|válido|
|scored_0381|E3|joint|267|individual 267|267|OPTIMAL|
|scored_0381|E4|repair|267||267|completo|
|scored_0382|E1|1|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0382|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0382|E1|6|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0382|E2|52-1|203||181|válido|
|scored_0382|E2|52-3|194||179|válido|
|scored_0382|E2|52-6|194||179|válido|
|scored_0382|E2|64-1|203||183|válido|
|scored_0382|E2|64-3|194||178|válido|
|scored_0382|E2|64-6|194||178|válido|
|scored_0382|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0382|E4|repair|194||194|completo|
|scored_0383|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0383|E1|3|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0383|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0383|E2|52-1|206||191|válido|
|scored_0383|E2|52-3|201||185|válido|
|scored_0383|E2|52-6|204||183|válido|
|scored_0383|E2|64-1|206||194|válido|
|scored_0383|E2|64-3|201||186|válido|
|scored_0383|E2|64-6|204||183|válido|
|scored_0383|E3|joint|201|individual 201|199|OPTIMAL|
|scored_0383|E4|repair|201||200|completo|
|scored_0384|E1|1|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0384|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0384|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0384|E2|52-1|201||187|válido|
|scored_0384|E2|52-3|194||181|válido|
|scored_0384|E2|52-6|189||180|válido|
|scored_0384|E2|64-1|201||185|válido|
|scored_0384|E2|64-3|194||182|válido|
|scored_0384|E2|64-6|189||182|válido|
|scored_0384|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0384|E4|repair|189||188|completo|
|scored_0385|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0385|E1|3|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0385|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0385|E2|52-1|215||199|válido|
|scored_0385|E2|52-3|204||189|válido|
|scored_0385|E2|52-6|207||189|válido|
|scored_0385|E2|64-1|215||199|válido|
|scored_0385|E2|64-3|204||190|válido|
|scored_0385|E2|64-6|207||188|válido|
|scored_0385|E3|joint|204|individual 204|202|OPTIMAL|
|scored_0385|E4|repair|204||204|completo|
|scored_0386|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0386|E1|3|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0386|E1|6|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0386|E2|52-1|270||250|válido|
|scored_0386|E2|52-3|263||240|válido|
|scored_0386|E2|52-6|258||244|válido|
|scored_0386|E2|64-1|270||250|válido|
|scored_0386|E2|64-3|263||241|válido|
|scored_0386|E2|64-6|258||245|válido|
|scored_0386|E3|joint|258|individual 257|257|OPTIMAL|
|scored_0386|E4|repair|258||258|completo|
|scored_0387|E1|1|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0387|E1|3|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0387|E1|6|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0387|E2|52-1|39||39|válido|
|scored_0387|E2|52-3|39||39|válido|
|scored_0387|E2|52-6|39||39|válido|
|scored_0387|E2|64-1|39||39|válido|
|scored_0387|E2|64-3|39||39|válido|
|scored_0387|E2|64-6|39||39|válido|
|scored_0387|E3|joint|38|individual 38|38|OPTIMAL|
|scored_0387|E4|repair|38||38|completo|
|scored_0388|E1|1|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0388|E1|3|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0388|E1|6|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0388|E2|52-1|33||33|válido|
|scored_0388|E2|52-3|33||33|válido|
|scored_0388|E2|52-6|33||33|válido|
|scored_0388|E2|64-1|33||33|válido|
|scored_0388|E2|64-3|33||33|válido|
|scored_0388|E2|64-6|33||33|válido|
|scored_0388|E3|joint|33|individual 31|30|OPTIMAL|
|scored_0388|E4|repair|33||32|completo|
|scored_0389|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0389|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0389|E1|6|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0389|E2|52-1|236||222|válido|
|scored_0389|E2|52-3|232||221|válido|
|scored_0389|E2|52-6|234||219|válido|
|scored_0389|E2|64-1|236||223|válido|
|scored_0389|E2|64-3|232||224|válido|
|scored_0389|E2|64-6|234||219|válido|
|scored_0389|E3|joint|232|individual 231|229|OPTIMAL|
|scored_0389|E4|repair|232||232|completo|
|scored_0390|E1|1|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0390|E1|3|142|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0390|E1|6|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0390|E2|52-1|143||132|válido|
|scored_0390|E2|52-3|142||131|válido|
|scored_0390|E2|52-6|140||131|válido|
|scored_0390|E2|64-1|143||132|válido|
|scored_0390|E2|64-3|142||130|válido|
|scored_0390|E2|64-6|140||130|válido|
|scored_0390|E3|joint|140|individual 140|139|OPTIMAL|
|scored_0390|E4|repair|140||138|completo|
|scored_0391|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0391|E1|3|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0391|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0391|E2|52-1|319||283|válido|
|scored_0391|E2|52-3|301||274|válido|
|scored_0391|E2|52-6|289||270|válido|
|scored_0391|E2|64-1|319||285|válido|
|scored_0391|E2|64-3|301||275|válido|
|scored_0391|E2|64-6|289||270|válido|
|scored_0391|E3|joint|289|individual 289|289|rechazado por registros|
|scored_0391|E4|repair|289||289|completo|
|scored_0392|E1|1|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0392|E1|3|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0392|E1|6|151|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0392|E2|52-1|153||149|válido|
|scored_0392|E2|52-3|150||144|válido|
|scored_0392|E2|52-6|151||144|válido|
|scored_0392|E2|64-1|153||149|válido|
|scored_0392|E2|64-3|150||148|válido|
|scored_0392|E2|64-6|151||150|válido|
|scored_0392|E3|joint|150|individual 150|150|OPTIMAL|
|scored_0392|E4|repair|150||147|completo|
|scored_0393|E1|1|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0393|E1|3|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0393|E1|6|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0393|E2|52-1|35||35|válido|
|scored_0393|E2|52-3|35||35|válido|
|scored_0393|E2|52-6|35||35|válido|
|scored_0393|E2|64-1|35||35|válido|
|scored_0393|E2|64-3|35||35|válido|
|scored_0393|E2|64-6|35||35|válido|
|scored_0393|E3|joint|35|individual 35|35|OPTIMAL|
|scored_0393|E4|repair|35||35|completo|
|scored_0394|E1|1|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0394|E1|3|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0394|E1|6|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0394|E2|52-1|183||175|válido|
|scored_0394|E2|52-3|183||173|válido|
|scored_0394|E2|52-6|183||172|válido|
|scored_0394|E2|64-1|183||175|válido|
|scored_0394|E2|64-3|183||173|válido|
|scored_0394|E2|64-6|183||173|válido|
|scored_0394|E3|joint|183|individual 182|181|OPTIMAL|
|scored_0394|E4|repair|183||180|completo|
|scored_0395|E1|1|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0395|E1|3|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0395|E1|6|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0395|E2|52-1|117||115|válido|
|scored_0395|E2|52-3|117||115|válido|
|scored_0395|E2|52-6|117||115|válido|
|scored_0395|E2|64-1|117||116|válido|
|scored_0395|E2|64-3|117||116|válido|
|scored_0395|E2|64-6|117||116|válido|
|scored_0395|E3|joint|117|individual 117|117|OPTIMAL|
|scored_0395|E4|repair|117||115|completo|
|scored_0396|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0396|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0396|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0396|E2|52-1|50||50|válido|
|scored_0396|E2|52-3|50||50|válido|
|scored_0396|E2|52-6|50||50|válido|
|scored_0396|E2|64-1|50||50|válido|
|scored_0396|E2|64-3|50||50|válido|
|scored_0396|E2|64-6|50||50|válido|
|scored_0396|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0396|E4|repair|50||50|completo|
|scored_0397|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0397|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0397|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0397|E2|52-1|273||247|válido|
|scored_0397|E2|52-3|261||240|válido|
|scored_0397|E2|52-6|257||240|válido|
|scored_0397|E2|64-1|273||250|válido|
|scored_0397|E2|64-3|261||240|válido|
|scored_0397|E2|64-6|257||243|válido|
|scored_0397|E3|joint|257|individual 257|257|OPTIMAL|
|scored_0397|E4|repair|257||257|completo|
|scored_0398|E1|1|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0398|E1|3|277|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0398|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0398|E2|52-1|288||265|válido|
|scored_0398|E2|52-3|277||256|válido|
|scored_0398|E2|52-6|269||255|válido|
|scored_0398|E2|64-1|288||265|válido|
|scored_0398|E2|64-3|277||256|válido|
|scored_0398|E2|64-6|269||254|válido|
|scored_0398|E3|joint|269|individual 268|268|OPTIMAL|
|scored_0398|E4|repair|269||269|completo|
|scored_0399|E1|1|33|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0399|E1|3|33|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0399|E1|6|33|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0399|E2|52-1|33||33|válido|
|scored_0399|E2|52-3|33||33|válido|
|scored_0399|E2|52-6|33||33|válido|
|scored_0399|E2|64-1|33||33|válido|
|scored_0399|E2|64-3|33||33|válido|
|scored_0399|E2|64-6|33||33|válido|
|scored_0399|E3|joint|30|individual 28|28|OPTIMAL|
|scored_0399|E4|repair|30||29|completo|
|scored_0400|E1|1|314|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0400|E1|3|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0400|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0400|E2|52-1|314||297|válido|
|scored_0400|E2|52-3|307||284|válido|
|scored_0400|E2|52-6|303||313|válido|
|scored_0400|E2|64-1|314||299|válido|
|scored_0400|E2|64-3|307||284|válido|
|scored_0400|E2|64-6|303||330|válido|
|scored_0400|E3|joint|303|individual 302|300|OPTIMAL|
|scored_0400|E4|repair|303||303|completo|
|scored_0401|E1|1|42|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0401|E1|3|42|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0401|E1|6|42|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0401|E2|52-1|42||42|válido|
|scored_0401|E2|52-3|42||42|válido|
|scored_0401|E2|52-6|42||42|válido|
|scored_0401|E2|64-1|42||42|válido|
|scored_0401|E2|64-3|42||42|válido|
|scored_0401|E2|64-6|42||42|válido|
|scored_0401|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0401|E4|repair|42||41|completo|
|scored_0402|E1|1|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0402|E1|3|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0402|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0402|E2|52-1|124||120|válido|
|scored_0402|E2|52-3|123||120|válido|
|scored_0402|E2|52-6|123||120|válido|
|scored_0402|E2|64-1|124||120|válido|
|scored_0402|E2|64-3|123||120|válido|
|scored_0402|E2|64-6|123||120|válido|
|scored_0402|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0402|E4|repair|123||120|completo|
|scored_0403|E1|1|334|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0403|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0403|E1|6|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0403|E2|52-1|334||303|válido|
|scored_0403|E2|52-3|329||292|válido|
|scored_0403|E2|52-6|315||295|válido|
|scored_0403|E2|64-1|334||306|válido|
|scored_0403|E2|64-3|329||297|válido|
|scored_0403|E2|64-6|315||294|válido|
|scored_0403|E3|joint|315|individual 315|314|FEASIBLE|
|scored_0403|E4|repair|315||315|completo|
|scored_0404|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0404|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0404|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0404|E2|52-1|137||130|válido|
|scored_0404|E2|52-3|137||130|válido|
|scored_0404|E2|52-6|137||130|válido|
|scored_0404|E2|64-1|137||132|válido|
|scored_0404|E2|64-3|137||131|válido|
|scored_0404|E2|64-6|137||131|válido|
|scored_0404|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0404|E4|repair|137||137|completo|
|scored_0405|E1|1|404|postprocess 401, optimize_a4 401, nested_windows 401|401|válido|
|scored_0405|E1|3|380|postprocess 380, optimize_a4 380, nested_windows 380|380|válido|
|scored_0405|E1|6|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0405|E2|52-1|404||359|válido|
|scored_0405|E2|52-3|380||345|válido|
|scored_0405|E2|52-6|363||336|válido|
|scored_0405|E2|64-1|404||362|válido|
|scored_0405|E2|64-3|380||434|válido|
|scored_0405|E2|64-6|363||338|válido|
|scored_0405|E3|joint|363|individual 363|361|FEASIBLE|
|scored_0405|E4|repair|363||363|completo|
|scored_0406|E1|1|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0406|E1|3|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0406|E1|6|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0406|E2|52-1|189||176|válido|
|scored_0406|E2|52-3|184||173|válido|
|scored_0406|E2|52-6|183||176|válido|
|scored_0406|E2|64-1|189||174|válido|
|scored_0406|E2|64-3|184||172|válido|
|scored_0406|E2|64-6|183||173|válido|
|scored_0406|E3|joint|183|individual 183|183|OPTIMAL|
|scored_0406|E4|repair|183||181|completo|
|scored_0407|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0407|E1|3|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0407|E1|6|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0407|E2|52-1|218||202|válido|
|scored_0407|E2|52-3|209||199|válido|
|scored_0407|E2|52-6|208||198|válido|
|scored_0407|E2|64-1|218||201|válido|
|scored_0407|E2|64-3|209||198|válido|
|scored_0407|E2|64-6|208||199|válido|
|scored_0407|E3|joint|208|individual 208|207|OPTIMAL|
|scored_0407|E4|repair|208||208|completo|
|scored_0408|E1|1|228|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0408|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0408|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0408|E2|52-1|228||214|válido|
|scored_0408|E2|52-3|226||208|válido|
|scored_0408|E2|52-6|222||207|válido|
|scored_0408|E2|64-1|228||213|válido|
|scored_0408|E2|64-3|226||207|válido|
|scored_0408|E2|64-6|222||207|válido|
|scored_0408|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0408|E4|repair|222||221|completo|
|scored_0409|E1|1|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0409|E1|3|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0409|E1|6|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0409|E2|52-1|33||33|válido|
|scored_0409|E2|52-3|33||33|válido|
|scored_0409|E2|52-6|33||33|válido|
|scored_0409|E2|64-1|33||33|válido|
|scored_0409|E2|64-3|33||33|válido|
|scored_0409|E2|64-6|33||33|válido|
|scored_0409|E3|joint|31|individual 30|30|OPTIMAL|
|scored_0409|E4|repair|31||31|completo|
|scored_0410|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0410|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0410|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0410|E2|52-1|253||236|válido|
|scored_0410|E2|52-3|243||228|válido|
|scored_0410|E2|52-6|242||228|válido|
|scored_0410|E2|64-1|253||236|válido|
|scored_0410|E2|64-3|243||229|válido|
|scored_0410|E2|64-6|242||225|válido|
|scored_0410|E3|joint|242|individual 242|242|OPTIMAL|
|scored_0410|E4|repair|242||242|completo|
|scored_0411|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0411|E1|3|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0411|E1|6|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0411|E2|52-1|254||230|válido|
|scored_0411|E2|52-3|237||226|válido|
|scored_0411|E2|52-6|247||226|válido|
|scored_0411|E2|64-1|254||232|válido|
|scored_0411|E2|64-3|237||227|válido|
|scored_0411|E2|64-6|247||226|válido|
|scored_0411|E3|joint|237|individual 237|237|OPTIMAL|
|scored_0411|E4|repair|237||237|completo|
|scored_0412|E1|1|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0412|E1|3|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0412|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0412|E2|52-1|244||225|válido|
|scored_0412|E2|52-3|236||225|válido|
|scored_0412|E2|52-6|232||218|válido|
|scored_0412|E2|64-1|244||225|válido|
|scored_0412|E2|64-3|236||225|válido|
|scored_0412|E2|64-6|232||219|válido|
|scored_0412|E3|joint|232|individual 231|231|FEASIBLE|
|scored_0412|E4|repair|232||232|completo|
|scored_0413|E1|1|212|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0413|E1|3|209|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0413|E1|6|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0413|E2|52-1|212||200|válido|
|scored_0413|E2|52-3|209||193|válido|
|scored_0413|E2|52-6|206||193|válido|
|scored_0413|E2|64-1|212||201|válido|
|scored_0413|E2|64-3|209||194|válido|
|scored_0413|E2|64-6|206||191|válido|
|scored_0413|E3|joint|206|individual 206|206|OPTIMAL|
|scored_0413|E4|repair|206||206|completo|
|scored_0414|E1|1|177|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0414|E1|3|176|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0414|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0414|E2|52-1|177||170|válido|
|scored_0414|E2|52-3|176||169|válido|
|scored_0414|E2|52-6|174||163|válido|
|scored_0414|E2|64-1|177||172|válido|
|scored_0414|E2|64-3|176||170|válido|
|scored_0414|E2|64-6|174||165|válido|
|scored_0414|E3|joint|174|individual 174|173|OPTIMAL|
|scored_0414|E4|repair|174||171|completo|
|scored_0415|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0415|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0415|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0415|E2|52-1|206||190|válido|
|scored_0415|E2|52-3|202||190|válido|
|scored_0415|E2|52-6|198||188|válido|
|scored_0415|E2|64-1|206||190|válido|
|scored_0415|E2|64-3|202||190|válido|
|scored_0415|E2|64-6|198||189|válido|
|scored_0415|E3|joint|198|individual 198|198|OPTIMAL|
|scored_0415|E4|repair|198||198|completo|
|scored_0416|E1|1|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0416|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0416|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0416|E2|52-1|262||231|válido|
|scored_0416|E2|52-3|246||228|válido|
|scored_0416|E2|52-6|246||227|válido|
|scored_0416|E2|64-1|262||233|válido|
|scored_0416|E2|64-3|246||227|válido|
|scored_0416|E2|64-6|246||225|válido|
|scored_0416|E3|joint|246|individual 245|241|OPTIMAL|
|scored_0416|E4|repair|246||246|completo|
|scored_0417|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0417|E1|3|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0417|E1|6|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0417|E2|52-1|237||218|válido|
|scored_0417|E2|52-3|228||213|válido|
|scored_0417|E2|52-6|225||214|válido|
|scored_0417|E2|64-1|237||218|válido|
|scored_0417|E2|64-3|228||214|válido|
|scored_0417|E2|64-6|225||214|válido|
|scored_0417|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0417|E4|repair|225||225|completo|
|scored_0418|E1|1|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0418|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0418|E1|6|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0418|E2|52-1|161||154|válido|
|scored_0418|E2|52-3|157||153|válido|
|scored_0418|E2|52-6|157||150|válido|
|scored_0418|E2|64-1|161||153|válido|
|scored_0418|E2|64-3|157||149|válido|
|scored_0418|E2|64-6|157||149|válido|
|scored_0418|E3|joint|157|individual 156|156|OPTIMAL|
|scored_0418|E4|repair|157||155|completo|
|scored_0419|E1|1|32|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0419|E1|3|32|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0419|E1|6|32|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0419|E2|52-1|32||32|válido|
|scored_0419|E2|52-3|32||32|válido|
|scored_0419|E2|52-6|32||32|válido|
|scored_0419|E2|64-1|32||32|válido|
|scored_0419|E2|64-3|32||32|válido|
|scored_0419|E2|64-6|32||32|válido|
|scored_0419|E3|joint|32|individual 32|32|OPTIMAL|
|scored_0419|E4|repair|32||32|completo|
|scored_0420|E1|1|221|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0420|E1|3|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0420|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0420|E2|52-1|221||204|válido|
|scored_0420|E2|52-3|210||197|válido|
|scored_0420|E2|52-6|207||191|válido|
|scored_0420|E2|64-1|221||200|válido|
|scored_0420|E2|64-3|210||199|válido|
|scored_0420|E2|64-6|207||193|válido|
|scored_0420|E3|joint|207|individual 207|207|OPTIMAL|
|scored_0420|E4|repair|207||207|completo|
|scored_0421|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0421|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0421|E1|6|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0421|E2|52-1|215||203|válido|
|scored_0421|E2|52-3|213||199|válido|
|scored_0421|E2|52-6|205||196|válido|
|scored_0421|E2|64-1|215||201|válido|
|scored_0421|E2|64-3|213||200|válido|
|scored_0421|E2|64-6|205||196|válido|
|scored_0421|E3|joint|205|individual 205|204|OPTIMAL|
|scored_0421|E4|repair|205||201|completo|
|scored_0422|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0422|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0422|E1|6|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0422|E2|52-1|223||209|válido|
|scored_0422|E2|52-3|219||203|válido|
|scored_0422|E2|52-6|219||201|válido|
|scored_0422|E2|64-1|223||209|válido|
|scored_0422|E2|64-3|219||203|válido|
|scored_0422|E2|64-6|219||202|válido|
|scored_0422|E3|joint|219|individual 218|217|OPTIMAL|
|scored_0422|E4|repair|219||219|completo|
|scored_0423|E1|1|118|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0423|E1|3|118|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0423|E1|6|118|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0423|E2|52-1|118||116|válido|
|scored_0423|E2|52-3|118||116|válido|
|scored_0423|E2|52-6|118||116|válido|
|scored_0423|E2|64-1|118||116|válido|
|scored_0423|E2|64-3|118||116|válido|
|scored_0423|E2|64-6|118||116|válido|
|scored_0423|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0423|E4|repair|118||118|completo|
|scored_0424|E1|1|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0424|E1|3|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0424|E1|6|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0424|E2|52-1|322||289|válido|
|scored_0424|E2|52-3|306||278|válido|
|scored_0424|E2|52-6|296||275|válido|
|scored_0424|E2|64-1|322||289|válido|
|scored_0424|E2|64-3|306||278|válido|
|scored_0424|E2|64-6|296||293|válido|
|scored_0424|E3|joint|296|individual 296|296|FEASIBLE|
|scored_0424|E4|repair|296||296|completo|
|scored_0425|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0425|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0425|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0425|E2|52-1|41||41|válido|
|scored_0425|E2|52-3|41||41|válido|
|scored_0425|E2|52-6|41||41|válido|
|scored_0425|E2|64-1|41||41|válido|
|scored_0425|E2|64-3|41||41|válido|
|scored_0425|E2|64-6|41||41|válido|
|scored_0425|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0425|E4|repair|41||41|completo|
|scored_0426|E1|1|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0426|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0426|E1|6|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0426|E2|52-1|194||178|válido|
|scored_0426|E2|52-3|188||175|válido|
|scored_0426|E2|52-6|184||170|válido|
|scored_0426|E2|64-1|194||178|válido|
|scored_0426|E2|64-3|188||173|válido|
|scored_0426|E2|64-6|184||171|válido|
|scored_0426|E3|joint|184|individual 184|184|OPTIMAL|
|scored_0426|E4|repair|184||184|completo|
|scored_0427|E1|1|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0427|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0427|E1|6|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0427|E2|52-1|209||195|válido|
|scored_0427|E2|52-3|208||190|válido|
|scored_0427|E2|52-6|194||183|válido|
|scored_0427|E2|64-1|209||187|válido|
|scored_0427|E2|64-3|208||186|válido|
|scored_0427|E2|64-6|194||179|válido|
|scored_0427|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0427|E4|repair|194||193|completo|
|scored_0428|E1|1|310|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0428|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0428|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0428|E2|52-1|310||276|válido|
|scored_0428|E2|52-3|288||267|válido|
|scored_0428|E2|52-6|284||264|válido|
|scored_0428|E2|64-1|310||281|válido|
|scored_0428|E2|64-3|288||270|válido|
|scored_0428|E2|64-6|284||264|válido|
|scored_0428|E3|joint|284|individual 284|284|rechazado por registros|
|scored_0428|E4|repair|284||284|completo|
|scored_0429|E1|1|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0429|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0429|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0429|E2|52-1|150||142|válido|
|scored_0429|E2|52-3|148||142|válido|
|scored_0429|E2|52-6|148||141|válido|
|scored_0429|E2|64-1|150||144|válido|
|scored_0429|E2|64-3|148||144|válido|
|scored_0429|E2|64-6|148||143|válido|
|scored_0429|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0429|E4|repair|148||146|completo|
|scored_0430|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0430|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0430|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0430|E2|52-1|113||111|válido|
|scored_0430|E2|52-3|113||111|válido|
|scored_0430|E2|52-6|113||111|válido|
|scored_0430|E2|64-1|113||111|válido|
|scored_0430|E2|64-3|113||111|válido|
|scored_0430|E2|64-6|113||111|válido|
|scored_0430|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0430|E4|repair|113||110|completo|
|scored_0431|E1|1|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0431|E1|3|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0431|E1|6|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0431|E2|52-1|110||107|válido|
|scored_0431|E2|52-3|110||107|válido|
|scored_0431|E2|52-6|110||107|válido|
|scored_0431|E2|64-1|110||107|válido|
|scored_0431|E2|64-3|110||107|válido|
|scored_0431|E2|64-6|110||107|válido|
|scored_0431|E3|joint|110|individual 110|110|OPTIMAL|
|scored_0431|E4|repair|110||108|completo|
|scored_0432|E1|1|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0432|E1|3|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0432|E1|6|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0432|E2|52-1|187||175|válido|
|scored_0432|E2|52-3|182||172|válido|
|scored_0432|E2|52-6|179||172|válido|
|scored_0432|E2|64-1|187||176|válido|
|scored_0432|E2|64-3|182||170|válido|
|scored_0432|E2|64-6|179||172|válido|
|scored_0432|E3|joint|179|individual 179|179|OPTIMAL|
|scored_0432|E4|repair|179||177|completo|
|scored_0433|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0433|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0433|E1|6|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0433|E2|52-1|177||166|válido|
|scored_0433|E2|52-3|173||165|válido|
|scored_0433|E2|52-6|171||164|válido|
|scored_0433|E2|64-1|177||165|válido|
|scored_0433|E2|64-3|173||164|válido|
|scored_0433|E2|64-6|171||163|válido|
|scored_0433|E3|joint|171|individual 171|171|OPTIMAL|
|scored_0433|E4|repair|171||171|completo|
|scored_0434|E1|1|141|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0434|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0434|E1|6|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0434|E2|52-1|141||137|válido|
|scored_0434|E2|52-3|140||137|válido|
|scored_0434|E2|52-6|140||137|válido|
|scored_0434|E2|64-1|141||136|válido|
|scored_0434|E2|64-3|140||136|válido|
|scored_0434|E2|64-6|140||136|válido|
|scored_0434|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0434|E4|repair|140||139|completo|
|scored_0435|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0435|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0435|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0435|E2|52-1|205||203|válido|
|scored_0435|E2|52-3|205||194|válido|
|scored_0435|E2|52-6|203||191|válido|
|scored_0435|E2|64-1|205||202|válido|
|scored_0435|E2|64-3|205||195|válido|
|scored_0435|E2|64-6|203||193|válido|
|scored_0435|E3|joint|203|individual 202|201|OPTIMAL|
|scored_0435|E4|repair|203||202|completo|
|scored_0436|E1|1|210|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0436|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0436|E1|6|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0436|E2|52-1|210||181|válido|
|scored_0436|E2|52-3|196||175|válido|
|scored_0436|E2|52-6|193||181|válido|
|scored_0436|E2|64-1|210||176|válido|
|scored_0436|E2|64-3|196||177|válido|
|scored_0436|E2|64-6|193||178|válido|
|scored_0436|E3|joint|193|individual 193|193|OPTIMAL|
|scored_0436|E4|repair|193||191|completo|
|scored_0437|E1|1|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0437|E1|3|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0437|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0437|E2|52-1|232||216|válido|
|scored_0437|E2|52-3|216||211|válido|
|scored_0437|E2|52-6|219||209|válido|
|scored_0437|E2|64-1|232||216|válido|
|scored_0437|E2|64-3|216||210|válido|
|scored_0437|E2|64-6|219||209|válido|
|scored_0437|E3|joint|216|individual 216|216|OPTIMAL|
|scored_0437|E4|repair|216||216|completo|
|scored_0438|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0438|E1|3|274|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0438|E1|6|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0438|E2|52-1|289||264|válido|
|scored_0438|E2|52-3|274||253|válido|
|scored_0438|E2|52-6|266||245|válido|
|scored_0438|E2|64-1|289||264|válido|
|scored_0438|E2|64-3|274||256|válido|
|scored_0438|E2|64-6|266||247|válido|
|scored_0438|E3|joint|266|individual 266|264|OPTIMAL|
|scored_0438|E4|repair|266||266|completo|
|scored_0439|E1|1|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0439|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0439|E1|6|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0439|E2|52-1|211||204|válido|
|scored_0439|E2|52-3|213||197|válido|
|scored_0439|E2|52-6|211||198|válido|
|scored_0439|E2|64-1|211||205|válido|
|scored_0439|E2|64-3|213||199|válido|
|scored_0439|E2|64-6|211||194|válido|
|scored_0439|E3|joint|211|individual 210|210|OPTIMAL|
|scored_0439|E4|repair|211||210|completo|
|scored_0440|E1|1|258|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0440|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0440|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0440|E2|52-1|258||244|válido|
|scored_0440|E2|52-3|258||239|válido|
|scored_0440|E2|52-6|255||237|válido|
|scored_0440|E2|64-1|258||244|válido|
|scored_0440|E2|64-3|258||239|válido|
|scored_0440|E2|64-6|255||238|válido|
|scored_0440|E3|joint|255|individual 255|254|OPTIMAL|
|scored_0440|E4|repair|255||255|completo|
|scored_0441|E1|1|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0441|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0441|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0441|E2|52-1|259||235|válido|
|scored_0441|E2|52-3|250||234|válido|
|scored_0441|E2|52-6|241||226|válido|
|scored_0441|E2|64-1|259||236|válido|
|scored_0441|E2|64-3|250||233|válido|
|scored_0441|E2|64-6|241||226|válido|
|scored_0441|E3|joint|241|individual 241|239|OPTIMAL|
|scored_0441|E4|repair|241||241|completo|
|scored_0442|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0442|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0442|E1|6|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0442|E2|52-1|251||230|válido|
|scored_0442|E2|52-3|239||220|válido|
|scored_0442|E2|52-6|234||220|válido|
|scored_0442|E2|64-1|251||231|válido|
|scored_0442|E2|64-3|239||222|válido|
|scored_0442|E2|64-6|234||219|válido|
|scored_0442|E3|joint|234|individual 234|230|OPTIMAL|
|scored_0442|E4|repair|234||233|completo|
|scored_0443|E1|1|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0443|E1|3|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0443|E1|6|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0443|E2|52-1|45||45|válido|
|scored_0443|E2|52-3|45||45|válido|
|scored_0443|E2|52-6|45||45|válido|
|scored_0443|E2|64-1|45||45|válido|
|scored_0443|E2|64-3|45||45|válido|
|scored_0443|E2|64-6|45||45|válido|
|scored_0443|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0443|E4|repair|45||45|completo|
|scored_0444|E1|1|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0444|E1|3|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0444|E1|6|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0444|E2|52-1|337||313|válido|
|scored_0444|E2|52-3|326||303|válido|
|scored_0444|E2|52-6|323||298|válido|
|scored_0444|E2|64-1|337||314|válido|
|scored_0444|E2|64-3|326||309|válido|
|scored_0444|E2|64-6|323||300|válido|
|scored_0444|E3|joint|323|individual 323|319|FEASIBLE|
|scored_0444|E4|repair|323||323|completo|
|scored_0445|E1|1|325|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0445|E1|3|315|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0445|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0445|E2|52-1|325||308|válido|
|scored_0445|E2|52-3|315||289|válido|
|scored_0445|E2|52-6|311||292|válido|
|scored_0445|E2|64-1|325||307|válido|
|scored_0445|E2|64-3|315||293|válido|
|scored_0445|E2|64-6|311||292|válido|
|scored_0445|E3|joint|311|individual 311|310|OPTIMAL|
|scored_0445|E4|repair|311||311|completo|
|scored_0446|E1|1|348|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0446|E1|3|338|postprocess 338, optimize_a4 338, nested_windows 338|338|válido|
|scored_0446|E1|6|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0446|E2|52-1|348||327|válido|
|scored_0446|E2|52-3|338||314|válido|
|scored_0446|E2|52-6|329||310|válido|
|scored_0446|E2|64-1|348||327|válido|
|scored_0446|E2|64-3|338||315|válido|
|scored_0446|E2|64-6|329||310|válido|
|scored_0446|E3|joint|329|individual 327|327|FEASIBLE|
|scored_0446|E4|repair|329||329|completo|
|scored_0447|E1|1|318|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0447|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0447|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0447|E2|52-1|318||287|válido|
|scored_0447|E2|52-3|299||274|válido|
|scored_0447|E2|52-6|298||272|válido|
|scored_0447|E2|64-1|318||290|válido|
|scored_0447|E2|64-3|299||281|válido|
|scored_0447|E2|64-6|298||277|válido|
|scored_0447|E3|joint|298|individual 298|297|OPTIMAL|
|scored_0447|E4|repair|298||298|completo|
|scored_0448|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0448|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0448|E1|6|274|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0448|E2|52-1|289||269|válido|
|scored_0448|E2|52-3|277||260|válido|
|scored_0448|E2|52-6|274||260|válido|
|scored_0448|E2|64-1|289||269|válido|
|scored_0448|E2|64-3|277||261|válido|
|scored_0448|E2|64-6|274||260|válido|
|scored_0448|E3|joint|274|individual 274|274|OPTIMAL|
|scored_0448|E4|repair|274||274|completo|
|scored_0449|E1|1|128|postprocess 128, optimize_a4 128, nested_windows 128|128|válido|
|scored_0449|E1|3|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0449|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0449|E2|52-1|128||126|válido|
|scored_0449|E2|52-3|127||125|válido|
|scored_0449|E2|52-6|127||125|válido|
|scored_0449|E2|64-1|128||125|válido|
|scored_0449|E2|64-3|127||125|válido|
|scored_0449|E2|64-6|127||125|válido|
|scored_0449|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0449|E4|repair|127||124|completo|
|scored_0450|E1|1|384|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|scored_0450|E1|3|372|postprocess 372, optimize_a4 372, nested_windows 372|372|válido|
|scored_0450|E1|6|360|postprocess 360, optimize_a4 360, nested_windows 360|360|válido|
|scored_0450|E2|52-1|384||360|válido|
|scored_0450|E2|52-3|372||352|válido|
|scored_0450|E2|52-6|360||342|válido|
|scored_0450|E2|64-1|384||366|válido|
|scored_0450|E2|64-3|372||350|válido|
|scored_0450|E2|64-6|360||346|válido|
|scored_0450|E3|joint|360|individual 360|360|FEASIBLE|
|scored_0450|E4|repair|360||360|completo|
|scored_0451|E1|1|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0451|E1|3|154|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0451|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0451|E2|52-1|155||149|válido|
|scored_0451|E2|52-3|154||148|válido|
|scored_0451|E2|52-6|152||148|válido|
|scored_0451|E2|64-1|155||149|válido|
|scored_0451|E2|64-3|154||148|válido|
|scored_0451|E2|64-6|152||146|válido|
|scored_0451|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0451|E4|repair|152||149|completo|
|scored_0452|E1|1|294|postprocess 293, optimize_a4 293, nested_windows 293|293|válido|
|scored_0452|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0452|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0452|E2|52-1|294||271|válido|
|scored_0452|E2|52-3|277||253|válido|
|scored_0452|E2|52-6|269||254|válido|
|scored_0452|E2|64-1|294||272|válido|
|scored_0452|E2|64-3|277||253|válido|
|scored_0452|E2|64-6|269||256|válido|
|scored_0452|E3|joint|269|individual 269|269|OPTIMAL|
|scored_0452|E4|repair|269||269|completo|
|scored_0453|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0453|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0453|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0453|E2|52-1|200||190|válido|
|scored_0453|E2|52-3|194||185|válido|
|scored_0453|E2|52-6|198||185|válido|
|scored_0453|E2|64-1|200||186|válido|
|scored_0453|E2|64-3|194||184|válido|
|scored_0453|E2|64-6|198||184|válido|
|scored_0453|E3|joint|194|individual 194|193|OPTIMAL|
|scored_0453|E4|repair|194||192|completo|
|scored_0454|E1|1|292|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0454|E1|3|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0454|E1|6|275|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0454|E2|52-1|292||271|válido|
|scored_0454|E2|52-3|275||261|válido|
|scored_0454|E2|52-6|275||258|válido|
|scored_0454|E2|64-1|292||272|válido|
|scored_0454|E2|64-3|275||261|válido|
|scored_0454|E2|64-6|275||258|válido|
|scored_0454|E3|joint|275|individual 275|274|OPTIMAL|
|scored_0454|E4|repair|275||275|completo|
|scored_0455|E1|1|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0455|E1|3|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0455|E1|6|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0455|E2|52-1|39||39|válido|
|scored_0455|E2|52-3|39||39|válido|
|scored_0455|E2|52-6|39||39|válido|
|scored_0455|E2|64-1|39||39|válido|
|scored_0455|E2|64-3|39||39|válido|
|scored_0455|E2|64-6|39||39|válido|
|scored_0455|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0455|E4|repair|39||39|completo|
|scored_0456|E1|1|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0456|E1|3|156|postprocess 156, optimize_a4 155, nested_windows 155|155|válido|
|scored_0456|E1|6|156|postprocess 156, optimize_a4 155, nested_windows 155|155|válido|
|scored_0456|E2|52-1|157||149|válido|
|scored_0456|E2|52-3|156||149|válido|
|scored_0456|E2|52-6|156||149|válido|
|scored_0456|E2|64-1|157||150|válido|
|scored_0456|E2|64-3|156||148|válido|
|scored_0456|E2|64-6|156||148|válido|
|scored_0456|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0456|E4|repair|156||154|completo|
|scored_0457|E1|1|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0457|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0457|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0457|E2|52-1|191||181|válido|
|scored_0457|E2|52-3|188||176|válido|
|scored_0457|E2|52-6|186||175|válido|
|scored_0457|E2|64-1|191||180|válido|
|scored_0457|E2|64-3|188||176|válido|
|scored_0457|E2|64-6|186||176|válido|
|scored_0457|E3|joint|186|individual 186|185|OPTIMAL|
|scored_0457|E4|repair|186||185|completo|
|scored_0458|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0458|E1|3|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0458|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0458|E2|52-1|211||194|válido|
|scored_0458|E2|52-3|204||190|válido|
|scored_0458|E2|52-6|197||189|válido|
|scored_0458|E2|64-1|211||196|válido|
|scored_0458|E2|64-3|204||190|válido|
|scored_0458|E2|64-6|197||188|válido|
|scored_0458|E3|joint|197|individual 197|197|OPTIMAL|
|scored_0458|E4|repair|197||197|completo|
|scored_0459|E1|1|115|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0459|E1|3|115|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0459|E1|6|115|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0459|E2|52-1|115||109|válido|
|scored_0459|E2|52-3|115||109|válido|
|scored_0459|E2|52-6|115||109|válido|
|scored_0459|E2|64-1|115||109|válido|
|scored_0459|E2|64-3|115||109|válido|
|scored_0459|E2|64-6|115||109|válido|
|scored_0459|E3|joint|115|individual 115|115|OPTIMAL|
|scored_0459|E4|repair|115||112|completo|
|scored_0460|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0460|E1|3|217|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0460|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0460|E2|52-1|220||211|válido|
|scored_0460|E2|52-3|217||205|válido|
|scored_0460|E2|52-6|211||202|válido|
|scored_0460|E2|64-1|220||212|válido|
|scored_0460|E2|64-3|217||206|válido|
|scored_0460|E2|64-6|211||204|válido|
|scored_0460|E3|joint|211|individual 211|211|OPTIMAL|
|scored_0460|E4|repair|211||211|completo|
|scored_0461|E1|1|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0461|E1|3|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0461|E1|6|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0461|E2|52-1|46||46|válido|
|scored_0461|E2|52-3|46||46|válido|
|scored_0461|E2|52-6|46||46|válido|
|scored_0461|E2|64-1|46||46|válido|
|scored_0461|E2|64-3|46||46|válido|
|scored_0461|E2|64-6|46||46|válido|
|scored_0461|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0461|E4|repair|46||45|completo|
|scored_0462|E1|1|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0462|E1|3|155|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0462|E1|6|152|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0462|E2|52-1|155||147|válido|
|scored_0462|E2|52-3|155||147|válido|
|scored_0462|E2|52-6|152||146|válido|
|scored_0462|E2|64-1|155||146|válido|
|scored_0462|E2|64-3|155||146|válido|
|scored_0462|E2|64-6|152||145|válido|
|scored_0462|E3|joint|152|individual 152|151|OPTIMAL|
|scored_0462|E4|repair|152||150|completo|
|scored_0463|E1|1|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0463|E1|3|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0463|E1|6|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0463|E2|52-1|139||134|válido|
|scored_0463|E2|52-3|139||134|válido|
|scored_0463|E2|52-6|139||133|válido|
|scored_0463|E2|64-1|139||135|válido|
|scored_0463|E2|64-3|139||135|válido|
|scored_0463|E2|64-6|139||135|válido|
|scored_0463|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0463|E4|repair|139||135|completo|
|scored_0464|E1|1|105|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0464|E1|3|105|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0464|E1|6|104|postprocess 102, optimize_a4 102, nested_windows 102|102|válido|
|scored_0464|E2|52-1|105||102|válido|
|scored_0464|E2|52-3|105||102|válido|
|scored_0464|E2|52-6|104||102|válido|
|scored_0464|E2|64-1|105||102|válido|
|scored_0464|E2|64-3|105||102|válido|
|scored_0464|E2|64-6|104||102|válido|
|scored_0464|E3|joint|104|individual 104|104|OPTIMAL|
|scored_0464|E4|repair|104||102|completo|
|scored_0465|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0465|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0465|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0465|E2|52-1|206||188|válido|
|scored_0465|E2|52-3|197||187|válido|
|scored_0465|E2|52-6|195||178|válido|
|scored_0465|E2|64-1|206||189|válido|
|scored_0465|E2|64-3|197||187|válido|
|scored_0465|E2|64-6|195||181|válido|
|scored_0465|E3|joint|195|individual 195|194|OPTIMAL|
|scored_0465|E4|repair|195||193|completo|
|scored_0466|E1|1|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0466|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0466|E1|6|253|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0466|E2|52-1|264||237|válido|
|scored_0466|E2|52-3|260||231|válido|
|scored_0466|E2|52-6|253||231|válido|
|scored_0466|E2|64-1|264||240|válido|
|scored_0466|E2|64-3|260||232|válido|
|scored_0466|E2|64-6|253||231|válido|
|scored_0466|E3|joint|253|individual 252|252|OPTIMAL|
|scored_0466|E4|repair|253||253|completo|
|scored_0467|E1|1|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0467|E1|3|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0467|E1|6|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0467|E2|52-1|130||123|válido|
|scored_0467|E2|52-3|129||122|válido|
|scored_0467|E2|52-6|129||122|válido|
|scored_0467|E2|64-1|130||123|válido|
|scored_0467|E2|64-3|129||123|válido|
|scored_0467|E2|64-6|129||123|válido|
|scored_0467|E3|joint|129|individual 129|129|OPTIMAL|
|scored_0467|E4|repair|129||124|completo|
|scored_0468|E1|1|215|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0468|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0468|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0468|E2|52-1|215||200|válido|
|scored_0468|E2|52-3|209||194|válido|
|scored_0468|E2|52-6|208||193|válido|
|scored_0468|E2|64-1|215||199|válido|
|scored_0468|E2|64-3|209||195|válido|
|scored_0468|E2|64-6|208||193|válido|
|scored_0468|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0468|E4|repair|208||208|completo|
|scored_0469|E1|1|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0469|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0469|E1|6|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0469|E2|52-1|111||111|válido|
|scored_0469|E2|52-3|111||111|válido|
|scored_0469|E2|52-6|111||111|válido|
|scored_0469|E2|64-1|111||111|válido|
|scored_0469|E2|64-3|111||111|válido|
|scored_0469|E2|64-6|111||111|válido|
|scored_0469|E3|joint|111|individual 111|111|OPTIMAL|
|scored_0469|E4|repair|111||107|completo|
|scored_0470|E1|1|374|postprocess 374, optimize_a4 374, nested_windows 374|374|válido|
|scored_0470|E1|3|346|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|scored_0470|E1|6|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0470|E2|52-1|374||341|válido|
|scored_0470|E2|52-3|346||329|válido|
|scored_0470|E2|52-6|347||327|válido|
|scored_0470|E2|64-1|374||344|válido|
|scored_0470|E2|64-3|346||589|válido|
|scored_0470|E2|64-6|347||430|válido|
|scored_0470|E3|joint|346|individual 345|344|FEASIBLE|
|scored_0470|E4|repair|346||346|completo|
|scored_0471|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0471|E1|3|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0471|E1|6|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0471|E2|52-1|133||126|válido|
|scored_0471|E2|52-3|132||126|válido|
|scored_0471|E2|52-6|132||126|válido|
|scored_0471|E2|64-1|133||126|válido|
|scored_0471|E2|64-3|132||126|válido|
|scored_0471|E2|64-6|132||126|válido|
|scored_0471|E3|joint|132|individual 132|132|OPTIMAL|
|scored_0471|E4|repair|132||130|completo|
|scored_0472|E1|1|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0472|E1|3|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0472|E1|6|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0472|E2|52-1|33||33|válido|
|scored_0472|E2|52-3|33||33|válido|
|scored_0472|E2|52-6|33||33|válido|
|scored_0472|E2|64-1|33||33|válido|
|scored_0472|E2|64-3|33||33|válido|
|scored_0472|E2|64-6|33||33|válido|
|scored_0472|E3|joint|33|individual 33|33|OPTIMAL|
|scored_0472|E4|repair|33||32|completo|
|scored_0473|E1|1|39|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0473|E1|3|39|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0473|E1|6|39|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0473|E2|52-1|39||39|válido|
|scored_0473|E2|52-3|39||39|válido|
|scored_0473|E2|52-6|39||39|válido|
|scored_0473|E2|64-1|39||39|válido|
|scored_0473|E2|64-3|39||39|válido|
|scored_0473|E2|64-6|39||39|válido|
|scored_0473|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0473|E4|repair|39||37|completo|
|scored_0474|E1|1|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0474|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0474|E1|6|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0474|E2|52-1|111||109|válido|
|scored_0474|E2|52-3|111||109|válido|
|scored_0474|E2|52-6|111||109|válido|
|scored_0474|E2|64-1|111||109|válido|
|scored_0474|E2|64-3|111||109|válido|
|scored_0474|E2|64-6|111||109|válido|
|scored_0474|E3|joint|111|individual 111|111|OPTIMAL|
|scored_0474|E4|repair|111||108|completo|
|scored_0475|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0475|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0475|E1|6|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0475|E2|52-1|289||267|válido|
|scored_0475|E2|52-3|278||261|válido|
|scored_0475|E2|52-6|264||248|válido|
|scored_0475|E2|64-1|289||267|válido|
|scored_0475|E2|64-3|278||258|válido|
|scored_0475|E2|64-6|264||248|válido|
|scored_0475|E3|joint|264|individual 264|263|OPTIMAL|
|scored_0475|E4|repair|264||264|completo|
|scored_0476|E1|1|303|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0476|E1|3|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0476|E1|6|285|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0476|E2|52-1|303||276|válido|
|scored_0476|E2|52-3|301||268|válido|
|scored_0476|E2|52-6|285||279|válido|
|scored_0476|E2|64-1|303||277|válido|
|scored_0476|E2|64-3|301||271|válido|
|scored_0476|E2|64-6|285||262|válido|
|scored_0476|E3|joint|285|individual 285|284|OPTIMAL|
|scored_0476|E4|repair|285||285|completo|
|scored_0477|E1|1|235|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0477|E1|3|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0477|E1|6|217|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0477|E2|52-1|235||215|válido|
|scored_0477|E2|52-3|228||206|válido|
|scored_0477|E2|52-6|217||209|válido|
|scored_0477|E2|64-1|235||212|válido|
|scored_0477|E2|64-3|228||206|válido|
|scored_0477|E2|64-6|217||211|válido|
|scored_0477|E3|joint|217|individual 217|216|OPTIMAL|
|scored_0477|E4|repair|217||217|completo|
|scored_0478|E1|1|109|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0478|E1|3|110|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0478|E1|6|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0478|E2|52-1|109||107|válido|
|scored_0478|E2|52-3|110||102|válido|
|scored_0478|E2|52-6|108||99|válido|
|scored_0478|E2|64-1|109||107|válido|
|scored_0478|E2|64-3|110||102|válido|
|scored_0478|E2|64-6|108||99|válido|
|scored_0478|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0478|E4|repair|108||108|completo|
|scored_0479|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0479|E1|3|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0479|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0479|E2|52-1|158||147|válido|
|scored_0479|E2|52-3|155||148|válido|
|scored_0479|E2|52-6|155||147|válido|
|scored_0479|E2|64-1|158||146|válido|
|scored_0479|E2|64-3|155||144|válido|
|scored_0479|E2|64-6|155||144|válido|
|scored_0479|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0479|E4|repair|155||152|completo|
|scored_0480|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0480|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0480|E1|6|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0480|E2|52-1|206||189|válido|
|scored_0480|E2|52-3|202||184|válido|
|scored_0480|E2|52-6|199||185|válido|
|scored_0480|E2|64-1|206||190|válido|
|scored_0480|E2|64-3|202||185|válido|
|scored_0480|E2|64-6|199||183|válido|
|scored_0480|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0480|E4|repair|199||198|completo|
|scored_0481|E1|1|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0481|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0481|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0481|E2|52-1|244||233|válido|
|scored_0481|E2|52-3|243||226|válido|
|scored_0481|E2|52-6|242||224|válido|
|scored_0481|E2|64-1|244||233|válido|
|scored_0481|E2|64-3|243||229|válido|
|scored_0481|E2|64-6|242||228|válido|
|scored_0481|E3|joint|242|individual 242|242|OPTIMAL|
|scored_0481|E4|repair|242||242|completo|
|scored_0482|E1|1|262|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0482|E1|3|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0482|E1|6|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0482|E2|52-1|262||239|válido|
|scored_0482|E2|52-3|255||238|válido|
|scored_0482|E2|52-6|247||235|válido|
|scored_0482|E2|64-1|262||240|válido|
|scored_0482|E2|64-3|255||238|válido|
|scored_0482|E2|64-6|247||236|válido|
|scored_0482|E3|joint|247|individual 247|246|OPTIMAL|
|scored_0482|E4|repair|247||246|completo|
|scored_0483|E1|1|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0483|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0483|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0483|E2|52-1|232||220|válido|
|scored_0483|E2|52-3|223||209|válido|
|scored_0483|E2|52-6|221||210|válido|
|scored_0483|E2|64-1|232||223|válido|
|scored_0483|E2|64-3|223||209|válido|
|scored_0483|E2|64-6|221||210|válido|
|scored_0483|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0483|E4|repair|221||217|completo|
|scored_0484|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0484|E1|3|190|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0484|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0484|E2|52-1|192||180|válido|
|scored_0484|E2|52-3|190||176|válido|
|scored_0484|E2|52-6|188||176|válido|
|scored_0484|E2|64-1|192||180|válido|
|scored_0484|E2|64-3|190||176|válido|
|scored_0484|E2|64-6|188||176|válido|
|scored_0484|E3|joint|188|individual 188|187|OPTIMAL|
|scored_0484|E4|repair|188||188|completo|
|scored_0485|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0485|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0485|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0485|E2|52-1|2||2|válido|
|scored_0485|E2|52-3|2||2|válido|
|scored_0485|E2|52-6|2||2|válido|
|scored_0485|E2|64-1|2||2|válido|
|scored_0485|E2|64-3|2||2|válido|
|scored_0485|E2|64-6|2||2|válido|
|scored_0485|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0485|E4|repair|2||2|completo|
|scored_0486|E1|1|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0486|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0486|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0486|E2|52-1|148||144|válido|
|scored_0486|E2|52-3|147||142|válido|
|scored_0486|E2|52-6|145||144|válido|
|scored_0486|E2|64-1|148||140|válido|
|scored_0486|E2|64-3|147||140|válido|
|scored_0486|E2|64-6|145||140|válido|
|scored_0486|E3|joint|145|individual 145|144|OPTIMAL|
|scored_0486|E4|repair|145||145|completo|
|scored_0487|E1|1|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0487|E1|3|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0487|E1|6|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0487|E2|52-1|44||44|válido|
|scored_0487|E2|52-3|44||44|válido|
|scored_0487|E2|52-6|44||44|válido|
|scored_0487|E2|64-1|44||44|válido|
|scored_0487|E2|64-3|44||44|válido|
|scored_0487|E2|64-6|44||44|válido|
|scored_0487|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0487|E4|repair|44||42|completo|
|scored_0488|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0488|E1|3|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0488|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0488|E2|52-1|199||180|válido|
|scored_0488|E2|52-3|193||183|válido|
|scored_0488|E2|52-6|189||179|válido|
|scored_0488|E2|64-1|199||184|válido|
|scored_0488|E2|64-3|193||179|válido|
|scored_0488|E2|64-6|189||177|válido|
|scored_0488|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0488|E4|repair|189||189|completo|
|scored_0489|E1|1|195|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0489|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0489|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0489|E2|52-1|195||183|válido|
|scored_0489|E2|52-3|191||182|válido|
|scored_0489|E2|52-6|191||180|válido|
|scored_0489|E2|64-1|195||180|válido|
|scored_0489|E2|64-3|191||180|válido|
|scored_0489|E2|64-6|191||179|válido|
|scored_0489|E3|joint|191|individual 191|190|OPTIMAL|
|scored_0489|E4|repair|191||190|completo|
|scored_0490|E1|1|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0490|E1|3|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0490|E1|6|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0490|E2|52-1|6||6|válido|
|scored_0490|E2|52-3|6||6|válido|
|scored_0490|E2|52-6|6||6|válido|
|scored_0490|E2|64-1|6||6|válido|
|scored_0490|E2|64-3|6||6|válido|
|scored_0490|E2|64-6|6||6|válido|
|scored_0490|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0490|E4|repair|4||4|completo|
|scored_0491|E1|1|269|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0491|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0491|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0491|E2|52-1|269||242|válido|
|scored_0491|E2|52-3|258||238|válido|
|scored_0491|E2|52-6|255||236|válido|
|scored_0491|E2|64-1|269||242|válido|
|scored_0491|E2|64-3|258||237|válido|
|scored_0491|E2|64-6|255||241|válido|
|scored_0491|E3|joint|255|individual 255|254|OPTIMAL|
|scored_0491|E4|repair|255||255|completo|
|scored_0492|E1|1|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0492|E1|3|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0492|E1|6|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0492|E2|52-1|52||52|válido|
|scored_0492|E2|52-3|52||52|válido|
|scored_0492|E2|52-6|52||52|válido|
|scored_0492|E2|64-1|52||52|válido|
|scored_0492|E2|64-3|52||52|válido|
|scored_0492|E2|64-6|52||52|válido|
|scored_0492|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0492|E4|repair|52||52|completo|
|scored_0493|E1|1|342|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|scored_0493|E1|3|356|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0493|E1|6|360|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0493|E2|52-1|342||313|válido|
|scored_0493|E2|52-3|356||311|válido|
|scored_0493|E2|52-6|360||301|válido|
|scored_0493|E2|64-1|342||315|válido|
|scored_0493|E2|64-3|356||316|válido|
|scored_0493|E2|64-6|360||306|válido|
|scored_0493|E3|joint|342|individual 329|324|OPTIMAL|
|scored_0493|E4|repair|342||342|completo|
|scored_0494|E1|1|320|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0494|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0494|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0494|E2|52-1|320||281|válido|
|scored_0494|E2|52-3|308||280|válido|
|scored_0494|E2|52-6|292||278|válido|
|scored_0494|E2|64-1|320||287|válido|
|scored_0494|E2|64-3|308||282|válido|
|scored_0494|E2|64-6|292||292|válido|
|scored_0494|E3|joint|292|individual 292|291|OPTIMAL|
|scored_0494|E4|repair|292||292|completo|
|scored_0495|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0495|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0495|E1|6|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0495|E2|52-1|153||144|válido|
|scored_0495|E2|52-3|153||144|válido|
|scored_0495|E2|52-6|153||144|válido|
|scored_0495|E2|64-1|153||144|válido|
|scored_0495|E2|64-3|153||144|válido|
|scored_0495|E2|64-6|153||143|válido|
|scored_0495|E3|joint|153|individual 153|153|OPTIMAL|
|scored_0495|E4|repair|153||150|completo|
|scored_0496|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0496|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0496|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0496|E2|52-1|177||171|válido|
|scored_0496|E2|52-3|174||166|válido|
|scored_0496|E2|52-6|172||166|válido|
|scored_0496|E2|64-1|177||168|válido|
|scored_0496|E2|64-3|174||160|válido|
|scored_0496|E2|64-6|172||160|válido|
|scored_0496|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0496|E4|repair|172||172|completo|
|scored_0497|E1|1|250|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0497|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0497|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0497|E2|52-1|250||229|válido|
|scored_0497|E2|52-3|231||218|válido|
|scored_0497|E2|52-6|229||218|válido|
|scored_0497|E2|64-1|250||228|válido|
|scored_0497|E2|64-3|231||216|válido|
|scored_0497|E2|64-6|229||216|válido|
|scored_0497|E3|joint|229|individual 229|227|OPTIMAL|
|scored_0497|E4|repair|229||229|completo|
|scored_0498|E1|1|211|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0498|E1|3|207|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0498|E1|6|209|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0498|E2|52-1|211||197|válido|
|scored_0498|E2|52-3|207||201|válido|
|scored_0498|E2|52-6|209||194|válido|
|scored_0498|E2|64-1|211||193|válido|
|scored_0498|E2|64-3|207||197|válido|
|scored_0498|E2|64-6|209||192|válido|
|scored_0498|E3|joint|207|individual 207|207|OPTIMAL|
|scored_0498|E4|repair|207||205|completo|
|scored_0499|E1|1|210|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0499|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0499|E1|6|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0499|E2|52-1|210||188|válido|
|scored_0499|E2|52-3|208||182|válido|
|scored_0499|E2|52-6|202||175|válido|
|scored_0499|E2|64-1|210||191|válido|
|scored_0499|E2|64-3|208||185|válido|
|scored_0499|E2|64-6|202||176|válido|
|scored_0499|E3|joint|202|individual 202|202|OPTIMAL|
|scored_0499|E4|repair|202||201|completo|
|scored_0500|E1|1|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0500|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0500|E1|6|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0500|E2|52-1|111||107|válido|
|scored_0500|E2|52-3|111||107|válido|
|scored_0500|E2|52-6|111||107|válido|
|scored_0500|E2|64-1|111||106|válido|
|scored_0500|E2|64-3|111||106|válido|
|scored_0500|E2|64-6|111||106|válido|
|scored_0500|E3|joint|111|individual 111|111|OPTIMAL|
|scored_0500|E4|repair|111||109|completo|
|scored_0501|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0501|E1|3|185|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0501|E1|6|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0501|E2|52-1|197||178|válido|
|scored_0501|E2|52-3|185||179|válido|
|scored_0501|E2|52-6|183||171|válido|
|scored_0501|E2|64-1|197||179|válido|
|scored_0501|E2|64-3|185||177|válido|
|scored_0501|E2|64-6|183||172|válido|
|scored_0501|E3|joint|183|individual 182|182|OPTIMAL|
|scored_0501|E4|repair|183||183|completo|
|scored_0502|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0502|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0502|E1|6|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0502|E2|52-1|216||201|válido|
|scored_0502|E2|52-3|215||201|válido|
|scored_0502|E2|52-6|213||202|válido|
|scored_0502|E2|64-1|216||204|válido|
|scored_0502|E2|64-3|215||201|válido|
|scored_0502|E2|64-6|213||202|válido|
|scored_0502|E3|joint|213|individual 213|211|OPTIMAL|
|scored_0502|E4|repair|213||213|completo|
|scored_0503|E1|1|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0503|E1|3|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0503|E1|6|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0503|E2|52-1|110||107|válido|
|scored_0503|E2|52-3|110||107|válido|
|scored_0503|E2|52-6|110||107|válido|
|scored_0503|E2|64-1|110||107|válido|
|scored_0503|E2|64-3|110||107|válido|
|scored_0503|E2|64-6|110||107|válido|
|scored_0503|E3|joint|110|individual 110|110|OPTIMAL|
|scored_0503|E4|repair|110||110|completo|
|scored_0504|E1|1|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0504|E1|3|254|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0504|E1|6|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0504|E2|52-1|267||243|válido|
|scored_0504|E2|52-3|254||238|válido|
|scored_0504|E2|52-6|253||238|válido|
|scored_0504|E2|64-1|267||243|válido|
|scored_0504|E2|64-3|254||240|válido|
|scored_0504|E2|64-6|253||238|válido|
|scored_0504|E3|joint|253|individual 252|251|OPTIMAL|
|scored_0504|E4|repair|253||253|completo|
|scored_0505|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0505|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0505|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0505|E2|52-1|163||154|válido|
|scored_0505|E2|52-3|162||152|válido|
|scored_0505|E2|52-6|160||153|válido|
|scored_0505|E2|64-1|163||153|válido|
|scored_0505|E2|64-3|162||150|válido|
|scored_0505|E2|64-6|160||150|válido|
|scored_0505|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0505|E4|repair|160||159|completo|
|scored_0506|E1|1|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0506|E1|3|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0506|E1|6|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0506|E2|52-1|159||149|válido|
|scored_0506|E2|52-3|158||147|válido|
|scored_0506|E2|52-6|158||147|válido|
|scored_0506|E2|64-1|159||149|válido|
|scored_0506|E2|64-3|158||147|válido|
|scored_0506|E2|64-6|158||147|válido|
|scored_0506|E3|joint|158|individual 157|157|OPTIMAL|
|scored_0506|E4|repair|158||156|completo|
|scored_0507|E1|1|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0507|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0507|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0507|E2|52-1|227||213|válido|
|scored_0507|E2|52-3|220||208|válido|
|scored_0507|E2|52-6|219||205|válido|
|scored_0507|E2|64-1|227||213|válido|
|scored_0507|E2|64-3|220||207|válido|
|scored_0507|E2|64-6|219||205|válido|
|scored_0507|E3|joint|219|individual 219|219|OPTIMAL|
|scored_0507|E4|repair|219||219|completo|
|scored_0508|E1|1|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0508|E1|3|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0508|E1|6|276|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0508|E2|52-1|297||262|válido|
|scored_0508|E2|52-3|276||259|válido|
|scored_0508|E2|52-6|276||254|válido|
|scored_0508|E2|64-1|297||263|válido|
|scored_0508|E2|64-3|276||265|válido|
|scored_0508|E2|64-6|276||252|válido|
|scored_0508|E3|joint|276|individual 276|271|OPTIMAL|
|scored_0508|E4|repair|276||276|completo|
|scored_0509|E1|1|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0509|E1|3|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0509|E1|6|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0509|E2|52-1|182||172|válido|
|scored_0509|E2|52-3|182||166|válido|
|scored_0509|E2|52-6|178||166|válido|
|scored_0509|E2|64-1|182||174|válido|
|scored_0509|E2|64-3|182||167|válido|
|scored_0509|E2|64-6|178||167|válido|
|scored_0509|E3|joint|178|individual 178|178|OPTIMAL|
|scored_0509|E4|repair|178||178|completo|
|scored_0510|E1|1|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0510|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0510|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0510|E2|52-1|116||111|válido|
|scored_0510|E2|52-3|116||111|válido|
|scored_0510|E2|52-6|116||108|válido|
|scored_0510|E2|64-1|116||111|válido|
|scored_0510|E2|64-3|116||111|válido|
|scored_0510|E2|64-6|116||108|válido|
|scored_0510|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0510|E4|repair|116||112|completo|
|scored_0511|E1|1|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0511|E1|3|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0511|E1|6|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0511|E2|52-1|333||317|válido|
|scored_0511|E2|52-3|321||307|válido|
|scored_0511|E2|52-6|326||306|válido|
|scored_0511|E2|64-1|333||314|válido|
|scored_0511|E2|64-3|321||305|válido|
|scored_0511|E2|64-6|326||304|válido|
|scored_0511|E3|joint|321|individual 321|318|OPTIMAL|
|scored_0511|E4|repair|321||321|completo|
|scored_0512|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0512|E1|3|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0512|E1|6|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0512|E2|52-1|206||191|válido|
|scored_0512|E2|52-3|198||190|válido|
|scored_0512|E2|52-6|194||187|válido|
|scored_0512|E2|64-1|206||196|válido|
|scored_0512|E2|64-3|198||190|válido|
|scored_0512|E2|64-6|194||190|válido|
|scored_0512|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0512|E4|repair|194||189|completo|
|scored_0513|E1|1|201|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0513|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0513|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0513|E2|52-1|201||190|válido|
|scored_0513|E2|52-3|195||183|válido|
|scored_0513|E2|52-6|195||183|válido|
|scored_0513|E2|64-1|201||190|válido|
|scored_0513|E2|64-3|195||183|válido|
|scored_0513|E2|64-6|195||183|válido|
|scored_0513|E3|joint|195|individual 195|195|OPTIMAL|
|scored_0513|E4|repair|195||195|completo|
|scored_0514|E1|1|306|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0514|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0514|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0514|E2|52-1|306||278|válido|
|scored_0514|E2|52-3|290||270|válido|
|scored_0514|E2|52-6|281||267|válido|
|scored_0514|E2|64-1|306||277|válido|
|scored_0514|E2|64-3|290||272|válido|
|scored_0514|E2|64-6|281||268|válido|
|scored_0514|E3|joint|281|individual 281|280|FEASIBLE|
|scored_0514|E4|repair|281||281|completo|
|scored_0515|E1|1|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0515|E1|3|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0515|E1|6|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0515|E2|52-1|39||39|válido|
|scored_0515|E2|52-3|39||39|válido|
|scored_0515|E2|52-6|39||39|válido|
|scored_0515|E2|64-1|39||39|válido|
|scored_0515|E2|64-3|39||39|válido|
|scored_0515|E2|64-6|39||39|válido|
|scored_0515|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0515|E4|repair|39||39|completo|
|scored_0516|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0516|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0516|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0516|E2|52-1|216||204|válido|
|scored_0516|E2|52-3|209||204|válido|
|scored_0516|E2|52-6|208||199|válido|
|scored_0516|E2|64-1|216||207|válido|
|scored_0516|E2|64-3|209||201|válido|
|scored_0516|E2|64-6|208||201|válido|
|scored_0516|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0516|E4|repair|208||208|completo|
|scored_0517|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0517|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0517|E1|6|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0517|E2|52-1|211||202|válido|
|scored_0517|E2|52-3|209||198|válido|
|scored_0517|E2|52-6|205||198|válido|
|scored_0517|E2|64-1|211||202|válido|
|scored_0517|E2|64-3|209||198|válido|
|scored_0517|E2|64-6|205||198|válido|
|scored_0517|E3|joint|205|individual 205|204|OPTIMAL|
|scored_0517|E4|repair|205||203|completo|
|scored_0518|E1|1|269|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0518|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0518|E1|6|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0518|E2|52-1|269||253|válido|
|scored_0518|E2|52-3|265||247|válido|
|scored_0518|E2|52-6|263||245|válido|
|scored_0518|E2|64-1|269||253|válido|
|scored_0518|E2|64-3|265||247|válido|
|scored_0518|E2|64-6|263||244|válido|
|scored_0518|E3|joint|263|individual 262|262|OPTIMAL|
|scored_0518|E4|repair|263||263|completo|
|scored_0519|E1|1|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0519|E1|3|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0519|E1|6|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0519|E2|52-1|118||115|válido|
|scored_0519|E2|52-3|118||113|válido|
|scored_0519|E2|52-6|118||113|válido|
|scored_0519|E2|64-1|118||114|válido|
|scored_0519|E2|64-3|118||113|válido|
|scored_0519|E2|64-6|118||113|válido|
|scored_0519|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0519|E4|repair|118||114|completo|
|scored_0520|E1|1|236|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0520|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0520|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0520|E2|52-1|236||222|válido|
|scored_0520|E2|52-3|231||222|válido|
|scored_0520|E2|52-6|229||219|válido|
|scored_0520|E2|64-1|236||223|válido|
|scored_0520|E2|64-3|231||222|válido|
|scored_0520|E2|64-6|229||222|válido|
|scored_0520|E3|joint|229|individual 229|229|OPTIMAL|
|scored_0520|E4|repair|229||228|completo|
|scored_0521|E1|1|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0521|E1|3|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0521|E1|6|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0521|E2|52-1|219||208|válido|
|scored_0521|E2|52-3|213||203|válido|
|scored_0521|E2|52-6|208||198|válido|
|scored_0521|E2|64-1|219||203|válido|
|scored_0521|E2|64-3|213||199|válido|
|scored_0521|E2|64-6|208||196|válido|
|scored_0521|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0521|E4|repair|208||208|completo|
|scored_0522|E1|1|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0522|E1|3|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0522|E1|6|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0522|E2|52-1|121||118|válido|
|scored_0522|E2|52-3|121||118|válido|
|scored_0522|E2|52-6|121||118|válido|
|scored_0522|E2|64-1|121||118|válido|
|scored_0522|E2|64-3|121||117|válido|
|scored_0522|E2|64-6|121||117|válido|
|scored_0522|E3|joint|121|individual 121|121|OPTIMAL|
|scored_0522|E4|repair|121||119|completo|
|scored_0523|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0523|E1|3|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0523|E1|6|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0523|E2|52-1|163||159|válido|
|scored_0523|E2|52-3|160||155|válido|
|scored_0523|E2|52-6|159||155|válido|
|scored_0523|E2|64-1|163||157|válido|
|scored_0523|E2|64-3|160||156|válido|
|scored_0523|E2|64-6|159||155|válido|
|scored_0523|E3|joint|159|individual 159|159|OPTIMAL|
|scored_0523|E4|repair|159||158|completo|
|scored_0524|E1|1|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0524|E1|3|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0524|E1|6|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0524|E2|52-1|184||163|válido|
|scored_0524|E2|52-3|176||163|válido|
|scored_0524|E2|52-6|173||160|válido|
|scored_0524|E2|64-1|184||163|válido|
|scored_0524|E2|64-3|176||161|válido|
|scored_0524|E2|64-6|173||160|válido|
|scored_0524|E3|joint|173|individual 173|173|OPTIMAL|
|scored_0524|E4|repair|173||170|completo|
|scored_0525|E1|1|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0525|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0525|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0525|E2|52-1|161||152|válido|
|scored_0525|E2|52-3|160||152|válido|
|scored_0525|E2|52-6|155||152|válido|
|scored_0525|E2|64-1|161||151|válido|
|scored_0525|E2|64-3|160||151|válido|
|scored_0525|E2|64-6|155||151|válido|
|scored_0525|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0525|E4|repair|155||154|completo|
|scored_0526|E1|1|300|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0526|E1|3|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0526|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0526|E2|52-1|300||282|válido|
|scored_0526|E2|52-3|289||271|válido|
|scored_0526|E2|52-6|284||265|válido|
|scored_0526|E2|64-1|300||282|válido|
|scored_0526|E2|64-3|289||271|válido|
|scored_0526|E2|64-6|284||267|válido|
|scored_0526|E3|joint|284|individual 283|283|FEASIBLE|
|scored_0526|E4|repair|284||284|completo|
|scored_0527|E1|1|315|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0527|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0527|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0527|E2|52-1|315||287|válido|
|scored_0527|E2|52-3|310||280|válido|
|scored_0527|E2|52-6|298||276|válido|
|scored_0527|E2|64-1|315||294|válido|
|scored_0527|E2|64-3|310||284|válido|
|scored_0527|E2|64-6|298||280|válido|
|scored_0527|E3|joint|298|individual 298|298|rechazado por registros|
|scored_0527|E4|repair|298||298|completo|
|scored_0528|E1|1|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0528|E1|3|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0528|E1|6|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0528|E2|52-1|183||175|válido|
|scored_0528|E2|52-3|181||174|válido|
|scored_0528|E2|52-6|180||170|válido|
|scored_0528|E2|64-1|183||174|válido|
|scored_0528|E2|64-3|181||171|válido|
|scored_0528|E2|64-6|180||171|válido|
|scored_0528|E3|joint|180|individual 180|180|OPTIMAL|
|scored_0528|E4|repair|180||180|completo|
|scored_0529|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0529|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0529|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0529|E2|52-1|204||188|válido|
|scored_0529|E2|52-3|206||192|válido|
|scored_0529|E2|52-6|199||186|válido|
|scored_0529|E2|64-1|204||192|válido|
|scored_0529|E2|64-3|206||194|válido|
|scored_0529|E2|64-6|199||190|válido|
|scored_0529|E3|joint|199|individual 198|196|OPTIMAL|
|scored_0529|E4|repair|199||198|completo|
|scored_0530|E1|1|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0530|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0530|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0530|E2|52-1|220||198|válido|
|scored_0530|E2|52-3|214||199|válido|
|scored_0530|E2|52-6|212||196|válido|
|scored_0530|E2|64-1|220||200|válido|
|scored_0530|E2|64-3|214||200|válido|
|scored_0530|E2|64-6|212||194|válido|
|scored_0530|E3|joint|212|individual 212|212|OPTIMAL|
|scored_0530|E4|repair|212||212|completo|
|scored_0531|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0531|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0531|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0531|E2|52-1|270||247|válido|
|scored_0531|E2|52-3|260||245|válido|
|scored_0531|E2|52-6|255||238|válido|
|scored_0531|E2|64-1|270||249|válido|
|scored_0531|E2|64-3|260||240|válido|
|scored_0531|E2|64-6|255||238|válido|
|scored_0531|E3|joint|255|individual 255|255|OPTIMAL|
|scored_0531|E4|repair|255||255|completo|
|scored_0532|E1|1|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0532|E1|3|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0532|E1|6|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0532|E2|52-1|107||108|válido|
|scored_0532|E2|52-3|107||108|válido|
|scored_0532|E2|52-6|107||108|válido|
|scored_0532|E2|64-1|107||105|válido|
|scored_0532|E2|64-3|107||105|válido|
|scored_0532|E2|64-6|107||105|válido|
|scored_0532|E3|joint|107|individual 107|107|OPTIMAL|
|scored_0532|E4|repair|107||104|completo|
|scored_0533|E1|1|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0533|E1|3|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0533|E1|6|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0533|E2|52-1|150||144|válido|
|scored_0533|E2|52-3|150||144|válido|
|scored_0533|E2|52-6|150||144|válido|
|scored_0533|E2|64-1|150||143|válido|
|scored_0533|E2|64-3|150||143|válido|
|scored_0533|E2|64-6|150||143|válido|
|scored_0533|E3|joint|150|individual 150|150|OPTIMAL|
|scored_0533|E4|repair|150||150|completo|
|scored_0534|E1|1|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0534|E1|3|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0534|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0534|E2|52-1|179||163|válido|
|scored_0534|E2|52-3|172||164|válido|
|scored_0534|E2|52-6|166||161|válido|
|scored_0534|E2|64-1|179||163|válido|
|scored_0534|E2|64-3|172||163|válido|
|scored_0534|E2|64-6|166||161|válido|
|scored_0534|E3|joint|166|individual 166|165|OPTIMAL|
|scored_0534|E4|repair|166||165|completo|
|scored_0535|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0535|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0535|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0535|E2|52-1|190||170|válido|
|scored_0535|E2|52-3|192||170|válido|
|scored_0535|E2|52-6|188||167|válido|
|scored_0535|E2|64-1|190||172|válido|
|scored_0535|E2|64-3|192||169|válido|
|scored_0535|E2|64-6|188||172|válido|
|scored_0535|E3|joint|188|individual 188|184|OPTIMAL|
|scored_0535|E4|repair|188||188|completo|
|scored_0536|E1|1|340|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0536|E1|3|335|postprocess 335, optimize_a4 335, nested_windows 335|335|válido|
|scored_0536|E1|6|319|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0536|E2|52-1|340||322|válido|
|scored_0536|E2|52-3|335||308|válido|
|scored_0536|E2|52-6|319||309|válido|
|scored_0536|E2|64-1|340||324|válido|
|scored_0536|E2|64-3|335||308|válido|
|scored_0536|E2|64-6|319||308|válido|
|scored_0536|E3|joint|319|individual 319|319|OPTIMAL|
|scored_0536|E4|repair|319||319|completo|
|scored_0537|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0537|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0537|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0537|E2|52-1|2||2|válido|
|scored_0537|E2|52-3|2||2|válido|
|scored_0537|E2|52-6|2||2|válido|
|scored_0537|E2|64-1|2||2|válido|
|scored_0537|E2|64-3|2||2|válido|
|scored_0537|E2|64-6|2||2|válido|
|scored_0537|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0537|E4|repair|2||2|completo|
|scored_0538|E1|1|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0538|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0538|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0538|E2|52-1|239||222|válido|
|scored_0538|E2|52-3|235||216|válido|
|scored_0538|E2|52-6|229||211|válido|
|scored_0538|E2|64-1|239||223|válido|
|scored_0538|E2|64-3|235||220|válido|
|scored_0538|E2|64-6|229||214|válido|
|scored_0538|E3|joint|229|individual 229|229|OPTIMAL|
|scored_0538|E4|repair|229||229|completo|
|scored_0539|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0539|E1|3|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0539|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0539|E2|52-1|123||119|válido|
|scored_0539|E2|52-3|122||119|válido|
|scored_0539|E2|52-6|122||119|válido|
|scored_0539|E2|64-1|123||119|válido|
|scored_0539|E2|64-3|122||119|válido|
|scored_0539|E2|64-6|122||119|válido|
|scored_0539|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0539|E4|repair|122||119|completo|
|scored_0540|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0540|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0540|E1|6|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0540|E2|52-1|228||219|válido|
|scored_0540|E2|52-3|226||215|válido|
|scored_0540|E2|52-6|225||214|válido|
|scored_0540|E2|64-1|228||219|válido|
|scored_0540|E2|64-3|226||215|válido|
|scored_0540|E2|64-6|225||212|válido|
|scored_0540|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0540|E4|repair|225||225|completo|
|scored_0541|E1|1|71|postprocess 71, optimize_a4 71, nested_windows 71|71|válido|
|scored_0541|E1|3|71|postprocess 71, optimize_a4 71, nested_windows 71|71|válido|
|scored_0541|E1|6|70|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0541|E2|52-1|71||69|válido|
|scored_0541|E2|52-3|71||69|válido|
|scored_0541|E2|52-6|70||69|válido|
|scored_0541|E2|64-1|71||69|válido|
|scored_0541|E2|64-3|71||69|válido|
|scored_0541|E2|64-6|70||69|válido|
|scored_0541|E3|joint|70|individual 70|70|OPTIMAL|
|scored_0541|E4|repair|70||69|completo|
|scored_0542|E1|1|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0542|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0542|E1|6|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0542|E2|52-1|147||141|válido|
|scored_0542|E2|52-3|143||140|válido|
|scored_0542|E2|52-6|140||140|válido|
|scored_0542|E2|64-1|147||139|válido|
|scored_0542|E2|64-3|143||136|válido|
|scored_0542|E2|64-6|140||136|válido|
|scored_0542|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0542|E4|repair|140||137|completo|
|scored_0543|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0543|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0543|E1|6|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0543|E2|52-1|133||127|válido|
|scored_0543|E2|52-3|133||127|válido|
|scored_0543|E2|52-6|133||127|válido|
|scored_0543|E2|64-1|133||128|válido|
|scored_0543|E2|64-3|133||127|válido|
|scored_0543|E2|64-6|133||127|válido|
|scored_0543|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0543|E4|repair|133||131|completo|
|scored_0544|E1|1|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0544|E1|3|235|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0544|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0544|E2|52-1|243||227|válido|
|scored_0544|E2|52-3|235||219|válido|
|scored_0544|E2|52-6|232||219|válido|
|scored_0544|E2|64-1|243||227|válido|
|scored_0544|E2|64-3|235||221|válido|
|scored_0544|E2|64-6|232||217|válido|
|scored_0544|E3|joint|232|individual 232|232|FEASIBLE|
|scored_0544|E4|repair|232||232|completo|
|scored_0545|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0545|E1|3|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0545|E1|6|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0545|E2|52-1|150||144|válido|
|scored_0545|E2|52-3|148||142|válido|
|scored_0545|E2|52-6|148||141|válido|
|scored_0545|E2|64-1|150||143|válido|
|scored_0545|E2|64-3|148||142|válido|
|scored_0545|E2|64-6|148||140|válido|
|scored_0545|E3|joint|148|individual 147|147|OPTIMAL|
|scored_0545|E4|repair|148||145|completo|
|scored_0546|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0546|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0546|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0546|E2|52-1|206||185|válido|
|scored_0546|E2|52-3|200||187|válido|
|scored_0546|E2|52-6|198||184|válido|
|scored_0546|E2|64-1|206||184|válido|
|scored_0546|E2|64-3|200||187|válido|
|scored_0546|E2|64-6|198||184|válido|
|scored_0546|E3|joint|198|individual 197|197|OPTIMAL|
|scored_0546|E4|repair|198||198|completo|
|scored_0547|E1|1|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0547|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0547|E1|6|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0547|E2|52-1|139||134|válido|
|scored_0547|E2|52-3|138||133|válido|
|scored_0547|E2|52-6|138||133|válido|
|scored_0547|E2|64-1|139||134|válido|
|scored_0547|E2|64-3|138||134|válido|
|scored_0547|E2|64-6|138||134|válido|
|scored_0547|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0547|E4|repair|138||137|completo|
|scored_0548|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0548|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0548|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0548|E2|52-1|50||50|válido|
|scored_0548|E2|52-3|50||50|válido|
|scored_0548|E2|52-6|50||50|válido|
|scored_0548|E2|64-1|50||50|válido|
|scored_0548|E2|64-3|50||50|válido|
|scored_0548|E2|64-6|50||50|válido|
|scored_0548|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0548|E4|repair|50||50|completo|
|scored_0549|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0549|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0549|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0549|E2|52-1|146||140|válido|
|scored_0549|E2|52-3|146||139|válido|
|scored_0549|E2|52-6|146||139|válido|
|scored_0549|E2|64-1|146||140|válido|
|scored_0549|E2|64-3|146||140|válido|
|scored_0549|E2|64-6|146||140|válido|
|scored_0549|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0549|E4|repair|146||146|completo|
|scored_0550|E1|1|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0550|E1|3|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0550|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0550|E2|52-1|146||141|válido|
|scored_0550|E2|52-3|143||134|válido|
|scored_0550|E2|52-6|136||133|válido|
|scored_0550|E2|64-1|146||143|válido|
|scored_0550|E2|64-3|143||135|válido|
|scored_0550|E2|64-6|136||134|válido|
|scored_0550|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0550|E4|repair|136||135|completo|
|scored_0551|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0551|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0551|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0551|E2|52-1|238||228|válido|
|scored_0551|E2|52-3|233||220|válido|
|scored_0551|E2|52-6|233||220|válido|
|scored_0551|E2|64-1|238||230|válido|
|scored_0551|E2|64-3|233||223|válido|
|scored_0551|E2|64-6|233||222|válido|
|scored_0551|E3|joint|233|individual 232|229|OPTIMAL|
|scored_0551|E4|repair|233||233|completo|
|scored_0552|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0552|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0552|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0552|E2|52-1|310||288|válido|
|scored_0552|E2|52-3|294||279|válido|
|scored_0552|E2|52-6|284||271|válido|
|scored_0552|E2|64-1|310||288|válido|
|scored_0552|E2|64-3|294||277|válido|
|scored_0552|E2|64-6|284||273|válido|
|scored_0552|E3|joint|284|individual 283|283|OPTIMAL|
|scored_0552|E4|repair|284||284|completo|
|scored_0553|E1|1|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|scored_0553|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0553|E1|6|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0553|E2|52-1|256||230|válido|
|scored_0553|E2|52-3|243||224|válido|
|scored_0553|E2|52-6|239||220|válido|
|scored_0553|E2|64-1|256||230|válido|
|scored_0553|E2|64-3|243||226|válido|
|scored_0553|E2|64-6|239||220|válido|
|scored_0553|E3|joint|239|individual 239|238|OPTIMAL|
|scored_0553|E4|repair|239||238|completo|
|scored_0554|E1|1|142|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0554|E1|3|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0554|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0554|E2|52-1|142||140|válido|
|scored_0554|E2|52-3|143||139|válido|
|scored_0554|E2|52-6|143||139|válido|
|scored_0554|E2|64-1|142||139|válido|
|scored_0554|E2|64-3|143||140|válido|
|scored_0554|E2|64-6|143||140|válido|
|scored_0554|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0554|E4|repair|142||142|completo|
|scored_0555|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0555|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0555|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0555|E2|52-1|169||161|válido|
|scored_0555|E2|52-3|171||157|válido|
|scored_0555|E2|52-6|168||157|válido|
|scored_0555|E2|64-1|169||160|válido|
|scored_0555|E2|64-3|171||155|válido|
|scored_0555|E2|64-6|168||155|válido|
|scored_0555|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0555|E4|repair|168||168|completo|
|scored_0556|E1|1|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0556|E1|3|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0556|E1|6|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0556|E2|52-1|41||41|válido|
|scored_0556|E2|52-3|41||41|válido|
|scored_0556|E2|52-6|41||41|válido|
|scored_0556|E2|64-1|41||41|válido|
|scored_0556|E2|64-3|41||41|válido|
|scored_0556|E2|64-6|41||41|válido|
|scored_0556|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0556|E4|repair|41||36|completo|
|scored_0557|E1|1|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0557|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0557|E1|6|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0557|E2|52-1|174||165|válido|
|scored_0557|E2|52-3|171||161|válido|
|scored_0557|E2|52-6|170||163|válido|
|scored_0557|E2|64-1|174||164|válido|
|scored_0557|E2|64-3|171||165|válido|
|scored_0557|E2|64-6|170||164|válido|
|scored_0557|E3|joint|170|individual 170|170|OPTIMAL|
|scored_0557|E4|repair|170||170|completo|
|scored_0558|E1|1|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0558|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0558|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0558|E2|52-1|161||154|válido|
|scored_0558|E2|52-3|160||152|válido|
|scored_0558|E2|52-6|160||153|válido|
|scored_0558|E2|64-1|161||152|válido|
|scored_0558|E2|64-3|160||151|válido|
|scored_0558|E2|64-6|160||149|válido|
|scored_0558|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0558|E4|repair|160||160|completo|
|scored_0559|E1|1|48|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0559|E1|3|48|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0559|E1|6|48|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0559|E2|52-1|48||48|válido|
|scored_0559|E2|52-3|48||48|válido|
|scored_0559|E2|52-6|48||48|válido|
|scored_0559|E2|64-1|48||48|válido|
|scored_0559|E2|64-3|48||48|válido|
|scored_0559|E2|64-6|48||48|válido|
|scored_0559|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0559|E4|repair|48||48|completo|
|scored_0560|E1|1|53|postprocess 53, optimize_a4 53, nested_windows 53|53|válido|
|scored_0560|E1|3|53|postprocess 53, optimize_a4 53, nested_windows 53|53|válido|
|scored_0560|E1|6|53|postprocess 53, optimize_a4 53, nested_windows 53|53|válido|
|scored_0560|E2|52-1|53||53|válido|
|scored_0560|E2|52-3|53||53|válido|
|scored_0560|E2|52-6|53||53|válido|
|scored_0560|E2|64-1|53||53|válido|
|scored_0560|E2|64-3|53||53|válido|
|scored_0560|E2|64-6|53||53|válido|
|scored_0560|E3|joint|53|individual 53|53|OPTIMAL|
|scored_0560|E4|repair|53||50|completo|
|scored_0561|E1|1|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0561|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0561|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0561|E2|52-1|131||124|válido|
|scored_0561|E2|52-3|131||124|válido|
|scored_0561|E2|52-6|131||124|válido|
|scored_0561|E2|64-1|131||125|válido|
|scored_0561|E2|64-3|131||125|válido|
|scored_0561|E2|64-6|131||125|válido|
|scored_0561|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0561|E4|repair|131||126|completo|
|scored_0562|E1|1|117|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0562|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0562|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0562|E2|52-1|117||117|válido|
|scored_0562|E2|52-3|116||117|válido|
|scored_0562|E2|52-6|116||117|válido|
|scored_0562|E2|64-1|117||116|válido|
|scored_0562|E2|64-3|116||116|válido|
|scored_0562|E2|64-6|116||116|válido|
|scored_0562|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0562|E4|repair|116||116|completo|
|scored_0563|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0563|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0563|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0563|E2|52-1|141||136|válido|
|scored_0563|E2|52-3|137||132|válido|
|scored_0563|E2|52-6|137||133|válido|
|scored_0563|E2|64-1|141||135|válido|
|scored_0563|E2|64-3|137||133|válido|
|scored_0563|E2|64-6|137||133|válido|
|scored_0563|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0563|E4|repair|137||137|completo|
|scored_0564|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0564|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0564|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0564|E2|52-1|41||41|válido|
|scored_0564|E2|52-3|41||41|válido|
|scored_0564|E2|52-6|41||41|válido|
|scored_0564|E2|64-1|41||41|válido|
|scored_0564|E2|64-3|41||41|válido|
|scored_0564|E2|64-6|41||41|válido|
|scored_0564|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0564|E4|repair|41||41|completo|
|scored_0565|E1|1|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0565|E1|3|294|postprocess 293, optimize_a4 293, nested_windows 293|293|válido|
|scored_0565|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0565|E2|52-1|302||280|válido|
|scored_0565|E2|52-3|294||279|válido|
|scored_0565|E2|52-6|287||265|válido|
|scored_0565|E2|64-1|302||283|válido|
|scored_0565|E2|64-3|294||277|válido|
|scored_0565|E2|64-6|287||267|válido|
|scored_0565|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0565|E4|repair|287||287|completo|
|scored_0566|E1|1|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0566|E1|3|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0566|E1|6|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0566|E2|52-1|33||33|válido|
|scored_0566|E2|52-3|33||33|válido|
|scored_0566|E2|52-6|33||33|válido|
|scored_0566|E2|64-1|33||33|válido|
|scored_0566|E2|64-3|33||33|válido|
|scored_0566|E2|64-6|33||33|válido|
|scored_0566|E3|joint|33|individual 33|33|OPTIMAL|
|scored_0566|E4|repair|33||31|completo|
|scored_0567|E1|1|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0567|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0567|E1|6|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0567|E2|52-1|156||148|válido|
|scored_0567|E2|52-3|153||147|válido|
|scored_0567|E2|52-6|150||145|válido|
|scored_0567|E2|64-1|156||141|válido|
|scored_0567|E2|64-3|153||141|válido|
|scored_0567|E2|64-6|150||140|válido|
|scored_0567|E3|joint|150|individual 150|150|OPTIMAL|
|scored_0567|E4|repair|150||148|completo|
|scored_0568|E1|1|156|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0568|E1|3|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0568|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0568|E2|52-1|156||140|válido|
|scored_0568|E2|52-3|150||140|válido|
|scored_0568|E2|52-6|148||140|válido|
|scored_0568|E2|64-1|156||140|válido|
|scored_0568|E2|64-3|150||140|válido|
|scored_0568|E2|64-6|148||140|válido|
|scored_0568|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0568|E4|repair|148||144|completo|
|scored_0569|E1|1|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0569|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0569|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0569|E2|52-1|208||197|válido|
|scored_0569|E2|52-3|202||191|válido|
|scored_0569|E2|52-6|199||190|válido|
|scored_0569|E2|64-1|208||198|válido|
|scored_0569|E2|64-3|202||190|válido|
|scored_0569|E2|64-6|199||187|válido|
|scored_0569|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0569|E4|repair|199||197|completo|
|scored_0570|E1|1|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0570|E1|3|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0570|E1|6|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0570|E2|52-1|119||114|válido|
|scored_0570|E2|52-3|119||114|válido|
|scored_0570|E2|52-6|119||114|válido|
|scored_0570|E2|64-1|119||112|válido|
|scored_0570|E2|64-3|119||112|válido|
|scored_0570|E2|64-6|119||112|válido|
|scored_0570|E3|joint|119|individual 119|119|OPTIMAL|
|scored_0570|E4|repair|119||113|completo|
|scored_0571|E1|1|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0571|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0571|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0571|E2|52-1|274||249|válido|
|scored_0571|E2|52-3|270||246|válido|
|scored_0571|E2|52-6|257||240|válido|
|scored_0571|E2|64-1|274||247|válido|
|scored_0571|E2|64-3|270||247|válido|
|scored_0571|E2|64-6|257||240|válido|
|scored_0571|E3|joint|257|individual 257|257|OPTIMAL|
|scored_0571|E4|repair|257||257|completo|
|scored_0572|E1|1|317|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0572|E1|3|308|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0572|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0572|E2|52-1|317||299|válido|
|scored_0572|E2|52-3|308||293|válido|
|scored_0572|E2|52-6|304||286|válido|
|scored_0572|E2|64-1|317||298|válido|
|scored_0572|E2|64-3|308||294|válido|
|scored_0572|E2|64-6|304||289|válido|
|scored_0572|E3|joint|304|individual 303|302|OPTIMAL|
|scored_0572|E4|repair|304||304|completo|
|scored_0573|E1|1|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0573|E1|3|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0573|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0573|E2|52-1|255||238|válido|
|scored_0573|E2|52-3|248||235|válido|
|scored_0573|E2|52-6|237||231|válido|
|scored_0573|E2|64-1|255||239|válido|
|scored_0573|E2|64-3|248||238|válido|
|scored_0573|E2|64-6|237||232|válido|
|scored_0573|E3|joint|237|individual 237|237|OPTIMAL|
|scored_0573|E4|repair|237||237|completo|
|scored_0574|E1|1|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0574|E1|3|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0574|E1|6|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0574|E2|52-1|136||128|válido|
|scored_0574|E2|52-3|136||128|válido|
|scored_0574|E2|52-6|136||131|válido|
|scored_0574|E2|64-1|136||128|válido|
|scored_0574|E2|64-3|136||128|válido|
|scored_0574|E2|64-6|136||128|válido|
|scored_0574|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0574|E4|repair|136||134|completo|
|scored_0575|E1|1|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0575|E1|3|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0575|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0575|E2|52-1|170||165|válido|
|scored_0575|E2|52-3|168||159|válido|
|scored_0575|E2|52-6|167||165|válido|
|scored_0575|E2|64-1|170||163|válido|
|scored_0575|E2|64-3|168||160|válido|
|scored_0575|E2|64-6|167||160|válido|
|scored_0575|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0575|E4|repair|167||166|completo|
|scored_0576|E1|1|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0576|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0576|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0576|E2|52-1|230||209|válido|
|scored_0576|E2|52-3|219||207|válido|
|scored_0576|E2|52-6|215||208|válido|
|scored_0576|E2|64-1|230||210|válido|
|scored_0576|E2|64-3|219||210|válido|
|scored_0576|E2|64-6|215||211|válido|
|scored_0576|E3|joint|215|individual 214|213|OPTIMAL|
|scored_0576|E4|repair|215||215|completo|
|scored_0577|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0577|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0577|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0577|E2|52-1|243||217|válido|
|scored_0577|E2|52-3|233||215|válido|
|scored_0577|E2|52-6|226||211|válido|
|scored_0577|E2|64-1|243||221|válido|
|scored_0577|E2|64-3|233||215|válido|
|scored_0577|E2|64-6|226||210|válido|
|scored_0577|E3|joint|226|individual 225|225|OPTIMAL|
|scored_0577|E4|repair|226||226|completo|
|scored_0578|E1|1|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0578|E1|3|171|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0578|E1|6|163|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0578|E2|52-1|171||160|válido|
|scored_0578|E2|52-3|171||160|válido|
|scored_0578|E2|52-6|163||154|válido|
|scored_0578|E2|64-1|171||166|válido|
|scored_0578|E2|64-3|171||161|válido|
|scored_0578|E2|64-6|163||159|válido|
|scored_0578|E3|joint|163|individual 163|162|OPTIMAL|
|scored_0578|E4|repair|163||163|completo|
|scored_0579|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0579|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0579|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0579|E2|52-1|223||215|válido|
|scored_0579|E2|52-3|225||209|válido|
|scored_0579|E2|52-6|220||204|válido|
|scored_0579|E2|64-1|223||216|válido|
|scored_0579|E2|64-3|225||213|válido|
|scored_0579|E2|64-6|220||205|válido|
|scored_0579|E3|joint|220|individual 219|219|OPTIMAL|
|scored_0579|E4|repair|220||220|completo|
|scored_0580|E1|1|329|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0580|E1|3|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0580|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0580|E2|52-1|329||300|válido|
|scored_0580|E2|52-3|312||289|válido|
|scored_0580|E2|52-6|303||282|válido|
|scored_0580|E2|64-1|329||299|válido|
|scored_0580|E2|64-3|312||290|válido|
|scored_0580|E2|64-6|303||283|válido|
|scored_0580|E3|joint|303|individual 302|301|OPTIMAL|
|scored_0580|E4|repair|303||303|completo|
|scored_0581|E1|1|351|postprocess 351, optimize_a4 351, nested_windows 351|351|válido|
|scored_0581|E1|3|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0581|E1|6|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0581|E2|52-1|351||324|válido|
|scored_0581|E2|52-3|339||316|válido|
|scored_0581|E2|52-6|329||314|válido|
|scored_0581|E2|64-1|351||327|válido|
|scored_0581|E2|64-3|339||319|válido|
|scored_0581|E2|64-6|329||313|válido|
|scored_0581|E3|joint|329|individual 329|329|FEASIBLE|
|scored_0581|E4|repair|329||329|completo|
|scored_0582|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0582|E1|3|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0582|E1|6|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0582|E2|52-1|254||234|válido|
|scored_0582|E2|52-3|247||228|válido|
|scored_0582|E2|52-6|243||228|válido|
|scored_0582|E2|64-1|254||238|válido|
|scored_0582|E2|64-3|247||230|válido|
|scored_0582|E2|64-6|243||226|válido|
|scored_0582|E3|joint|243|individual 242|241|OPTIMAL|
|scored_0582|E4|repair|243||243|completo|
|scored_0583|E1|1|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0583|E1|3|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0583|E1|6|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0583|E2|52-1|51||51|válido|
|scored_0583|E2|52-3|51||51|válido|
|scored_0583|E2|52-6|51||51|válido|
|scored_0583|E2|64-1|51||51|válido|
|scored_0583|E2|64-3|51||51|válido|
|scored_0583|E2|64-6|51||51|válido|
|scored_0583|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0583|E4|repair|51||51|completo|
|scored_0584|E1|1|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0584|E1|3|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0584|E1|6|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0584|E2|52-1|59||60|válido|
|scored_0584|E2|52-3|59||60|válido|
|scored_0584|E2|52-6|59||60|válido|
|scored_0584|E2|64-1|59||60|válido|
|scored_0584|E2|64-3|59||60|válido|
|scored_0584|E2|64-6|59||60|válido|
|scored_0584|E3|joint|59|individual 59|59|OPTIMAL|
|scored_0584|E4|repair|59||57|completo|
|scored_0585|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0585|E1|3|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0585|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0585|E2|52-1|248||226|válido|
|scored_0585|E2|52-3|244||229|válido|
|scored_0585|E2|52-6|241||226|válido|
|scored_0585|E2|64-1|248||230|válido|
|scored_0585|E2|64-3|244||226|válido|
|scored_0585|E2|64-6|241||228|válido|
|scored_0585|E3|joint|241|individual 241|237|FEASIBLE|
|scored_0585|E4|repair|241||241|completo|
|scored_0586|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0586|E1|3|112|postprocess 112, optimize_a4 112, nested_windows 112|112|válido|
|scored_0586|E1|6|112|postprocess 112, optimize_a4 112, nested_windows 112|112|válido|
|scored_0586|E2|52-1|113||105|válido|
|scored_0586|E2|52-3|112||107|válido|
|scored_0586|E2|52-6|112||107|válido|
|scored_0586|E2|64-1|113||105|válido|
|scored_0586|E2|64-3|112||105|válido|
|scored_0586|E2|64-6|112||105|válido|
|scored_0586|E3|joint|112|individual 112|112|OPTIMAL|
|scored_0586|E4|repair|112||110|completo|
|scored_0587|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0587|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0587|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0587|E2|52-1|202||190|válido|
|scored_0587|E2|52-3|196||187|válido|
|scored_0587|E2|52-6|195||186|válido|
|scored_0587|E2|64-1|202||187|válido|
|scored_0587|E2|64-3|196||182|válido|
|scored_0587|E2|64-6|195||185|válido|
|scored_0587|E3|joint|195|individual 194|194|OPTIMAL|
|scored_0587|E4|repair|195||195|completo|
|scored_0588|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0588|E1|3|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0588|E1|6|184|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0588|E2|52-1|190||176|válido|
|scored_0588|E2|52-3|186||174|válido|
|scored_0588|E2|52-6|184||172|válido|
|scored_0588|E2|64-1|190||175|válido|
|scored_0588|E2|64-3|186||172|válido|
|scored_0588|E2|64-6|184||170|válido|
|scored_0588|E3|joint|184|individual 184|183|OPTIMAL|
|scored_0588|E4|repair|184||184|completo|
|scored_0589|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0589|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0589|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0589|E2|52-1|113||114|válido|
|scored_0589|E2|52-3|113||114|válido|
|scored_0589|E2|52-6|113||114|válido|
|scored_0589|E2|64-1|113||114|válido|
|scored_0589|E2|64-3|113||114|válido|
|scored_0589|E2|64-6|113||114|válido|
|scored_0589|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0589|E4|repair|113||111|completo|
|scored_0590|E1|1|189|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0590|E1|3|181|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0590|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0590|E2|52-1|189||180|válido|
|scored_0590|E2|52-3|181||176|válido|
|scored_0590|E2|52-6|181||173|válido|
|scored_0590|E2|64-1|189||179|válido|
|scored_0590|E2|64-3|181||174|válido|
|scored_0590|E2|64-6|181||171|válido|
|scored_0590|E3|joint|181|individual 180|180|OPTIMAL|
|scored_0590|E4|repair|181||179|completo|
|scored_0591|E1|1|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0591|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0591|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0591|E2|52-1|166||160|válido|
|scored_0591|E2|52-3|166||159|válido|
|scored_0591|E2|52-6|165||157|válido|
|scored_0591|E2|64-1|166||161|válido|
|scored_0591|E2|64-3|166||158|válido|
|scored_0591|E2|64-6|165||157|válido|
|scored_0591|E3|joint|165|individual 164|164|OPTIMAL|
|scored_0591|E4|repair|165||163|completo|
|scored_0592|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0592|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0592|E1|6|274|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0592|E2|52-1|298||271|válido|
|scored_0592|E2|52-3|281||262|válido|
|scored_0592|E2|52-6|274||259|válido|
|scored_0592|E2|64-1|298||273|válido|
|scored_0592|E2|64-3|281||264|válido|
|scored_0592|E2|64-6|274||260|válido|
|scored_0592|E3|joint|274|individual 273|273|OPTIMAL|
|scored_0592|E4|repair|274||274|completo|
|scored_0593|E1|1|320|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0593|E1|3|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0593|E1|6|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0593|E2|52-1|320||306|válido|
|scored_0593|E2|52-3|316||294|válido|
|scored_0593|E2|52-6|307||282|válido|
|scored_0593|E2|64-1|320||306|válido|
|scored_0593|E2|64-3|316||294|válido|
|scored_0593|E2|64-6|307||285|válido|
|scored_0593|E3|joint|307|individual 306|305|FEASIBLE|
|scored_0593|E4|repair|307||307|completo|
|scored_0594|E1|1|325|postprocess 325, optimize_a4 325, nested_windows 325|325|válido|
|scored_0594|E1|3|315|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0594|E1|6|309|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0594|E2|52-1|325||297|válido|
|scored_0594|E2|52-3|315||288|válido|
|scored_0594|E2|52-6|309||284|válido|
|scored_0594|E2|64-1|325||297|válido|
|scored_0594|E2|64-3|315||288|válido|
|scored_0594|E2|64-6|309||285|válido|
|scored_0594|E3|joint|309|individual 308|308|rechazado por registros|
|scored_0594|E4|repair|309||309|completo|
|scored_0595|E1|1|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0595|E1|3|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0595|E1|6|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0595|E2|52-1|148||146|válido|
|scored_0595|E2|52-3|147||147|válido|
|scored_0595|E2|52-6|147||145|válido|
|scored_0595|E2|64-1|148||146|válido|
|scored_0595|E2|64-3|147||147|válido|
|scored_0595|E2|64-6|147||145|válido|
|scored_0595|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0595|E4|repair|147||146|completo|
|scored_0596|E1|1|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0596|E1|3|164|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0596|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0596|E2|52-1|169||161|válido|
|scored_0596|E2|52-3|164||153|válido|
|scored_0596|E2|52-6|163||153|válido|
|scored_0596|E2|64-1|169||161|válido|
|scored_0596|E2|64-3|164||154|válido|
|scored_0596|E2|64-6|163||153|válido|
|scored_0596|E3|joint|163|individual 162|162|OPTIMAL|
|scored_0596|E4|repair|163||163|completo|
|scored_0597|E1|1|5|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|scored_0597|E1|3|5|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|scored_0597|E1|6|5|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|scored_0597|E2|52-1|5||5|válido|
|scored_0597|E2|52-3|5||5|válido|
|scored_0597|E2|52-6|5||5|válido|
|scored_0597|E2|64-1|5||5|válido|
|scored_0597|E2|64-3|5||5|válido|
|scored_0597|E2|64-6|5||5|válido|
|scored_0597|E3|joint|3|individual 3|3|OPTIMAL|
|scored_0597|E4|repair|3||3|completo|
|scored_0598|E1|1|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0598|E1|3|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0598|E1|6|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0598|E2|52-1|103||101|válido|
|scored_0598|E2|52-3|103||101|válido|
|scored_0598|E2|52-6|103||101|válido|
|scored_0598|E2|64-1|103||101|válido|
|scored_0598|E2|64-3|103||101|válido|
|scored_0598|E2|64-6|103||101|válido|
|scored_0598|E3|joint|103|individual 103|103|OPTIMAL|
|scored_0598|E4|repair|103||101|completo|
|scored_0599|E1|1|237|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0599|E1|3|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0599|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0599|E2|52-1|237||228|válido|
|scored_0599|E2|52-3|230||202|válido|
|scored_0599|E2|52-6|226||196|válido|
|scored_0599|E2|64-1|237||228|válido|
|scored_0599|E2|64-3|230||202|válido|
|scored_0599|E2|64-6|226||200|válido|
|scored_0599|E3|joint|226|individual 225|224|OPTIMAL|
|scored_0599|E4|repair|226||226|completo|
|scored_0600|E1|1|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0600|E1|3|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0600|E1|6|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0600|E2|52-1|3||3|válido|
|scored_0600|E2|52-3|3||3|válido|
|scored_0600|E2|52-6|3||3|válido|
|scored_0600|E2|64-1|3||3|válido|
|scored_0600|E2|64-3|3||3|válido|
|scored_0600|E2|64-6|3||3|válido|
|scored_0600|E3|joint|3|individual 3|3|OPTIMAL|
|scored_0600|E4|repair|3||3|completo|
|scored_0601|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0601|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0601|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0601|E2|52-1|153||150|válido|
|scored_0601|E2|52-3|152||150|válido|
|scored_0601|E2|52-6|152||147|válido|
|scored_0601|E2|64-1|153||148|válido|
|scored_0601|E2|64-3|152||148|válido|
|scored_0601|E2|64-6|152||148|válido|
|scored_0601|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0601|E4|repair|152||152|completo|
|scored_0602|E1|1|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0602|E1|3|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0602|E1|6|137|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0602|E2|52-1|139||136|válido|
|scored_0602|E2|52-3|139||135|válido|
|scored_0602|E2|52-6|137||135|válido|
|scored_0602|E2|64-1|139||136|válido|
|scored_0602|E2|64-3|139||135|válido|
|scored_0602|E2|64-6|137||134|válido|
|scored_0602|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0602|E4|repair|137||135|completo|
|scored_0603|E1|1|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0603|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0603|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0603|E2|52-1|208||197|válido|
|scored_0603|E2|52-3|203||192|válido|
|scored_0603|E2|52-6|202||190|válido|
|scored_0603|E2|64-1|208||194|válido|
|scored_0603|E2|64-3|203||190|válido|
|scored_0603|E2|64-6|202||189|válido|
|scored_0603|E3|joint|202|individual 201|201|OPTIMAL|
|scored_0603|E4|repair|202||200|completo|
|scored_0604|E1|1|178|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0604|E1|3|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0604|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0604|E2|52-1|178||159|válido|
|scored_0604|E2|52-3|178||159|válido|
|scored_0604|E2|52-6|172||160|válido|
|scored_0604|E2|64-1|178||159|válido|
|scored_0604|E2|64-3|178||158|válido|
|scored_0604|E2|64-6|172||161|válido|
|scored_0604|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0604|E4|repair|172||166|completo|
|scored_0605|E1|1|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0605|E1|3|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0605|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0605|E2|52-1|221||210|válido|
|scored_0605|E2|52-3|224||208|válido|
|scored_0605|E2|52-6|221||207|válido|
|scored_0605|E2|64-1|221||209|válido|
|scored_0605|E2|64-3|224||207|válido|
|scored_0605|E2|64-6|221||207|válido|
|scored_0605|E3|joint|221|individual 221|219|OPTIMAL|
|scored_0605|E4|repair|221||221|completo|
|scored_0606|E1|1|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0606|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0606|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0606|E2|52-1|233||219|válido|
|scored_0606|E2|52-3|226||210|válido|
|scored_0606|E2|52-6|223||207|válido|
|scored_0606|E2|64-1|233||216|válido|
|scored_0606|E2|64-3|226||208|válido|
|scored_0606|E2|64-6|223||205|válido|
|scored_0606|E3|joint|223|individual 223|221|OPTIMAL|
|scored_0606|E4|repair|223||223|completo|
|scored_0607|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0607|E1|3|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0607|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0607|E2|52-1|168||157|válido|
|scored_0607|E2|52-3|166||156|válido|
|scored_0607|E2|52-6|163||156|válido|
|scored_0607|E2|64-1|168||156|válido|
|scored_0607|E2|64-3|166||156|válido|
|scored_0607|E2|64-6|163||155|válido|
|scored_0607|E3|joint|163|individual 163|163|OPTIMAL|
|scored_0607|E4|repair|163||161|completo|
|scored_0608|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0608|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0608|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0608|E2|52-1|207||192|válido|
|scored_0608|E2|52-3|199||193|válido|
|scored_0608|E2|52-6|201||187|válido|
|scored_0608|E2|64-1|207||189|válido|
|scored_0608|E2|64-3|199||191|válido|
|scored_0608|E2|64-6|201||188|válido|
|scored_0608|E3|joint|199|individual 198|198|OPTIMAL|
|scored_0608|E4|repair|199||197|completo|
|scored_0609|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0609|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0609|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0609|E2|52-1|210||195|válido|
|scored_0609|E2|52-3|206||192|válido|
|scored_0609|E2|52-6|201||188|válido|
|scored_0609|E2|64-1|210||195|válido|
|scored_0609|E2|64-3|206||192|válido|
|scored_0609|E2|64-6|201||189|válido|
|scored_0609|E3|joint|201|individual 200|200|OPTIMAL|
|scored_0609|E4|repair|201||201|completo|
|scored_0610|E1|1|393|postprocess 393, optimize_a4 393, nested_windows 393|393|válido|
|scored_0610|E1|3|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0610|E1|6|360|postprocess 360, optimize_a4 360, nested_windows 360|360|válido|
|scored_0610|E2|52-1|393||351|válido|
|scored_0610|E2|52-3|370||335|válido|
|scored_0610|E2|52-6|360||336|válido|
|scored_0610|E2|64-1|393||347|válido|
|scored_0610|E2|64-3|370||339|válido|
|scored_0610|E2|64-6|360||336|válido|
|scored_0610|E3|joint|360|individual 360|358|OPTIMAL|
|scored_0610|E4|repair|360||360|completo|
|scored_0611|E1|1|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0611|E1|3|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0611|E1|6|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0611|E2|52-1|42||42|válido|
|scored_0611|E2|52-3|42||42|válido|
|scored_0611|E2|52-6|42||42|válido|
|scored_0611|E2|64-1|42||42|válido|
|scored_0611|E2|64-3|42||42|válido|
|scored_0611|E2|64-6|42||42|válido|
|scored_0611|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0611|E4|repair|42||41|completo|
|scored_0612|E1|1|361|postprocess 361, optimize_a4 361, nested_windows 361|361|válido|
|scored_0612|E1|3|342|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|scored_0612|E1|6|346|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|scored_0612|E2|52-1|361||335|válido|
|scored_0612|E2|52-3|342||324|válido|
|scored_0612|E2|52-6|346||321|válido|
|scored_0612|E2|64-1|361||339|válido|
|scored_0612|E2|64-3|342||323|válido|
|scored_0612|E2|64-6|346||325|válido|
|scored_0612|E3|joint|342|individual 341|338|OPTIMAL|
|scored_0612|E4|repair|342||342|completo|
|scored_0613|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0613|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0613|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0613|E2|52-1|137||133|válido|
|scored_0613|E2|52-3|137||133|válido|
|scored_0613|E2|52-6|137||133|válido|
|scored_0613|E2|64-1|137||135|válido|
|scored_0613|E2|64-3|137||134|válido|
|scored_0613|E2|64-6|137||134|válido|
|scored_0613|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0613|E4|repair|137||135|completo|
|scored_0614|E1|1|32|postprocess 31, optimize_a4 30, nested_windows 30|30|válido|
|scored_0614|E1|3|32|postprocess 31, optimize_a4 30, nested_windows 30|30|válido|
|scored_0614|E1|6|32|postprocess 31, optimize_a4 30, nested_windows 30|30|válido|
|scored_0614|E2|52-1|32||32|válido|
|scored_0614|E2|52-3|32||32|válido|
|scored_0614|E2|52-6|32||32|válido|
|scored_0614|E2|64-1|32||32|válido|
|scored_0614|E2|64-3|32||32|válido|
|scored_0614|E2|64-6|32||32|válido|
|scored_0614|E3|joint|30|individual 30|30|OPTIMAL|
|scored_0614|E4|repair|30||28|completo|
|scored_0615|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0615|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0615|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0615|E2|52-1|146||140|válido|
|scored_0615|E2|52-3|143||140|válido|
|scored_0615|E2|52-6|143||140|válido|
|scored_0615|E2|64-1|146||140|válido|
|scored_0615|E2|64-3|143||140|válido|
|scored_0615|E2|64-6|143||141|válido|
|scored_0615|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0615|E4|repair|143||141|completo|
|scored_0616|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0616|E1|3|204|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0616|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0616|E2|52-1|215||190|válido|
|scored_0616|E2|52-3|204||186|válido|
|scored_0616|E2|52-6|199||186|válido|
|scored_0616|E2|64-1|215||190|válido|
|scored_0616|E2|64-3|204||186|válido|
|scored_0616|E2|64-6|199||186|válido|
|scored_0616|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0616|E4|repair|199||197|completo|
|scored_0617|E1|1|138|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0617|E1|3|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0617|E1|6|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0617|E2|52-1|138||132|válido|
|scored_0617|E2|52-3|139||131|válido|
|scored_0617|E2|52-6|137||129|válido|
|scored_0617|E2|64-1|138||131|válido|
|scored_0617|E2|64-3|139||130|válido|
|scored_0617|E2|64-6|137||129|válido|
|scored_0617|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0617|E4|repair|137||137|completo|
|scored_0618|E1|1|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0618|E1|3|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0618|E1|6|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0618|E2|52-1|136||132|válido|
|scored_0618|E2|52-3|134||130|válido|
|scored_0618|E2|52-6|134||130|válido|
|scored_0618|E2|64-1|136||132|válido|
|scored_0618|E2|64-3|134||131|válido|
|scored_0618|E2|64-6|134||131|válido|
|scored_0618|E3|joint|134|individual 134|134|OPTIMAL|
|scored_0618|E4|repair|134||130|completo|
|scored_0619|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0619|E1|3|196|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0619|E1|6|204|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0619|E2|52-1|202||193|válido|
|scored_0619|E2|52-3|196||193|válido|
|scored_0619|E2|52-6|204||193|válido|
|scored_0619|E2|64-1|202||197|válido|
|scored_0619|E2|64-3|196||196|válido|
|scored_0619|E2|64-6|204||191|válido|
|scored_0619|E3|joint|196|individual 196|196|OPTIMAL|
|scored_0619|E4|repair|196||195|completo|
|scored_0620|E1|1|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0620|E1|3|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0620|E1|6|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0620|E2|52-1|52||52|válido|
|scored_0620|E2|52-3|52||52|válido|
|scored_0620|E2|52-6|52||52|válido|
|scored_0620|E2|64-1|52||52|válido|
|scored_0620|E2|64-3|52||52|válido|
|scored_0620|E2|64-6|52||52|válido|
|scored_0620|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0620|E4|repair|52||50|completo|
|scored_0621|E1|1|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0621|E1|3|230|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0621|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0621|E2|52-1|241||222|válido|
|scored_0621|E2|52-3|230||214|válido|
|scored_0621|E2|52-6|227||217|válido|
|scored_0621|E2|64-1|241||221|válido|
|scored_0621|E2|64-3|230||217|válido|
|scored_0621|E2|64-6|227||219|válido|
|scored_0621|E3|joint|227|individual 227|227|OPTIMAL|
|scored_0621|E4|repair|227||227|completo|
|scored_0622|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0622|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0622|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0622|E2|52-1|233||213|válido|
|scored_0622|E2|52-3|226||210|válido|
|scored_0622|E2|52-6|221||201|válido|
|scored_0622|E2|64-1|233||219|válido|
|scored_0622|E2|64-3|226||210|válido|
|scored_0622|E2|64-6|221||203|válido|
|scored_0622|E3|joint|221|individual 221|220|OPTIMAL|
|scored_0622|E4|repair|221||220|completo|
|scored_0623|E1|1|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0623|E1|3|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0623|E1|6|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0623|E2|52-1|103||102|válido|
|scored_0623|E2|52-3|103||102|válido|
|scored_0623|E2|52-6|103||102|válido|
|scored_0623|E2|64-1|103||104|válido|
|scored_0623|E2|64-3|103||104|válido|
|scored_0623|E2|64-6|103||104|válido|
|scored_0623|E3|joint|103|individual 103|103|OPTIMAL|
|scored_0623|E4|repair|103||102|completo|
|scored_0624|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0624|E1|3|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0624|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0624|E2|52-1|159||148|válido|
|scored_0624|E2|52-3|158||145|válido|
|scored_0624|E2|52-6|156||145|válido|
|scored_0624|E2|64-1|159||149|válido|
|scored_0624|E2|64-3|158||148|válido|
|scored_0624|E2|64-6|156||146|válido|
|scored_0624|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0624|E4|repair|156||154|completo|
|scored_0625|E1|1|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0625|E1|3|126|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0625|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0625|E2|52-1|126||124|válido|
|scored_0625|E2|52-3|126||122|válido|
|scored_0625|E2|52-6|123||122|válido|
|scored_0625|E2|64-1|126||123|válido|
|scored_0625|E2|64-3|126||123|válido|
|scored_0625|E2|64-6|123||123|válido|
|scored_0625|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0625|E4|repair|123||122|completo|
|scored_0626|E1|1|247|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0626|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0626|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0626|E2|52-1|247||226|válido|
|scored_0626|E2|52-3|235||219|válido|
|scored_0626|E2|52-6|228||218|válido|
|scored_0626|E2|64-1|247||221|válido|
|scored_0626|E2|64-3|235||217|válido|
|scored_0626|E2|64-6|228||218|válido|
|scored_0626|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0626|E4|repair|228||228|completo|
|scored_0627|E1|1|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0627|E1|3|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0627|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0627|E2|52-1|184||175|válido|
|scored_0627|E2|52-3|184||169|válido|
|scored_0627|E2|52-6|181||170|válido|
|scored_0627|E2|64-1|184||176|válido|
|scored_0627|E2|64-3|184||171|válido|
|scored_0627|E2|64-6|181||172|válido|
|scored_0627|E3|joint|181|individual 180|178|OPTIMAL|
|scored_0627|E4|repair|181||178|completo|
|scored_0628|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0628|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0628|E1|6|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0628|E2|52-1|295||273|válido|
|scored_0628|E2|52-3|278||261|válido|
|scored_0628|E2|52-6|276||256|válido|
|scored_0628|E2|64-1|295||272|válido|
|scored_0628|E2|64-3|278||261|válido|
|scored_0628|E2|64-6|276||255|válido|
|scored_0628|E3|joint|276|individual 276|274|FEASIBLE|
|scored_0628|E4|repair|276||276|completo|
|scored_0629|E1|1|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0629|E1|3|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0629|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0629|E2|52-1|214||199|válido|
|scored_0629|E2|52-3|211||198|válido|
|scored_0629|E2|52-6|211||196|válido|
|scored_0629|E2|64-1|214||201|válido|
|scored_0629|E2|64-3|211||196|válido|
|scored_0629|E2|64-6|211||197|válido|
|scored_0629|E3|joint|211|individual 211|210|OPTIMAL|
|scored_0629|E4|repair|211||211|completo|
|scored_0630|E1|1|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0630|E1|3|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0630|E1|6|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0630|E2|52-1|44||44|válido|
|scored_0630|E2|52-3|44||44|válido|
|scored_0630|E2|52-6|44||44|válido|
|scored_0630|E2|64-1|44||44|válido|
|scored_0630|E2|64-3|44||44|válido|
|scored_0630|E2|64-6|44||44|válido|
|scored_0630|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0630|E4|repair|44||42|completo|
|scored_0631|E1|1|175|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0631|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0631|E1|6|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0631|E2|52-1|175||167|válido|
|scored_0631|E2|52-3|173||166|válido|
|scored_0631|E2|52-6|173||165|válido|
|scored_0631|E2|64-1|175||166|válido|
|scored_0631|E2|64-3|173||165|válido|
|scored_0631|E2|64-6|173||165|válido|
|scored_0631|E3|joint|173|individual 173|172|OPTIMAL|
|scored_0631|E4|repair|173||170|completo|
|scored_0632|E1|1|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0632|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0632|E1|6|256|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0632|E2|52-1|272||245|válido|
|scored_0632|E2|52-3|260||241|válido|
|scored_0632|E2|52-6|256||238|válido|
|scored_0632|E2|64-1|272||246|válido|
|scored_0632|E2|64-3|260||242|válido|
|scored_0632|E2|64-6|256||236|válido|
|scored_0632|E3|joint|256|individual 256|256|OPTIMAL|
|scored_0632|E4|repair|256||256|completo|
|scored_0633|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0633|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0633|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0633|E2|52-1|228||219|válido|
|scored_0633|E2|52-3|225||208|válido|
|scored_0633|E2|52-6|223||207|válido|
|scored_0633|E2|64-1|228||214|válido|
|scored_0633|E2|64-3|225||209|válido|
|scored_0633|E2|64-6|223||207|válido|
|scored_0633|E3|joint|223|individual 223|223|OPTIMAL|
|scored_0633|E4|repair|223||223|completo|
|scored_0634|E1|1|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0634|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0634|E1|6|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0634|E2|52-1|231||216|válido|
|scored_0634|E2|52-3|219||207|válido|
|scored_0634|E2|52-6|220||207|válido|
|scored_0634|E2|64-1|231||217|válido|
|scored_0634|E2|64-3|219||209|válido|
|scored_0634|E2|64-6|220||210|válido|
|scored_0634|E3|joint|219|individual 219|219|OPTIMAL|
|scored_0634|E4|repair|219||219|completo|
|scored_0635|E1|1|50|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0635|E1|3|50|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0635|E1|6|50|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0635|E2|52-1|50||50|válido|
|scored_0635|E2|52-3|50||50|válido|
|scored_0635|E2|52-6|50||50|válido|
|scored_0635|E2|64-1|50||50|válido|
|scored_0635|E2|64-3|50||50|válido|
|scored_0635|E2|64-6|50||50|válido|
|scored_0635|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0635|E4|repair|50||46|completo|
|scored_0636|E1|1|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0636|E1|3|335|postprocess 335, optimize_a4 335, nested_windows 335|335|válido|
|scored_0636|E1|6|331|postprocess 331, optimize_a4 331, nested_windows 331|331|válido|
|scored_0636|E2|52-1|353||329|válido|
|scored_0636|E2|52-3|335||316|válido|
|scored_0636|E2|52-6|331||314|válido|
|scored_0636|E2|64-1|353||330|válido|
|scored_0636|E2|64-3|335||317|válido|
|scored_0636|E2|64-6|331||315|válido|
|scored_0636|E3|joint|331|individual 331|329|OPTIMAL|
|scored_0636|E4|repair|331||331|completo|
|scored_0637|E1|1|43|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0637|E1|3|43|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0637|E1|6|43|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0637|E2|52-1|43||43|válido|
|scored_0637|E2|52-3|43||43|válido|
|scored_0637|E2|52-6|43||43|válido|
|scored_0637|E2|64-1|43||43|válido|
|scored_0637|E2|64-3|43||43|válido|
|scored_0637|E2|64-6|43||43|válido|
|scored_0637|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0637|E4|repair|43||43|completo|
|scored_0638|E1|1|380|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|scored_0638|E1|3|373|postprocess 372, optimize_a4 372, nested_windows 372|372|válido|
|scored_0638|E1|6|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0638|E2|52-1|380||351|válido|
|scored_0638|E2|52-3|373||342|válido|
|scored_0638|E2|52-6|363||341|válido|
|scored_0638|E2|64-1|380||357|válido|
|scored_0638|E2|64-3|373||347|válido|
|scored_0638|E2|64-6|363||341|válido|
|scored_0638|E3|joint|363|individual 362|362|FEASIBLE|
|scored_0638|E4|repair|363||363|completo|
|scored_0639|E1|1|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0639|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0639|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0639|E2|52-1|164||157|válido|
|scored_0639|E2|52-3|165||155|válido|
|scored_0639|E2|52-6|162||153|válido|
|scored_0639|E2|64-1|164||153|válido|
|scored_0639|E2|64-3|165||153|válido|
|scored_0639|E2|64-6|162||153|válido|
|scored_0639|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0639|E4|repair|162||160|completo|
|scored_0640|E1|1|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0640|E1|3|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0640|E1|6|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0640|E2|52-1|302||276|válido|
|scored_0640|E2|52-3|283||266|válido|
|scored_0640|E2|52-6|285||260|válido|
|scored_0640|E2|64-1|302||276|válido|
|scored_0640|E2|64-3|283||265|válido|
|scored_0640|E2|64-6|285||261|válido|
|scored_0640|E3|joint|283|individual 283|281|OPTIMAL|
|scored_0640|E4|repair|283||283|completo|
|scored_0641|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0641|E1|3|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0641|E1|6|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0641|E2|52-1|150||143|válido|
|scored_0641|E2|52-3|150||142|válido|
|scored_0641|E2|52-6|149||142|válido|
|scored_0641|E2|64-1|150||142|válido|
|scored_0641|E2|64-3|150||139|válido|
|scored_0641|E2|64-6|149||139|válido|
|scored_0641|E3|joint|149|individual 149|149|OPTIMAL|
|scored_0641|E4|repair|149||149|completo|
|scored_0642|E1|1|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0642|E1|3|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0642|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0642|E2|52-1|161||153|válido|
|scored_0642|E2|52-3|161||153|válido|
|scored_0642|E2|52-6|161||152|válido|
|scored_0642|E2|64-1|161||154|válido|
|scored_0642|E2|64-3|161||152|válido|
|scored_0642|E2|64-6|161||152|válido|
|scored_0642|E3|joint|161|individual 161|161|OPTIMAL|
|scored_0642|E4|repair|161||158|completo|
|scored_0643|E1|1|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0643|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0643|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0643|E2|52-1|266||233|válido|
|scored_0643|E2|52-3|250||238|válido|
|scored_0643|E2|52-6|255||229|válido|
|scored_0643|E2|64-1|266||241|válido|
|scored_0643|E2|64-3|250||238|válido|
|scored_0643|E2|64-6|255||229|válido|
|scored_0643|E3|joint|250|individual 250|247|OPTIMAL|
|scored_0643|E4|repair|250||244|completo|
|scored_0644|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0644|E1|3|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0644|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0644|E2|52-1|192||182|válido|
|scored_0644|E2|52-3|184||176|válido|
|scored_0644|E2|52-6|186||176|válido|
|scored_0644|E2|64-1|192||182|válido|
|scored_0644|E2|64-3|184||174|válido|
|scored_0644|E2|64-6|186||174|válido|
|scored_0644|E3|joint|184|individual 184|183|OPTIMAL|
|scored_0644|E4|repair|184||184|completo|
|scored_0645|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0645|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0645|E1|6|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0645|E2|52-1|273||248|válido|
|scored_0645|E2|52-3|265||246|válido|
|scored_0645|E2|52-6|260||247|válido|
|scored_0645|E2|64-1|273||249|válido|
|scored_0645|E2|64-3|265||247|válido|
|scored_0645|E2|64-6|260||246|válido|
|scored_0645|E3|joint|260|individual 260|259|OPTIMAL|
|scored_0645|E4|repair|260||260|completo|
|scored_0646|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0646|E1|3|227|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0646|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0646|E2|52-1|233||213|válido|
|scored_0646|E2|52-3|227||204|válido|
|scored_0646|E2|52-6|227||206|válido|
|scored_0646|E2|64-1|233||212|válido|
|scored_0646|E2|64-3|227||205|válido|
|scored_0646|E2|64-6|227||203|válido|
|scored_0646|E3|joint|227|individual 227|227|OPTIMAL|
|scored_0646|E4|repair|227||226|completo|
|scored_0647|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0647|E1|3|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0647|E1|6|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0647|E2|52-1|296||273|válido|
|scored_0647|E2|52-3|284||264|válido|
|scored_0647|E2|52-6|277||259|válido|
|scored_0647|E2|64-1|296||275|válido|
|scored_0647|E2|64-3|284||267|válido|
|scored_0647|E2|64-6|277||260|válido|
|scored_0647|E3|joint|277|individual 277|277|OPTIMAL|
|scored_0647|E4|repair|277||277|completo|
|scored_0648|E1|1|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0648|E1|3|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0648|E1|6|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0648|E2|52-1|127||127|válido|
|scored_0648|E2|52-3|126||127|válido|
|scored_0648|E2|52-6|126||127|válido|
|scored_0648|E2|64-1|127||129|válido|
|scored_0648|E2|64-3|126||128|válido|
|scored_0648|E2|64-6|126||128|válido|
|scored_0648|E3|joint|126|individual 126|126|OPTIMAL|
|scored_0648|E4|repair|126||124|completo|
|scored_0649|E1|1|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0649|E1|3|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0649|E1|6|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0649|E2|52-1|154||152|válido|
|scored_0649|E2|52-3|154||152|válido|
|scored_0649|E2|52-6|154||151|válido|
|scored_0649|E2|64-1|154||152|válido|
|scored_0649|E2|64-3|154||152|válido|
|scored_0649|E2|64-6|154||152|válido|
|scored_0649|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0649|E4|repair|154||154|completo|
|scored_0650|E1|1|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0650|E1|3|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0650|E1|6|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0650|E2|52-1|45||45|válido|
|scored_0650|E2|52-3|45||45|válido|
|scored_0650|E2|52-6|45||45|válido|
|scored_0650|E2|64-1|45||45|válido|
|scored_0650|E2|64-3|45||45|válido|
|scored_0650|E2|64-6|45||45|válido|
|scored_0650|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0650|E4|repair|45||44|completo|
|scored_0651|E1|1|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0651|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0651|E1|6|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0651|E2|52-1|241||223|válido|
|scored_0651|E2|52-3|235||209|válido|
|scored_0651|E2|52-6|228||206|válido|
|scored_0651|E2|64-1|241||222|válido|
|scored_0651|E2|64-3|235||212|válido|
|scored_0651|E2|64-6|228||208|válido|
|scored_0651|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0651|E4|repair|228||228|completo|
|scored_0652|E1|1|130|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0652|E1|3|130|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0652|E1|6|130|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0652|E2|52-1|130||127|válido|
|scored_0652|E2|52-3|130||127|válido|
|scored_0652|E2|52-6|130||127|válido|
|scored_0652|E2|64-1|130||127|válido|
|scored_0652|E2|64-3|130||127|válido|
|scored_0652|E2|64-6|130||127|válido|
|scored_0652|E3|joint|130|individual 130|130|OPTIMAL|
|scored_0652|E4|repair|130||127|completo|
|scored_0653|E1|1|233|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0653|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0653|E1|6|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0653|E2|52-1|233||219|válido|
|scored_0653|E2|52-3|226||216|válido|
|scored_0653|E2|52-6|228||212|válido|
|scored_0653|E2|64-1|233||218|válido|
|scored_0653|E2|64-3|226||219|válido|
|scored_0653|E2|64-6|228||209|válido|
|scored_0653|E3|joint|226|individual 225|225|OPTIMAL|
|scored_0653|E4|repair|226||224|completo|
|scored_0654|E1|1|48|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0654|E1|3|48|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0654|E1|6|48|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0654|E2|52-1|48||48|válido|
|scored_0654|E2|52-3|48||48|válido|
|scored_0654|E2|52-6|48||48|válido|
|scored_0654|E2|64-1|48||48|válido|
|scored_0654|E2|64-3|48||48|válido|
|scored_0654|E2|64-6|48||48|válido|
|scored_0654|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0654|E4|repair|48||44|completo|
|scored_0655|E1|1|367|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0655|E1|3|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0655|E1|6|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0655|E2|52-1|367||341|válido|
|scored_0655|E2|52-3|353||325|válido|
|scored_0655|E2|52-6|345||326|válido|
|scored_0655|E2|64-1|367||346|válido|
|scored_0655|E2|64-3|353||326|válido|
|scored_0655|E2|64-6|345||327|válido|
|scored_0655|E3|joint|345|individual 345|343|OPTIMAL|
|scored_0655|E4|repair|345||345|completo|
|scored_0656|E1|1|51|postprocess 48, optimize_a4 47, nested_windows 47|47|válido|
|scored_0656|E1|3|51|postprocess 48, optimize_a4 47, nested_windows 47|47|válido|
|scored_0656|E1|6|51|postprocess 48, optimize_a4 47, nested_windows 47|47|válido|
|scored_0656|E2|52-1|51||51|válido|
|scored_0656|E2|52-3|51||51|válido|
|scored_0656|E2|52-6|51||51|válido|
|scored_0656|E2|64-1|51||51|válido|
|scored_0656|E2|64-3|51||51|válido|
|scored_0656|E2|64-6|51||51|válido|
|scored_0656|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0656|E4|repair|51||51|completo|
|scored_0657|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0657|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0657|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0657|E2|52-1|125||122|válido|
|scored_0657|E2|52-3|125||122|válido|
|scored_0657|E2|52-6|124||122|válido|
|scored_0657|E2|64-1|125||122|válido|
|scored_0657|E2|64-3|125||122|válido|
|scored_0657|E2|64-6|124||122|válido|
|scored_0657|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0657|E4|repair|124||120|completo|
|scored_0658|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0658|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0658|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0658|E2|52-1|235||219|válido|
|scored_0658|E2|52-3|232||218|válido|
|scored_0658|E2|52-6|227||215|válido|
|scored_0658|E2|64-1|235||222|válido|
|scored_0658|E2|64-3|232||217|válido|
|scored_0658|E2|64-6|227||216|válido|
|scored_0658|E3|joint|227|individual 227|227|OPTIMAL|
|scored_0658|E4|repair|227||227|completo|
|scored_0659|E1|1|110|postprocess 110, optimize_a4 109, nested_windows 109|109|válido|
|scored_0659|E1|3|110|postprocess 110, optimize_a4 109, nested_windows 109|109|válido|
|scored_0659|E1|6|110|postprocess 110, optimize_a4 109, nested_windows 109|109|válido|
|scored_0659|E2|52-1|110||109|válido|
|scored_0659|E2|52-3|110||109|válido|
|scored_0659|E2|52-6|110||109|válido|
|scored_0659|E2|64-1|110||108|válido|
|scored_0659|E2|64-3|110||108|válido|
|scored_0659|E2|64-6|110||108|válido|
|scored_0659|E3|joint|110|individual 110|110|OPTIMAL|
|scored_0659|E4|repair|110||110|completo|
|scored_0660|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0660|E1|3|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0660|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0660|E2|52-1|238||218|válido|
|scored_0660|E2|52-3|230||217|válido|
|scored_0660|E2|52-6|226||215|válido|
|scored_0660|E2|64-1|238||219|válido|
|scored_0660|E2|64-3|230||221|válido|
|scored_0660|E2|64-6|226||213|válido|
|scored_0660|E3|joint|226|individual 226|225|OPTIMAL|
|scored_0660|E4|repair|226||226|completo|
|scored_0661|E1|1|314|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0661|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0661|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0661|E2|52-1|314||295|válido|
|scored_0661|E2|52-3|299||286|válido|
|scored_0661|E2|52-6|297||282|válido|
|scored_0661|E2|64-1|314||297|válido|
|scored_0661|E2|64-3|299||290|válido|
|scored_0661|E2|64-6|297||357|válido|
|scored_0661|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0661|E4|repair|297||297|completo|
|scored_0662|E1|1|238|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0662|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0662|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0662|E2|52-1|238||217|válido|
|scored_0662|E2|52-3|229||205|válido|
|scored_0662|E2|52-6|224||207|válido|
|scored_0662|E2|64-1|238||218|válido|
|scored_0662|E2|64-3|229||209|válido|
|scored_0662|E2|64-6|224||209|válido|
|scored_0662|E3|joint|224|individual 224|224|OPTIMAL|
|scored_0662|E4|repair|224||224|completo|
|scored_0663|E1|1|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0663|E1|3|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0663|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0663|E2|52-1|257||246|válido|
|scored_0663|E2|52-3|252||228|válido|
|scored_0663|E2|52-6|244||225|válido|
|scored_0663|E2|64-1|257||245|válido|
|scored_0663|E2|64-3|252||231|válido|
|scored_0663|E2|64-6|244||224|válido|
|scored_0663|E3|joint|244|individual 244|244|OPTIMAL|
|scored_0663|E4|repair|244||244|completo|
|scored_0664|E1|1|296|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0664|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0664|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0664|E2|52-1|296||278|válido|
|scored_0664|E2|52-3|277||269|válido|
|scored_0664|E2|52-6|278||287|válido|
|scored_0664|E2|64-1|296||278|válido|
|scored_0664|E2|64-3|277||269|válido|
|scored_0664|E2|64-6|278||294|válido|
|scored_0664|E3|joint|277|individual 276|275|OPTIMAL|
|scored_0664|E4|repair|277||277|completo|
|scored_0665|E1|1|127|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0665|E1|3|127|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0665|E1|6|127|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0665|E2|52-1|127||118|válido|
|scored_0665|E2|52-3|127||118|válido|
|scored_0665|E2|52-6|127||118|válido|
|scored_0665|E2|64-1|127||118|válido|
|scored_0665|E2|64-3|127||118|válido|
|scored_0665|E2|64-6|127||118|válido|
|scored_0665|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0665|E4|repair|127||121|completo|
|scored_0666|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0666|E1|3|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0666|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0666|E2|52-1|207||195|válido|
|scored_0666|E2|52-3|201||188|válido|
|scored_0666|E2|52-6|198||187|válido|
|scored_0666|E2|64-1|207||195|válido|
|scored_0666|E2|64-3|201||190|válido|
|scored_0666|E2|64-6|198||186|válido|
|scored_0666|E3|joint|198|individual 197|196|OPTIMAL|
|scored_0666|E4|repair|198||198|completo|
|scored_0667|E1|1|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0667|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0667|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0667|E2|52-1|228||214|válido|
|scored_0667|E2|52-3|226||213|válido|
|scored_0667|E2|52-6|224||212|válido|
|scored_0667|E2|64-1|228||217|válido|
|scored_0667|E2|64-3|226||210|válido|
|scored_0667|E2|64-6|224||211|válido|
|scored_0667|E3|joint|224|individual 224|224|OPTIMAL|
|scored_0667|E4|repair|224||223|completo|
|scored_0668|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0668|E1|3|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0668|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0668|E2|52-1|158||149|válido|
|scored_0668|E2|52-3|157||149|válido|
|scored_0668|E2|52-6|155||149|válido|
|scored_0668|E2|64-1|158||151|válido|
|scored_0668|E2|64-3|157||151|válido|
|scored_0668|E2|64-6|155||149|válido|
|scored_0668|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0668|E4|repair|155||154|completo|
|scored_0669|E1|1|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0669|E1|3|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0669|E1|6|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0669|E2|52-1|186||175|válido|
|scored_0669|E2|52-3|183||167|válido|
|scored_0669|E2|52-6|179||161|válido|
|scored_0669|E2|64-1|186||174|válido|
|scored_0669|E2|64-3|183||169|válido|
|scored_0669|E2|64-6|179||166|válido|
|scored_0669|E3|joint|179|individual 179|179|OPTIMAL|
|scored_0669|E4|repair|179||176|completo|
|scored_0670|E1|1|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0670|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0670|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0670|E2|52-1|252||233|válido|
|scored_0670|E2|52-3|242||228|válido|
|scored_0670|E2|52-6|241||222|válido|
|scored_0670|E2|64-1|252||232|válido|
|scored_0670|E2|64-3|242||227|válido|
|scored_0670|E2|64-6|241||224|válido|
|scored_0670|E3|joint|241|individual 240|239|OPTIMAL|
|scored_0670|E4|repair|241||241|completo|
|scored_0671|E1|1|273|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0671|E1|3|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0671|E1|6|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0671|E2|52-1|273||247|válido|
|scored_0671|E2|52-3|269||240|válido|
|scored_0671|E2|52-6|259||242|válido|
|scored_0671|E2|64-1|273||246|válido|
|scored_0671|E2|64-3|269||244|válido|
|scored_0671|E2|64-6|259||242|válido|
|scored_0671|E3|joint|259|individual 259|258|OPTIMAL|
|scored_0671|E4|repair|259||259|completo|
|scored_0672|E1|1|74|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0672|E1|3|75|postprocess 72, optimize_a4 72, nested_windows 72|72|válido|
|scored_0672|E1|6|75|postprocess 72, optimize_a4 72, nested_windows 72|72|válido|
|scored_0672|E2|52-1|74||74|válido|
|scored_0672|E2|52-3|75||75|válido|
|scored_0672|E2|52-6|75||75|válido|
|scored_0672|E2|64-1|74||74|válido|
|scored_0672|E2|64-3|75||75|válido|
|scored_0672|E2|64-6|75||75|válido|
|scored_0672|E3|joint|74|individual 74|74|OPTIMAL|
|scored_0672|E4|repair|74||74|completo|
|scored_0673|E1|1|224|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0673|E1|3|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0673|E1|6|222|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0673|E2|52-1|224||213|válido|
|scored_0673|E2|52-3|224||212|válido|
|scored_0673|E2|52-6|222||208|válido|
|scored_0673|E2|64-1|224||213|válido|
|scored_0673|E2|64-3|224||210|válido|
|scored_0673|E2|64-6|222||207|válido|
|scored_0673|E3|joint|222|individual 222|219|OPTIMAL|
|scored_0673|E4|repair|222||222|completo|
|scored_0674|E1|1|168|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0674|E1|3|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0674|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0674|E2|52-1|168||164|válido|
|scored_0674|E2|52-3|167||161|válido|
|scored_0674|E2|52-6|166||156|válido|
|scored_0674|E2|64-1|168||162|válido|
|scored_0674|E2|64-3|167||161|válido|
|scored_0674|E2|64-6|166||155|válido|
|scored_0674|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0674|E4|repair|166||163|completo|
|scored_0675|E1|1|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0675|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0675|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0675|E2|52-1|201||181|válido|
|scored_0675|E2|52-3|200||185|válido|
|scored_0675|E2|52-6|195||184|válido|
|scored_0675|E2|64-1|201||181|válido|
|scored_0675|E2|64-3|200||184|válido|
|scored_0675|E2|64-6|195||183|válido|
|scored_0675|E3|joint|195|individual 195|194|OPTIMAL|
|scored_0675|E4|repair|195||188|completo|
|scored_0676|E1|1|7|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|scored_0676|E1|3|7|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|scored_0676|E1|6|7|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|scored_0676|E2|52-1|7||7|válido|
|scored_0676|E2|52-3|7||7|válido|
|scored_0676|E2|52-6|7||7|válido|
|scored_0676|E2|64-1|7||7|válido|
|scored_0676|E2|64-3|7||7|válido|
|scored_0676|E2|64-6|7||7|válido|
|scored_0676|E3|joint|6|individual 6|6|OPTIMAL|
|scored_0676|E4|repair|6||6|completo|
|scored_0677|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0677|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0677|E1|6|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0677|E2|52-1|318||298|válido|
|scored_0677|E2|52-3|303||286|válido|
|scored_0677|E2|52-6|296||285|válido|
|scored_0677|E2|64-1|318||297|válido|
|scored_0677|E2|64-3|303||289|válido|
|scored_0677|E2|64-6|296||287|válido|
|scored_0677|E3|joint|296|individual 296|296|FEASIBLE|
|scored_0677|E4|repair|296||296|completo|
|scored_0678|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0678|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0678|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0678|E2|52-1|229||212|válido|
|scored_0678|E2|52-3|223||207|válido|
|scored_0678|E2|52-6|221||206|válido|
|scored_0678|E2|64-1|229||213|válido|
|scored_0678|E2|64-3|223||208|válido|
|scored_0678|E2|64-6|221||207|válido|
|scored_0678|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0678|E4|repair|221||221|completo|
|scored_0679|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0679|E1|3|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0679|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0679|E2|52-1|159||147|válido|
|scored_0679|E2|52-3|155||147|válido|
|scored_0679|E2|52-6|151||147|válido|
|scored_0679|E2|64-1|159||146|válido|
|scored_0679|E2|64-3|155||146|válido|
|scored_0679|E2|64-6|151||146|válido|
|scored_0679|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0679|E4|repair|151||147|completo|
|scored_0680|E1|1|287|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0680|E1|3|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0680|E1|6|266|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0680|E2|52-1|287||253|válido|
|scored_0680|E2|52-3|274||250|válido|
|scored_0680|E2|52-6|266||245|válido|
|scored_0680|E2|64-1|287||255|válido|
|scored_0680|E2|64-3|274||249|válido|
|scored_0680|E2|64-6|266||246|válido|
|scored_0680|E3|joint|266|individual 265|265|OPTIMAL|
|scored_0680|E4|repair|266||266|completo|
|scored_0681|E1|1|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0681|E1|3|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0681|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0681|E2|52-1|312||273|válido|
|scored_0681|E2|52-3|287||255|válido|
|scored_0681|E2|52-6|282||254|válido|
|scored_0681|E2|64-1|312||273|válido|
|scored_0681|E2|64-3|287||258|válido|
|scored_0681|E2|64-6|282||255|válido|
|scored_0681|E3|joint|282|individual 282|280|FEASIBLE|
|scored_0681|E4|repair|282||282|completo|
|scored_0682|E1|1|381|postprocess 381, optimize_a4 381, nested_windows 381|381|válido|
|scored_0682|E1|3|354|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0682|E1|6|351|postprocess 351, optimize_a4 351, nested_windows 351|351|válido|
|scored_0682|E2|52-1|381||346|válido|
|scored_0682|E2|52-3|354||334|válido|
|scored_0682|E2|52-6|351||327|válido|
|scored_0682|E2|64-1|381||354|válido|
|scored_0682|E2|64-3|354||341|válido|
|scored_0682|E2|64-6|351||329|válido|
|scored_0682|E3|joint|351|individual 351|346|OPTIMAL|
|scored_0682|E4|repair|351||351|completo|
|scored_0683|E1|1|193|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0683|E1|3|191|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0683|E1|6|191|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0683|E2|52-1|193||181|válido|
|scored_0683|E2|52-3|191||180|válido|
|scored_0683|E2|52-6|191||180|válido|
|scored_0683|E2|64-1|193||182|válido|
|scored_0683|E2|64-3|191||182|válido|
|scored_0683|E2|64-6|191||182|válido|
|scored_0683|E3|joint|191|individual 190|190|OPTIMAL|
|scored_0683|E4|repair|191||189|completo|
|scored_0684|E1|1|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0684|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0684|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0684|E2|52-1|143||140|válido|
|scored_0684|E2|52-3|143||140|válido|
|scored_0684|E2|52-6|143||139|válido|
|scored_0684|E2|64-1|143||139|válido|
|scored_0684|E2|64-3|143||139|válido|
|scored_0684|E2|64-6|143||139|válido|
|scored_0684|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0684|E4|repair|143||141|completo|
|scored_0685|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0685|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0685|E1|6|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0685|E2|52-1|243||230|válido|
|scored_0685|E2|52-3|238||220|válido|
|scored_0685|E2|52-6|233||220|válido|
|scored_0685|E2|64-1|243||231|válido|
|scored_0685|E2|64-3|238||226|válido|
|scored_0685|E2|64-6|233||222|válido|
|scored_0685|E3|joint|233|individual 233|233|OPTIMAL|
|scored_0685|E4|repair|233||233|completo|
|scored_0686|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0686|E1|3|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0686|E1|6|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0686|E2|52-1|125||117|válido|
|scored_0686|E2|52-3|125||117|válido|
|scored_0686|E2|52-6|125||117|válido|
|scored_0686|E2|64-1|125||119|válido|
|scored_0686|E2|64-3|125||119|válido|
|scored_0686|E2|64-6|125||119|válido|
|scored_0686|E3|joint|125|individual 125|125|OPTIMAL|
|scored_0686|E4|repair|125||123|completo|
|scored_0687|E1|1|324|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0687|E1|3|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0687|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0687|E2|52-1|324||304|válido|
|scored_0687|E2|52-3|313||290|válido|
|scored_0687|E2|52-6|303||286|válido|
|scored_0687|E2|64-1|324||305|válido|
|scored_0687|E2|64-3|313||293|válido|
|scored_0687|E2|64-6|303||286|válido|
|scored_0687|E3|joint|303|individual 303|303|OPTIMAL|
|scored_0687|E4|repair|303||303|completo|
|scored_0688|E1|1|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0688|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0688|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0688|E2|52-1|227||210|válido|
|scored_0688|E2|52-3|220||202|válido|
|scored_0688|E2|52-6|214||204|válido|
|scored_0688|E2|64-1|227||207|válido|
|scored_0688|E2|64-3|220||203|válido|
|scored_0688|E2|64-6|214||206|válido|
|scored_0688|E3|joint|214|individual 214|213|OPTIMAL|
|scored_0688|E4|repair|214||214|completo|
|scored_0689|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0689|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0689|E1|6|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0689|E2|52-1|254||233|válido|
|scored_0689|E2|52-3|241||227|válido|
|scored_0689|E2|52-6|243||225|válido|
|scored_0689|E2|64-1|254||236|válido|
|scored_0689|E2|64-3|241||228|válido|
|scored_0689|E2|64-6|243||228|válido|
|scored_0689|E3|joint|241|individual 241|240|OPTIMAL|
|scored_0689|E4|repair|241||241|completo|
|scored_0690|E1|1|381|postprocess 381, optimize_a4 381, nested_windows 381|381|válido|
|scored_0690|E1|3|365|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0690|E1|6|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0690|E2|52-1|381||351|válido|
|scored_0690|E2|52-3|365||340|válido|
|scored_0690|E2|52-6|350||331|válido|
|scored_0690|E2|64-1|381||353|válido|
|scored_0690|E2|64-3|365||343|válido|
|scored_0690|E2|64-6|350||333|válido|
|scored_0690|E3|joint|350|individual 350|350|FEASIBLE|
|scored_0690|E4|repair|350||350|completo|
|scored_0691|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0691|E1|3|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0691|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0691|E2|52-1|192||178|válido|
|scored_0691|E2|52-3|189||174|válido|
|scored_0691|E2|52-6|186||177|válido|
|scored_0691|E2|64-1|192||180|válido|
|scored_0691|E2|64-3|189||176|válido|
|scored_0691|E2|64-6|186||176|válido|
|scored_0691|E3|joint|186|individual 186|186|OPTIMAL|
|scored_0691|E4|repair|186||185|completo|
|scored_0692|E1|1|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0692|E1|3|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0692|E1|6|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0692|E2|52-1|291||263|válido|
|scored_0692|E2|52-3|268||254|válido|
|scored_0692|E2|52-6|270||249|válido|
|scored_0692|E2|64-1|291||262|válido|
|scored_0692|E2|64-3|268||255|válido|
|scored_0692|E2|64-6|270||249|válido|
|scored_0692|E3|joint|268|individual 268|265|OPTIMAL|
|scored_0692|E4|repair|268||268|completo|
|scored_0693|E1|1|236|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0693|E1|3|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0693|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0693|E2|52-1|236||213|válido|
|scored_0693|E2|52-3|234||208|válido|
|scored_0693|E2|52-6|228||209|válido|
|scored_0693|E2|64-1|236||213|válido|
|scored_0693|E2|64-3|234||211|válido|
|scored_0693|E2|64-6|228||209|válido|
|scored_0693|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0693|E4|repair|228||227|completo|
|scored_0694|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0694|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0694|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0694|E2|52-1|219||201|válido|
|scored_0694|E2|52-3|208||197|válido|
|scored_0694|E2|52-6|207||193|válido|
|scored_0694|E2|64-1|219||200|válido|
|scored_0694|E2|64-3|208||198|válido|
|scored_0694|E2|64-6|207||192|válido|
|scored_0694|E3|joint|207|individual 207|206|OPTIMAL|
|scored_0694|E4|repair|207||207|completo|
|scored_0695|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0695|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0695|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0695|E2|52-1|155||147|válido|
|scored_0695|E2|52-3|152||145|válido|
|scored_0695|E2|52-6|151||144|válido|
|scored_0695|E2|64-1|155||145|válido|
|scored_0695|E2|64-3|152||145|válido|
|scored_0695|E2|64-6|151||142|válido|
|scored_0695|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0695|E4|repair|151||150|completo|
|scored_0696|E1|1|106|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0696|E1|3|106|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0696|E1|6|106|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0696|E2|52-1|106||104|válido|
|scored_0696|E2|52-3|106||104|válido|
|scored_0696|E2|52-6|106||104|válido|
|scored_0696|E2|64-1|106||104|válido|
|scored_0696|E2|64-3|106||104|válido|
|scored_0696|E2|64-6|106||104|válido|
|scored_0696|E3|joint|106|individual 106|106|OPTIMAL|
|scored_0696|E4|repair|106||101|completo|
|scored_0697|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0697|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0697|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0697|E2|52-1|144||137|válido|
|scored_0697|E2|52-3|144||135|válido|
|scored_0697|E2|52-6|143||135|válido|
|scored_0697|E2|64-1|144||137|válido|
|scored_0697|E2|64-3|144||135|válido|
|scored_0697|E2|64-6|143||135|válido|
|scored_0697|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0697|E4|repair|143||138|completo|
|scored_0698|E1|1|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0698|E1|3|324|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0698|E1|6|315|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0698|E2|52-1|347||302|válido|
|scored_0698|E2|52-3|324||294|válido|
|scored_0698|E2|52-6|315||291|válido|
|scored_0698|E2|64-1|347||309|válido|
|scored_0698|E2|64-3|324||294|válido|
|scored_0698|E2|64-6|315||293|válido|
|scored_0698|E3|joint|315|individual 315|315|rechazado por registros|
|scored_0698|E4|repair|315||315|completo|
|scored_0699|E1|1|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0699|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0699|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0699|E2|52-1|196||177|válido|
|scored_0699|E2|52-3|191||175|válido|
|scored_0699|E2|52-6|190||176|válido|
|scored_0699|E2|64-1|196||177|válido|
|scored_0699|E2|64-3|191||175|válido|
|scored_0699|E2|64-6|190||176|válido|
|scored_0699|E3|joint|190|individual 189|189|OPTIMAL|
|scored_0699|E4|repair|190||187|completo|
|scored_0700|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0700|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0700|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0700|E2|52-1|233||221|válido|
|scored_0700|E2|52-3|225||212|válido|
|scored_0700|E2|52-6|220||213|válido|
|scored_0700|E2|64-1|233||220|válido|
|scored_0700|E2|64-3|225||213|válido|
|scored_0700|E2|64-6|220||213|válido|
|scored_0700|E3|joint|220|individual 220|220|OPTIMAL|
|scored_0700|E4|repair|220||220|completo|
|scored_0701|E1|1|41|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0701|E1|3|41|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0701|E1|6|41|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0701|E2|52-1|41||41|válido|
|scored_0701|E2|52-3|41||41|válido|
|scored_0701|E2|52-6|41||41|válido|
|scored_0701|E2|64-1|41||41|válido|
|scored_0701|E2|64-3|41||41|válido|
|scored_0701|E2|64-6|41||41|válido|
|scored_0701|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0701|E4|repair|41||41|completo|
|scored_0702|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0702|E1|3|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0702|E1|6|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0702|E2|52-1|306||277|válido|
|scored_0702|E2|52-3|279||267|válido|
|scored_0702|E2|52-6|272||262|válido|
|scored_0702|E2|64-1|306||277|válido|
|scored_0702|E2|64-3|279||267|válido|
|scored_0702|E2|64-6|272||263|válido|
|scored_0702|E3|joint|272|individual 272|272|OPTIMAL|
|scored_0702|E4|repair|272||272|completo|
|scored_0703|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0703|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0703|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0703|E2|52-1|235||214|válido|
|scored_0703|E2|52-3|229||214|válido|
|scored_0703|E2|52-6|223||214|válido|
|scored_0703|E2|64-1|235||217|válido|
|scored_0703|E2|64-3|229||215|válido|
|scored_0703|E2|64-6|223||214|válido|
|scored_0703|E3|joint|223|individual 223|223|OPTIMAL|
|scored_0703|E4|repair|223||223|completo|
|scored_0704|E1|1|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0704|E1|3|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0704|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0704|E2|52-1|166||161|válido|
|scored_0704|E2|52-3|165||156|válido|
|scored_0704|E2|52-6|166||156|válido|
|scored_0704|E2|64-1|166||158|válido|
|scored_0704|E2|64-3|165||157|válido|
|scored_0704|E2|64-6|166||157|válido|
|scored_0704|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0704|E4|repair|165||162|completo|
|scored_0705|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0705|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0705|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0705|E2|52-1|218||200|válido|
|scored_0705|E2|52-3|209||192|válido|
|scored_0705|E2|52-6|208||191|válido|
|scored_0705|E2|64-1|218||202|válido|
|scored_0705|E2|64-3|209||192|válido|
|scored_0705|E2|64-6|208||192|válido|
|scored_0705|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0705|E4|repair|208||208|completo|
|scored_0706|E1|1|263|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0706|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0706|E1|6|247|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0706|E2|52-1|263||242|válido|
|scored_0706|E2|52-3|261||237|válido|
|scored_0706|E2|52-6|247||229|válido|
|scored_0706|E2|64-1|263||241|válido|
|scored_0706|E2|64-3|261||235|válido|
|scored_0706|E2|64-6|247||228|válido|
|scored_0706|E3|joint|247|individual 247|247|OPTIMAL|
|scored_0706|E4|repair|247||247|completo|
|scored_0707|E1|1|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0707|E1|3|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0707|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0707|E2|52-1|122||117|válido|
|scored_0707|E2|52-3|122||117|válido|
|scored_0707|E2|52-6|122||117|válido|
|scored_0707|E2|64-1|122||119|válido|
|scored_0707|E2|64-3|122||119|válido|
|scored_0707|E2|64-6|122||118|válido|
|scored_0707|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0707|E4|repair|122||117|completo|
|scored_0708|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0708|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0708|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0708|E2|52-1|202||194|válido|
|scored_0708|E2|52-3|199||186|válido|
|scored_0708|E2|52-6|198||190|válido|
|scored_0708|E2|64-1|202||195|válido|
|scored_0708|E2|64-3|199||188|válido|
|scored_0708|E2|64-6|198||188|válido|
|scored_0708|E3|joint|198|individual 198|196|OPTIMAL|
|scored_0708|E4|repair|198||198|completo|
|scored_0709|E1|1|178|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0709|E1|3|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0709|E1|6|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0709|E2|52-1|178||162|válido|
|scored_0709|E2|52-3|169||159|válido|
|scored_0709|E2|52-6|169||158|válido|
|scored_0709|E2|64-1|178||168|válido|
|scored_0709|E2|64-3|169||165|válido|
|scored_0709|E2|64-6|169||160|válido|
|scored_0709|E3|joint|169|individual 169|169|OPTIMAL|
|scored_0709|E4|repair|169||166|completo|
|scored_0710|E1|1|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0710|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0710|E1|6|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0710|E2|52-1|160||152|válido|
|scored_0710|E2|52-3|159||148|válido|
|scored_0710|E2|52-6|159||148|válido|
|scored_0710|E2|64-1|160||152|válido|
|scored_0710|E2|64-3|159||148|válido|
|scored_0710|E2|64-6|159||148|válido|
|scored_0710|E3|joint|159|individual 158|157|OPTIMAL|
|scored_0710|E4|repair|159||157|completo|
|scored_0711|E1|1|31|postprocess 29, optimize_a4 29, nested_windows 29|29|válido|
|scored_0711|E1|3|31|postprocess 29, optimize_a4 29, nested_windows 29|29|válido|
|scored_0711|E1|6|31|postprocess 29, optimize_a4 29, nested_windows 29|29|válido|
|scored_0711|E2|52-1|31||31|válido|
|scored_0711|E2|52-3|31||31|válido|
|scored_0711|E2|52-6|31||31|válido|
|scored_0711|E2|64-1|31||31|válido|
|scored_0711|E2|64-3|31||31|válido|
|scored_0711|E2|64-6|31||31|válido|
|scored_0711|E3|joint|31|individual 31|31|OPTIMAL|
|scored_0711|E4|repair|31||31|completo|
|scored_0712|E1|1|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0712|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0712|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0712|E2|52-1|269||243|válido|
|scored_0712|E2|52-3|258||235|válido|
|scored_0712|E2|52-6|255||237|válido|
|scored_0712|E2|64-1|269||244|válido|
|scored_0712|E2|64-3|258||233|válido|
|scored_0712|E2|64-6|255||238|válido|
|scored_0712|E3|joint|255|individual 255|254|OPTIMAL|
|scored_0712|E4|repair|255||255|completo|
|scored_0713|E1|1|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0713|E1|3|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0713|E1|6|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0713|E2|52-1|130||122|válido|
|scored_0713|E2|52-3|130||122|válido|
|scored_0713|E2|52-6|130||122|válido|
|scored_0713|E2|64-1|130||124|válido|
|scored_0713|E2|64-3|130||124|válido|
|scored_0713|E2|64-6|130||124|válido|
|scored_0713|E3|joint|130|individual 130|130|OPTIMAL|
|scored_0713|E4|repair|130||127|completo|
|scored_0714|E1|1|271|postprocess 271, optimize_a4 271, nested_windows 271|271|válido|
|scored_0714|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0714|E1|6|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|scored_0714|E2|52-1|271||244|válido|
|scored_0714|E2|52-3|261||243|válido|
|scored_0714|E2|52-6|256||232|válido|
|scored_0714|E2|64-1|271||251|válido|
|scored_0714|E2|64-3|261||247|válido|
|scored_0714|E2|64-6|256||241|válido|
|scored_0714|E3|joint|256|individual 256|254|OPTIMAL|
|scored_0714|E4|repair|256||254|completo|
|scored_0715|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0715|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0715|E1|6|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0715|E2|52-1|219||207|válido|
|scored_0715|E2|52-3|214||197|válido|
|scored_0715|E2|52-6|210||200|válido|
|scored_0715|E2|64-1|219||206|válido|
|scored_0715|E2|64-3|214||197|válido|
|scored_0715|E2|64-6|210||199|válido|
|scored_0715|E3|joint|210|individual 210|209|OPTIMAL|
|scored_0715|E4|repair|210||210|completo|
|scored_0716|E1|1|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0716|E1|3|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0716|E1|6|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0716|E2|52-1|154||151|válido|
|scored_0716|E2|52-3|154||148|válido|
|scored_0716|E2|52-6|154||148|válido|
|scored_0716|E2|64-1|154||150|válido|
|scored_0716|E2|64-3|154||150|válido|
|scored_0716|E2|64-6|154||150|válido|
|scored_0716|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0716|E4|repair|154||151|completo|
|scored_0717|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0717|E1|3|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0717|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0717|E2|52-1|165||156|válido|
|scored_0717|E2|52-3|164||155|válido|
|scored_0717|E2|52-6|160||155|válido|
|scored_0717|E2|64-1|165||153|válido|
|scored_0717|E2|64-3|164||153|válido|
|scored_0717|E2|64-6|160||153|válido|
|scored_0717|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0717|E4|repair|160||156|completo|
|scored_0718|E1|1|188|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0718|E1|3|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0718|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0718|E2|52-1|188||177|válido|
|scored_0718|E2|52-3|183||173|válido|
|scored_0718|E2|52-6|183||173|válido|
|scored_0718|E2|64-1|188||176|válido|
|scored_0718|E2|64-3|183||171|válido|
|scored_0718|E2|64-6|183||171|válido|
|scored_0718|E3|joint|183|individual 182|180|OPTIMAL|
|scored_0718|E4|repair|183||181|completo|
|scored_0719|E1|1|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0719|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0719|E1|6|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0719|E2|52-1|353||332|válido|
|scored_0719|E2|52-3|337||320|válido|
|scored_0719|E2|52-6|330||309|válido|
|scored_0719|E2|64-1|353||338|válido|
|scored_0719|E2|64-3|337||326|válido|
|scored_0719|E2|64-6|330||315|válido|
|scored_0719|E3|joint|330|individual 330|330|OPTIMAL|
|scored_0719|E4|repair|330||330|completo|
|scored_0720|E1|1|38|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0720|E1|3|38|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0720|E1|6|38|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0720|E2|52-1|38||38|válido|
|scored_0720|E2|52-3|38||38|válido|
|scored_0720|E2|52-6|38||38|válido|
|scored_0720|E2|64-1|38||38|válido|
|scored_0720|E2|64-3|38||38|válido|
|scored_0720|E2|64-6|38||38|válido|
|scored_0720|E3|joint|38|individual 38|38|OPTIMAL|
|scored_0720|E4|repair|38||38|completo|
|scored_0721|E1|1|6|postprocess 6, optimize_a4 4, nested_windows 4|4|válido|
|scored_0721|E1|3|6|postprocess 6, optimize_a4 4, nested_windows 4|4|válido|
|scored_0721|E1|6|6|postprocess 6, optimize_a4 4, nested_windows 4|4|válido|
|scored_0721|E2|52-1|6||6|válido|
|scored_0721|E2|52-3|6||6|válido|
|scored_0721|E2|52-6|6||6|válido|
|scored_0721|E2|64-1|6||6|válido|
|scored_0721|E2|64-3|6||6|válido|
|scored_0721|E2|64-6|6||6|válido|
|scored_0721|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0721|E4|repair|4||4|completo|
|scored_0722|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0722|E1|3|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0722|E1|6|210|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0722|E2|52-1|226||207|válido|
|scored_0722|E2|52-3|221||198|válido|
|scored_0722|E2|52-6|210||198|válido|
|scored_0722|E2|64-1|226||212|válido|
|scored_0722|E2|64-3|221||204|válido|
|scored_0722|E2|64-6|210||211|válido|
|scored_0722|E3|joint|210|individual 208|208|OPTIMAL|
|scored_0722|E4|repair|210||208|completo|
|scored_0723|E1|1|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0723|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0723|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0723|E2|52-1|132||125|válido|
|scored_0723|E2|52-3|131||125|válido|
|scored_0723|E2|52-6|131||125|válido|
|scored_0723|E2|64-1|132||125|válido|
|scored_0723|E2|64-3|131||125|válido|
|scored_0723|E2|64-6|131||125|válido|
|scored_0723|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0723|E4|repair|131||129|completo|
|scored_0724|E1|1|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0724|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0724|E1|6|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0724|E2|52-1|221||210|válido|
|scored_0724|E2|52-3|218||206|válido|
|scored_0724|E2|52-6|218||199|válido|
|scored_0724|E2|64-1|221||211|válido|
|scored_0724|E2|64-3|218||207|válido|
|scored_0724|E2|64-6|218||200|válido|
|scored_0724|E3|joint|218|individual 217|217|OPTIMAL|
|scored_0724|E4|repair|218||218|completo|
|scored_0725|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0725|E1|3|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0725|E1|6|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0725|E2|52-1|319||289|válido|
|scored_0725|E2|52-3|300||281|válido|
|scored_0725|E2|52-6|302||273|válido|
|scored_0725|E2|64-1|319||354|válido|
|scored_0725|E2|64-3|300||279|válido|
|scored_0725|E2|64-6|302||275|válido|
|scored_0725|E3|joint|300|individual 299|298|OPTIMAL|
|scored_0725|E4|repair|300||300|completo|
|scored_0726|E1|1|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0726|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0726|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0726|E2|52-1|199||190|válido|
|scored_0726|E2|52-3|196||186|válido|
|scored_0726|E2|52-6|191||186|válido|
|scored_0726|E2|64-1|199||190|válido|
|scored_0726|E2|64-3|196||187|válido|
|scored_0726|E2|64-6|191||186|válido|
|scored_0726|E3|joint|191|individual 191|189|OPTIMAL|
|scored_0726|E4|repair|191||191|completo|
|scored_0727|E1|1|108|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0727|E1|3|108|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0727|E1|6|108|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0727|E2|52-1|108||105|válido|
|scored_0727|E2|52-3|108||105|válido|
|scored_0727|E2|52-6|108||105|válido|
|scored_0727|E2|64-1|108||105|válido|
|scored_0727|E2|64-3|108||105|válido|
|scored_0727|E2|64-6|108||105|válido|
|scored_0727|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0727|E4|repair|108||103|completo|
|scored_0728|E1|1|375|postprocess 375, optimize_a4 375, nested_windows 375|375|válido|
|scored_0728|E1|3|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|scored_0728|E1|6|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0728|E2|52-1|375||337|válido|
|scored_0728|E2|52-3|343||322|válido|
|scored_0728|E2|52-6|339||315|válido|
|scored_0728|E2|64-1|375||337|válido|
|scored_0728|E2|64-3|343||322|válido|
|scored_0728|E2|64-6|339||361|válido|
|scored_0728|E3|joint|339|individual 339|339|FEASIBLE|
|scored_0728|E4|repair|339||339|completo|
|scored_0729|E1|1|425|postprocess 425, optimize_a4 425, nested_windows 425|425|válido|
|scored_0729|E1|3|393|postprocess 393, optimize_a4 393, nested_windows 393|393|válido|
|scored_0729|E1|6|384|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|scored_0729|E2|52-1|425||372|válido|
|scored_0729|E2|52-3|393||355|válido|
|scored_0729|E2|52-6|384||352|válido|
|scored_0729|E2|64-1|425||375|válido|
|scored_0729|E2|64-3|393||360|válido|
|scored_0729|E2|64-6|384||355|válido|
|scored_0729|E3|joint|384|individual 382|381|FEASIBLE|
|scored_0729|E4|repair|384||384|completo|
|scored_0730|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0730|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0730|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0730|E2|52-1|176||165|válido|
|scored_0730|E2|52-3|171||164|válido|
|scored_0730|E2|52-6|167||161|válido|
|scored_0730|E2|64-1|176||165|válido|
|scored_0730|E2|64-3|171||167|válido|
|scored_0730|E2|64-6|167||163|válido|
|scored_0730|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0730|E4|repair|167||161|completo|
|scored_0731|E1|1|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0731|E1|3|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0731|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0731|E2|52-1|227||199|válido|
|scored_0731|E2|52-3|220||195|válido|
|scored_0731|E2|52-6|208||195|válido|
|scored_0731|E2|64-1|227||200|válido|
|scored_0731|E2|64-3|220||199|válido|
|scored_0731|E2|64-6|208||197|válido|
|scored_0731|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0731|E4|repair|208||207|completo|
|scored_0732|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0732|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0732|E1|6|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0732|E2|52-1|163||157|válido|
|scored_0732|E2|52-3|159||154|válido|
|scored_0732|E2|52-6|158||154|válido|
|scored_0732|E2|64-1|163||156|válido|
|scored_0732|E2|64-3|159||154|válido|
|scored_0732|E2|64-6|158||154|válido|
|scored_0732|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0732|E4|repair|158||156|completo|
|scored_0733|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0733|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0733|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0733|E2|52-1|123||121|válido|
|scored_0733|E2|52-3|116||116|válido|
|scored_0733|E2|52-6|116||117|válido|
|scored_0733|E2|64-1|123||121|válido|
|scored_0733|E2|64-3|116||116|válido|
|scored_0733|E2|64-6|116||117|válido|
|scored_0733|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0733|E4|repair|116||116|completo|
|scored_0734|E1|1|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0734|E1|3|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0734|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0734|E2|52-1|193||180|válido|
|scored_0734|E2|52-3|184||179|válido|
|scored_0734|E2|52-6|182||178|válido|
|scored_0734|E2|64-1|193||179|válido|
|scored_0734|E2|64-3|184||178|válido|
|scored_0734|E2|64-6|182||179|válido|
|scored_0734|E3|joint|182|individual 182|182|OPTIMAL|
|scored_0734|E4|repair|182||182|completo|
|scored_0735|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0735|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0735|E1|6|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0735|E2|52-1|158||150|válido|
|scored_0735|E2|52-3|157||148|válido|
|scored_0735|E2|52-6|157||148|válido|
|scored_0735|E2|64-1|158||147|válido|
|scored_0735|E2|64-3|157||145|válido|
|scored_0735|E2|64-6|157||145|válido|
|scored_0735|E3|joint|157|individual 157|157|OPTIMAL|
|scored_0735|E4|repair|157||155|completo|
|scored_0736|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0736|E1|3|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0736|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0736|E2|52-1|223||203|válido|
|scored_0736|E2|52-3|211||202|válido|
|scored_0736|E2|52-6|208||196|válido|
|scored_0736|E2|64-1|223||206|válido|
|scored_0736|E2|64-3|211||203|válido|
|scored_0736|E2|64-6|208||195|válido|
|scored_0736|E3|joint|208|individual 207|203|OPTIMAL|
|scored_0736|E4|repair|208||206|completo|
|scored_0737|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0737|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0737|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0737|E2|52-1|151||148|válido|
|scored_0737|E2|52-3|151||148|válido|
|scored_0737|E2|52-6|148||148|válido|
|scored_0737|E2|64-1|151||147|válido|
|scored_0737|E2|64-3|151||147|válido|
|scored_0737|E2|64-6|148||147|válido|
|scored_0737|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0737|E4|repair|148||144|completo|
|scored_0738|E1|1|160|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0738|E1|3|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0738|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0738|E2|52-1|160||147|válido|
|scored_0738|E2|52-3|155||147|válido|
|scored_0738|E2|52-6|152||146|válido|
|scored_0738|E2|64-1|160||147|válido|
|scored_0738|E2|64-3|155||147|válido|
|scored_0738|E2|64-6|152||146|válido|
|scored_0738|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0738|E4|repair|152||151|completo|
|scored_0739|E1|1|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0739|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0739|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0739|E2|52-1|208||194|válido|
|scored_0739|E2|52-3|202||191|válido|
|scored_0739|E2|52-6|202||191|válido|
|scored_0739|E2|64-1|208||193|válido|
|scored_0739|E2|64-3|202||192|válido|
|scored_0739|E2|64-6|202||187|válido|
|scored_0739|E3|joint|202|individual 201|201|OPTIMAL|
|scored_0739|E4|repair|202||202|completo|
|scored_0740|E1|1|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0740|E1|3|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0740|E1|6|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0740|E2|52-1|180||169|válido|
|scored_0740|E2|52-3|178||168|válido|
|scored_0740|E2|52-6|176||167|válido|
|scored_0740|E2|64-1|180||167|válido|
|scored_0740|E2|64-3|178||166|válido|
|scored_0740|E2|64-6|176||164|válido|
|scored_0740|E3|joint|176|individual 176|176|OPTIMAL|
|scored_0740|E4|repair|176||175|completo|
|scored_0741|E1|1|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0741|E1|3|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0741|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0741|E2|52-1|186||173|válido|
|scored_0741|E2|52-3|183||168|válido|
|scored_0741|E2|52-6|182||167|válido|
|scored_0741|E2|64-1|186||178|válido|
|scored_0741|E2|64-3|183||168|válido|
|scored_0741|E2|64-6|182||168|válido|
|scored_0741|E3|joint|182|individual 182|180|OPTIMAL|
|scored_0741|E4|repair|182||182|completo|
|scored_0742|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0742|E1|3|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0742|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0742|E2|52-1|247||230|válido|
|scored_0742|E2|52-3|240||229|válido|
|scored_0742|E2|52-6|242||224|válido|
|scored_0742|E2|64-1|247||234|válido|
|scored_0742|E2|64-3|240||231|válido|
|scored_0742|E2|64-6|242||230|válido|
|scored_0742|E3|joint|240|individual 239|237|OPTIMAL|
|scored_0742|E4|repair|240||240|completo|
|scored_0743|E1|1|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0743|E1|3|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0743|E1|6|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0743|E2|52-1|140||138|válido|
|scored_0743|E2|52-3|140||138|válido|
|scored_0743|E2|52-6|140||134|válido|
|scored_0743|E2|64-1|140||133|válido|
|scored_0743|E2|64-3|140||133|válido|
|scored_0743|E2|64-6|140||132|válido|
|scored_0743|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0743|E4|repair|140||138|completo|
|scored_0744|E1|1|397|postprocess 397, optimize_a4 397, nested_windows 397|397|válido|
|scored_0744|E1|3|378|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0744|E1|6|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0744|E2|52-1|397||365|válido|
|scored_0744|E2|52-3|378||354|válido|
|scored_0744|E2|52-6|370||343|válido|
|scored_0744|E2|64-1|397||369|válido|
|scored_0744|E2|64-3|378||353|válido|
|scored_0744|E2|64-6|370||349|válido|
|scored_0744|E3|joint|370|individual 369|367|FEASIBLE|
|scored_0744|E4|repair|370||370|completo|
|scored_0745|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0745|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0745|E1|6|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0745|E2|52-1|247||238|válido|
|scored_0745|E2|52-3|239||223|válido|
|scored_0745|E2|52-6|235||220|válido|
|scored_0745|E2|64-1|247||239|válido|
|scored_0745|E2|64-3|239||226|válido|
|scored_0745|E2|64-6|235||222|válido|
|scored_0745|E3|joint|235|individual 235|234|FEASIBLE|
|scored_0745|E4|repair|235||232|completo|
|scored_0746|E1|1|209|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0746|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0746|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0746|E2|52-1|209||188|válido|
|scored_0746|E2|52-3|205||184|válido|
|scored_0746|E2|52-6|202||187|válido|
|scored_0746|E2|64-1|209||194|válido|
|scored_0746|E2|64-3|205||190|válido|
|scored_0746|E2|64-6|202||189|válido|
|scored_0746|E3|joint|202|individual 201|200|OPTIMAL|
|scored_0746|E4|repair|202||202|completo|
|scored_0747|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0747|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0747|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0747|E2|52-1|210||195|válido|
|scored_0747|E2|52-3|209||191|válido|
|scored_0747|E2|52-6|204||191|válido|
|scored_0747|E2|64-1|210||194|válido|
|scored_0747|E2|64-3|209||190|válido|
|scored_0747|E2|64-6|204||193|válido|
|scored_0747|E3|joint|204|individual 204|204|OPTIMAL|
|scored_0747|E4|repair|204||204|completo|
|scored_0748|E1|1|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0748|E1|3|314|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0748|E1|6|306|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0748|E2|52-1|327||301|válido|
|scored_0748|E2|52-3|314||296|válido|
|scored_0748|E2|52-6|306||288|válido|
|scored_0748|E2|64-1|327||301|válido|
|scored_0748|E2|64-3|314||299|válido|
|scored_0748|E2|64-6|306||291|válido|
|scored_0748|E3|joint|306|individual 306|306|OPTIMAL|
|scored_0748|E4|repair|306||306|completo|
|scored_0749|E1|1|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0749|E1|3|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0749|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0749|E2|52-1|124||119|válido|
|scored_0749|E2|52-3|124||119|válido|
|scored_0749|E2|52-6|124||118|válido|
|scored_0749|E2|64-1|124||118|válido|
|scored_0749|E2|64-3|124||118|válido|
|scored_0749|E2|64-6|124||118|válido|
|scored_0749|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0749|E4|repair|124||122|completo|
|scored_0750|E1|1|149|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0750|E1|3|149|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0750|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0750|E2|52-1|149||143|válido|
|scored_0750|E2|52-3|149||142|válido|
|scored_0750|E2|52-6|146||141|válido|
|scored_0750|E2|64-1|149||142|válido|
|scored_0750|E2|64-3|149||141|válido|
|scored_0750|E2|64-6|146||141|válido|
|scored_0750|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0750|E4|repair|146||143|completo|
|scored_0751|E1|1|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0751|E1|3|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0751|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0751|E2|52-1|321||278|válido|
|scored_0751|E2|52-3|291||272|válido|
|scored_0751|E2|52-6|284||268|válido|
|scored_0751|E2|64-1|321||278|válido|
|scored_0751|E2|64-3|291||271|válido|
|scored_0751|E2|64-6|284||270|válido|
|scored_0751|E3|joint|284|individual 283|283|OPTIMAL|
|scored_0751|E4|repair|284||284|completo|
|scored_0752|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0752|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0752|E1|6|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0752|E2|52-1|133||128|válido|
|scored_0752|E2|52-3|133||128|válido|
|scored_0752|E2|52-6|132||128|válido|
|scored_0752|E2|64-1|133||129|válido|
|scored_0752|E2|64-3|133||129|válido|
|scored_0752|E2|64-6|132||129|válido|
|scored_0752|E3|joint|132|individual 132|132|OPTIMAL|
|scored_0752|E4|repair|132||132|completo|
|scored_0753|E1|1|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0753|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0753|E1|6|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0753|E2|52-1|126||122|válido|
|scored_0753|E2|52-3|125||122|válido|
|scored_0753|E2|52-6|125||122|válido|
|scored_0753|E2|64-1|126||122|válido|
|scored_0753|E2|64-3|125||122|válido|
|scored_0753|E2|64-6|125||122|válido|
|scored_0753|E3|joint|125|individual 125|125|OPTIMAL|
|scored_0753|E4|repair|125||123|completo|
|scored_0754|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0754|E1|3|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0754|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0754|E2|52-1|236||216|válido|
|scored_0754|E2|52-3|227||212|válido|
|scored_0754|E2|52-6|221||208|válido|
|scored_0754|E2|64-1|236||219|válido|
|scored_0754|E2|64-3|227||212|válido|
|scored_0754|E2|64-6|221||211|válido|
|scored_0754|E3|joint|221|individual 220|220|OPTIMAL|
|scored_0754|E4|repair|221||220|completo|
|scored_0755|E1|1|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0755|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0755|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0755|E2|52-1|150||147|válido|
|scored_0755|E2|52-3|148||147|válido|
|scored_0755|E2|52-6|147||144|válido|
|scored_0755|E2|64-1|150||147|válido|
|scored_0755|E2|64-3|148||147|válido|
|scored_0755|E2|64-6|147||147|válido|
|scored_0755|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0755|E4|repair|147||142|completo|
|scored_0756|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0756|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0756|E1|6|201|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0756|E2|52-1|211||197|válido|
|scored_0756|E2|52-3|205||191|válido|
|scored_0756|E2|52-6|201||191|válido|
|scored_0756|E2|64-1|211||197|válido|
|scored_0756|E2|64-3|205||192|válido|
|scored_0756|E2|64-6|201||192|válido|
|scored_0756|E3|joint|201|individual 201|201|OPTIMAL|
|scored_0756|E4|repair|201||201|completo|
|scored_0757|E1|1|175|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0757|E1|3|174|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0757|E1|6|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0757|E2|52-1|175||163|válido|
|scored_0757|E2|52-3|174||163|válido|
|scored_0757|E2|52-6|170||163|válido|
|scored_0757|E2|64-1|175||165|válido|
|scored_0757|E2|64-3|174||164|válido|
|scored_0757|E2|64-6|170||163|válido|
|scored_0757|E3|joint|170|individual 170|170|OPTIMAL|
|scored_0757|E4|repair|170||168|completo|
|scored_0758|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0758|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0758|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0758|E2|52-1|145||137|válido|
|scored_0758|E2|52-3|143||137|válido|
|scored_0758|E2|52-6|143||138|válido|
|scored_0758|E2|64-1|145||136|válido|
|scored_0758|E2|64-3|143||137|válido|
|scored_0758|E2|64-6|143||137|válido|
|scored_0758|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0758|E4|repair|143||141|completo|
|scored_0759|E1|1|283|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0759|E1|3|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0759|E1|6|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0759|E2|52-1|283||258|válido|
|scored_0759|E2|52-3|276||253|válido|
|scored_0759|E2|52-6|268||244|válido|
|scored_0759|E2|64-1|283||259|válido|
|scored_0759|E2|64-3|276||255|válido|
|scored_0759|E2|64-6|268||244|válido|
|scored_0759|E3|joint|268|individual 268|268|OPTIMAL|
|scored_0759|E4|repair|268||268|completo|
|scored_0760|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0760|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0760|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0760|E2|52-1|146||141|válido|
|scored_0760|E2|52-3|146||138|válido|
|scored_0760|E2|52-6|146||138|válido|
|scored_0760|E2|64-1|146||138|válido|
|scored_0760|E2|64-3|146||137|válido|
|scored_0760|E2|64-6|146||137|válido|
|scored_0760|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0760|E4|repair|146||146|completo|
|scored_0761|E1|1|128|postprocess 128, optimize_a4 128, nested_windows 128|128|válido|
|scored_0761|E1|3|128|postprocess 128, optimize_a4 128, nested_windows 128|128|válido|
|scored_0761|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0761|E2|52-1|128||125|válido|
|scored_0761|E2|52-3|128||125|válido|
|scored_0761|E2|52-6|127||124|válido|
|scored_0761|E2|64-1|128||125|válido|
|scored_0761|E2|64-3|128||123|válido|
|scored_0761|E2|64-6|127||123|válido|
|scored_0761|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0761|E4|repair|127||124|completo|
|scored_0762|E1|1|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|scored_0762|E1|3|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0762|E1|6|361|postprocess 361, optimize_a4 361, nested_windows 361|361|válido|
|scored_0762|E2|52-1|385||358|válido|
|scored_0762|E2|52-3|364||341|válido|
|scored_0762|E2|52-6|361||347|válido|
|scored_0762|E2|64-1|385||363|válido|
|scored_0762|E2|64-3|364||342|válido|
|scored_0762|E2|64-6|361||348|válido|
|scored_0762|E3|joint|361|individual 360|359|FEASIBLE|
|scored_0762|E4|repair|361||361|completo|
|scored_0763|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0763|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0763|E1|6|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0763|E2|52-1|119||116|válido|
|scored_0763|E2|52-3|120||115|válido|
|scored_0763|E2|52-6|121||115|válido|
|scored_0763|E2|64-1|119||116|válido|
|scored_0763|E2|64-3|120||115|válido|
|scored_0763|E2|64-6|121||115|válido|
|scored_0763|E3|joint|119|individual 119|119|OPTIMAL|
|scored_0763|E4|repair|119||119|completo|
|scored_0764|E1|1|291|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0764|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0764|E1|6|266|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0764|E2|52-1|291||253|válido|
|scored_0764|E2|52-3|270||249|válido|
|scored_0764|E2|52-6|266||249|válido|
|scored_0764|E2|64-1|291||250|válido|
|scored_0764|E2|64-3|270||251|válido|
|scored_0764|E2|64-6|266||248|válido|
|scored_0764|E3|joint|266|individual 266|266|OPTIMAL|
|scored_0764|E4|repair|266||266|completo|
|scored_0765|E1|1|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0765|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0765|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0765|E2|52-1|255||237|válido|
|scored_0765|E2|52-3|249||228|válido|
|scored_0765|E2|52-6|246||231|válido|
|scored_0765|E2|64-1|255||237|válido|
|scored_0765|E2|64-3|249||230|válido|
|scored_0765|E2|64-6|246||234|válido|
|scored_0765|E3|joint|246|individual 246|244|OPTIMAL|
|scored_0765|E4|repair|246||245|completo|
|scored_0766|E1|1|295|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0766|E1|3|281|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0766|E1|6|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0766|E2|52-1|295||270|válido|
|scored_0766|E2|52-3|281||263|válido|
|scored_0766|E2|52-6|279||262|válido|
|scored_0766|E2|64-1|295||271|válido|
|scored_0766|E2|64-3|281||269|válido|
|scored_0766|E2|64-6|279||263|válido|
|scored_0766|E3|joint|279|individual 279|277|OPTIMAL|
|scored_0766|E4|repair|279||279|completo|
|scored_0767|E1|1|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0767|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0767|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0767|E2|52-1|161||153|válido|
|scored_0767|E2|52-3|157||150|válido|
|scored_0767|E2|52-6|152||149|válido|
|scored_0767|E2|64-1|161||153|válido|
|scored_0767|E2|64-3|157||150|válido|
|scored_0767|E2|64-6|152||149|válido|
|scored_0767|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0767|E4|repair|152||151|completo|
|scored_0768|E1|1|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0768|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0768|E1|6|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0768|E2|52-1|165||155|válido|
|scored_0768|E2|52-3|162||156|válido|
|scored_0768|E2|52-6|158||152|válido|
|scored_0768|E2|64-1|165||156|válido|
|scored_0768|E2|64-3|162||157|válido|
|scored_0768|E2|64-6|158||153|válido|
|scored_0768|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0768|E4|repair|158||158|completo|
|scored_0769|E1|1|172|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0769|E1|3|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0769|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0769|E2|52-1|172||150|válido|
|scored_0769|E2|52-3|169||145|válido|
|scored_0769|E2|52-6|168||144|válido|
|scored_0769|E2|64-1|172||147|válido|
|scored_0769|E2|64-3|169||148|válido|
|scored_0769|E2|64-6|168||148|válido|
|scored_0769|E3|joint|168|individual 168|167|OPTIMAL|
|scored_0769|E4|repair|168||164|completo|
|scored_0770|E1|1|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0770|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0770|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0770|E2|52-1|209||191|válido|
|scored_0770|E2|52-3|202||184|válido|
|scored_0770|E2|52-6|197||191|válido|
|scored_0770|E2|64-1|209||191|válido|
|scored_0770|E2|64-3|202||188|válido|
|scored_0770|E2|64-6|197||194|válido|
|scored_0770|E3|joint|197|individual 197|196|OPTIMAL|
|scored_0770|E4|repair|197||197|completo|
|scored_0771|E1|1|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0771|E1|3|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0771|E1|6|340|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0771|E2|52-1|358||339|válido|
|scored_0771|E2|52-3|348||322|válido|
|scored_0771|E2|52-6|340||321|válido|
|scored_0771|E2|64-1|358||344|válido|
|scored_0771|E2|64-3|348||326|válido|
|scored_0771|E2|64-6|340||325|válido|
|scored_0771|E3|joint|340|individual 340|340|OPTIMAL|
|scored_0771|E4|repair|340||340|completo|
|scored_0772|E1|1|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0772|E1|3|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0772|E1|6|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0772|E2|52-1|159||156|válido|
|scored_0772|E2|52-3|155||150|válido|
|scored_0772|E2|52-6|155||148|válido|
|scored_0772|E2|64-1|159||156|válido|
|scored_0772|E2|64-3|155||149|válido|
|scored_0772|E2|64-6|155||149|válido|
|scored_0772|E3|joint|155|individual 154|154|OPTIMAL|
|scored_0772|E4|repair|155||151|completo|
|scored_0773|E1|1|63|postprocess 60, optimize_a4 60, nested_windows 60|60|válido|
|scored_0773|E1|3|63|postprocess 60, optimize_a4 60, nested_windows 60|60|válido|
|scored_0773|E1|6|63|postprocess 60, optimize_a4 60, nested_windows 60|60|válido|
|scored_0773|E2|52-1|63||63|válido|
|scored_0773|E2|52-3|63||63|válido|
|scored_0773|E2|52-6|63||63|válido|
|scored_0773|E2|64-1|63||63|válido|
|scored_0773|E2|64-3|63||63|válido|
|scored_0773|E2|64-6|63||63|válido|
|scored_0773|E3|joint|63|individual 63|63|OPTIMAL|
|scored_0773|E4|repair|63||63|completo|
|scored_0774|E1|1|401|postprocess 399, optimize_a4 399, nested_windows 399|399|válido|
|scored_0774|E1|3|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|scored_0774|E1|6|374|postprocess 374, optimize_a4 374, nested_windows 374|374|válido|
|scored_0774|E2|52-1|401||366|válido|
|scored_0774|E2|52-3|385||352|válido|
|scored_0774|E2|52-6|374||344|válido|
|scored_0774|E2|64-1|401||368|válido|
|scored_0774|E2|64-3|385||353|válido|
|scored_0774|E2|64-6|374||344|válido|
|scored_0774|E3|joint|374|individual 374|374|OPTIMAL|
|scored_0774|E4|repair|374||374|completo|
|scored_0775|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0775|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0775|E1|6|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0775|E2|52-1|211||194|válido|
|scored_0775|E2|52-3|197||196|válido|
|scored_0775|E2|52-6|196||186|válido|
|scored_0775|E2|64-1|211||188|válido|
|scored_0775|E2|64-3|197||197|válido|
|scored_0775|E2|64-6|196||186|válido|
|scored_0775|E3|joint|196|individual 196|196|OPTIMAL|
|scored_0775|E4|repair|196||194|completo|
|scored_0776|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0776|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0776|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0776|E2|52-1|298||286|válido|
|scored_0776|E2|52-3|299||282|válido|
|scored_0776|E2|52-6|292||273|válido|
|scored_0776|E2|64-1|298||284|válido|
|scored_0776|E2|64-3|299||283|válido|
|scored_0776|E2|64-6|292||279|válido|
|scored_0776|E3|joint|292|individual 292|291|OPTIMAL|
|scored_0776|E4|repair|292||292|completo|
|scored_0777|E1|1|49|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0777|E1|3|49|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0777|E1|6|49|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0777|E2|52-1|49||50|válido|
|scored_0777|E2|52-3|49||50|válido|
|scored_0777|E2|52-6|49||50|válido|
|scored_0777|E2|64-1|49||50|válido|
|scored_0777|E2|64-3|49||50|válido|
|scored_0777|E2|64-6|49||50|válido|
|scored_0777|E3|joint|49|individual 49|49|OPTIMAL|
|scored_0777|E4|repair|49||49|completo|
|scored_0778|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0778|E1|3|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0778|E1|6|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0778|E2|52-1|176||163|válido|
|scored_0778|E2|52-3|172||163|válido|
|scored_0778|E2|52-6|171||163|válido|
|scored_0778|E2|64-1|176||167|válido|
|scored_0778|E2|64-3|172||165|válido|
|scored_0778|E2|64-6|171||165|válido|
|scored_0778|E3|joint|171|individual 171|171|OPTIMAL|
|scored_0778|E4|repair|171||170|completo|
|scored_0779|E1|1|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0779|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0779|E1|6|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0779|E2|52-1|213||199|válido|
|scored_0779|E2|52-3|206||179|válido|
|scored_0779|E2|52-6|200||180|válido|
|scored_0779|E2|64-1|213||199|válido|
|scored_0779|E2|64-3|206||179|válido|
|scored_0779|E2|64-6|200||180|válido|
|scored_0779|E3|joint|200|individual 200|200|OPTIMAL|
|scored_0779|E4|repair|200||199|completo|
|scored_0780|E1|1|379|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0780|E1|3|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0780|E1|6|359|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0780|E2|52-1|379||353|válido|
|scored_0780|E2|52-3|358||331|válido|
|scored_0780|E2|52-6|359||333|válido|
|scored_0780|E2|64-1|379||355|válido|
|scored_0780|E2|64-3|358||336|válido|
|scored_0780|E2|64-6|359||333|válido|
|scored_0780|E3|joint|358|individual 354|353|FEASIBLE|
|scored_0780|E4|repair|358||358|completo|
|scored_0781|E1|1|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0781|E1|3|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0781|E1|6|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0781|E2|52-1|348||322|válido|
|scored_0781|E2|52-3|336||311|válido|
|scored_0781|E2|52-6|327||305|válido|
|scored_0781|E2|64-1|348||325|válido|
|scored_0781|E2|64-3|336||313|válido|
|scored_0781|E2|64-6|327||307|válido|
|scored_0781|E3|joint|327|individual 326|326|FEASIBLE|
|scored_0781|E4|repair|327||327|completo|
|scored_0782|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0782|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0782|E1|6|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0782|E2|52-1|206||190|válido|
|scored_0782|E2|52-3|200||188|válido|
|scored_0782|E2|52-6|194||189|válido|
|scored_0782|E2|64-1|206||190|válido|
|scored_0782|E2|64-3|200||188|válido|
|scored_0782|E2|64-6|194||186|válido|
|scored_0782|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0782|E4|repair|194||194|completo|
|scored_0783|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0783|E1|3|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0783|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0783|E2|52-1|151||140|válido|
|scored_0783|E2|52-3|147||140|válido|
|scored_0783|E2|52-6|148||139|válido|
|scored_0783|E2|64-1|151||139|válido|
|scored_0783|E2|64-3|147||139|válido|
|scored_0783|E2|64-6|148||139|válido|
|scored_0783|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0783|E4|repair|147||146|completo|
|scored_0784|E1|1|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0784|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0784|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0784|E2|52-1|322||289|válido|
|scored_0784|E2|52-3|303||275|válido|
|scored_0784|E2|52-6|297||281|válido|
|scored_0784|E2|64-1|322||290|válido|
|scored_0784|E2|64-3|303||278|válido|
|scored_0784|E2|64-6|297||282|válido|
|scored_0784|E3|joint|297|individual 297|294|FEASIBLE|
|scored_0784|E4|repair|297||297|completo|
|scored_0785|E1|1|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0785|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0785|E1|6|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0785|E2|52-1|140||134|válido|
|scored_0785|E2|52-3|140||134|válido|
|scored_0785|E2|52-6|140||134|válido|
|scored_0785|E2|64-1|140||134|válido|
|scored_0785|E2|64-3|140||134|válido|
|scored_0785|E2|64-6|140||134|válido|
|scored_0785|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0785|E4|repair|140||138|completo|
|scored_0786|E1|1|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0786|E1|3|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0786|E1|6|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0786|E2|52-1|347||321|válido|
|scored_0786|E2|52-3|327||309|válido|
|scored_0786|E2|52-6|326||299|válido|
|scored_0786|E2|64-1|347||325|válido|
|scored_0786|E2|64-3|327||308|válido|
|scored_0786|E2|64-6|326||304|válido|
|scored_0786|E3|joint|326|individual 325|323|FEASIBLE|
|scored_0786|E4|repair|326||326|completo|
|scored_0787|E1|1|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0787|E1|3|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0787|E1|6|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0787|E2|52-1|134||127|válido|
|scored_0787|E2|52-3|134||127|válido|
|scored_0787|E2|52-6|133||127|válido|
|scored_0787|E2|64-1|134||126|válido|
|scored_0787|E2|64-3|134||126|válido|
|scored_0787|E2|64-6|133||126|válido|
|scored_0787|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0787|E4|repair|133||133|completo|
|scored_0788|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0788|E1|3|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0788|E1|6|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0788|E2|52-1|319||294|válido|
|scored_0788|E2|52-3|311||287|válido|
|scored_0788|E2|52-6|300||286|válido|
|scored_0788|E2|64-1|319||295|válido|
|scored_0788|E2|64-3|311||291|válido|
|scored_0788|E2|64-6|300||285|válido|
|scored_0788|E3|joint|300|individual 300|300|OPTIMAL|
|scored_0788|E4|repair|300||300|completo|
|scored_0789|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0789|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0789|E1|6|226|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0789|E2|52-1|250||226|válido|
|scored_0789|E2|52-3|239||226|válido|
|scored_0789|E2|52-6|226||219|válido|
|scored_0789|E2|64-1|250||229|válido|
|scored_0789|E2|64-3|239||225|válido|
|scored_0789|E2|64-6|226||220|válido|
|scored_0789|E3|joint|226|individual 225|225|OPTIMAL|
|scored_0789|E4|repair|226||226|completo|
|scored_0790|E1|1|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0790|E1|3|370|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0790|E1|6|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0790|E2|52-1|336||289|válido|
|scored_0790|E2|52-3|370||274|válido|
|scored_0790|E2|52-6|301||272|válido|
|scored_0790|E2|64-1|336||293|válido|
|scored_0790|E2|64-3|370||277|válido|
|scored_0790|E2|64-6|301||277|válido|
|scored_0790|E3|joint|301|individual 301|300|OPTIMAL|
|scored_0790|E4|repair|301||301|completo|
|scored_0791|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0791|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0791|E1|6|204|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0791|E2|52-1|218||202|válido|
|scored_0791|E2|52-3|207||199|válido|
|scored_0791|E2|52-6|204||193|válido|
|scored_0791|E2|64-1|218||203|válido|
|scored_0791|E2|64-3|207||198|válido|
|scored_0791|E2|64-6|204||192|válido|
|scored_0791|E3|joint|204|individual 204|202|OPTIMAL|
|scored_0791|E4|repair|204||203|completo|
|scored_0792|E1|1|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0792|E1|3|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0792|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0792|E2|52-1|182||170|válido|
|scored_0792|E2|52-3|177||169|válido|
|scored_0792|E2|52-6|174||168|válido|
|scored_0792|E2|64-1|182||174|válido|
|scored_0792|E2|64-3|177||167|válido|
|scored_0792|E2|64-6|174||169|válido|
|scored_0792|E3|joint|174|individual 174|173|OPTIMAL|
|scored_0792|E4|repair|174||174|completo|
|scored_0793|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0793|E1|3|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0793|E1|6|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0793|E2|52-1|318||284|válido|
|scored_0793|E2|52-3|306||273|válido|
|scored_0793|E2|52-6|302||277|válido|
|scored_0793|E2|64-1|318||287|válido|
|scored_0793|E2|64-3|306||273|válido|
|scored_0793|E2|64-6|302||278|válido|
|scored_0793|E3|joint|302|individual 302|302|FEASIBLE|
|scored_0793|E4|repair|302||302|completo|
|scored_0794|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0794|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0794|E1|6|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0794|E2|52-1|240||227|válido|
|scored_0794|E2|52-3|229||213|válido|
|scored_0794|E2|52-6|223||210|válido|
|scored_0794|E2|64-1|240||230|válido|
|scored_0794|E2|64-3|229||213|válido|
|scored_0794|E2|64-6|223||210|válido|
|scored_0794|E3|joint|223|individual 222|222|OPTIMAL|
|scored_0794|E4|repair|223||223|completo|
|scored_0795|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0795|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0795|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0795|E2|52-1|248||231|válido|
|scored_0795|E2|52-3|241||225|válido|
|scored_0795|E2|52-6|238||221|válido|
|scored_0795|E2|64-1|248||226|válido|
|scored_0795|E2|64-3|241||225|válido|
|scored_0795|E2|64-6|238||220|válido|
|scored_0795|E3|joint|238|individual 238|238|OPTIMAL|
|scored_0795|E4|repair|238||238|completo|
|scored_0796|E1|1|320|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0796|E1|3|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0796|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0796|E2|52-1|320||293|válido|
|scored_0796|E2|52-3|302||278|válido|
|scored_0796|E2|52-6|297||278|válido|
|scored_0796|E2|64-1|320||293|válido|
|scored_0796|E2|64-3|302||279|válido|
|scored_0796|E2|64-6|297||278|válido|
|scored_0796|E3|joint|297|individual 296|295|FEASIBLE|
|scored_0796|E4|repair|297||297|completo|
|scored_0797|E1|1|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0797|E1|3|94|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0797|E1|6|91|postprocess 91, optimize_a4 91, nested_windows 91|91|válido|
|scored_0797|E2|52-1|90||91|válido|
|scored_0797|E2|52-3|94||94|válido|
|scored_0797|E2|52-6|91||90|válido|
|scored_0797|E2|64-1|90||91|válido|
|scored_0797|E2|64-3|94||94|válido|
|scored_0797|E2|64-6|91||90|válido|
|scored_0797|E3|joint|90|individual 90|90|OPTIMAL|
|scored_0797|E4|repair|90||90|completo|
|scored_0798|E1|1|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0798|E1|3|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0798|E1|6|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0798|E2|52-1|4||4|válido|
|scored_0798|E2|52-3|4||4|válido|
|scored_0798|E2|52-6|4||4|válido|
|scored_0798|E2|64-1|4||4|válido|
|scored_0798|E2|64-3|4||4|válido|
|scored_0798|E2|64-6|4||4|válido|
|scored_0798|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0798|E4|repair|2||2|completo|
|scored_0799|E1|1|329|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0799|E1|3|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0799|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0799|E2|52-1|329||301|válido|
|scored_0799|E2|52-3|315||290|válido|
|scored_0799|E2|52-6|311||286|válido|
|scored_0799|E2|64-1|329||302|válido|
|scored_0799|E2|64-3|315||298|válido|
|scored_0799|E2|64-6|311||285|válido|
|scored_0799|E3|joint|311|individual 311|310|FEASIBLE|
|scored_0799|E4|repair|311||311|completo|
|scored_0800|E1|1|104|postprocess 104, optimize_a4 103, nested_windows 103|103|válido|
|scored_0800|E1|3|104|postprocess 104, optimize_a4 103, nested_windows 103|103|válido|
|scored_0800|E1|6|104|postprocess 104, optimize_a4 103, nested_windows 103|103|válido|
|scored_0800|E2|52-1|104||103|válido|
|scored_0800|E2|52-3|104||103|válido|
|scored_0800|E2|52-6|104||103|válido|
|scored_0800|E2|64-1|104||102|válido|
|scored_0800|E2|64-3|104||102|válido|
|scored_0800|E2|64-6|104||102|válido|
|scored_0800|E3|joint|104|individual 104|104|OPTIMAL|
|scored_0800|E4|repair|104||102|completo|
|scored_0801|E1|1|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0801|E1|3|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0801|E1|6|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0801|E2|52-1|129||127|válido|
|scored_0801|E2|52-3|129||126|válido|
|scored_0801|E2|52-6|129||126|válido|
|scored_0801|E2|64-1|129||127|válido|
|scored_0801|E2|64-3|129||126|válido|
|scored_0801|E2|64-6|129||126|válido|
|scored_0801|E3|joint|129|individual 127|127|OPTIMAL|
|scored_0801|E4|repair|129||125|completo|
|scored_0802|E1|1|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0802|E1|3|249|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0802|E1|6|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0802|E2|52-1|264||243|válido|
|scored_0802|E2|52-3|249||235|válido|
|scored_0802|E2|52-6|249||232|válido|
|scored_0802|E2|64-1|264||245|válido|
|scored_0802|E2|64-3|249||235|válido|
|scored_0802|E2|64-6|249||233|válido|
|scored_0802|E3|joint|249|individual 249|247|OPTIMAL|
|scored_0802|E4|repair|249||249|completo|
|scored_0803|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0803|E1|3|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0803|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0803|E2|52-1|192||178|válido|
|scored_0803|E2|52-3|184||177|válido|
|scored_0803|E2|52-6|187||177|válido|
|scored_0803|E2|64-1|192||177|válido|
|scored_0803|E2|64-3|184||174|válido|
|scored_0803|E2|64-6|187||174|válido|
|scored_0803|E3|joint|184|individual 184|184|OPTIMAL|
|scored_0803|E4|repair|184||184|completo|
|scored_0804|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0804|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0804|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0804|E2|52-1|145||144|válido|
|scored_0804|E2|52-3|145||143|válido|
|scored_0804|E2|52-6|145||143|válido|
|scored_0804|E2|64-1|145||144|válido|
|scored_0804|E2|64-3|145||144|válido|
|scored_0804|E2|64-6|145||144|válido|
|scored_0804|E3|joint|145|individual 145|145|OPTIMAL|
|scored_0804|E4|repair|145||141|completo|
|scored_0805|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0805|E1|3|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0805|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0805|E2|52-1|226||200|válido|
|scored_0805|E2|52-3|219||200|válido|
|scored_0805|E2|52-6|214||200|válido|
|scored_0805|E2|64-1|226||200|válido|
|scored_0805|E2|64-3|219||201|válido|
|scored_0805|E2|64-6|214||201|válido|
|scored_0805|E3|joint|214|individual 213|213|OPTIMAL|
|scored_0805|E4|repair|214||214|completo|
|scored_0806|E1|1|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0806|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0806|E1|6|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0806|E2|52-1|121||121|válido|
|scored_0806|E2|52-3|120||120|válido|
|scored_0806|E2|52-6|120||120|válido|
|scored_0806|E2|64-1|121||121|válido|
|scored_0806|E2|64-3|120||120|válido|
|scored_0806|E2|64-6|120||120|válido|
|scored_0806|E3|joint|120|individual 120|120|OPTIMAL|
|scored_0806|E4|repair|120||119|completo|
|scored_0807|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0807|E1|3|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0807|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0807|E2|52-1|197||184|válido|
|scored_0807|E2|52-3|187||179|válido|
|scored_0807|E2|52-6|186||177|válido|
|scored_0807|E2|64-1|197||186|válido|
|scored_0807|E2|64-3|187||184|válido|
|scored_0807|E2|64-6|186||176|válido|
|scored_0807|E3|joint|186|individual 186|184|OPTIMAL|
|scored_0807|E4|repair|186||183|completo|
|scored_0808|E1|1|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0808|E1|3|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0808|E1|6|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0808|E2|52-1|363||332|válido|
|scored_0808|E2|52-3|339||310|válido|
|scored_0808|E2|52-6|339||307|válido|
|scored_0808|E2|64-1|363||332|válido|
|scored_0808|E2|64-3|339||313|válido|
|scored_0808|E2|64-6|339||305|válido|
|scored_0808|E3|joint|339|individual 337|336|FEASIBLE|
|scored_0808|E4|repair|339||339|completo|
|scored_0809|E1|1|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0809|E1|3|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0809|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0809|E2|52-1|185||171|válido|
|scored_0809|E2|52-3|182||166|válido|
|scored_0809|E2|52-6|181||166|válido|
|scored_0809|E2|64-1|185||168|válido|
|scored_0809|E2|64-3|182||170|válido|
|scored_0809|E2|64-6|181||170|válido|
|scored_0809|E3|joint|181|individual 181|180|OPTIMAL|
|scored_0809|E4|repair|181||180|completo|
|scored_0810|E1|1|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0810|E1|3|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0810|E1|6|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0810|E2|52-1|45||45|válido|
|scored_0810|E2|52-3|45||45|válido|
|scored_0810|E2|52-6|45||45|válido|
|scored_0810|E2|64-1|45||45|válido|
|scored_0810|E2|64-3|45||45|válido|
|scored_0810|E2|64-6|45||45|válido|
|scored_0810|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0810|E4|repair|45||41|completo|
|scored_0811|E1|1|115|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0811|E1|3|115|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0811|E1|6|115|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0811|E2|52-1|115||113|válido|
|scored_0811|E2|52-3|115||113|válido|
|scored_0811|E2|52-6|115||113|válido|
|scored_0811|E2|64-1|115||113|válido|
|scored_0811|E2|64-3|115||113|válido|
|scored_0811|E2|64-6|115||113|válido|
|scored_0811|E3|joint|115|individual 115|115|OPTIMAL|
|scored_0811|E4|repair|115||114|completo|
|scored_0812|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0812|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0812|E1|6|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0812|E2|52-1|133||131|válido|
|scored_0812|E2|52-3|133||131|válido|
|scored_0812|E2|52-6|133||131|válido|
|scored_0812|E2|64-1|133||131|válido|
|scored_0812|E2|64-3|133||131|válido|
|scored_0812|E2|64-6|133||131|válido|
|scored_0812|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0812|E4|repair|133||133|completo|
|scored_0813|E1|1|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0813|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0813|E1|6|295|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0813|E2|52-1|326||286|válido|
|scored_0813|E2|52-3|303||285|válido|
|scored_0813|E2|52-6|295||279|válido|
|scored_0813|E2|64-1|326||289|válido|
|scored_0813|E2|64-3|303||284|válido|
|scored_0813|E2|64-6|295||279|válido|
|scored_0813|E3|joint|295|individual 295|295|OPTIMAL|
|scored_0813|E4|repair|295||295|completo|
|scored_0814|E1|1|256|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0814|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0814|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0814|E2|52-1|256||218|válido|
|scored_0814|E2|52-3|231||212|válido|
|scored_0814|E2|52-6|222||203|válido|
|scored_0814|E2|64-1|256||219|válido|
|scored_0814|E2|64-3|231||204|válido|
|scored_0814|E2|64-6|222||200|válido|
|scored_0814|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0814|E4|repair|222||222|completo|
|scored_0815|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0815|E1|3|290|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0815|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0815|E2|52-1|303||272|válido|
|scored_0815|E2|52-3|290||261|válido|
|scored_0815|E2|52-6|284||262|válido|
|scored_0815|E2|64-1|303||276|válido|
|scored_0815|E2|64-3|290||264|válido|
|scored_0815|E2|64-6|284||264|válido|
|scored_0815|E3|joint|284|individual 281|280|OPTIMAL|
|scored_0815|E4|repair|284||283|completo|
|scored_0816|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0816|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0816|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0816|E2|52-1|141||133|válido|
|scored_0816|E2|52-3|140||133|válido|
|scored_0816|E2|52-6|139||133|válido|
|scored_0816|E2|64-1|141||133|válido|
|scored_0816|E2|64-3|140||133|válido|
|scored_0816|E2|64-6|139||133|válido|
|scored_0816|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0816|E4|repair|139||135|completo|
|scored_0817|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0817|E1|3|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0817|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0817|E2|52-1|236||218|válido|
|scored_0817|E2|52-3|225||214|válido|
|scored_0817|E2|52-6|226||209|válido|
|scored_0817|E2|64-1|236||219|válido|
|scored_0817|E2|64-3|225||215|válido|
|scored_0817|E2|64-6|226||209|válido|
|scored_0817|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0817|E4|repair|225||225|completo|
|scored_0818|E1|1|34|postprocess 33, optimize_a4 32, nested_windows 32|32|válido|
|scored_0818|E1|3|34|postprocess 33, optimize_a4 32, nested_windows 32|32|válido|
|scored_0818|E1|6|34|postprocess 33, optimize_a4 32, nested_windows 32|32|válido|
|scored_0818|E2|52-1|34||34|válido|
|scored_0818|E2|52-3|34||34|válido|
|scored_0818|E2|52-6|34||34|válido|
|scored_0818|E2|64-1|34||34|válido|
|scored_0818|E2|64-3|34||34|válido|
|scored_0818|E2|64-6|34||34|válido|
|scored_0818|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0818|E4|repair|34||34|completo|
|scored_0819|E1|1|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0819|E1|3|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0819|E1|6|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0819|E2|52-1|327||308|válido|
|scored_0819|E2|52-3|306||296|válido|
|scored_0819|E2|52-6|310||282|válido|
|scored_0819|E2|64-1|327||308|válido|
|scored_0819|E2|64-3|306||297|válido|
|scored_0819|E2|64-6|310||282|válido|
|scored_0819|E3|joint|306|individual 305|304|FEASIBLE|
|scored_0819|E4|repair|306||306|completo|
|scored_0820|E1|1|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0820|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0820|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0820|E2|52-1|326||287|válido|
|scored_0820|E2|52-3|308||276|válido|
|scored_0820|E2|52-6|304||276|válido|
|scored_0820|E2|64-1|326||288|válido|
|scored_0820|E2|64-3|308||276|válido|
|scored_0820|E2|64-6|304||277|válido|
|scored_0820|E3|joint|304|individual 304|303|OPTIMAL|
|scored_0820|E4|repair|304||302|completo|
|scored_0821|E1|1|133|postprocess 133, optimize_a4 132, nested_windows 132|132|válido|
|scored_0821|E1|3|133|postprocess 133, optimize_a4 132, nested_windows 132|132|válido|
|scored_0821|E1|6|133|postprocess 133, optimize_a4 132, nested_windows 132|132|válido|
|scored_0821|E2|52-1|133||127|válido|
|scored_0821|E2|52-3|133||127|válido|
|scored_0821|E2|52-6|133||127|válido|
|scored_0821|E2|64-1|133||127|válido|
|scored_0821|E2|64-3|133||127|válido|
|scored_0821|E2|64-6|133||127|válido|
|scored_0821|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0821|E4|repair|133||133|completo|
|scored_0822|E1|1|48|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0822|E1|3|48|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0822|E1|6|48|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0822|E2|52-1|48||48|válido|
|scored_0822|E2|52-3|48||48|válido|
|scored_0822|E2|52-6|48||48|válido|
|scored_0822|E2|64-1|48||48|válido|
|scored_0822|E2|64-3|48||48|válido|
|scored_0822|E2|64-6|48||48|válido|
|scored_0822|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0822|E4|repair|48||48|completo|
|scored_0823|E1|1|232|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0823|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0823|E1|6|221|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0823|E2|52-1|232||211|válido|
|scored_0823|E2|52-3|226||208|válido|
|scored_0823|E2|52-6|221||209|válido|
|scored_0823|E2|64-1|232||218|válido|
|scored_0823|E2|64-3|226||208|válido|
|scored_0823|E2|64-6|221||192|válido|
|scored_0823|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0823|E4|repair|221||219|completo|
|scored_0824|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0824|E1|3|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0824|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0824|E2|52-1|192||178|válido|
|scored_0824|E2|52-3|187||176|válido|
|scored_0824|E2|52-6|187||178|válido|
|scored_0824|E2|64-1|192||178|válido|
|scored_0824|E2|64-3|187||177|válido|
|scored_0824|E2|64-6|187||177|válido|
|scored_0824|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0824|E4|repair|187||187|completo|
|scored_0825|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0825|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0825|E1|6|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0825|E2|52-1|155||151|válido|
|scored_0825|E2|52-3|153||151|válido|
|scored_0825|E2|52-6|153||151|válido|
|scored_0825|E2|64-1|155||152|válido|
|scored_0825|E2|64-3|153||152|válido|
|scored_0825|E2|64-6|153||152|válido|
|scored_0825|E3|joint|153|individual 153|153|OPTIMAL|
|scored_0825|E4|repair|153||152|completo|
|scored_0826|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0826|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0826|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0826|E2|52-1|301||270|válido|
|scored_0826|E2|52-3|294||271|válido|
|scored_0826|E2|52-6|284||259|válido|
|scored_0826|E2|64-1|301||277|válido|
|scored_0826|E2|64-3|294||268|válido|
|scored_0826|E2|64-6|284||261|válido|
|scored_0826|E3|joint|284|individual 283|283|OPTIMAL|
|scored_0826|E4|repair|284||284|completo|
|scored_0827|E1|1|281|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0827|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0827|E1|6|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0827|E2|52-1|281||249|válido|
|scored_0827|E2|52-3|265||246|válido|
|scored_0827|E2|52-6|260||241|válido|
|scored_0827|E2|64-1|281||253|válido|
|scored_0827|E2|64-3|265||245|válido|
|scored_0827|E2|64-6|260||264|válido|
|scored_0827|E3|joint|260|individual 260|256|OPTIMAL|
|scored_0827|E4|repair|260||259|completo|
|scored_0828|E1|1|45|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0828|E1|3|45|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0828|E1|6|45|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0828|E2|52-1|45||45|válido|
|scored_0828|E2|52-3|45||45|válido|
|scored_0828|E2|52-6|45||45|válido|
|scored_0828|E2|64-1|45||45|válido|
|scored_0828|E2|64-3|45||45|válido|
|scored_0828|E2|64-6|45||45|válido|
|scored_0828|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0828|E4|repair|45||45|completo|
|scored_0829|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0829|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0829|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0829|E2|52-1|237||220|válido|
|scored_0829|E2|52-3|232||212|válido|
|scored_0829|E2|52-6|232||213|válido|
|scored_0829|E2|64-1|237||221|válido|
|scored_0829|E2|64-3|232||215|válido|
|scored_0829|E2|64-6|232||212|válido|
|scored_0829|E3|joint|232|individual 231|229|OPTIMAL|
|scored_0829|E4|repair|232||232|completo|
|scored_0830|E1|1|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0830|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0830|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0830|E2|52-1|132||130|válido|
|scored_0830|E2|52-3|131||128|válido|
|scored_0830|E2|52-6|131||127|válido|
|scored_0830|E2|64-1|132||130|válido|
|scored_0830|E2|64-3|131||127|válido|
|scored_0830|E2|64-6|131||127|válido|
|scored_0830|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0830|E4|repair|131||130|completo|
|scored_0831|E1|1|121|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|scored_0831|E1|3|121|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|scored_0831|E1|6|121|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|scored_0831|E2|52-1|121||119|válido|
|scored_0831|E2|52-3|121||119|válido|
|scored_0831|E2|52-6|121||119|válido|
|scored_0831|E2|64-1|121||119|válido|
|scored_0831|E2|64-3|121||119|válido|
|scored_0831|E2|64-6|121||119|válido|
|scored_0831|E3|joint|121|individual 121|121|OPTIMAL|
|scored_0831|E4|repair|121||119|completo|
|scored_0832|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0832|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0832|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0832|E2|52-1|250||231|válido|
|scored_0832|E2|52-3|250||230|válido|
|scored_0832|E2|52-6|246||230|válido|
|scored_0832|E2|64-1|250||233|válido|
|scored_0832|E2|64-3|250||230|válido|
|scored_0832|E2|64-6|246||235|válido|
|scored_0832|E3|joint|246|individual 246|243|OPTIMAL|
|scored_0832|E4|repair|246||246|completo|
|scored_0833|E1|1|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0833|E1|3|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0833|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0833|E2|52-1|170||162|válido|
|scored_0833|E2|52-3|167||165|válido|
|scored_0833|E2|52-6|165||157|válido|
|scored_0833|E2|64-1|170||161|válido|
|scored_0833|E2|64-3|167||160|válido|
|scored_0833|E2|64-6|165||157|válido|
|scored_0833|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0833|E4|repair|165||165|completo|
|scored_0834|E1|1|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0834|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0834|E1|6|283|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0834|E2|52-1|300||276|válido|
|scored_0834|E2|52-3|294||262|válido|
|scored_0834|E2|52-6|283||267|válido|
|scored_0834|E2|64-1|300||277|válido|
|scored_0834|E2|64-3|294||262|válido|
|scored_0834|E2|64-6|283||266|válido|
|scored_0834|E3|joint|283|individual 282|282|OPTIMAL|
|scored_0834|E4|repair|283||283|completo|
|scored_0835|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0835|E1|3|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0835|E1|6|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0835|E2|52-1|151||144|válido|
|scored_0835|E2|52-3|150||143|válido|
|scored_0835|E2|52-6|142||142|válido|
|scored_0835|E2|64-1|151||141|válido|
|scored_0835|E2|64-3|150||141|válido|
|scored_0835|E2|64-6|142||143|válido|
|scored_0835|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0835|E4|repair|142||142|completo|
|scored_0836|E1|1|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0836|E1|3|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0836|E1|6|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0836|E2|52-1|257||242|válido|
|scored_0836|E2|52-3|251||234|válido|
|scored_0836|E2|52-6|249||232|válido|
|scored_0836|E2|64-1|257||243|válido|
|scored_0836|E2|64-3|251||235|válido|
|scored_0836|E2|64-6|249||233|válido|
|scored_0836|E3|joint|249|individual 249|249|OPTIMAL|
|scored_0836|E4|repair|249||249|completo|
|scored_0837|E1|1|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0837|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0837|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0837|E2|52-1|311||294|válido|
|scored_0837|E2|52-3|304||288|válido|
|scored_0837|E2|52-6|295||289|válido|
|scored_0837|E2|64-1|311||294|válido|
|scored_0837|E2|64-3|304||336|válido|
|scored_0837|E2|64-6|295||289|válido|
|scored_0837|E3|joint|295|individual 295|295|rechazado por registros|
|scored_0837|E4|repair|295||295|completo|
|scored_0838|E1|1|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0838|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0838|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0838|E2|52-1|186||176|válido|
|scored_0838|E2|52-3|185||172|válido|
|scored_0838|E2|52-6|181||172|válido|
|scored_0838|E2|64-1|186||176|válido|
|scored_0838|E2|64-3|185||172|válido|
|scored_0838|E2|64-6|181||172|válido|
|scored_0838|E3|joint|181|individual 181|181|OPTIMAL|
|scored_0838|E4|repair|181||181|completo|
|scored_0839|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0839|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0839|E1|6|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0839|E2|52-1|235||218|válido|
|scored_0839|E2|52-3|231||215|válido|
|scored_0839|E2|52-6|231||212|válido|
|scored_0839|E2|64-1|235||218|válido|
|scored_0839|E2|64-3|231||216|válido|
|scored_0839|E2|64-6|231||210|válido|
|scored_0839|E3|joint|231|individual 231|231|OPTIMAL|
|scored_0839|E4|repair|231||231|completo|
|scored_0840|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0840|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0840|E1|6|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0840|E2|52-1|330||300|válido|
|scored_0840|E2|52-3|308||292|válido|
|scored_0840|E2|52-6|302||283|válido|
|scored_0840|E2|64-1|330||301|válido|
|scored_0840|E2|64-3|308||290|válido|
|scored_0840|E2|64-6|302||282|válido|
|scored_0840|E3|joint|302|individual 302|301|OPTIMAL|
|scored_0840|E4|repair|302||302|completo|
|scored_0841|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0841|E1|3|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0841|E1|6|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0841|E2|52-1|228||203|válido|
|scored_0841|E2|52-3|211||203|válido|
|scored_0841|E2|52-6|209||190|válido|
|scored_0841|E2|64-1|228||204|válido|
|scored_0841|E2|64-3|211||206|válido|
|scored_0841|E2|64-6|209||195|válido|
|scored_0841|E3|joint|209|individual 208|206|OPTIMAL|
|scored_0841|E4|repair|209||203|completo|
|scored_0842|E1|1|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0842|E1|3|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0842|E1|6|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0842|E2|52-1|39||39|válido|
|scored_0842|E2|52-3|39||39|válido|
|scored_0842|E2|52-6|39||39|válido|
|scored_0842|E2|64-1|39||39|válido|
|scored_0842|E2|64-3|39||39|válido|
|scored_0842|E2|64-6|39||39|válido|
|scored_0842|E3|joint|38|individual 38|37|OPTIMAL|
|scored_0842|E4|repair|38||36|completo|
|scored_0843|E1|1|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0843|E1|3|190|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0843|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0843|E2|52-1|193||178|válido|
|scored_0843|E2|52-3|190||180|válido|
|scored_0843|E2|52-6|190||178|válido|
|scored_0843|E2|64-1|193||179|válido|
|scored_0843|E2|64-3|190||181|válido|
|scored_0843|E2|64-6|190||177|válido|
|scored_0843|E3|joint|190|individual 189|188|OPTIMAL|
|scored_0843|E4|repair|190||190|completo|
|scored_0844|E1|1|359|postprocess 359, optimize_a4 359, nested_windows 359|359|válido|
|scored_0844|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0844|E1|6|330|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0844|E2|52-1|359||332|válido|
|scored_0844|E2|52-3|337||319|válido|
|scored_0844|E2|52-6|330||318|válido|
|scored_0844|E2|64-1|359||333|válido|
|scored_0844|E2|64-3|337||320|válido|
|scored_0844|E2|64-6|330||318|válido|
|scored_0844|E3|joint|330|individual 330|329|FEASIBLE|
|scored_0844|E4|repair|330||330|completo|
|scored_0845|E1|1|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0845|E1|3|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0845|E1|6|288|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0845|E2|52-1|313||281|válido|
|scored_0845|E2|52-3|289||278|válido|
|scored_0845|E2|52-6|288||265|válido|
|scored_0845|E2|64-1|313||284|válido|
|scored_0845|E2|64-3|289||277|válido|
|scored_0845|E2|64-6|288||269|válido|
|scored_0845|E3|joint|288|individual 288|287|FEASIBLE|
|scored_0845|E4|repair|288||288|completo|
|scored_0846|E1|1|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0846|E1|3|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0846|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0846|E2|52-1|129||127|válido|
|scored_0846|E2|52-3|129||128|válido|
|scored_0846|E2|52-6|127||128|válido|
|scored_0846|E2|64-1|129||127|válido|
|scored_0846|E2|64-3|129||128|válido|
|scored_0846|E2|64-6|127||128|válido|
|scored_0846|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0846|E4|repair|127||125|completo|
|scored_0847|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0847|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0847|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0847|E2|52-1|238||227|válido|
|scored_0847|E2|52-3|234||216|válido|
|scored_0847|E2|52-6|229||220|válido|
|scored_0847|E2|64-1|238||225|válido|
|scored_0847|E2|64-3|234||218|válido|
|scored_0847|E2|64-6|229||222|válido|
|scored_0847|E3|joint|229|individual 229|229|OPTIMAL|
|scored_0847|E4|repair|229||228|completo|
|scored_0848|E1|1|357|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0848|E1|3|334|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0848|E1|6|338|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0848|E2|52-1|357||324|válido|
|scored_0848|E2|52-3|334||312|válido|
|scored_0848|E2|52-6|338||312|válido|
|scored_0848|E2|64-1|357||327|válido|
|scored_0848|E2|64-3|334||312|válido|
|scored_0848|E2|64-6|338||312|válido|
|scored_0848|E3|joint|334|individual 334|331|FEASIBLE|
|scored_0848|E4|repair|334||334|completo|
|scored_0849|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0849|E1|3|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0849|E1|6|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0849|E2|52-1|270||249|válido|
|scored_0849|E2|52-3|259||240|válido|
|scored_0849|E2|52-6|258||243|válido|
|scored_0849|E2|64-1|270||244|válido|
|scored_0849|E2|64-3|259||241|válido|
|scored_0849|E2|64-6|258||240|válido|
|scored_0849|E3|joint|258|individual 257|256|OPTIMAL|
|scored_0849|E4|repair|258||258|completo|
|scored_0850|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0850|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0850|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0850|E2|52-1|240||222|válido|
|scored_0850|E2|52-3|234||219|válido|
|scored_0850|E2|52-6|232||218|válido|
|scored_0850|E2|64-1|240||224|válido|
|scored_0850|E2|64-3|234||219|válido|
|scored_0850|E2|64-6|232||219|válido|
|scored_0850|E3|joint|232|individual 232|229|OPTIMAL|
|scored_0850|E4|repair|232||232|completo|
|scored_0851|E1|1|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0851|E1|3|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0851|E1|6|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0851|E2|52-1|44||44|válido|
|scored_0851|E2|52-3|44||44|válido|
|scored_0851|E2|52-6|44||44|válido|
|scored_0851|E2|64-1|44||44|válido|
|scored_0851|E2|64-3|44||44|válido|
|scored_0851|E2|64-6|44||44|válido|
|scored_0851|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0851|E4|repair|44||43|completo|
|scored_0852|E1|1|224|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0852|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0852|E1|6|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0852|E2|52-1|224||207|válido|
|scored_0852|E2|52-3|214||199|válido|
|scored_0852|E2|52-6|210||199|válido|
|scored_0852|E2|64-1|224||206|válido|
|scored_0852|E2|64-3|214||198|válido|
|scored_0852|E2|64-6|210||198|válido|
|scored_0852|E3|joint|210|individual 210|210|OPTIMAL|
|scored_0852|E4|repair|210||209|completo|
|scored_0853|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0853|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0853|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0853|E2|52-1|236||227|válido|
|scored_0853|E2|52-3|235||225|válido|
|scored_0853|E2|52-6|232||218|válido|
|scored_0853|E2|64-1|236||227|válido|
|scored_0853|E2|64-3|235||224|válido|
|scored_0853|E2|64-6|232||220|válido|
|scored_0853|E3|joint|232|individual 232|231|OPTIMAL|
|scored_0853|E4|repair|232||232|completo|
|scored_0854|E1|1|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0854|E1|3|216|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0854|E1|6|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0854|E2|52-1|221||207|válido|
|scored_0854|E2|52-3|216||203|válido|
|scored_0854|E2|52-6|213||201|válido|
|scored_0854|E2|64-1|221||206|válido|
|scored_0854|E2|64-3|216||199|válido|
|scored_0854|E2|64-6|213||197|válido|
|scored_0854|E3|joint|213|individual 213|212|OPTIMAL|
|scored_0854|E4|repair|213||213|completo|
|scored_0855|E1|1|93|postprocess 93, optimize_a4 93, nested_windows 93|93|válido|
|scored_0855|E1|3|93|postprocess 93, optimize_a4 93, nested_windows 93|93|válido|
|scored_0855|E1|6|93|postprocess 93, optimize_a4 93, nested_windows 93|93|válido|
|scored_0855|E2|52-1|93||91|válido|
|scored_0855|E2|52-3|93||91|válido|
|scored_0855|E2|52-6|93||91|válido|
|scored_0855|E2|64-1|93||91|válido|
|scored_0855|E2|64-3|93||91|válido|
|scored_0855|E2|64-6|93||91|válido|
|scored_0855|E3|joint|93|individual 93|93|OPTIMAL|
|scored_0855|E4|repair|93||91|completo|
|scored_0856|E1|1|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0856|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0856|E1|6|285|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0856|E2|52-1|313||287|válido|
|scored_0856|E2|52-3|286||272|válido|
|scored_0856|E2|52-6|285||268|válido|
|scored_0856|E2|64-1|313||286|válido|
|scored_0856|E2|64-3|286||272|válido|
|scored_0856|E2|64-6|285||267|válido|
|scored_0856|E3|joint|285|individual 285|282|FEASIBLE|
|scored_0856|E4|repair|285||285|completo|
|scored_0857|E1|1|357|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0857|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0857|E1|6|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0857|E2|52-1|357||295|válido|
|scored_0857|E2|52-3|308||286|válido|
|scored_0857|E2|52-6|299||280|válido|
|scored_0857|E2|64-1|357||293|válido|
|scored_0857|E2|64-3|308||286|válido|
|scored_0857|E2|64-6|299||281|válido|
|scored_0857|E3|joint|299|individual 299|299|OPTIMAL|
|scored_0857|E4|repair|299||299|completo|
|scored_0858|E1|1|57|postprocess 56, optimize_a4 56, nested_windows 56|56|válido|
|scored_0858|E1|3|57|postprocess 56, optimize_a4 56, nested_windows 56|56|válido|
|scored_0858|E1|6|57|postprocess 56, optimize_a4 56, nested_windows 56|56|válido|
|scored_0858|E2|52-1|57||57|válido|
|scored_0858|E2|52-3|57||57|válido|
|scored_0858|E2|52-6|57||57|válido|
|scored_0858|E2|64-1|57||57|válido|
|scored_0858|E2|64-3|57||57|válido|
|scored_0858|E2|64-6|57||57|válido|
|scored_0858|E3|joint|57|individual 57|57|OPTIMAL|
|scored_0858|E4|repair|57||57|completo|
|scored_0859|E1|1|310|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0859|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0859|E1|6|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0859|E2|52-1|310||288|válido|
|scored_0859|E2|52-3|288||274|válido|
|scored_0859|E2|52-6|279||267|válido|
|scored_0859|E2|64-1|310||289|válido|
|scored_0859|E2|64-3|288||280|válido|
|scored_0859|E2|64-6|279||268|válido|
|scored_0859|E3|joint|279|individual 278|278|FEASIBLE|
|scored_0859|E4|repair|279||279|completo|
|scored_0860|E1|1|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0860|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0860|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0860|E2|52-1|194||183|válido|
|scored_0860|E2|52-3|188||181|válido|
|scored_0860|E2|52-6|188||180|válido|
|scored_0860|E2|64-1|194||184|válido|
|scored_0860|E2|64-3|188||181|válido|
|scored_0860|E2|64-6|188||181|válido|
|scored_0860|E3|joint|188|individual 188|188|OPTIMAL|
|scored_0860|E4|repair|188||187|completo|
|scored_0861|E1|1|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0861|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0861|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0861|E2|52-1|213||202|válido|
|scored_0861|E2|52-3|213||198|válido|
|scored_0861|E2|52-6|211||199|válido|
|scored_0861|E2|64-1|213||204|válido|
|scored_0861|E2|64-3|213||197|válido|
|scored_0861|E2|64-6|211||204|válido|
|scored_0861|E3|joint|211|individual 211|210|OPTIMAL|
|scored_0861|E4|repair|211||211|completo|
|scored_0862|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0862|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0862|E1|6|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0862|E2|52-1|169||165|válido|
|scored_0862|E2|52-3|169||165|válido|
|scored_0862|E2|52-6|169||165|válido|
|scored_0862|E2|64-1|169||169|válido|
|scored_0862|E2|64-3|169||168|válido|
|scored_0862|E2|64-6|169||166|válido|
|scored_0862|E3|joint|169|individual 168|168|OPTIMAL|
|scored_0862|E4|repair|169||166|completo|
|scored_0863|E1|1|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0863|E1|3|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0863|E1|6|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0863|E2|52-1|45||45|válido|
|scored_0863|E2|52-3|45||45|válido|
|scored_0863|E2|52-6|45||45|válido|
|scored_0863|E2|64-1|45||45|válido|
|scored_0863|E2|64-3|45||45|válido|
|scored_0863|E2|64-6|45||45|válido|
|scored_0863|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0863|E4|repair|45||44|completo|
|scored_0864|E1|1|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0864|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0864|E1|6|250|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0864|E2|52-1|270||238|válido|
|scored_0864|E2|52-3|249||235|válido|
|scored_0864|E2|52-6|250||230|válido|
|scored_0864|E2|64-1|270||243|válido|
|scored_0864|E2|64-3|249||234|válido|
|scored_0864|E2|64-6|250||231|válido|
|scored_0864|E3|joint|249|individual 249|244|OPTIMAL|
|scored_0864|E4|repair|249||249|completo|
|scored_0865|E1|1|335|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0865|E1|3|320|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0865|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0865|E2|52-1|335||306|válido|
|scored_0865|E2|52-3|320||298|válido|
|scored_0865|E2|52-6|311||294|válido|
|scored_0865|E2|64-1|335||310|válido|
|scored_0865|E2|64-3|320||299|válido|
|scored_0865|E2|64-6|311||293|válido|
|scored_0865|E3|joint|311|individual 310|309|OPTIMAL|
|scored_0865|E4|repair|311||311|completo|
|scored_0866|E1|1|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0866|E1|3|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0866|E1|6|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0866|E2|52-1|44||44|válido|
|scored_0866|E2|52-3|44||44|válido|
|scored_0866|E2|52-6|44||44|válido|
|scored_0866|E2|64-1|44||44|válido|
|scored_0866|E2|64-3|44||44|válido|
|scored_0866|E2|64-6|44||44|válido|
|scored_0866|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0866|E4|repair|44||43|completo|
|scored_0867|E1|1|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0867|E1|3|117|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0867|E1|6|117|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0867|E2|52-1|119||113|válido|
|scored_0867|E2|52-3|117||114|válido|
|scored_0867|E2|52-6|117||111|válido|
|scored_0867|E2|64-1|119||113|válido|
|scored_0867|E2|64-3|117||114|válido|
|scored_0867|E2|64-6|117||111|válido|
|scored_0867|E3|joint|117|individual 117|117|OPTIMAL|
|scored_0867|E4|repair|117||117|completo|
|scored_0868|E1|1|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0868|E1|3|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0868|E1|6|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0868|E2|52-1|43||43|válido|
|scored_0868|E2|52-3|43||43|válido|
|scored_0868|E2|52-6|43||43|válido|
|scored_0868|E2|64-1|43||43|válido|
|scored_0868|E2|64-3|43||43|válido|
|scored_0868|E2|64-6|43||43|válido|
|scored_0868|E3|joint|42|individual 40|40|OPTIMAL|
|scored_0868|E4|repair|42||42|completo|
|scored_0869|E1|1|174|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0869|E1|3|165|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0869|E1|6|165|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0869|E2|52-1|174||159|válido|
|scored_0869|E2|52-3|165||158|válido|
|scored_0869|E2|52-6|165||156|válido|
|scored_0869|E2|64-1|174||159|válido|
|scored_0869|E2|64-3|165||158|válido|
|scored_0869|E2|64-6|165||157|válido|
|scored_0869|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0869|E4|repair|165||163|completo|
|scored_0870|E1|1|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0870|E1|3|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0870|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0870|E2|52-1|310||283|válido|
|scored_0870|E2|52-3|296||272|válido|
|scored_0870|E2|52-6|287||269|válido|
|scored_0870|E2|64-1|310||282|válido|
|scored_0870|E2|64-3|296||274|válido|
|scored_0870|E2|64-6|287||271|válido|
|scored_0870|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0870|E4|repair|287||287|completo|
|scored_0871|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0871|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0871|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0871|E2|52-1|153||142|válido|
|scored_0871|E2|52-3|151||142|válido|
|scored_0871|E2|52-6|151||140|válido|
|scored_0871|E2|64-1|153||142|válido|
|scored_0871|E2|64-3|151||142|válido|
|scored_0871|E2|64-6|151||140|válido|
|scored_0871|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0871|E4|repair|151||149|completo|
|scored_0872|E1|1|139|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0872|E1|3|137|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0872|E1|6|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0872|E2|52-1|139||132|válido|
|scored_0872|E2|52-3|137||132|válido|
|scored_0872|E2|52-6|136||132|válido|
|scored_0872|E2|64-1|139||132|válido|
|scored_0872|E2|64-3|137||132|válido|
|scored_0872|E2|64-6|136||132|válido|
|scored_0872|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0872|E4|repair|136||132|completo|
|scored_0873|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0873|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0873|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0873|E2|52-1|50||50|válido|
|scored_0873|E2|52-3|50||50|válido|
|scored_0873|E2|52-6|50||50|válido|
|scored_0873|E2|64-1|50||50|válido|
|scored_0873|E2|64-3|50||50|válido|
|scored_0873|E2|64-6|50||50|válido|
|scored_0873|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0873|E4|repair|50||50|completo|
|scored_0874|E1|1|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0874|E1|3|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0874|E1|6|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0874|E2|52-1|47||47|válido|
|scored_0874|E2|52-3|47||47|válido|
|scored_0874|E2|52-6|47||47|válido|
|scored_0874|E2|64-1|47||47|válido|
|scored_0874|E2|64-3|47||47|válido|
|scored_0874|E2|64-6|47||47|válido|
|scored_0874|E3|joint|47|individual 47|47|OPTIMAL|
|scored_0874|E4|repair|47||47|completo|
|scored_0875|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0875|E1|3|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0875|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0875|E2|52-1|220||206|válido|
|scored_0875|E2|52-3|217||204|válido|
|scored_0875|E2|52-6|214||197|válido|
|scored_0875|E2|64-1|220||207|válido|
|scored_0875|E2|64-3|217||204|válido|
|scored_0875|E2|64-6|214||197|válido|
|scored_0875|E3|joint|214|individual 214|213|OPTIMAL|
|scored_0875|E4|repair|214||214|completo|
|scored_0876|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0876|E1|3|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0876|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0876|E2|52-1|289||257|válido|
|scored_0876|E2|52-3|272||255|válido|
|scored_0876|E2|52-6|269||245|válido|
|scored_0876|E2|64-1|289||261|válido|
|scored_0876|E2|64-3|272||256|válido|
|scored_0876|E2|64-6|269||246|válido|
|scored_0876|E3|joint|269|individual 268|267|OPTIMAL|
|scored_0876|E4|repair|269||269|completo|
|scored_0877|E1|1|79|postprocess 78, optimize_a4 78, nested_windows 78|78|válido|
|scored_0877|E1|3|79|postprocess 78, optimize_a4 78, nested_windows 78|78|válido|
|scored_0877|E1|6|79|postprocess 78, optimize_a4 78, nested_windows 78|78|válido|
|scored_0877|E2|52-1|79||77|válido|
|scored_0877|E2|52-3|79||77|válido|
|scored_0877|E2|52-6|79||77|válido|
|scored_0877|E2|64-1|79||77|válido|
|scored_0877|E2|64-3|79||77|válido|
|scored_0877|E2|64-6|79||77|válido|
|scored_0877|E3|joint|79|individual 79|79|OPTIMAL|
|scored_0877|E4|repair|79||75|completo|
|scored_0878|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0878|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0878|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0878|E2|52-1|219||203|válido|
|scored_0878|E2|52-3|215||194|válido|
|scored_0878|E2|52-6|212||193|válido|
|scored_0878|E2|64-1|219||203|válido|
|scored_0878|E2|64-3|215||195|válido|
|scored_0878|E2|64-6|212||195|válido|
|scored_0878|E3|joint|212|individual 212|212|OPTIMAL|
|scored_0878|E4|repair|212||212|completo|
|scored_0879|E1|1|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0879|E1|3|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0879|E1|6|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0879|E2|52-1|138||132|válido|
|scored_0879|E2|52-3|138||131|válido|
|scored_0879|E2|52-6|138||130|válido|
|scored_0879|E2|64-1|138||130|válido|
|scored_0879|E2|64-3|138||130|válido|
|scored_0879|E2|64-6|138||129|válido|
|scored_0879|E3|joint|138|individual 136|135|OPTIMAL|
|scored_0879|E4|repair|138||137|completo|
|scored_0880|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0880|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0880|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0880|E2|52-1|50||50|válido|
|scored_0880|E2|52-3|50||50|válido|
|scored_0880|E2|52-6|50||50|válido|
|scored_0880|E2|64-1|50||50|válido|
|scored_0880|E2|64-3|50||50|válido|
|scored_0880|E2|64-6|50||50|válido|
|scored_0880|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0880|E4|repair|50||50|completo|
|scored_0881|E1|1|114|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0881|E1|3|114|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0881|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0881|E2|52-1|114||111|válido|
|scored_0881|E2|52-3|114||111|válido|
|scored_0881|E2|52-6|113||111|válido|
|scored_0881|E2|64-1|114||111|válido|
|scored_0881|E2|64-3|114||111|válido|
|scored_0881|E2|64-6|113||111|válido|
|scored_0881|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0881|E4|repair|113||109|completo|
|scored_0882|E1|1|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0882|E1|3|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0882|E1|6|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0882|E2|52-1|37||37|válido|
|scored_0882|E2|52-3|37||37|válido|
|scored_0882|E2|52-6|37||37|válido|
|scored_0882|E2|64-1|37||37|válido|
|scored_0882|E2|64-3|37||37|válido|
|scored_0882|E2|64-6|37||37|válido|
|scored_0882|E3|joint|36|individual 35|35|OPTIMAL|
|scored_0882|E4|repair|36||32|completo|
|scored_0883|E1|1|10|postprocess 8, optimize_a4 7, nested_windows 7|7|válido|
|scored_0883|E1|3|10|postprocess 8, optimize_a4 7, nested_windows 7|7|válido|
|scored_0883|E1|6|10|postprocess 8, optimize_a4 7, nested_windows 7|7|válido|
|scored_0883|E2|52-1|10||10|válido|
|scored_0883|E2|52-3|10||10|válido|
|scored_0883|E2|52-6|10||10|válido|
|scored_0883|E2|64-1|10||10|válido|
|scored_0883|E2|64-3|10||10|válido|
|scored_0883|E2|64-6|10||10|válido|
|scored_0883|E3|joint|7|individual 7|7|OPTIMAL|
|scored_0883|E4|repair|7||7|completo|
|scored_0884|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0884|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0884|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0884|E2|52-1|144||139|válido|
|scored_0884|E2|52-3|144||139|válido|
|scored_0884|E2|52-6|144||138|válido|
|scored_0884|E2|64-1|144||138|válido|
|scored_0884|E2|64-3|144||138|válido|
|scored_0884|E2|64-6|144||137|válido|
|scored_0884|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0884|E4|repair|144||143|completo|
|scored_0885|E1|1|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0885|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0885|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0885|E2|52-1|225||207|válido|
|scored_0885|E2|52-3|218||209|válido|
|scored_0885|E2|52-6|211||202|válido|
|scored_0885|E2|64-1|225||209|válido|
|scored_0885|E2|64-3|218||206|válido|
|scored_0885|E2|64-6|211||204|válido|
|scored_0885|E3|joint|211|individual 211|211|OPTIMAL|
|scored_0885|E4|repair|211||210|completo|
|scored_0886|E1|1|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0886|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0886|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0886|E2|52-1|214||199|válido|
|scored_0886|E2|52-3|207||194|válido|
|scored_0886|E2|52-6|203||195|válido|
|scored_0886|E2|64-1|214||201|válido|
|scored_0886|E2|64-3|207||191|válido|
|scored_0886|E2|64-6|203||191|válido|
|scored_0886|E3|joint|203|individual 203|203|OPTIMAL|
|scored_0886|E4|repair|203||203|completo|
|scored_0887|E1|1|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0887|E1|3|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0887|E1|6|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0887|E2|52-1|4||4|válido|
|scored_0887|E2|52-3|4||4|válido|
|scored_0887|E2|52-6|4||4|válido|
|scored_0887|E2|64-1|4||4|válido|
|scored_0887|E2|64-3|4||4|válido|
|scored_0887|E2|64-6|4||4|válido|
|scored_0887|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0887|E4|repair|4||4|completo|
|scored_0888|E1|1|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0888|E1|3|164|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0888|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0888|E2|52-1|168||161|válido|
|scored_0888|E2|52-3|164||161|válido|
|scored_0888|E2|52-6|162||158|válido|
|scored_0888|E2|64-1|168||161|válido|
|scored_0888|E2|64-3|164||161|válido|
|scored_0888|E2|64-6|162||156|válido|
|scored_0888|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0888|E4|repair|162||159|completo|
|scored_0889|E1|1|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0889|E1|3|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0889|E1|6|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0889|E2|52-1|142||137|válido|
|scored_0889|E2|52-3|141||137|válido|
|scored_0889|E2|52-6|141||137|válido|
|scored_0889|E2|64-1|142||136|válido|
|scored_0889|E2|64-3|141||136|válido|
|scored_0889|E2|64-6|141||136|válido|
|scored_0889|E3|joint|141|individual 141|141|OPTIMAL|
|scored_0889|E4|repair|141||139|completo|
|scored_0890|E1|1|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0890|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0890|E1|6|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0890|E2|52-1|120||117|válido|
|scored_0890|E2|52-3|120||117|válido|
|scored_0890|E2|52-6|120||117|válido|
|scored_0890|E2|64-1|120||116|válido|
|scored_0890|E2|64-3|120||116|válido|
|scored_0890|E2|64-6|120||116|válido|
|scored_0890|E3|joint|120|individual 120|120|OPTIMAL|
|scored_0890|E4|repair|120||115|completo|
|scored_0891|E1|1|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0891|E1|3|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0891|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0891|E2|52-1|139||138|válido|
|scored_0891|E2|52-3|139||137|válido|
|scored_0891|E2|52-6|139||137|válido|
|scored_0891|E2|64-1|139||136|válido|
|scored_0891|E2|64-3|139||136|válido|
|scored_0891|E2|64-6|139||136|válido|
|scored_0891|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0891|E4|repair|139||135|completo|
|scored_0892|E1|1|29|postprocess 27, optimize_a4 27, nested_windows 27|27|válido|
|scored_0892|E1|3|29|postprocess 27, optimize_a4 27, nested_windows 27|27|válido|
|scored_0892|E1|6|29|postprocess 27, optimize_a4 27, nested_windows 27|27|válido|
|scored_0892|E2|52-1|29||29|válido|
|scored_0892|E2|52-3|29||29|válido|
|scored_0892|E2|52-6|29||29|válido|
|scored_0892|E2|64-1|29||29|válido|
|scored_0892|E2|64-3|29||29|válido|
|scored_0892|E2|64-6|29||29|válido|
|scored_0892|E3|joint|22|individual 22|22|OPTIMAL|
|scored_0892|E4|repair|22||21|completo|
|scored_0893|E1|1|177|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0893|E1|3|174|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0893|E1|6|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0893|E2|52-1|177||163|válido|
|scored_0893|E2|52-3|174||163|válido|
|scored_0893|E2|52-6|169||163|válido|
|scored_0893|E2|64-1|177||166|válido|
|scored_0893|E2|64-3|174||163|válido|
|scored_0893|E2|64-6|169||161|válido|
|scored_0893|E3|joint|169|individual 169|169|OPTIMAL|
|scored_0893|E4|repair|169||167|completo|
|scored_0894|E1|1|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0894|E1|3|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0894|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0894|E2|52-1|173||158|válido|
|scored_0894|E2|52-3|168||156|válido|
|scored_0894|E2|52-6|166||154|válido|
|scored_0894|E2|64-1|173||160|válido|
|scored_0894|E2|64-3|168||156|válido|
|scored_0894|E2|64-6|166||153|válido|
|scored_0894|E3|joint|166|individual 165|165|OPTIMAL|
|scored_0894|E4|repair|166||164|completo|
|scored_0895|E1|1|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0895|E1|3|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0895|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0895|E2|52-1|162||152|válido|
|scored_0895|E2|52-3|160||149|válido|
|scored_0895|E2|52-6|160||147|válido|
|scored_0895|E2|64-1|162||150|válido|
|scored_0895|E2|64-3|160||147|válido|
|scored_0895|E2|64-6|160||147|válido|
|scored_0895|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0895|E4|repair|160||160|completo|
|scored_0896|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0896|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0896|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0896|E2|52-1|251||223|válido|
|scored_0896|E2|52-3|234||224|válido|
|scored_0896|E2|52-6|233||216|válido|
|scored_0896|E2|64-1|251||229|válido|
|scored_0896|E2|64-3|234||225|válido|
|scored_0896|E2|64-6|233||219|válido|
|scored_0896|E3|joint|233|individual 232|230|OPTIMAL|
|scored_0896|E4|repair|233||232|completo|
|scored_0897|E1|1|51|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0897|E1|3|51|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0897|E1|6|51|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0897|E2|52-1|51||51|válido|
|scored_0897|E2|52-3|51||51|válido|
|scored_0897|E2|52-6|51||51|válido|
|scored_0897|E2|64-1|51||51|válido|
|scored_0897|E2|64-3|51||51|válido|
|scored_0897|E2|64-6|51||51|válido|
|scored_0897|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0897|E4|repair|51||51|completo|
|scored_0898|E1|1|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0898|E1|3|109|postprocess 109, optimize_a4 109, nested_windows 109|109|válido|
|scored_0898|E1|6|109|postprocess 109, optimize_a4 109, nested_windows 109|109|válido|
|scored_0898|E2|52-1|108||103|válido|
|scored_0898|E2|52-3|109||103|válido|
|scored_0898|E2|52-6|109||103|válido|
|scored_0898|E2|64-1|108||103|válido|
|scored_0898|E2|64-3|109||103|válido|
|scored_0898|E2|64-6|109||103|válido|
|scored_0898|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0898|E4|repair|108||106|completo|
|scored_0899|E1|1|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0899|E1|3|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0899|E1|6|309|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0899|E2|52-1|336||307|válido|
|scored_0899|E2|52-3|311||292|válido|
|scored_0899|E2|52-6|309||289|válido|
|scored_0899|E2|64-1|336||309|válido|
|scored_0899|E2|64-3|311||291|válido|
|scored_0899|E2|64-6|309||290|válido|
|scored_0899|E3|joint|309|individual 309|304|OPTIMAL|
|scored_0899|E4|repair|309||309|completo|
|scored_0900|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0900|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0900|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0900|E2|52-1|235||212|válido|
|scored_0900|E2|52-3|226||209|válido|
|scored_0900|E2|52-6|219||211|válido|
|scored_0900|E2|64-1|235||211|válido|
|scored_0900|E2|64-3|226||208|válido|
|scored_0900|E2|64-6|219||211|válido|
|scored_0900|E3|joint|219|individual 219|219|FEASIBLE|
|scored_0900|E4|repair|219||219|completo|
|scored_0901|E1|1|27|postprocess 24, optimize_a4 24, nested_windows 24|24|válido|
|scored_0901|E1|3|27|postprocess 24, optimize_a4 24, nested_windows 24|24|válido|
|scored_0901|E1|6|27|postprocess 24, optimize_a4 24, nested_windows 24|24|válido|
|scored_0901|E2|52-1|27||27|válido|
|scored_0901|E2|52-3|27||27|válido|
|scored_0901|E2|52-6|27||27|válido|
|scored_0901|E2|64-1|27||27|válido|
|scored_0901|E2|64-3|27||27|válido|
|scored_0901|E2|64-6|27||27|válido|
|scored_0901|E3|joint|24|individual 24|24|OPTIMAL|
|scored_0901|E4|repair|24||21|completo|
|scored_0902|E1|1|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0902|E1|3|289|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0902|E1|6|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0902|E2|52-1|302||278|válido|
|scored_0902|E2|52-3|289||265|válido|
|scored_0902|E2|52-6|283||262|válido|
|scored_0902|E2|64-1|302||278|válido|
|scored_0902|E2|64-3|289||265|válido|
|scored_0902|E2|64-6|283||262|válido|
|scored_0902|E3|joint|283|individual 283|283|OPTIMAL|
|scored_0902|E4|repair|283||283|completo|
|scored_0903|E1|1|216|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0903|E1|3|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0903|E1|6|201|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0903|E2|52-1|216||193|válido|
|scored_0903|E2|52-3|209||190|válido|
|scored_0903|E2|52-6|201||192|válido|
|scored_0903|E2|64-1|216||191|válido|
|scored_0903|E2|64-3|209||192|válido|
|scored_0903|E2|64-6|201||193|válido|
|scored_0903|E3|joint|201|individual 201|200|OPTIMAL|
|scored_0903|E4|repair|201||201|completo|
|scored_0904|E1|1|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0904|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0904|E1|6|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0904|E2|52-1|191||175|válido|
|scored_0904|E2|52-3|185||172|válido|
|scored_0904|E2|52-6|178||169|válido|
|scored_0904|E2|64-1|191||177|válido|
|scored_0904|E2|64-3|185||174|válido|
|scored_0904|E2|64-6|178||173|válido|
|scored_0904|E3|joint|178|individual 178|178|OPTIMAL|
|scored_0904|E4|repair|178||178|completo|
|scored_0905|E1|1|341|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0905|E1|3|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0905|E1|6|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0905|E2|52-1|341||307|válido|
|scored_0905|E2|52-3|319||299|válido|
|scored_0905|E2|52-6|317||296|válido|
|scored_0905|E2|64-1|341||310|válido|
|scored_0905|E2|64-3|319||300|válido|
|scored_0905|E2|64-6|317||296|válido|
|scored_0905|E3|joint|317|individual 317|314|FEASIBLE|
|scored_0905|E4|repair|317||317|completo|
|scored_0906|E1|1|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0906|E1|3|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0906|E1|6|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0906|E2|52-1|279||263|válido|
|scored_0906|E2|52-3|263||260|válido|
|scored_0906|E2|52-6|263||258|válido|
|scored_0906|E2|64-1|279||264|válido|
|scored_0906|E2|64-3|263||258|válido|
|scored_0906|E2|64-6|263||258|válido|
|scored_0906|E3|joint|263|individual 263|261|OPTIMAL|
|scored_0906|E4|repair|263||263|completo|
|scored_0907|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0907|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0907|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0907|E2|52-1|155||149|válido|
|scored_0907|E2|52-3|152||146|válido|
|scored_0907|E2|52-6|152||145|válido|
|scored_0907|E2|64-1|155||150|válido|
|scored_0907|E2|64-3|152||149|válido|
|scored_0907|E2|64-6|152||147|válido|
|scored_0907|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0907|E4|repair|152||151|completo|
|scored_0908|E1|1|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0908|E1|3|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0908|E1|6|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0908|E2|52-1|44||44|válido|
|scored_0908|E2|52-3|44||44|válido|
|scored_0908|E2|52-6|44||44|válido|
|scored_0908|E2|64-1|44||44|válido|
|scored_0908|E2|64-3|44||44|válido|
|scored_0908|E2|64-6|44||44|válido|
|scored_0908|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0908|E4|repair|44||44|completo|
|scored_0909|E1|1|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0909|E1|3|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0909|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0909|E2|52-1|222||206|válido|
|scored_0909|E2|52-3|216||200|válido|
|scored_0909|E2|52-6|219||201|válido|
|scored_0909|E2|64-1|222||206|válido|
|scored_0909|E2|64-3|216||199|válido|
|scored_0909|E2|64-6|219||201|válido|
|scored_0909|E3|joint|216|individual 216|216|OPTIMAL|
|scored_0909|E4|repair|216||216|completo|
|scored_0910|E1|1|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0910|E1|3|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0910|E1|6|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0910|E2|52-1|46||46|válido|
|scored_0910|E2|52-3|46||46|válido|
|scored_0910|E2|52-6|46||46|válido|
|scored_0910|E2|64-1|46||46|válido|
|scored_0910|E2|64-3|46||46|válido|
|scored_0910|E2|64-6|46||46|válido|
|scored_0910|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0910|E4|repair|46||46|completo|
|scored_0911|E1|1|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0911|E1|3|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0911|E1|6|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0911|E2|52-1|112||107|válido|
|scored_0911|E2|52-3|112||107|válido|
|scored_0911|E2|52-6|112||107|válido|
|scored_0911|E2|64-1|112||107|válido|
|scored_0911|E2|64-3|112||107|válido|
|scored_0911|E2|64-6|112||107|válido|
|scored_0911|E3|joint|112|individual 112|112|OPTIMAL|
|scored_0911|E4|repair|112||111|completo|
|scored_0912|E1|1|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0912|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0912|E1|6|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0912|E2|52-1|291||262|válido|
|scored_0912|E2|52-3|281||265|válido|
|scored_0912|E2|52-6|275||260|válido|
|scored_0912|E2|64-1|291||261|válido|
|scored_0912|E2|64-3|281||267|válido|
|scored_0912|E2|64-6|275||262|válido|
|scored_0912|E3|joint|275|individual 275|275|OPTIMAL|
|scored_0912|E4|repair|275||275|completo|
|scored_0913|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0913|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0913|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0913|E2|52-1|41||41|válido|
|scored_0913|E2|52-3|41||41|válido|
|scored_0913|E2|52-6|41||41|válido|
|scored_0913|E2|64-1|41||41|válido|
|scored_0913|E2|64-3|41||41|válido|
|scored_0913|E2|64-6|41||41|válido|
|scored_0913|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0913|E4|repair|34||34|completo|
|scored_0914|E1|1|54|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0914|E1|3|54|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0914|E1|6|54|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0914|E2|52-1|54||54|válido|
|scored_0914|E2|52-3|54||54|válido|
|scored_0914|E2|52-6|54||54|válido|
|scored_0914|E2|64-1|54||54|válido|
|scored_0914|E2|64-3|54||54|válido|
|scored_0914|E2|64-6|54||54|válido|
|scored_0914|E3|joint|54|individual 54|54|OPTIMAL|
|scored_0914|E4|repair|54||50|completo|
|scored_0915|E1|1|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0915|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0915|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0915|E2|52-1|266||247|válido|
|scored_0915|E2|52-3|258||239|válido|
|scored_0915|E2|52-6|255||235|válido|
|scored_0915|E2|64-1|266||246|válido|
|scored_0915|E2|64-3|258||241|válido|
|scored_0915|E2|64-6|255||238|válido|
|scored_0915|E3|joint|255|individual 253|253|OPTIMAL|
|scored_0915|E4|repair|255||255|completo|
|scored_0916|E1|1|324|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0916|E1|3|308|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0916|E1|6|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0916|E2|52-1|324||299|válido|
|scored_0916|E2|52-3|308||290|válido|
|scored_0916|E2|52-6|302||286|válido|
|scored_0916|E2|64-1|324||299|válido|
|scored_0916|E2|64-3|308||291|válido|
|scored_0916|E2|64-6|302||284|válido|
|scored_0916|E3|joint|302|individual 301|299|FEASIBLE|
|scored_0916|E4|repair|302||302|completo|
|scored_0917|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0917|E1|3|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0917|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0917|E2|52-1|306||275|válido|
|scored_0917|E2|52-3|295||268|válido|
|scored_0917|E2|52-6|287||265|válido|
|scored_0917|E2|64-1|306||275|válido|
|scored_0917|E2|64-3|295||267|válido|
|scored_0917|E2|64-6|287||264|válido|
|scored_0917|E3|joint|287|individual 286|286|OPTIMAL|
|scored_0917|E4|repair|287||287|completo|
|scored_0918|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0918|E1|3|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0918|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0918|E2|52-1|303||281|válido|
|scored_0918|E2|52-3|292||273|válido|
|scored_0918|E2|52-6|284||266|válido|
|scored_0918|E2|64-1|303||281|válido|
|scored_0918|E2|64-3|292||269|válido|
|scored_0918|E2|64-6|284||265|válido|
|scored_0918|E3|joint|284|individual 284|284|FEASIBLE|
|scored_0918|E4|repair|284||284|completo|
|scored_0919|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0919|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0919|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0919|E2|52-1|205||198|válido|
|scored_0919|E2|52-3|203||194|válido|
|scored_0919|E2|52-6|199||193|válido|
|scored_0919|E2|64-1|205||189|válido|
|scored_0919|E2|64-3|203||188|válido|
|scored_0919|E2|64-6|199||187|válido|
|scored_0919|E3|joint|199|individual 199|198|OPTIMAL|
|scored_0919|E4|repair|199||198|completo|
|scored_0920|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0920|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0920|E1|6|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0920|E2|52-1|192||182|válido|
|scored_0920|E2|52-3|188||182|válido|
|scored_0920|E2|52-6|187||182|válido|
|scored_0920|E2|64-1|192||182|válido|
|scored_0920|E2|64-3|188||182|válido|
|scored_0920|E2|64-6|187||182|válido|
|scored_0920|E3|joint|187|individual 187|186|OPTIMAL|
|scored_0920|E4|repair|187||187|completo|
|scored_0921|E1|1|231|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0921|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0921|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0921|E2|52-1|231||219|válido|
|scored_0921|E2|52-3|219||207|válido|
|scored_0921|E2|52-6|215||209|válido|
|scored_0921|E2|64-1|231||216|válido|
|scored_0921|E2|64-3|219||207|válido|
|scored_0921|E2|64-6|215||209|válido|
|scored_0921|E3|joint|215|individual 215|215|OPTIMAL|
|scored_0921|E4|repair|215||215|completo|
|scored_0922|E1|1|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0922|E1|3|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0922|E1|6|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0922|E2|52-1|35||35|válido|
|scored_0922|E2|52-3|35||35|válido|
|scored_0922|E2|52-6|35||35|válido|
|scored_0922|E2|64-1|35||35|válido|
|scored_0922|E2|64-3|35||35|válido|
|scored_0922|E2|64-6|35||35|válido|
|scored_0922|E3|joint|35|individual 35|35|OPTIMAL|
|scored_0922|E4|repair|35||35|completo|
|scored_0923|E1|1|281|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0923|E1|3|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0923|E1|6|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0923|E2|52-1|281||260|válido|
|scored_0923|E2|52-3|267||248|válido|
|scored_0923|E2|52-6|262||247|válido|
|scored_0923|E2|64-1|281||264|válido|
|scored_0923|E2|64-3|267||249|válido|
|scored_0923|E2|64-6|262||248|válido|
|scored_0923|E3|joint|262|individual 262|262|OPTIMAL|
|scored_0923|E4|repair|262||262|completo|
|scored_0924|E1|1|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0924|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0924|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0924|E2|52-1|147||142|válido|
|scored_0924|E2|52-3|147||142|válido|
|scored_0924|E2|52-6|146||142|válido|
|scored_0924|E2|64-1|147||144|válido|
|scored_0924|E2|64-3|147||143|válido|
|scored_0924|E2|64-6|146||143|válido|
|scored_0924|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0924|E4|repair|146||145|completo|
|scored_0925|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0925|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0925|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0925|E2|52-1|220||203|válido|
|scored_0925|E2|52-3|207||198|válido|
|scored_0925|E2|52-6|204||196|válido|
|scored_0925|E2|64-1|220||203|válido|
|scored_0925|E2|64-3|207||198|válido|
|scored_0925|E2|64-6|204||196|válido|
|scored_0925|E3|joint|204|individual 202|201|OPTIMAL|
|scored_0925|E4|repair|204||202|completo|
|scored_0926|E1|1|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0926|E1|3|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0926|E1|6|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0926|E2|52-1|33||33|válido|
|scored_0926|E2|52-3|33||33|válido|
|scored_0926|E2|52-6|33||33|válido|
|scored_0926|E2|64-1|33||33|válido|
|scored_0926|E2|64-3|33||33|válido|
|scored_0926|E2|64-6|33||33|válido|
|scored_0926|E3|joint|33|individual 33|33|OPTIMAL|
|scored_0926|E4|repair|33||32|completo|
|scored_0927|E1|1|185|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0927|E1|3|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0927|E1|6|181|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0927|E2|52-1|185||174|válido|
|scored_0927|E2|52-3|183||171|válido|
|scored_0927|E2|52-6|181||171|válido|
|scored_0927|E2|64-1|185||173|válido|
|scored_0927|E2|64-3|183||173|válido|
|scored_0927|E2|64-6|181||171|válido|
|scored_0927|E3|joint|181|individual 181|181|OPTIMAL|
|scored_0927|E4|repair|181||181|completo|
|scored_0928|E1|1|46|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0928|E1|3|46|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0928|E1|6|46|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0928|E2|52-1|46||46|válido|
|scored_0928|E2|52-3|46||46|válido|
|scored_0928|E2|52-6|46||46|válido|
|scored_0928|E2|64-1|46||46|válido|
|scored_0928|E2|64-3|46||46|válido|
|scored_0928|E2|64-6|46||46|válido|
|scored_0928|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0928|E4|repair|46||42|completo|
|scored_0929|E1|1|198|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0929|E1|3|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0929|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0929|E2|52-1|198||184|válido|
|scored_0929|E2|52-3|193||172|válido|
|scored_0929|E2|52-6|188||176|válido|
|scored_0929|E2|64-1|198||182|válido|
|scored_0929|E2|64-3|193||178|válido|
|scored_0929|E2|64-6|188||175|válido|
|scored_0929|E3|joint|188|individual 188|185|OPTIMAL|
|scored_0929|E4|repair|188||187|completo|
|scored_0930|E1|1|112|postprocess 111, optimize_a4 109, nested_windows 109|109|válido|
|scored_0930|E1|3|112|postprocess 111, optimize_a4 109, nested_windows 109|109|válido|
|scored_0930|E1|6|112|postprocess 111, optimize_a4 109, nested_windows 109|109|válido|
|scored_0930|E2|52-1|112||109|válido|
|scored_0930|E2|52-3|112||108|válido|
|scored_0930|E2|52-6|112||108|válido|
|scored_0930|E2|64-1|112||110|válido|
|scored_0930|E2|64-3|112||110|válido|
|scored_0930|E2|64-6|112||110|válido|
|scored_0930|E3|joint|112|individual 112|112|OPTIMAL|
|scored_0930|E4|repair|112||106|completo|
|scored_0931|E1|1|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0931|E1|3|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0931|E1|6|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0931|E2|52-1|4||4|válido|
|scored_0931|E2|52-3|4||4|válido|
|scored_0931|E2|52-6|4||4|válido|
|scored_0931|E2|64-1|4||4|válido|
|scored_0931|E2|64-3|4||4|válido|
|scored_0931|E2|64-6|4||4|válido|
|scored_0931|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0931|E4|repair|2||2|completo|
|scored_0932|E1|1|335|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0932|E1|3|320|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0932|E1|6|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0932|E2|52-1|335||307|válido|
|scored_0932|E2|52-3|320||283|válido|
|scored_0932|E2|52-6|308||288|válido|
|scored_0932|E2|64-1|335||312|válido|
|scored_0932|E2|64-3|320||284|válido|
|scored_0932|E2|64-6|308||287|válido|
|scored_0932|E3|joint|308|individual 308|308|OPTIMAL|
|scored_0932|E4|repair|308||308|completo|
|scored_0933|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0933|E1|3|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0933|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0933|E2|52-1|229||205|válido|
|scored_0933|E2|52-3|217||203|válido|
|scored_0933|E2|52-6|215||197|válido|
|scored_0933|E2|64-1|229||206|válido|
|scored_0933|E2|64-3|217||206|válido|
|scored_0933|E2|64-6|215||200|válido|
|scored_0933|E3|joint|215|individual 215|214|OPTIMAL|
|scored_0933|E4|repair|215||215|completo|
|scored_0934|E1|1|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0934|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0934|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0934|E2|52-1|193||181|válido|
|scored_0934|E2|52-3|191||183|válido|
|scored_0934|E2|52-6|190||179|válido|
|scored_0934|E2|64-1|193||182|válido|
|scored_0934|E2|64-3|191||183|válido|
|scored_0934|E2|64-6|190||177|válido|
|scored_0934|E3|joint|190|individual 189|189|OPTIMAL|
|scored_0934|E4|repair|190||189|completo|
|scored_0935|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0935|E1|3|229|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0935|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0935|E2|52-1|237||221|válido|
|scored_0935|E2|52-3|229||213|válido|
|scored_0935|E2|52-6|229||213|válido|
|scored_0935|E2|64-1|237||222|válido|
|scored_0935|E2|64-3|229||212|válido|
|scored_0935|E2|64-6|229||214|válido|
|scored_0935|E3|joint|229|individual 228|226|OPTIMAL|
|scored_0935|E4|repair|229||229|completo|
|scored_0936|E1|1|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0936|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0936|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0936|E2|52-1|209||190|válido|
|scored_0936|E2|52-3|202||189|válido|
|scored_0936|E2|52-6|201||192|válido|
|scored_0936|E2|64-1|209||192|válido|
|scored_0936|E2|64-3|202||187|válido|
|scored_0936|E2|64-6|201||192|válido|
|scored_0936|E3|joint|201|individual 201|201|OPTIMAL|
|scored_0936|E4|repair|201||201|completo|
|scored_0937|E1|1|41|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0937|E1|3|41|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0937|E1|6|41|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0937|E2|52-1|41||41|válido|
|scored_0937|E2|52-3|41||41|válido|
|scored_0937|E2|52-6|41||41|válido|
|scored_0937|E2|64-1|41||41|válido|
|scored_0937|E2|64-3|41||41|válido|
|scored_0937|E2|64-6|41||41|válido|
|scored_0937|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0937|E4|repair|41||41|completo|
|scored_0938|E1|1|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0938|E1|3|181|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0938|E1|6|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0938|E2|52-1|180||175|válido|
|scored_0938|E2|52-3|181||170|válido|
|scored_0938|E2|52-6|175||172|válido|
|scored_0938|E2|64-1|180||167|válido|
|scored_0938|E2|64-3|181||165|válido|
|scored_0938|E2|64-6|175||165|válido|
|scored_0938|E3|joint|175|individual 173|172|OPTIMAL|
|scored_0938|E4|repair|175||174|completo|
|scored_0939|E1|1|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|scored_0939|E1|3|331|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0939|E1|6|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0939|E2|52-1|369||321|válido|
|scored_0939|E2|52-3|331||309|válido|
|scored_0939|E2|52-6|322||289|válido|
|scored_0939|E2|64-1|369||324|válido|
|scored_0939|E2|64-3|331||310|válido|
|scored_0939|E2|64-6|322||295|válido|
|scored_0939|E3|joint|322|individual 320|320|rechazado por registros|
|scored_0939|E4|repair|322||322|completo|
|scored_0940|E1|1|178|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0940|E1|3|175|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0940|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0940|E2|52-1|178||164|válido|
|scored_0940|E2|52-3|175||159|válido|
|scored_0940|E2|52-6|167||157|válido|
|scored_0940|E2|64-1|178||164|válido|
|scored_0940|E2|64-3|175||160|válido|
|scored_0940|E2|64-6|167||158|válido|
|scored_0940|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0940|E4|repair|167||167|completo|
|scored_0941|E1|1|334|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0941|E1|3|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0941|E1|6|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0941|E2|52-1|334||307|válido|
|scored_0941|E2|52-3|323||299|válido|
|scored_0941|E2|52-6|315||290|válido|
|scored_0941|E2|64-1|334||307|válido|
|scored_0941|E2|64-3|323||302|válido|
|scored_0941|E2|64-6|315||293|válido|
|scored_0941|E3|joint|315|individual 315|314|FEASIBLE|
|scored_0941|E4|repair|315||315|completo|
|scored_0942|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0942|E1|3|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0942|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0942|E2|52-1|200||190|válido|
|scored_0942|E2|52-3|193||189|válido|
|scored_0942|E2|52-6|188||176|válido|
|scored_0942|E2|64-1|200||193|válido|
|scored_0942|E2|64-3|193||189|válido|
|scored_0942|E2|64-6|188||179|válido|
|scored_0942|E3|joint|188|individual 188|188|OPTIMAL|
|scored_0942|E4|repair|188||184|completo|
|scored_0943|E1|1|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0943|E1|3|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0943|E1|6|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0943|E2|52-1|118||117|válido|
|scored_0943|E2|52-3|118||117|válido|
|scored_0943|E2|52-6|118||117|válido|
|scored_0943|E2|64-1|118||117|válido|
|scored_0943|E2|64-3|118||117|válido|
|scored_0943|E2|64-6|118||117|válido|
|scored_0943|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0943|E4|repair|118||117|completo|
|scored_0944|E1|1|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0944|E1|3|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0944|E1|6|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0944|E2|52-1|312||288|válido|
|scored_0944|E2|52-3|298||280|válido|
|scored_0944|E2|52-6|296||280|válido|
|scored_0944|E2|64-1|312||291|válido|
|scored_0944|E2|64-3|298||281|válido|
|scored_0944|E2|64-6|296||281|válido|
|scored_0944|E3|joint|296|individual 295|294|OPTIMAL|
|scored_0944|E4|repair|296||296|completo|
|scored_0945|E1|1|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0945|E1|3|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0945|E1|6|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0945|E2|52-1|43||43|válido|
|scored_0945|E2|52-3|43||43|válido|
|scored_0945|E2|52-6|43||43|válido|
|scored_0945|E2|64-1|43||43|válido|
|scored_0945|E2|64-3|43||43|válido|
|scored_0945|E2|64-6|43||43|válido|
|scored_0945|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0945|E4|repair|43||37|completo|
|scored_0946|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0946|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0946|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0946|E2|52-1|303||273|válido|
|scored_0946|E2|52-3|286||267|válido|
|scored_0946|E2|52-6|278||267|válido|
|scored_0946|E2|64-1|303||276|válido|
|scored_0946|E2|64-3|286||271|válido|
|scored_0946|E2|64-6|278||269|válido|
|scored_0946|E3|joint|278|individual 278|277|OPTIMAL|
|scored_0946|E4|repair|278||278|completo|
|scored_0947|E1|1|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0947|E1|3|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0947|E1|6|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0947|E2|52-1|122||118|válido|
|scored_0947|E2|52-3|122||118|válido|
|scored_0947|E2|52-6|122||118|válido|
|scored_0947|E2|64-1|122||120|válido|
|scored_0947|E2|64-3|122||120|válido|
|scored_0947|E2|64-6|122||120|válido|
|scored_0947|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0947|E4|repair|122||121|completo|
|scored_0948|E1|1|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0948|E1|3|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0948|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0948|E2|52-1|172||167|válido|
|scored_0948|E2|52-3|172||164|válido|
|scored_0948|E2|52-6|168||163|válido|
|scored_0948|E2|64-1|172||166|válido|
|scored_0948|E2|64-3|172||165|válido|
|scored_0948|E2|64-6|168||163|válido|
|scored_0948|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0948|E4|repair|168||166|completo|
|scored_0949|E1|1|113|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0949|E1|3|113|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0949|E1|6|113|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0949|E2|52-1|113||109|válido|
|scored_0949|E2|52-3|113||109|válido|
|scored_0949|E2|52-6|113||109|válido|
|scored_0949|E2|64-1|113||109|válido|
|scored_0949|E2|64-3|113||109|válido|
|scored_0949|E2|64-6|113||109|válido|
|scored_0949|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0949|E4|repair|113||109|completo|
|scored_0950|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0950|E1|3|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0950|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0950|E2|52-1|168||162|válido|
|scored_0950|E2|52-3|167||159|válido|
|scored_0950|E2|52-6|166||159|válido|
|scored_0950|E2|64-1|168||159|válido|
|scored_0950|E2|64-3|167||156|válido|
|scored_0950|E2|64-6|166||156|válido|
|scored_0950|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0950|E4|repair|166||166|completo|
|scored_0951|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0951|E1|3|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0951|E1|6|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0951|E2|52-1|119||115|válido|
|scored_0951|E2|52-3|118||115|válido|
|scored_0951|E2|52-6|118||114|válido|
|scored_0951|E2|64-1|119||117|válido|
|scored_0951|E2|64-3|118||117|válido|
|scored_0951|E2|64-6|118||115|válido|
|scored_0951|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0951|E4|repair|118||118|completo|
|scored_0952|E1|1|298|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0952|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0952|E1|6|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0952|E2|52-1|298||279|válido|
|scored_0952|E2|52-3|286||267|válido|
|scored_0952|E2|52-6|283||270|válido|
|scored_0952|E2|64-1|298||279|válido|
|scored_0952|E2|64-3|286||268|válido|
|scored_0952|E2|64-6|283||271|válido|
|scored_0952|E3|joint|283|individual 283|283|OPTIMAL|
|scored_0952|E4|repair|283||283|completo|
|scored_0953|E1|1|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0953|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0953|E1|6|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0953|E2|52-1|243||229|válido|
|scored_0953|E2|52-3|242||231|válido|
|scored_0953|E2|52-6|234||224|válido|
|scored_0953|E2|64-1|243||230|válido|
|scored_0953|E2|64-3|242||231|válido|
|scored_0953|E2|64-6|234||226|válido|
|scored_0953|E3|joint|234|individual 234|233|OPTIMAL|
|scored_0953|E4|repair|234||234|completo|
|scored_0954|E1|1|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0954|E1|3|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0954|E1|6|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0954|E2|52-1|105||102|válido|
|scored_0954|E2|52-3|105||102|válido|
|scored_0954|E2|52-6|105||102|válido|
|scored_0954|E2|64-1|105||103|válido|
|scored_0954|E2|64-3|105||103|válido|
|scored_0954|E2|64-6|105||103|válido|
|scored_0954|E3|joint|105|individual 105|105|OPTIMAL|
|scored_0954|E4|repair|105||99|completo|
|scored_0955|E1|1|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0955|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0955|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0955|E2|52-1|116||113|válido|
|scored_0955|E2|52-3|116||113|válido|
|scored_0955|E2|52-6|116||113|válido|
|scored_0955|E2|64-1|116||113|válido|
|scored_0955|E2|64-3|116||113|válido|
|scored_0955|E2|64-6|116||113|válido|
|scored_0955|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0955|E4|repair|116||113|completo|
|scored_0956|E1|1|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0956|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0956|E1|6|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0956|E2|52-1|316||290|válido|
|scored_0956|E2|52-3|310||279|válido|
|scored_0956|E2|52-6|297||287|válido|
|scored_0956|E2|64-1|316||292|válido|
|scored_0956|E2|64-3|310||281|válido|
|scored_0956|E2|64-6|297||288|válido|
|scored_0956|E3|joint|297|individual 296|294|FEASIBLE|
|scored_0956|E4|repair|297||297|completo|
|scored_0957|E1|1|42|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0957|E1|3|42|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0957|E1|6|42|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0957|E2|52-1|42||42|válido|
|scored_0957|E2|52-3|42||42|válido|
|scored_0957|E2|52-6|42||42|válido|
|scored_0957|E2|64-1|42||42|válido|
|scored_0957|E2|64-3|42||42|válido|
|scored_0957|E2|64-6|42||42|válido|
|scored_0957|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0957|E4|repair|42||42|completo|
|scored_0958|E1|1|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0958|E1|3|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0958|E1|6|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0958|E2|52-1|41||41|válido|
|scored_0958|E2|52-3|41||41|válido|
|scored_0958|E2|52-6|41||41|válido|
|scored_0958|E2|64-1|41||41|válido|
|scored_0958|E2|64-3|41||41|válido|
|scored_0958|E2|64-6|41||41|válido|
|scored_0958|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0958|E4|repair|41||41|completo|
|scored_0959|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0959|E1|3|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0959|E1|6|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0959|E2|52-1|330||299|válido|
|scored_0959|E2|52-3|313||297|válido|
|scored_0959|E2|52-6|313||289|válido|
|scored_0959|E2|64-1|330||300|válido|
|scored_0959|E2|64-3|313||294|válido|
|scored_0959|E2|64-6|313||290|válido|
|scored_0959|E3|joint|313|individual 312|312|OPTIMAL|
|scored_0959|E4|repair|313||313|completo|
|scored_0960|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0960|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0960|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0960|E2|52-1|163||159|válido|
|scored_0960|E2|52-3|162||157|válido|
|scored_0960|E2|52-6|162||157|válido|
|scored_0960|E2|64-1|163||160|válido|
|scored_0960|E2|64-3|162||159|válido|
|scored_0960|E2|64-6|162||158|válido|
|scored_0960|E3|joint|162|individual 161|161|OPTIMAL|
|scored_0960|E4|repair|162||161|completo|
|scored_0961|E1|1|166|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0961|E1|3|164|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0961|E1|6|164|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0961|E2|52-1|166||157|válido|
|scored_0961|E2|52-3|164||155|válido|
|scored_0961|E2|52-6|164||155|válido|
|scored_0961|E2|64-1|166||155|válido|
|scored_0961|E2|64-3|164||155|válido|
|scored_0961|E2|64-6|164||155|válido|
|scored_0961|E3|joint|164|individual 163|163|OPTIMAL|
|scored_0961|E4|repair|164||164|completo|
|scored_0962|E1|1|48|postprocess 47, optimize_a4 47, nested_windows 47|47|válido|
|scored_0962|E1|3|48|postprocess 47, optimize_a4 47, nested_windows 47|47|válido|
|scored_0962|E1|6|48|postprocess 47, optimize_a4 47, nested_windows 47|47|válido|
|scored_0962|E2|52-1|48||48|válido|
|scored_0962|E2|52-3|48||48|válido|
|scored_0962|E2|52-6|48||48|válido|
|scored_0962|E2|64-1|48||48|válido|
|scored_0962|E2|64-3|48||48|válido|
|scored_0962|E2|64-6|48||48|válido|
|scored_0962|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0962|E4|repair|48||47|completo|
|scored_0963|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0963|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0963|E1|6|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0963|E2|52-1|165||158|válido|
|scored_0963|E2|52-3|166||158|válido|
|scored_0963|E2|52-6|165||155|válido|
|scored_0963|E2|64-1|165||160|válido|
|scored_0963|E2|64-3|166||156|válido|
|scored_0963|E2|64-6|165||156|válido|
|scored_0963|E3|joint|165|individual 164|164|OPTIMAL|
|scored_0963|E4|repair|165||164|completo|
|scored_0964|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0964|E1|3|288|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0964|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0964|E2|52-1|296||274|válido|
|scored_0964|E2|52-3|288||258|válido|
|scored_0964|E2|52-6|281||258|válido|
|scored_0964|E2|64-1|296||274|válido|
|scored_0964|E2|64-3|288||262|válido|
|scored_0964|E2|64-6|281||262|válido|
|scored_0964|E3|joint|281|individual 281|280|FEASIBLE|
|scored_0964|E4|repair|281||281|completo|
|scored_0965|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0965|E1|3|275|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0965|E1|6|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0965|E2|52-1|295||265|válido|
|scored_0965|E2|52-3|275||256|válido|
|scored_0965|E2|52-6|272||257|válido|
|scored_0965|E2|64-1|295||265|válido|
|scored_0965|E2|64-3|275||258|válido|
|scored_0965|E2|64-6|272||254|válido|
|scored_0965|E3|joint|272|individual 271|269|OPTIMAL|
|scored_0965|E4|repair|272||272|completo|
|scored_0966|E1|1|114|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0966|E1|3|114|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0966|E1|6|114|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0966|E2|52-1|114||110|válido|
|scored_0966|E2|52-3|114||110|válido|
|scored_0966|E2|52-6|114||110|válido|
|scored_0966|E2|64-1|114||110|válido|
|scored_0966|E2|64-3|114||110|válido|
|scored_0966|E2|64-6|114||110|válido|
|scored_0966|E3|joint|114|individual 114|114|OPTIMAL|
|scored_0966|E4|repair|114||114|completo|
|scored_0967|E1|1|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0967|E1|3|146|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0967|E1|6|145|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0967|E2|52-1|146||138|válido|
|scored_0967|E2|52-3|146||138|válido|
|scored_0967|E2|52-6|145||138|válido|
|scored_0967|E2|64-1|146||138|válido|
|scored_0967|E2|64-3|146||138|válido|
|scored_0967|E2|64-6|145||138|válido|
|scored_0967|E3|joint|145|individual 145|145|OPTIMAL|
|scored_0967|E4|repair|145||145|completo|
|scored_0968|E1|1|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0968|E1|3|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0968|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0968|E2|52-1|224||209|válido|
|scored_0968|E2|52-3|221||207|válido|
|scored_0968|E2|52-6|220||206|válido|
|scored_0968|E2|64-1|224||209|válido|
|scored_0968|E2|64-3|221||207|válido|
|scored_0968|E2|64-6|220||205|válido|
|scored_0968|E3|joint|220|individual 220|220|OPTIMAL|
|scored_0968|E4|repair|220||219|completo|
|scored_0969|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0969|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0969|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0969|E2|52-1|205||187|válido|
|scored_0969|E2|52-3|195||184|válido|
|scored_0969|E2|52-6|193||184|válido|
|scored_0969|E2|64-1|205||188|válido|
|scored_0969|E2|64-3|195||184|válido|
|scored_0969|E2|64-6|193||184|válido|
|scored_0969|E3|joint|193|individual 193|193|OPTIMAL|
|scored_0969|E4|repair|193||193|completo|
|scored_0970|E1|1|330|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0970|E1|3|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0970|E1|6|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0970|E2|52-1|330||309|válido|
|scored_0970|E2|52-3|315||301|válido|
|scored_0970|E2|52-6|305||289|válido|
|scored_0970|E2|64-1|330||311|válido|
|scored_0970|E2|64-3|315||299|válido|
|scored_0970|E2|64-6|305||292|válido|
|scored_0970|E3|joint|305|individual 305|305|FEASIBLE|
|scored_0970|E4|repair|305||305|completo|
|scored_0971|E1|1|27|postprocess 23, optimize_a4 23, nested_windows 23|23|válido|
|scored_0971|E1|3|27|postprocess 23, optimize_a4 23, nested_windows 23|23|válido|
|scored_0971|E1|6|27|postprocess 23, optimize_a4 23, nested_windows 23|23|válido|
|scored_0971|E2|52-1|27||27|válido|
|scored_0971|E2|52-3|27||27|válido|
|scored_0971|E2|52-6|27||27|válido|
|scored_0971|E2|64-1|27||27|válido|
|scored_0971|E2|64-3|27||27|válido|
|scored_0971|E2|64-6|27||27|válido|
|scored_0971|E3|joint|27|individual 27|26|OPTIMAL|
|scored_0971|E4|repair|27||24|completo|
|scored_0972|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0972|E1|3|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0972|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0972|E2|52-1|163||156|válido|
|scored_0972|E2|52-3|161||155|válido|
|scored_0972|E2|52-6|161||155|válido|
|scored_0972|E2|64-1|163||157|válido|
|scored_0972|E2|64-3|161||156|válido|
|scored_0972|E2|64-6|161||155|válido|
|scored_0972|E3|joint|161|individual 161|161|OPTIMAL|
|scored_0972|E4|repair|161||159|completo|
|scored_0973|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0973|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0973|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0973|E2|52-1|146||136|válido|
|scored_0973|E2|52-3|146||136|válido|
|scored_0973|E2|52-6|146||136|válido|
|scored_0973|E2|64-1|146||136|válido|
|scored_0973|E2|64-3|146||136|válido|
|scored_0973|E2|64-6|146||136|válido|
|scored_0973|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0973|E4|repair|146||142|completo|
|scored_0974|E1|1|325|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0974|E1|3|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0974|E1|6|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0974|E2|52-1|325||302|válido|
|scored_0974|E2|52-3|319||294|válido|
|scored_0974|E2|52-6|310||286|válido|
|scored_0974|E2|64-1|325||311|válido|
|scored_0974|E2|64-3|319||296|válido|
|scored_0974|E2|64-6|310||289|válido|
|scored_0974|E3|joint|310|individual 310|309|FEASIBLE|
|scored_0974|E4|repair|310||310|completo|
|scored_0975|E1|1|171|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0975|E1|3|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0975|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0975|E2|52-1|171||157|válido|
|scored_0975|E2|52-3|167||158|válido|
|scored_0975|E2|52-6|166||158|válido|
|scored_0975|E2|64-1|171||159|válido|
|scored_0975|E2|64-3|167||161|válido|
|scored_0975|E2|64-6|166||161|válido|
|scored_0975|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0975|E4|repair|166||163|completo|
|scored_0976|E1|1|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0976|E1|3|154|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0976|E1|6|154|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0976|E2|52-1|157||152|válido|
|scored_0976|E2|52-3|154||151|válido|
|scored_0976|E2|52-6|154||151|válido|
|scored_0976|E2|64-1|157||152|válido|
|scored_0976|E2|64-3|154||152|válido|
|scored_0976|E2|64-6|154||152|válido|
|scored_0976|E3|joint|154|individual 153|153|OPTIMAL|
|scored_0976|E4|repair|154||154|completo|
|scored_0977|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0977|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0977|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0977|E2|52-1|229||214|válido|
|scored_0977|E2|52-3|220||210|válido|
|scored_0977|E2|52-6|217||207|válido|
|scored_0977|E2|64-1|229||216|válido|
|scored_0977|E2|64-3|220||210|válido|
|scored_0977|E2|64-6|217||208|válido|
|scored_0977|E3|joint|217|individual 217|217|OPTIMAL|
|scored_0977|E4|repair|217||217|completo|
|scored_0978|E1|1|139|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0978|E1|3|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0978|E1|6|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0978|E2|52-1|139||132|válido|
|scored_0978|E2|52-3|138||133|válido|
|scored_0978|E2|52-6|138||133|válido|
|scored_0978|E2|64-1|139||132|válido|
|scored_0978|E2|64-3|138||133|válido|
|scored_0978|E2|64-6|138||133|válido|
|scored_0978|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0978|E4|repair|138||136|completo|
|scored_0979|E1|1|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0979|E1|3|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0979|E1|6|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0979|E2|52-1|257||227|válido|
|scored_0979|E2|52-3|245||221|válido|
|scored_0979|E2|52-6|239||220|válido|
|scored_0979|E2|64-1|257||228|válido|
|scored_0979|E2|64-3|245||222|válido|
|scored_0979|E2|64-6|239||221|válido|
|scored_0979|E3|joint|239|individual 238|238|OPTIMAL|
|scored_0979|E4|repair|239||238|completo|
|scored_0980|E1|1|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0980|E1|3|171|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0980|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0980|E2|52-1|172||161|válido|
|scored_0980|E2|52-3|171||159|válido|
|scored_0980|E2|52-6|168||158|válido|
|scored_0980|E2|64-1|172||158|válido|
|scored_0980|E2|64-3|171||157|válido|
|scored_0980|E2|64-6|168||157|válido|
|scored_0980|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0980|E4|repair|168||167|completo|
|scored_0981|E1|1|36|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0981|E1|3|36|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0981|E1|6|36|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0981|E2|52-1|36||36|válido|
|scored_0981|E2|52-3|36||36|válido|
|scored_0981|E2|52-6|36||36|válido|
|scored_0981|E2|64-1|36||36|válido|
|scored_0981|E2|64-3|36||36|válido|
|scored_0981|E2|64-6|36||36|válido|
|scored_0981|E3|joint|36|individual 36|36|OPTIMAL|
|scored_0981|E4|repair|36||36|completo|
|scored_0982|E1|1|395|postprocess 394, optimize_a4 394, nested_windows 394|394|válido|
|scored_0982|E1|3|371|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0982|E1|6|368|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0982|E2|52-1|395||355|válido|
|scored_0982|E2|52-3|371||342|válido|
|scored_0982|E2|52-6|368||342|válido|
|scored_0982|E2|64-1|395||358|válido|
|scored_0982|E2|64-3|371||344|válido|
|scored_0982|E2|64-6|368||345|válido|
|scored_0982|E3|joint|368|individual 368|363|OPTIMAL|
|scored_0982|E4|repair|368||368|completo|
|scored_0983|E1|1|191|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0983|E1|3|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0983|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0983|E2|52-1|191||177|válido|
|scored_0983|E2|52-3|186||171|válido|
|scored_0983|E2|52-6|185||174|válido|
|scored_0983|E2|64-1|191||179|válido|
|scored_0983|E2|64-3|186||173|válido|
|scored_0983|E2|64-6|185||171|válido|
|scored_0983|E3|joint|185|individual 185|184|OPTIMAL|
|scored_0983|E4|repair|185||185|completo|
|scored_0984|E1|1|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0984|E1|3|182|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0984|E1|6|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0984|E2|52-1|181||168|válido|
|scored_0984|E2|52-3|182||171|válido|
|scored_0984|E2|52-6|177||170|válido|
|scored_0984|E2|64-1|181||169|válido|
|scored_0984|E2|64-3|182||169|válido|
|scored_0984|E2|64-6|177||169|válido|
|scored_0984|E3|joint|177|individual 177|175|OPTIMAL|
|scored_0984|E4|repair|177||177|completo|
|scored_0985|E1|1|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0985|E1|3|176|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0985|E1|6|177|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0985|E2|52-1|180||170|válido|
|scored_0985|E2|52-3|176||167|válido|
|scored_0985|E2|52-6|177||168|válido|
|scored_0985|E2|64-1|180||171|válido|
|scored_0985|E2|64-3|176||168|válido|
|scored_0985|E2|64-6|177||165|válido|
|scored_0985|E3|joint|176|individual 176|176|OPTIMAL|
|scored_0985|E4|repair|176||172|completo|
|scored_0986|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0986|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0986|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0986|E2|52-1|237||217|válido|
|scored_0986|E2|52-3|234||214|válido|
|scored_0986|E2|52-6|224||212|válido|
|scored_0986|E2|64-1|237||220|válido|
|scored_0986|E2|64-3|234||219|válido|
|scored_0986|E2|64-6|224||212|válido|
|scored_0986|E3|joint|224|individual 223|223|OPTIMAL|
|scored_0986|E4|repair|224||222|completo|
|scored_0987|E1|1|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0987|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0987|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0987|E2|52-1|203||185|válido|
|scored_0987|E2|52-3|188||182|válido|
|scored_0987|E2|52-6|187||181|válido|
|scored_0987|E2|64-1|203||185|válido|
|scored_0987|E2|64-3|188||182|válido|
|scored_0987|E2|64-6|187||177|válido|
|scored_0987|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0987|E4|repair|187||185|completo|
|scored_0988|E1|1|196|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0988|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0988|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0988|E2|52-1|196||185|válido|
|scored_0988|E2|52-3|192||187|válido|
|scored_0988|E2|52-6|191||184|válido|
|scored_0988|E2|64-1|196||181|válido|
|scored_0988|E2|64-3|192||186|válido|
|scored_0988|E2|64-6|191||187|válido|
|scored_0988|E3|joint|191|individual 190|186|OPTIMAL|
|scored_0988|E4|repair|191||188|completo|
|scored_0989|E1|1|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0989|E1|3|107|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0989|E1|6|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0989|E2|52-1|108||108|válido|
|scored_0989|E2|52-3|107||101|válido|
|scored_0989|E2|52-6|105||102|válido|
|scored_0989|E2|64-1|108||108|válido|
|scored_0989|E2|64-3|107||101|válido|
|scored_0989|E2|64-6|105||102|válido|
|scored_0989|E3|joint|105|individual 105|105|OPTIMAL|
|scored_0989|E4|repair|105||105|completo|
|scored_0990|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0990|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0990|E1|6|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0990|E2|52-1|243||230|válido|
|scored_0990|E2|52-3|238||230|válido|
|scored_0990|E2|52-6|243||222|válido|
|scored_0990|E2|64-1|243||232|válido|
|scored_0990|E2|64-3|238||230|válido|
|scored_0990|E2|64-6|243||226|válido|
|scored_0990|E3|joint|238|individual 238|238|OPTIMAL|
|scored_0990|E4|repair|238||237|completo|
|scored_0991|E1|1|253|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0991|E1|3|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0991|E1|6|232|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0991|E2|52-1|253||229|válido|
|scored_0991|E2|52-3|240||224|válido|
|scored_0991|E2|52-6|232||219|válido|
|scored_0991|E2|64-1|253||234|válido|
|scored_0991|E2|64-3|240||224|válido|
|scored_0991|E2|64-6|232||218|válido|
|scored_0991|E3|joint|232|individual 232|232|OPTIMAL|
|scored_0991|E4|repair|232||232|completo|
|scored_0992|E1|1|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0992|E1|3|357|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0992|E1|6|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0992|E2|52-1|364||341|válido|
|scored_0992|E2|52-3|357||328|válido|
|scored_0992|E2|52-6|348||326|válido|
|scored_0992|E2|64-1|364||345|válido|
|scored_0992|E2|64-3|357||329|válido|
|scored_0992|E2|64-6|348||326|válido|
|scored_0992|E3|joint|348|individual 348|348|OPTIMAL|
|scored_0992|E4|repair|348||348|completo|
|scored_0993|E1|1|316|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0993|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0993|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0993|E2|52-1|316||285|válido|
|scored_0993|E2|52-3|304||276|válido|
|scored_0993|E2|52-6|292||274|válido|
|scored_0993|E2|64-1|316||287|válido|
|scored_0993|E2|64-3|304||276|válido|
|scored_0993|E2|64-6|292||275|válido|
|scored_0993|E3|joint|292|individual 291|290|OPTIMAL|
|scored_0993|E4|repair|292||291|completo|
|scored_0994|E1|1|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0994|E1|3|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0994|E1|6|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0994|E2|52-1|220||203|válido|
|scored_0994|E2|52-3|214||195|válido|
|scored_0994|E2|52-6|216||200|válido|
|scored_0994|E2|64-1|220||202|válido|
|scored_0994|E2|64-3|214||194|válido|
|scored_0994|E2|64-6|216||197|válido|
|scored_0994|E3|joint|214|individual 212|209|OPTIMAL|
|scored_0994|E4|repair|214||211|completo|
|scored_0995|E1|1|198|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0995|E1|3|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0995|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0995|E2|52-1|198||184|válido|
|scored_0995|E2|52-3|190||182|válido|
|scored_0995|E2|52-6|187||181|válido|
|scored_0995|E2|64-1|198||186|válido|
|scored_0995|E2|64-3|190||178|válido|
|scored_0995|E2|64-6|187||178|válido|
|scored_0995|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0995|E4|repair|187||187|completo|
|scored_0996|E1|1|174|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0996|E1|3|171|postprocess 169, optimize_a4 168, nested_windows 168|168|válido|
|scored_0996|E1|6|174|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0996|E2|52-1|174||163|válido|
|scored_0996|E2|52-3|171||161|válido|
|scored_0996|E2|52-6|174||160|válido|
|scored_0996|E2|64-1|174||162|válido|
|scored_0996|E2|64-3|171||162|válido|
|scored_0996|E2|64-6|174||162|válido|
|scored_0996|E3|joint|171|individual 171|171|OPTIMAL|
|scored_0996|E4|repair|171||170|completo|
|scored_0997|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0997|E1|3|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0997|E1|6|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0997|E2|52-1|177||168|válido|
|scored_0997|E2|52-3|177||163|válido|
|scored_0997|E2|52-6|177||161|válido|
|scored_0997|E2|64-1|177||163|válido|
|scored_0997|E2|64-3|177||161|válido|
|scored_0997|E2|64-6|177||162|válido|
|scored_0997|E3|joint|177|individual 177|175|OPTIMAL|
|scored_0997|E4|repair|177||176|completo|
|scored_0998|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0998|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0998|E1|6|279|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0998|E2|52-1|295||269|válido|
|scored_0998|E2|52-3|278||259|válido|
|scored_0998|E2|52-6|279||255|válido|
|scored_0998|E2|64-1|295||270|válido|
|scored_0998|E2|64-3|278||258|válido|
|scored_0998|E2|64-6|279||256|válido|
|scored_0998|E3|joint|278|individual 278|276|OPTIMAL|
|scored_0998|E4|repair|278||278|completo|
|scored_0999|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0999|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0999|E1|6|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0999|E2|52-1|145||134|válido|
|scored_0999|E2|52-3|138||133|válido|
|scored_0999|E2|52-6|138||132|válido|
|scored_0999|E2|64-1|145||134|válido|
|scored_0999|E2|64-3|138||135|válido|
|scored_0999|E2|64-6|138||133|válido|
|scored_0999|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0999|E4|repair|138||134|completo|
|scored_1000|E1|1|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_1000|E1|3|323|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_1000|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_1000|E2|52-1|345||312|válido|
|scored_1000|E2|52-3|323||297|válido|
|scored_1000|E2|52-6|311||299|válido|
|scored_1000|E2|64-1|345||313|válido|
|scored_1000|E2|64-3|323||297|válido|
|scored_1000|E2|64-6|311||300|válido|
|scored_1000|E3|joint|311|individual 311|311|OPTIMAL|
|scored_1000|E4|repair|311||311|completo|